# NEOARM full-column inversion with one multiresolution cubic specimen

This notebook is an executable design study for identifying the NEOARM column from raw lens-control perturbations, images of one registered MAG*I*CAL-like specimen, vacuum-beam measurements, and selected diffraction patterns. Calibrated sample-plane position/angle states are retained only in a clearly labelled synthetic/reference branch; they are not required by the preferred no-commanded-tilt instrument route. The specimen is one invariant physical object. Magnification changes the illuminated footprint and selects a certified tile and level of detail; it never redraws the specimen.

The default model is a finite hard-edge solenoid driven directly by raw DAC coordinates. It is a low-dimensional, symplectic effective model, not a claim that a NEOARM field is rectangular. Thin rotating lenses initialize the fit; normalized Glaser fields are admitted only when held-out data justify the extra freedom; generic symplectic maps remain predictive fallbacks. The corrector is disabled and objective controls are fixed.

`SMOKE = True` runs the compact algebraic checks and a three-parameter fit. Every admitted Gaussian forward render is fail-closed on the JAX/Pallas GPU kernel: install the `inversion,gpu` extras and select the `TemGymCore (GPU)` kernel. Full 256×256 renders and the 64-start fit remain checkpointed and opt-in. Smoke results are not an instrument calibration.

In [ ]:
from __future__ import annotations
import csv, gc, hashlib, importlib.metadata as importlib_metadata, json, os, tomllib, urllib.request
from collections import Counter
from dataclasses import replace
from pathlib import Path
# Share GPU memory with checkpointed million-beamlet work instead of preallocating it; an explicit operator setting wins.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("JAX_ENABLE_X64", "true")
import jax
import jaxlib
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, Polygon, Rectangle
from scipy import ndimage
from scipy.optimize import least_squares
jax.config.update("jax_enable_x64", True)
np.set_printoptions(precision=5, suppress=True)

from temgym_core.components import ElectromagneticLens
from temgym_core.constants import compute_Rc_from_voltage, energy2wavelength
from temgym_core.evaluate import gpu_kernel_available
from temgym_core.gaussian import make_gaussian
from temgym_core.microscope_inversion import (
    AcquisitionState, CalibratedSourceState, FrameObservationBlock, HeterogeneousInversionDataset, IlluminationProjection2D, InversionDataset, calibrated_source_basis, compact_30_state_plan, compact_illumination_moment_experiment,
    EquivalentSolutionFamily, d_optimal_logdet, diagnose_multistart_basins, equivalent_solution_family, equivalent_solutions,
    drift_matrix, fit_dataset, fit_heterogeneous_dataset, fit_identifiable_subspace, greedy_d_optimal_selection,
    glaser_solenoid_matrix, hard_edge_solenoid_matrix, illumination_covariance_design_matrix, jax_jacobian, jacobian_svd,
    predict_illumination_experiment, profile_likelihood, response_observable_vector, scaled_identifiable_subspace, standard_illumination_projections, thin_lens_matrix,
)
from temgym_core.critical_line_calibration import (
    AxialPlaneCovarianceBlock, DeflectorCommandKickMap, EntranceCovarianceChart, HardEdgeInternalPlaneConstraint, PHASE_SPACE_COVARIANCE_OBSERVABLE_NAMES, audit_axial_plane_covariance_nuisance_identifiability, audit_condenser_identifiability, audit_entrance_covariance_nuisance_identifiability, axial_plane_covariance_observables, axial_plane_covariance_observables_with_entrance_covariance, central_deflector_command_difference, condenser_aperture_edge_scan, condenser_aperture_sweep, condenser_independent_reconnaissance_response, condenser_reconnaissance_frame_observables, condenser_reconnaissance_response,
    critical_line_scan, greedy_condenser_probe_design, hard_edge_matrix_to_plane, linear_zero_crossings, restrict_hard_edge_internal_plane_bounds, restrict_hard_edge_internal_planes_bounds, three_normal_aperture_edge_design,
    position_moments, sample_perforation_edge_scan, sample_plane_probe_observables, sample_plane_probe_observables_with_entrance_covariance,
)
from temgym_core.condenser_branch_b import (
    EXPECTED_BRANCH_B_PAYLOAD_SHA256, NORMALIZED_CONTROL_WARNING, BranchBLikelihood, BranchBObservations, load_branch_b_experiment, make_synthetic_branch_b_observations,
)
from temgym_core.condenser_astigmatic_design import load_astigmatic_condenser_design, recompute_astigmatic_design_audit
from temgym_core.condenser_astigmatic_likelihood import make_astigmatic_condenser_likelihood, make_synthetic_astigmatic_observations
from temgym_core.condenser_astigmatic_adapter import make_astigmatic_full_column_adapter
from temgym_core.condenser_astigmatic_holdout import astigmatic_training_residual, audit_astigmatic_held_out_training, evaluate_astigmatic_held_out, make_astigmatic_held_out_split
from temgym_core.multiscale_sample import (
    CubicActionField2D, MAGICALSampleAtlas, SampleLevel2D, render_sample_view,
    select_sample_view, validate_sample_level, validate_sample_view_derivatives,
)
from temgym_core.neoarm_calibration import (
    DetectorGeometry, DetectorNoiseModel, FramePrediction, GaussianRefinementSnapshot, RefinementRegion, aggregate_checkpoint_observables_for_fit, apply_transfer_matrix_to_gaussians, assess_gaussian_slice_safety, build_acquisition_manifest, build_heterogeneous_dataset_from_checkpoint, build_multiresolution_gaussian_topology,
    collins_propagate_2d, conjugate_gaussian_grid_reference, estimate_gaussian_workload, gaussian_cardinal_coefficients,
    calibrated_state_centroid_covariance_px2, design_nominal_magnification_dacs, design_signed_object_defocus_dacs,
    evaluate_held_out_metrics, extract_image_observables,
    fit_fourier_affine, frozen_topology_fingerprint, hard_edge_column_from_config,
    interact_gaussian_with_sample, known_fourier_features_from_sample_view, predict_frame, propagate_gaussians_through_column, reexpand_transmitted_field, render_frame_sequence_checkpointed, render_gaussian_detector, render_gaussian_detector_plane, shared_fourier_calibration_priors_from_dataset, simulate_detector_counts,
    validate_forward_models, validate_gaussian_refinement_sequence,
)
from temgym_core.neoarm_diffraction import (
    bind_native_diffraction_requests, render_native_diffraction_detector,
)
from temgym_core.diffraction_metrology import extract_registered_bragg_spot_offsets

from temgym_core.grid_wave_reference import (
    GridWaveOptions, GridWaveSamplingError, assess_grid_reference_convergence,
    certify_radial_symplectic_map, poisson_read_total_standard_error,
    round_solenoid_grid_reference,
)
from temgym_core.layer_metrology import (
    layer_observable_covariance_augmentation, registered_layer_metrology_from_sample_view,
    registered_layer_period_metrology_from_sample_view, registered_layer_reference_residuals_px,
)
from temgym_core.topology_validation import (
    PhysicalRenderSignature, PhysicalValidationEvidence, PhysicalValidationLedger,
    PhysicalValidationRequirements, PhysicalValidationThresholds, analytic_single_gaussian_detector_reference,
    assess_fisher_parameter_stability,
    assess_nuisance_profiled_optical_stability,
    certify_single_gaussian_perforation, stable_validation_signature,
)

from temgym_core.physical_observables import (
    IndexedDiffractionBraggSpots, PhysicalObservableBlockSpecification,
    fourier_observable_covariance_augmentation, imaging_reciprocal_vectors_from_feature_set,
    indexed_diffraction_bragg_spots_from_feature_set, make_calibration_aware_physical_predictor,
    predict_physical_observable_blocks, specifications_from_aggregated_dataset, strip_shared_multilayer_certificate_covariance,
)
from temgym_core.physical_nuisance import (
    GlobalObservationLayout, SharedLatentNuisanceLikelihood, diagnose_blockwise_identifiability, diagnose_blockwise_nuisance_identifiability, diagnose_nuisance_identifiability, evaluate_held_out_shared_likelihood,
)
from temgym_core.identifiability_reporting import assess_matched_model_recovery, verify_local_null_families
from temgym_core.nuisance_profiled_subspace import compare_scaled_active_subspaces, fit_nuisance_profiled_polytope, nuisance_profiled_identifiable_subspace, profile_nuisance_profiled_polytope_direction
from temgym_core.plotting import (
    ColumnElement, ColumnSection, PositionAngleTrace, basis_trace_events, export_ray_figure,
    gaussian_basis_envelope, plot_neoarm_ray_diagram, plot_neoarm_ray_small_multiples,
)

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists(): return candidate
    raise FileNotFoundError("Run this notebook from inside the TemGymCore checkout.")

REPO = find_repo_root(Path.cwd())
CONFIG_PATH = REPO / "examples/lens_inversion/neoarm_inversion_config.toml"
GENERATED = REPO / "examples/generated/neoarm_inversion"
GENERATED.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open("rb") as stream: config = tomllib.load(stream)
CONDENSER_ROUTE_ALLOWED_VALUES=("synthetic_astigmatic_no_tilt","synthetic_branch_a_conditional","synthetic_branch_b_unconditional","instrument_astigmatic_no_tilt","instrument_branch_a_conditional","instrument_branch_b_unconditional")
CONDENSER_ROUTE_DEFAULT=str(config.get("experiment",{}).get("condenser_route","synthetic_astigmatic_no_tilt")); CONDENSER_ROUTE=os.environ.get("NEOARM_CONDENSER_ROUTE",CONDENSER_ROUTE_DEFAULT).strip().lower()
if CONDENSER_ROUTE not in CONDENSER_ROUTE_ALLOWED_VALUES: raise ValueError(f"NEOARM_CONDENSER_ROUTE must be one of {CONDENSER_ROUTE_ALLOWED_VALUES}, got {CONDENSER_ROUTE!r}.")
CONDENSER_ROUTE_BRANCH="ASTIGMATIC" if "astigmatic" in CONDENSER_ROUTE else ("A" if "branch_a" in CONDENSER_ROUTE else "B"); CONDENSER_ROUTE_IS_SYNTHETIC=CONDENSER_ROUTE.startswith("synthetic_"); CONDENSER_ROUTE_IS_INSTRUMENT=CONDENSER_ROUTE.startswith("instrument_"); CONDENSER_ROUTE_REQUIRES_OBJ_PREFIELD_ANCHOR=CONDENSER_ROUTE_BRANCH=="A"
def environment_flag(name,default=False):
    raw=os.environ.get(name)
    if raw is None: return bool(default)
    normalized=raw.strip().lower()
    if normalized in {"1","true","yes","on"}: return True
    if normalized in {"0","false","no","off"}: return False
    raise ValueError(f"{name} must be a boolean flag, got {raw!r}.")
SMOKE=environment_flag("NEOARM_SMOKE",True); RUN_PHYSICAL_DEMO_SUBSET=environment_flag("NEOARM_RUN_PHYSICAL_DEMO_SUBSET",True); RUN_FULL_RENDERS=environment_flag("NEOARM_RUN_FULL_RENDERS",False); RUN_PHYSICAL_GLOBAL_FIT=environment_flag("NEOARM_RUN_PHYSICAL_GLOBAL_FIT",False); RUN_TRANSFER_RESPONSE_GLOBAL_FIT=environment_flag("NEOARM_RUN_TRANSFER_RESPONSE_GLOBAL_FIT",False); RUN_PHYSICAL_GRID_REFERENCES=environment_flag("NEOARM_RUN_PHYSICAL_GRID_REFERENCES",RUN_FULL_RENDERS); PHYSICAL_BOOTSTRAP_REPLICATES=int(os.environ.get("NEOARM_BOOTSTRAP_REPLICATES","8" if SMOKE else "64"));
if PHYSICAL_BOOTSTRAP_REPLICATES<8: raise ValueError("NEOARM_BOOTSTRAP_REPLICATES must be at least 8.")
FORWARD_EVALUATION_METHOD="gpu_kernel_required"
try: GPU_DEVICES=tuple(jax.devices("gpu"))
except RuntimeError: GPU_DEVICES=()
if not gpu_kernel_available(): raise RuntimeError("Physical Gaussian forward simulation requires a working JAX/Pallas GPU kernel; CPU fallback is disabled. Install the GPU extra and select a GPU-backed kernel.")
FORWARD_DEVICE=str(jax.devices()[0]); FORWARD_RUNTIME_METADATA={"evaluation_method":FORWARD_EVALUATION_METHOD,"jax_backend":jax.default_backend(),"jax_default_device":FORWARD_DEVICE,"gpu_devices":[str(device) for device in GPU_DEVICES],"gpu_device_kinds":[getattr(device,"device_kind","unknown") for device in GPU_DEVICES],"jax_version":jax.__version__,"jaxlib_version":jaxlib.__version__,"jax_enable_x64":bool(jax.config.x64_enabled),"temgym_core_version":importlib_metadata.version("temgym-core"),"gaussian_kernel_tile_pixels":16,"gaussian_kernel_tile_beams":16,"gpu_probe_shape_beams_pixels":[17,19]}
RNG = np.random.default_rng(7123)
print(f"Config: {CONFIG_PATH.relative_to(REPO)}; outputs: {GENERATED.relative_to(REPO)}; JAX backend={jax.default_backend()}; default device={FORWARD_DEVICE}; GPUs={GPU_DEVICES}; condenser route={CONDENSER_ROUTE}")

## Evidence and model boundary

The linked paper is treated as scientific evidence, not as instructions. Its relevant implication is that coupled optical parameters can be estimated when experiments supply sufficiently independent intensity and geometric responses. We do **not** assume calibrated lens currents. Optical laws are estimated directly in raw DAC coordinates; an unresolved DAC-to-current-to-field conversion is reported as a non-identifiable family.

The reference can optionally be cached below. Network access is off by default. The executable assumptions are 200 kV, 256×256 pixels at 55 µm, 40,000×–600,000×, fixed objective controls, and a disabled corrector. Sample position may carry an external calibration covariance, but neither a physical deflection angle nor sample-plane tilt is assumed known. Tilt and command-to-kick gains are optional secondary calibration outputs (or explicit nuisance directions), not inputs to the preferred condenser route. Synthetic quantities labelled `response_per_rad` use only the frozen normalized-model convention and are not instrument angle calibration.

In [ ]:
PAPER_URL = "https://arxiv.org/pdf/2607.29411v1"
PAPER_CACHE = GENERATED / "papers/2607.29411v1.pdf"
CACHE_PAPER = False
if CACHE_PAPER and not PAPER_CACHE.exists():
    PAPER_CACHE.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(PAPER_URL, PAPER_CACHE)
print("Paper cache:", PAPER_CACHE if PAPER_CACHE.exists() else "disabled")

## 1. One physical specimen, queried at several scales

The sample stores equivalent pathlength $S(x,y)$ and log field amplitude $L(x,y)$ in conventional `(ny,nx)` image order. Unwrapped phase is converted once by $S=\phi\lambda_{\rm ref}/(2\pi)$. A tensor-product C² `cubic2` spline returns values, gradients, and Hessians without differentiating through `angle`. Pose is separate from raster data, and derivative tensors are rotated analytically into microscope coordinates.

The atlas registers a vacuum perforation, four Si/SiGe multilayer regions, and a silicon $\{111\}$ region. Every region has an antialiased multiresolution pyramid in the same certified physical coordinate system.

In [ ]:
beam_cfg, detector_cfg = config["beam"], config["detector"]
experiment_cfg, certificate_cfg = config["experiment"], config["sample"]["certificate"]
detector = DetectorGeometry(
    shape_yx=tuple(detector_cfg["shape_yx"]), pixel_size_yx_m=tuple(detector_cfg["pixel_size_yx_m"]),
    z_m=detector_cfg["z_m"], mtf_sigma_px=detector_cfg["mtf_sigma_px"],
)
column = hard_edge_column_from_config(config)
atlas = MAGICALSampleAtlas.from_config(config,region_shape_yx=tuple(config["sample"]["region_shape_yx"]),authoritative_multilayer_shape_yx=tuple(config["sample"]["authoritative_multilayer_shape_yx"]),global_support_margin_m=float(config["sample"]["atlas_support_margin_m"]))
expected_regions = {"perforation", *(f"multilayer_region_{i}" for i in range(1,5)), "silicon_lattice_region"}
assert set(atlas.regions) == expected_regions
assert np.isclose(atlas.silicon_lattice_region.lattice_spacing_m, certificate_cfg["si_111_spacing_m"], rtol=2e-3)
print(f"Specimen {atlas.certificate.version}; {len(atlas.regions)} regions; {column.n_parameters} free column parameters")
for name, region in atlas.regions.items():
    scale = region.lattice_spacing_m or region.characteristic_scale_m
    print(f"  {name:24s} {region.kind:16s} scale={scale*1e9:9.5f} nm")

In [ ]:
level0 = atlas.pyramid.levels[0]
extent_nm = np.asarray(level0.bounds_xyxy_m)*1e9
region_colors = {"perforation":"#0072B2", "multilayer":"#D55E00", "silicon_lattice":"#009E73"}
fig, axes = plt.subplots(1,2,figsize=(11.5,4.6),constrained_layout=True)
for ax,(image,title,cmap) in zip(axes,((level0.pathlength_yx_m*1e12,"Equivalent pathlength [pm]","coolwarm"),(level0.log_amplitude_yx,"Log field amplitude","magma")),strict=True):
    artist=ax.imshow(image,origin="lower",extent=extent_nm,cmap=cmap,interpolation="nearest")
    for region in atlas.regions.values():
        x0,x1,y0,y1=np.asarray(region.bounds_xyxy_m)*1e9
        ax.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,lw=1.2,ec=region_colors[region.kind]))
        ax.text(x0,y1+2,region.name.replace("_region",""),fontsize=6.7,color=region_colors[region.kind])
    ax.set(title=title,xlabel="sample x [nm]",ylabel="sample y [nm]",aspect="equal")
    fig.colorbar(artist,ax=ax,shrink=.8)
fig.suptitle("Invariant registered MAG*I*CAL-like atlas",fontsize=13); plt.show()

### Automatic footprint, feature, and LOD selection

The inverse image of all four detector corners through $J_s=\partial(x_d,y_d)/\partial(x_s,y_s)$ defines the specimen footprint under rotation and anisotropy. Margins cover lens wobble, calibrated input perturbations, Gaussian support, pose uncertainty, and cubic halos. The coarsest accepted level obeys

$$\Delta_{\rm level}\leq\min(p_{\rm specimen}/2,\;w_{\min}/3,\;P_{\min}/8).$$

The known stage coordinate targets a certified region. Moving the stage changes sample pose, not specimen identity. Selection occurs outside JIT and is frozen over the allowed parameter envelope.

A large $|A|$ is not by itself an imaging mode. Every nominal raw-DAC row is therefore solved against both the requested isotropic $|A|$ and specimen conjugacy $B\simeq0$. The solver fails with diagnostics if the declared safe DAC box cannot satisfy both; this prevents a high-magnification but defocused transfer map from producing deceptively smooth 'sample images'.

In [ ]:
magnifications=np.asarray(experiment_cfg["magnifications"],float)
nominal_design=design_nominal_magnification_dacs(
    column,column.nominal_parameters,magnifications,path_bounds=tuple(experiment_cfg["nominal_dac_search_bounds"]),search_points=int(experiment_cfg["nominal_dac_search_points"]),multistart_points=int(experiment_cfg["nominal_dac_multistart_points"]),specimen_conjugacy_b_norm_atol=float(experiment_cfg["nominal_conjugacy_b_atol_m"]))
nominal_dacs=np.asarray(nominal_design.dac_values)
signed_defocus_design=design_signed_object_defocus_dacs(column,column.nominal_parameters,nominal_dacs[-1],feature_spacing_m=float(certificate_cfg["si_111_spacing_m"]),electron_voltage_v=float(beam_cfg["voltage_v"]),selected_channels=("IL2","IL3"),selected_dac_half_width=.05,preserve_rotation=False)
assert signed_defocus_design.all_states_feasible and np.max(signed_defocus_design.magnification_relative_errors)<1e-9
print(f"600k signed phase-contrast design: defocus={signed_defocus_design.target_object_defocus_m*1e9} nm; controls={signed_defocus_design.selected_channel_names}; raw-DAC perturbations={signed_defocus_design.dac_perturbations[:,[column.channel_names.index(name) for name in signed_defocus_design.selected_channel_names]]}; rotation changes={signed_defocus_design.cumulative_rotation_changes_rad*1e6} µrad")
nominal_J={int(m):np.asarray(column.sample_position_jacobian(column.nominal_parameters,dac)) for m,dac in zip(magnifications,nominal_dacs,strict=True)}
sample_origin_xy=np.asarray(atlas.pose_translation_xy_m); sample_rotation=float(atlas.pose_rotation_rad); sample_tilt_xy=np.asarray(atlas.pose_tilt_xy_rad)
anisotropy=.006; detector_rotation=np.deg2rad(-1.7)
R_detector=np.array([[np.cos(detector_rotation),-np.sin(detector_rotation)],[np.sin(detector_rotation),np.cos(detector_rotation)]])
detector_affine=R_detector @ np.diag([1+anisotropy,1-anisotropy])/np.sqrt(1-anisotropy**2)
detector=replace(detector,affine_xy=detector_affine)  # one calibration shared by selection, surrogate, covariance, and physical rendering
def known_J(m): return detector_affine @ nominal_J[int(m)]
half=np.array([detector.shape_yx[1]*detector.pixel_size_yx_m[1],detector.shape_yx[0]*detector.pixel_size_yx_m[0]])/2
corners=np.array([[-half[0],-half[1]],[half[0],-half[1]],[half[0],half[1]],[-half[0],half[1]]])
views={}; records=[]; footprints={}
for m in magnifications:
    J_s=known_J(m)
    view=select_sample_view(
        atlas,transfer_jacobian_xy=J_s,detector_shape_yx=detector.shape_yx,detector_pixel_size_yx_m=detector.pixel_size_yx_m,
        sample_centre_xy_m=sample_origin_xy,sample_rotation_rad=sample_rotation,sample_tilt_xy_rad=sample_tilt_xy,region_name=None,gaussian_radius_min_m=3.4e-9,
        lens_wobble_margin_m=.15e-9,calibrated_input_margin_m=1.10e-9,gaussian_support_margin_m=.10e-9,
        pose_uncertainty_margin_m=.10e-9,interpolation_halo_margin_m=.05e-9,boundary=None,z=beam_cfg["sample_z_m"],
    )
    key=int(m); views[key]=view; s=view.selection; translation=np.asarray(s.sample_translation_xy_m)
    specimen_world_corners=corners@np.linalg.inv(J_s).T+sample_origin_xy
    footprints[key]=np.asarray(view.world_to_local(jnp.asarray(specimen_world_corners)))
    records.append(dict(magnification=key,target="automatic",family=s.feature_family,selected=s.selected_region_name or "global_atlas",regions=", ".join(s.region_names) or "—",unresolved=", ".join(s.unresolved_region_names) or "—",features=", ".join(s.feature_names) or "—",level=s.level_index,spacing_nm=1e9*s.selected_spacing_m,required_nm=1e9*s.required_spacing_m,pixel_nm=1e9*s.specimen_pixel_min_m,projected_px=s.projected_feature_pixels,intervals=s.visible_feature_intervals,margin_nm=1e9*s.total_margin_m,margin_components=tuple((label,1e9*value) for label,value in s.margin_components_m),feature_ok=s.meets_feature_sampling,resolution_ok=s.meets_resolution,certified=s.meets_feature_sampling and s.meets_resolution))
for target,achieved,coordinate,b_norm in zip(nominal_design.target_magnifications,nominal_design.achieved_magnifications,nominal_design.path_coordinates,nominal_design.specimen_conjugacy_b_norms,strict=True):
    print(f"Focused DAC solve {target:8.0f}×: achieved={achieved:11.3f}× seed-path={coordinate:+.6f} rel.err={abs(achieved/target-1):.2e} ||B||={b_norm:.3e} m")
assert nominal_design.all_targets_feasible and np.max(np.abs(nominal_design.achieved_magnifications/magnifications-1))<1e-4
for r in records:
    projected="—" if r['projected_px'] is None else f"{r['projected_px']:.2f}px"; intervals="—" if r['intervals'] is None else f"{r['intervals']:.2f}"
    margins=",".join(f"{name}:{value:.2f}" for name,value in r['margin_components'])
    print(f"{r['magnification']/1000:6.0f}k {r['family']:25s} tile={r['selected']} LOD={r['level']} Δ={r['spacing_nm']:.4f}/{r['required_nm']:.4f} nm projected={projected} intervals={intervals} margins[nm]={margins} unresolved={r['unresolved']} feature={r['feature_ok']} resolution={r['resolution_ok']} pass={r['certified']}")
assert all(r["certified"] for r in records), "A finer certified tile or smaller envelope is required."

fig,ax=plt.subplots(figsize=(7.5,6.8),constrained_layout=True); ax.imshow(level0.log_amplitude_yx,origin="lower",extent=extent_nm,cmap="Greys",alpha=.4)
for color,m in zip(plt.cm.viridis(np.linspace(.12,.9,len(magnifications))),magnifications,strict=True):
    ax.add_patch(Polygon(footprints[int(m)]*1e9,closed=True,fill=False,ec=color,lw=2,label=f"{m/1000:.0f}k"))
for region in atlas.regions.values():
    x0,x1,y0,y1=np.asarray(region.bounds_xyxy_m)*1e9; ax.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,ec=region_colors[region.kind],lw=.8,ls=":"))
ax.set(xlabel="local sample x [nm]",ylabel="local sample y [nm]",title="Inverse-mapped detector footprints on one specimen",aspect="equal"); ax.legend(frameon=False,ncols=3); plt.show()

## 2. Cubic values, gradients, Hessians, and boundaries

Gaussian propagation consumes a local quadratic action. Interpolating a wrapped complex field and then differentiating `angle` would insert discontinuities at phase wraps and unstable derivatives near zeros. The primary constructors therefore retain unwrapped phase-equivalent pathlength and log field amplitude separately. One explicit spline evaluation supplies $S,S_x,S_y,S_{xx},S_{xy},S_{yy}$ and the equivalent six quantities for $L$.

The following asymmetric, non-square analytic field exercises coordinate order, anisotropic pixels, multiple unwrapped cycles, pose, explicit derivatives, and `vacuum`, `constant`, and `periodic` boundaries.

In [ ]:
wavelength_m=float(np.asarray(energy2wavelength(beam_cfg["voltage_v"])))
ny_demo,nx_demo=25,37; dy_demo,dx_demo=.72e-9,.48e-9
x_demo=(np.arange(nx_demo)-(nx_demo-1)/2)*dx_demo; y_demo=(np.arange(ny_demo)-(ny_demo-1)/2)*dy_demo
xx_demo,yy_demo=np.meshgrid(x_demo,y_demo,indexing="xy")
phase_unwrapped=5.4*np.pi*xx_demo/np.ptp(x_demo)+.45*np.sin(2*np.pi*yy_demo/np.ptp(y_demo))
amplitude_demo=.92-.18*np.exp(-((xx_demo/3e-9)**2+(yy_demo/2.2e-9)**2))
demo_level=SampleLevel2D.from_amplitude_and_unwrapped_phase(
    amplitude_demo,phase_unwrapped,pixel_size_yx_m=(dy_demo,dx_demo),wavelength_m=wavelength_m,name="unwrapped-demo")
boundary_views={mode:CubicActionField2D.from_level(demo_level,boundary=mode,rotation_rad=np.deg2rad(11)) for mode in ("vacuum","constant","periodic")}
point=jnp.asarray([.83e-9,-1.31e-9]); jet=boundary_views["vacuum"].action_jet(point)
check=validate_sample_view_derivatives(boundary_views["vacuum"],point)
print("S [pm]",float(jet.pathlength.value*1e12),"  ∇S",np.asarray(jet.pathlength.gradient_xy))
print("H(S) [1/m]\n",np.asarray(jet.pathlength.hessian_xy))
print("explicit-vs-JAX max errors:",float(check.gradient_max_abs_error),float(check.hessian_max_abs_error))

view300=views[300_000]; hx=detector.shape_yx[1]*detector.pixel_size_yx_m[1]/600_000; hy=detector.shape_yx[0]*detector.pixel_size_yx_m[0]/600_000
xq=sample_origin_xy[0]+np.linspace(-.96*hx,.96*hx,45); yq=sample_origin_xy[1]+np.linspace(-.96*hy,.96*hy,41); xxq,yyq=np.meshgrid(xq,yq,indexing="xy")
atlas_jet=view300.action_jet(jnp.asarray(np.stack((xxq,yyq),axis=-1)))
path=np.asarray(atlas_jet.pathlength.value); logamp=np.asarray(atlas_jet.log_amplitude.value)
quantities=[
 (path*1e12,"S [pm]","coolwarm"),(np.linalg.norm(np.asarray(atlas_jet.pathlength.gradient_xy),axis=-1),"|∇S| [m/m]","viridis"),(np.linalg.norm(np.asarray(atlas_jet.pathlength.hessian_xy),axis=(-2,-1))*1e-9,"|H(S)| [nm⁻¹]","magma"),
 (logamp,"L = log amplitude","magma"),(np.linalg.norm(np.asarray(atlas_jet.log_amplitude.gradient_xy),axis=-1)*1e-9,"|∇L| [nm⁻¹]","viridis"),(np.linalg.norm(np.asarray(atlas_jet.log_amplitude.hessian_xy),axis=(-2,-1))*1e-18,"|H(L)| [nm⁻²]","magma")]
fig,axes=plt.subplots(2,3,figsize=(12.2,7),constrained_layout=True); extent=[xq[0]*1e9,xq[-1]*1e9,yq[0]*1e9,yq[-1]*1e9]
for ax,(q,title,cmap) in zip(axes.flat,quantities,strict=True):
    art=ax.imshow(q,origin="lower",extent=extent,cmap=cmap,aspect="equal"); ax.set(title=title,xlabel="world x [nm]",ylabel="world y [nm]"); fig.colorbar(art,ax=ax,shrink=.76)
fig.suptitle("Explicit C² action jets at the selected 300k view",fontsize=13); plt.show()

line_x=np.linspace(1.35*x_demo[0],1.35*x_demo[-1],301); points=jnp.stack((jnp.asarray(line_x),jnp.zeros_like(jnp.asarray(line_x))),axis=-1)
fig,ax=plt.subplots(figsize=(8.4,3.4),constrained_layout=True)
for mode,view in boundary_views.items(): ax.plot(line_x*1e9,np.asarray(view.action_jet(points).pathlength.value)*1e12,label=mode,lw=1.8)
ax.axvspan(x_demo[0]*1e9,x_demo[-1]*1e9,color="#CBD5E1",alpha=.25,label="raster")
ax.set(xlabel="query x [nm]",ylabel="S [pm]",title="Explicit boundary behavior"); ax.legend(frameon=False,ncols=4); plt.show()

### Five detector-frame previews

These images inverse-map native detector pixels through the solved hard-edge Jacobian and calibrated detector affine, evaluate the frozen cubic complex transmission, apply a declared 10 nm specimen-plane Fresnel defocus and detector MTF, and draw Poisson-plus-read noise. This remains a fast surrogate rather than the independently validated Gaussian render. The bottom row is a robust high-pass Fourier residual with the **certificate-predicted** vectors circled; it exposes whether the registered lattice or multilayer frequency is actually recoverable at the declared dose instead of assuming that it is. Image moments and Fourier peaks exercise intensity, centroid, covariance, orientation, frequency-vector, crop, sampling, dose, and metadata bookkeeping. All previews retain the native 256×256 geometry and a shared real-space display scale; every panel includes an honest specimen-plane scale bar.

In [ ]:
preview_shape=detector.shape_yx; preview_images={}; preview_expected={}; preview_observables={}; preview_certified_vectors={}; preview_spectral_peak_z={}
incident_intensity_sum=float(np.prod(preview_shape))  # one fixed vacuum-beam calibration for all frames
preview_dose=.25*float(detector_cfg["surrogate_dose_electrons"]); surrogate_defocus_m=float(detector_cfg["surrogate_phase_contrast_defocus_m"])
xdet=np.linspace(detector.x_coords_m[0],detector.x_coords_m[-1],preview_shape[1]); ydet=np.linspace(detector.y_coords_m[0],detector.y_coords_m[-1],preview_shape[0])
xxdet,yydet=np.meshgrid(xdet,ydet,indexing="xy"); detector_points=np.stack((xxdet,yydet),axis=-1)
fx_detector=np.fft.fftfreq(preview_shape[1],d=float(np.diff(xdet).mean())); fy_detector=np.fft.fftfreq(preview_shape[0],d=float(np.diff(ydet).mean())); qx_detector,qy_detector=np.meshgrid(fx_detector,fy_detector,indexing="xy")
preview_fov_xy_m=np.asarray(detector.pixel_size_yx_m)[::-1]*np.asarray(preview_shape)[::-1]; fft_centre_yx=np.asarray(preview_shape)//2
fig,axes=plt.subplots(2,len(magnifications),figsize=(15.2,6.2),constrained_layout=True)
for i,m in enumerate(magnifications):
    ax=axes[0,i]
    sample_points=detector_points@np.linalg.inv(known_J(m)).T+sample_origin_xy
    actions=np.asarray(views[int(m)].evaluate_actions(jnp.asarray(sample_points))); transmission=np.exp(actions[...,1]+2j*np.pi*actions[...,0]/wavelength_m)
    J_s=known_J(m); qx_sample=J_s[0,0]*qx_detector+J_s[1,0]*qy_detector; qy_sample=J_s[0,1]*qx_detector+J_s[1,1]*qy_detector
    fresnel=np.exp(-1j*np.pi*wavelength_m*surrogate_defocus_m*(qx_sample**2+qy_sample**2)); propagated=np.fft.ifft2(np.fft.fft2(transmission)*fresnel)
    expected=ndimage.gaussian_filter(np.abs(propagated)**2,detector.mtf_sigma_px); preview_expected[int(m)]=expected
    counts=simulate_detector_counts(expected,dose_electrons=preview_dose,incident_intensity_sum=incident_intensity_sum,read_noise_e=float(detector_cfg["surrogate_read_noise_e"]),seed=120+i); preview_images[int(m)]=counts
    preview_observables[int(m)]=extract_image_observables(np.clip(counts,0,None),n_fourier_vectors=6)
    selection=views[int(m)].selection; label=(selection.selected_region_name or "global_atlas").replace('_region',''); ax.imshow(counts,cmap="gray",origin="lower"); ax.set_title(f"{m/1000:.0f}k\n{label}",fontsize=9); ax.set(xticks=[],yticks=[])
    sample_m_per_horizontal_pixel=np.linalg.norm(np.linalg.inv(J_s)@np.asarray((detector.pixel_size_yx_m[1],0.)))
    bar_candidates_nm=np.asarray((.5,1,2,5,10,20,50,100)); bar_nm=float(bar_candidates_nm[bar_candidates_nm<=.22*preview_shape[1]*sample_m_per_horizontal_pixel*1e9][-1]); bar_px=bar_nm*1e-9/sample_m_per_horizontal_pixel
    ax.plot((10,10+bar_px),(10,10),color="white",lw=2.2,solid_capstyle="butt"); ax.text(10+.5*bar_px,15,f"{bar_nm:g} nm",color="white",ha="center",va="bottom",fontsize=7)
    window=np.outer(np.hanning(preview_shape[0]),np.hanning(preview_shape[1])); log_spectrum=np.log1p(np.abs(np.fft.fftshift(np.fft.fft2((counts-counts.mean())*window))))
    spectral_residual=log_spectrum-ndimage.gaussian_filter(log_spectrum,7.); spectral_sigma=1.4826*np.median(np.abs(spectral_residual-np.median(spectral_residual))); spectral_z=spectral_residual/max(spectral_sigma,np.finfo(float).eps)
    spacing=selection.routed_feature_scale_m; g_local=np.empty((0,2)) if spacing is None else np.asarray(((1/spacing,0.),(0.,1/spacing)) if selection.feature_family=="silicon_lattice_vectors" else ((1/spacing,0.),))
    projection=np.asarray(views[int(m)].projection_world_from_local); g_world=(np.linalg.inv(projection).T@g_local.T).T; g_detector=np.linalg.solve(J_s.T,g_world.T).T if len(g_world) else g_world; vectors_yx=(g_detector*preview_fov_xy_m)[:,::-1]; preview_certified_vectors[int(m)]=vectors_yx
    peak_z=[]; fft_ax=axes[1,i]; fft_ax.imshow(spectral_z,origin="lower",cmap="magma",vmin=-2,vmax=8)
    for vector_yx in vectors_yx:
        for sign in (-1,1):
            target=fft_centre_yx+sign*vector_yx; iy,ix=np.rint(target).astype(int)
            if 2<=iy<preview_shape[0]-2 and 2<=ix<preview_shape[1]-2:
                peak_z.append(float(np.max(spectral_z[iy-2:iy+3,ix-2:ix+3]))); fft_ax.scatter(target[1],target[0],s=42,facecolors="none",edgecolors="#56B4E9",linewidths=1.2)
    preview_spectral_peak_z[int(m)]=tuple(peak_z); peak_label="—" if not peak_z else f"{max(peak_z):.1f}σ"; fft_ax.set(title=f"○ certified ±g\npeak {peak_label}",xticks=[],yticks=[]); fft_ax.title.set_fontsize(8)
preview_vmax=float(np.percentile(np.stack(tuple(preview_images.values())),99.8))
for ax in axes[0]: ax.images[0].set_clim(0,preview_vmax)
fft_colorbar=fig.colorbar(axes[1,0].images[0],ax=list(axes[1]),shrink=.78,pad=.012); fft_colorbar.set_label("local Fourier residual [robust σ]",fontsize=8)
fig.suptitle("Noisy weak-phase detector previews and the Fourier evidence used for inversion",fontsize=12)
preview_exports=export_ray_figure(fig,GENERATED/"neoarm_multimag_noisy_previews",metadata={"title":"NEOARM multimag sample-image previews","magnifications":magnifications,"detector_shape_yx":detector.shape_yx,"phase_contrast_defocus_m":surrogate_defocus_m,"shared_display_range_counts":[0,preview_vmax],"certified_vectors_yx_bins":preview_certified_vectors,"certificate_peak_robust_z":preview_spectral_peak_z,"scope":"weak-phase crop, feature-routing, and certificate-frequency recoverability check"})
print("Preview exports:",preview_exports); plt.show()
print("image-derived summaries (native detector pixels):")
for m in magnifications:
    obs=preview_observables[int(m)]; fourier=np.array2string(obs.fourier_vectors_yx_px,precision=1,suppress_small=True)
    targets=np.array2string(preview_certified_vectors[int(m)],precision=1,suppress_small=True); peak=max(preview_spectral_peak_z[int(m)],default=np.nan)
    print(f"  {m:8.0f}× I={obs.total_intensity:9.1f} centroid={obs.centroid_xy_px} σ={obs.principal_sigma_px} angle={np.rad2deg(obs.principal_rotation_rad):7.2f}° extracted(y,x)={fourier} certificate(y,x)={targets} peak={peak:.2f} robust σ")

## 3. Gaussian interaction and an independent Collins reference

The Gaussian route has no fixed beam-count ceiling. It begins with at least 8–10 centers per shortest retained period, refines high-power regions, freezes the topology across compared states, and streams detector tiles. A work estimate is produced before full rendering.

A smooth patch uses the cubic value/gradient/Hessian action only when the illumination-weighted phase residual is below 0.1 rad and log-amplitude residual below 0.02. A Gaussian crossing an interface uses exact grid multiplication and Gaussian cardinal re-expansion. The route is chosen before JIT and cannot switch inside the loss.

The GPU smoke benchmark below uses a small off-knot patch from the selected 300,000× atlas view and a co-rotating drift segment. It tests cubic multiplication, Gaussian re-expansion, absolute relative power, and propagation against independent Simpson-quadrature Collins integration; it is not presented as a full-column image benchmark. Production images enter the optical fit only after the same gates pass for the frozen full-column transfer matrix and experimental noise model.

In [ ]:
si111=float(certificate_cfg["si_111_spacing_m"])
topology=build_multiresolution_gaussian_topology(
    (-12e-9,12e-9,-12e-9,12e-9),23e-9,centres_per_period=10,overlap_factor=experiment_cfg["gaussian_overlap_factor"],
    refinement_regions=(RefinementRegion((-6e-9,6e-9,-6e-9,6e-9),si111,"Si {111}"),),max_beamlets=None)
benchmark_count=min(256,topology.beam_count); benchmark_centres=np.asarray(topology.centres_xy_m[:benchmark_count]); benchmark_detector=DetectorGeometry(shape_yx=(64,64),pixel_size_yx_m=tuple(np.asarray(detector.pixel_size_yx_m)*4),z_m=beam_cfg["sample_z_m"],affine_xy=detector.affine_xy)
benchmark_beam=make_gaussian(x=benchmark_centres[:,0],y=benchmark_centres[:,1],dx=np.zeros(benchmark_count),dy=np.zeros(benchmark_count),z=np.full(benchmark_count,beam_cfg["sample_z_m"]),voltage=np.full(benchmark_count,beam_cfg["voltage_v"]),waist_x=np.full(benchmark_count,.02),waist_y=np.full(benchmark_count,.02))
_=render_gaussian_detector(benchmark_beam,np.eye(4),benchmark_detector,tile_rows=32,beam_batch_size=128,method=FORWARD_EVALUATION_METHOD); benchmark_render=render_gaussian_detector(benchmark_beam,np.eye(4),benchmark_detector,tile_rows=32,beam_batch_size=128,method=FORWARD_EVALUATION_METHOD)
measured_interactions_per_second=benchmark_count*np.prod(benchmark_detector.shape_yx)/max(benchmark_render.elapsed_s,np.finfo(float).eps); work=estimate_gaussian_workload(topology,detector,tile_shape_yx=(32,32),beam_chunk=256,measured_interactions_per_second=measured_interactions_per_second)
print("Frozen topology:",topology.metadata); print(work.human_summary()); print(f"Interactions: {work.interaction_count:,}; measured smoke throughput: {measured_interactions_per_second/1e6:.2f} million interactions/s")

axis=np.linspace(-2.4e-9,2.4e-9,7); vx_relative,vy_relative=np.meshgrid(axis,axis,indexing="xy"); vx=vx_relative+sample_origin_xy[0]; vy=vy_relative+sample_origin_xy[1]; centres=np.stack((vx.ravel(),vy.ravel()),axis=-1)
bundle=make_gaussian(x=centres[:,0],y=centres[:,1],dx=np.zeros(49),dy=np.zeros(49),z=np.full(49,beam_cfg["sample_z_m"]),voltage=np.full(49,beam_cfg["voltage_v"]),waist_x=np.full(49,.62e-9),waist_y=np.full(49,.62e-9))
adequacy=validate_sample_level(view300,bundle,phase_tol=experiment_cfg["local_phase_residual_limit_rad"],amplitude_tol=experiment_cfg["local_log_amplitude_residual_limit"],support_radius=2)
diagnostic=getattr(adequacy,"diagnostic",adequacy); route=getattr(adequacy,"recommended_route",getattr(adequacy,"interaction_mode","inspect"))
phase_res=np.asarray(diagnostic.phase_max_abs).reshape(7,7); amp_res=np.asarray(diagnostic.amplitude_max_abs).reshape(7,7)
scan_x=sample_origin_xy[0]+np.linspace(-10e-9,10e-9,401); scan_points=jnp.stack((jnp.asarray(scan_x),jnp.full_like(jnp.asarray(scan_x),sample_origin_xy[1])),axis=-1)
scan_jet=view300.action_jet(scan_points); interface_x=scan_x[np.argmax(np.linalg.norm(np.asarray(scan_jet.log_amplitude.gradient_xy),axis=-1))]
interface_centres=np.array([[interface_x+dx,sample_origin_xy[1]+dy] for dx in (-.8e-9,0,.8e-9) for dy in (-.8e-9,0,.8e-9)])
interface_bundle=make_gaussian(x=interface_centres[:,0],y=interface_centres[:,1],dx=np.zeros(9),dy=np.zeros(9),z=np.full(9,beam_cfg["sample_z_m"]),voltage=np.full(9,beam_cfg["voltage_v"]),waist_x=np.full(9,2.2e-9),waist_y=np.full(9,2.2e-9))
interface_adequacy=validate_sample_level(view300,interface_bundle,phase_tol=experiment_cfg["local_phase_residual_limit_rad"],amplitude_tol=experiment_cfg["local_log_amplitude_residual_limit"],support_radius=2)
print("Smooth-patch route:",route,"; maxima:",phase_res.max(),amp_res.max())
print(f"Interface at x={interface_x*1e9:.3f} nm routes to {interface_adequacy.recommended_route}; accepted topology remains frozen.")
fig,axes=plt.subplots(1,3,figsize=(12,3.7),constrained_layout=True); actions=np.asarray(view300.evaluate_actions(jnp.asarray(np.stack((vx,vy),axis=-1))))
axes[0].imshow(actions[...,1],origin="lower",extent=[-2.4,2.4,-2.4,2.4],cmap="magma"); axes[0].scatter((centres[:,0]-sample_origin_xy[0])*1e9,(centres[:,1]-sample_origin_xy[1])*1e9,s=7,c="white")
for centre in centres: axes[0].add_patch(Circle(tuple((centre-sample_origin_xy)*1e9),.62,fill=False,ec="white",lw=.45,alpha=.7))
axes[0].set_title("Gaussian centers and 1/e radii")
for ax,image,title,limit in ((axes[1],phase_res,"phase residual [rad]",.1),(axes[2],amp_res,"log-amplitude residual",.02)):
    art=ax.imshow(image,origin="lower",extent=[-2.4,2.4,-2.4,2.4],cmap="viridis"); ax.contour(np.linspace(-2.4,2.4,7),np.linspace(-2.4,2.4,7),image,levels=[limit],colors="#D55E00"); ax.set_title(title); fig.colorbar(art,ax=ax,shrink=.78)
for ax in axes: ax.set(xlabel="x [nm]",ylabel="y [nm]",aspect="equal")
plt.show()

In [ ]:
# Exact multiplication by the cubic sample, followed by a small Gaussian re-expansion.
n_in=17; xin=interface_x+np.linspace(-4e-9,4e-9,n_in); yin=sample_origin_xy[1]+np.linspace(-4e-9,4e-9,n_in); xxi,yyi=np.meshgrid(xin,yin,indexing="xy")
incident=np.exp(-(((xxi-interface_x)/2.4e-9)**2+((yyi-sample_origin_xy[1])/2.0e-9)**2))*np.exp(0.18j*(xxi-interface_x)/1e-9)
width_over=(1.25,1.25)
reexpanded,transmitted=reexpand_transmitted_field(incident,view300,xin,yin,voltage=beam_cfg["voltage_v"],z_m=beam_cfg["sample_z_m"],width_over_step_yx=width_over)
coeff=gaussian_cardinal_coefficients(transmitted,width_over_step_yx=width_over,kernel_tolerance=1e-13,ridge=1e-10)
Ky=np.exp(-((np.arange(n_in)[:,None]-np.arange(n_in)[None,:])/width_over[0])**2); Kx=np.exp(-((np.arange(n_in)[:,None]-np.arange(n_in)[None,:])/width_over[1])**2)
reconstructed=Ky@coeff@Kx.T
dxin=np.diff(xin).mean(); dyin=np.diff(yin).mean(); xoff=np.linspace(xin[0]+.37*dxin,xin[-1]-.37*dxin,29); yoff=np.linspace(yin[0]+.29*dyin,yin[-1]-.29*dyin,31)
xxoff,yyoff=np.meshgrid(xoff,yoff,indexing="xy"); incident_off=np.exp(-(((xxoff-interface_x)/2.4e-9)**2+((yyoff-sample_origin_xy[1])/2.0e-9)**2))*np.exp(.18j*(xxoff-interface_x)/1e-9)
off_actions=np.asarray(view300.evaluate_actions(jnp.asarray(np.stack((xxoff,yyoff),axis=-1))))
transmitted_off=incident_off*np.exp(off_actions[...,1]+2j*np.pi*off_actions[...,0]/wavelength_m)
Ky_off=np.exp(-((yoff[:,None]-yin[None,:])/(width_over[0]*dyin))**2); Kx_off=np.exp(-((xoff[:,None]-xin[None,:])/(width_over[1]*dxin))**2)
reconstructed_off=Ky_off@coeff@Kx_off.T

propagation_distance=.022; validation_detector=DetectorGeometry(shape_yx=(17,17),pixel_size_yx_m=(4.5e-6,4.5e-6),z_m=beam_cfg["sample_z_m"]+propagation_distance)
gaussian_render=render_gaussian_detector(reexpanded,drift_matrix(propagation_distance),validation_detector,tile_rows=17,beam_batch_size=64,method=FORWARD_EVALUATION_METHOD)
collins_field=collins_propagate_2d(transmitted,xin,yin,validation_detector.x_coords_m,validation_detector.y_coords_m,abcd_x=(1,propagation_distance,0,1),abcd_y=(1,propagation_distance,0,1),wavelength_m=wavelength_m)
I_gauss_raw=gaussian_render.intensity_yx; I_grid_raw=np.abs(collins_field)**2
reference_intensity_scale=max(float(I_grid_raw.max()),np.finfo(float).tiny)
I_gauss=I_gauss_raw/reference_intensity_scale; I_grid=I_grid_raw/reference_intensity_scale  # one reference calibration, never separate frame normalization
difference=I_gauss-I_grid
sample_field_noise_sigma=.02  # declared complex-field noise scale for this dimensionless calibration
obs_gaussian=extract_image_observables(I_gauss); obs_grid=extract_image_observables(I_grid)
def friedel_peak_discrepancy(candidate,reference):
    candidate=np.asarray(candidate); reference=np.asarray(reference)
    return max(min(min(np.linalg.norm(c-r),np.linalg.norm(c+r)) for c in candidate) for r in reference) if len(candidate) and len(reference) else np.inf
scale_gaussian=np.sqrt(np.prod(obs_gaussian.principal_sigma_px)); scale_grid=np.sqrt(np.prod(obs_grid.principal_sigma_px))
rotation_difference=abs(np.angle(np.exp(1j*(obs_gaussian.principal_rotation_rad-obs_grid.principal_rotation_rad))))
observable_residuals={
    "scale":abs(scale_gaussian/scale_grid-1)/.05,
    "rotation":rotation_difference/np.deg2rad(2),
    "centroid":np.linalg.norm(np.asarray(obs_gaussian.centroid_xy_px)-np.asarray(obs_grid.centroid_xy_px))/.5,
    "fourier_vector":friedel_peak_discrepancy(obs_gaussian.fourier_vectors_yx_px,obs_grid.fourier_vectors_yx_px)/2,
    "intensity":abs(obs_gaussian.total_intensity/obs_grid.total_intensity-1)/.05,
}
pair_validation=validate_forward_models(
    I_gauss,I_grid,noise_sigma_yx=.03,candidate_sample_field_yx=reconstructed_off,
    reference_sample_field_yx=transmitted_off,sample_field_noise_sigma=sample_field_noise_sigma,
    observable_residuals_in_sigma=observable_residuals,require_complete=False)
validation=validate_forward_models(
    I_gauss,I_grid,noise_sigma_yx=.03,candidate_sample_field_yx=reconstructed_off,
    reference_sample_field_yx=transmitted_off,sample_field_noise_sigma=sample_field_noise_sigma,
    observable_residuals_in_sigma=observable_residuals,parameter_change_in_sigma=None,require_complete=True)
wave_images_admitted=validation.passed
print(f"sample-field discrepancy: {pair_validation.sample_field_rms_in_noise_sigma:.3e} noise σ (noise scale={sample_field_noise_sigma:g})")
print("observable discrepancies [experimental σ]:",observable_residuals,"missing gates:",validation.missing_gates)
print(f"detector discrepancy: {pair_validation.detector_rms_in_noise_sigma:.3f} noise σ; pairwise gate={pair_validation.passed}; admitted={wave_images_admitted}")

fig,axes=plt.subplots(2,3,figsize=(11.5,7),constrained_layout=True)
for ax,image,title,cmap in ((axes[0,0],np.abs(transmitted_off),"exact atlas transmission, off-knot","magma"),(axes[0,1],np.abs(reconstructed_off),"Gaussian reconstruction, off-knot","magma"),(axes[0,2],np.abs(reconstructed_off-transmitted_off),"off-knot sample-plane |error|","viridis"),(axes[1,0],I_grid,"Collins reference","gray"),(axes[1,1],I_gauss,"Gaussian detector","gray"),(axes[1,2],difference,"Gaussian − Collins","coolwarm")):
    art=ax.imshow(image,origin="lower",cmap=cmap); ax.set(title=title,xticks=[],yticks=[]); fig.colorbar(art,ax=ax,shrink=.72)
fig.suptitle("Exact cubic multiplication, re-expansion, and independent propagation",fontsize=13); plt.show()
mid=len(yoff)//2; fig_profile,profile_axes=plt.subplots(1,2,figsize=(9.2,3.3),constrained_layout=True)
profile_axes[0].plot(xoff*1e9,np.abs(transmitted_off[mid]),color="#0072B2",lw=2,label="cubic atlas"); profile_axes[0].plot(xoff*1e9,np.abs(reconstructed_off[mid]),color="#D55E00",ls="--",label="Gaussian expansion")
profile_axes[1].plot(xoff*1e9,np.unwrap(np.angle(transmitted_off[mid])),color="#0072B2",lw=2); profile_axes[1].plot(xoff*1e9,np.unwrap(np.angle(reconstructed_off[mid])),color="#D55E00",ls="--")
profile_axes[0].set(ylabel="field amplitude",title="Off-knot central profile"); profile_axes[1].set(ylabel="unwrapped phase [rad]",title="Phase-equivalent path response")
for ax in profile_axes: ax.set_xlabel("sample x [nm]"); ax.grid(alpha=.2)
profile_axes[0].legend(frameon=False); plt.show()

if not wave_images_admitted:
    print("Image observables remain excluded: refine the sample grid/Gaussian topology until the 0.2-noise-σ gate passes.")

### Successive-topology admission, including fitted-parameter stability

A single Gaussian/Collins comparison is not enough: a forward image can look converged while the inverse estimate still moves. The next cell uses one tapered physical field from the registered atlas, one fixed off-knot sample grid, one fixed detector grid, and three strictly denser Gaussian expansions. Each topology independently estimates the same effective propagation distance from the independent fine-grid Collins image. Admission requires two successive transitions to pass every field, detector, observable, topology, and $<0.1\sigma$ parameter-change gate. No image from the Gaussian model enters the inversion unless both this refinement sequence and the final Collins comparison pass.

In [ ]:
refinement_extent_m=8e-9; refinement_sizes=(33,41,49); refinement_width_over=(1.25,1.25)
refinement_detector=DetectorGeometry(shape_yx=(25,25),pixel_size_yx_m=(3e-6,3e-6),z_m=beam_cfg["sample_z_m"]+propagation_distance)
def atlas_transmitted_field(x_axis,y_axis):
    xx,yy=np.meshgrid(x_axis,y_axis,indexing="xy")
    illumination=np.exp(-(((xx-interface_x)/2.4e-9)**2+((yy-sample_origin_xy[1])/2.0e-9)**2))*np.exp(.18j*(xx-interface_x)/1e-9)
    actions=np.asarray(view300.evaluate_actions(jnp.asarray(np.stack((xx,yy),axis=-1))))
    return illumination*np.exp(actions[...,1]+2j*np.pi*actions[...,0]/wavelength_m)

# This fine Collins field is independent of every Gaussian topology below.
reference_x=interface_x+np.linspace(-refinement_extent_m,refinement_extent_m,257); reference_y=sample_origin_xy[1]+np.linspace(-refinement_extent_m,refinement_extent_m,257)
reference_sample_field=atlas_transmitted_field(reference_x,reference_y)
reference_collins_field=collins_propagate_2d(reference_sample_field,reference_x,reference_y,refinement_detector.x_coords_m,refinement_detector.y_coords_m,abcd_x=(1,propagation_distance,0,1),abcd_y=(1,propagation_distance,0,1),wavelength_m=wavelength_m)
refinement_reference_raw=np.abs(reference_collins_field)**2; refinement_intensity_scale=max(float(refinement_reference_raw.max()),np.finfo(float).tiny); refinement_reference=refinement_reference_raw/refinement_intensity_scale
validation_x=interface_x+np.linspace(-.9625*refinement_extent_m,.9625*refinement_extent_m,83); validation_y=sample_origin_xy[1]+np.linspace(-.9625*refinement_extent_m,.9625*refinement_extent_m,85)
validation_sample_exact=atlas_transmitted_field(validation_x,validation_y)
refinement_detector_noise=.03; refinement_sample_noise=.02; refinement_snapshots=[]; refinement_products=[]
def canonical_friedel(vector):
    vector=np.asarray(vector,dtype=float).copy(); pivot=int(np.argmax(np.abs(vector)))
    return -vector if vector[pivot]<0 else vector
for n_side in refinement_sizes:
    x_nodes=interface_x+np.linspace(-refinement_extent_m,refinement_extent_m,n_side); y_nodes=sample_origin_xy[1]+np.linspace(-refinement_extent_m,refinement_extent_m,n_side)
    xx_nodes,yy_nodes=np.meshgrid(x_nodes,y_nodes,indexing="xy"); incident_nodes=np.exp(-(((xx_nodes-interface_x)/2.4e-9)**2+((yy_nodes-sample_origin_xy[1])/2.0e-9)**2))*np.exp(.18j*(xx_nodes-interface_x)/1e-9)
    beam_nodes,transmitted_nodes=reexpand_transmitted_field(incident_nodes,view300,x_nodes,y_nodes,voltage=beam_cfg["voltage_v"],z_m=beam_cfg["sample_z_m"],width_over_step_yx=refinement_width_over)
    coefficient_nodes=gaussian_cardinal_coefficients(transmitted_nodes,width_over_step_yx=refinement_width_over,kernel_tolerance=1e-13,ridge=1e-10)
    dx_nodes=np.diff(x_nodes).mean(); dy_nodes=np.diff(y_nodes).mean(); kernel_y=np.exp(-((validation_y[:,None]-y_nodes[None,:])/(refinement_width_over[0]*dy_nodes))**2); kernel_x=np.exp(-((validation_x[:,None]-x_nodes[None,:])/(refinement_width_over[1]*dx_nodes))**2)
    reconstructed_validation=kernel_y@coefficient_nodes@kernel_x.T
    def refinement_render(distance_m):
        return render_gaussian_detector(beam_nodes,drift_matrix(distance_m),refinement_detector,tile_rows=25,beam_batch_size=256,method=FORWARD_EVALUATION_METHOD).intensity_yx/refinement_intensity_scale
    candidate_image=refinement_render(propagation_distance); distance_step=20e-6; image_derivative=(refinement_render(propagation_distance+distance_step)-refinement_render(propagation_distance-distance_step))/(2*distance_step)
    distance_estimate=propagation_distance-np.sum(image_derivative*(candidate_image-refinement_reference))/np.sum(image_derivative**2); distance_standard_error=refinement_detector_noise/np.sqrt(np.sum(image_derivative**2))
    obs=extract_image_observables(candidate_image,n_fourier_vectors=1); spectral_vector=canonical_friedel(obs.fourier_vectors_yx_px[0])
    physical_observables={"scale":np.sqrt(np.prod(obs.principal_sigma_px)),"rotation":np.rad2deg(obs.principal_rotation_rad),"centroid":np.asarray(obs.centroid_xy_px),"fourier_vector":spectral_vector,"intensity":obs.total_intensity}
    snapshot=GaussianRefinementSnapshot(topology_id=f"registered-interface-{n_side}x{n_side}",beam_count=n_side*n_side,detector_intensity_yx=candidate_image,sample_field_yx=reconstructed_validation,observables=physical_observables,fit_parameters=np.asarray((distance_estimate,)),fit_standard_errors=np.asarray((distance_standard_error,)),identifiable_parameter_mask=np.asarray((True,)))
    refinement_snapshots.append(snapshot); refinement_products.append((n_side,candidate_image,reconstructed_validation,distance_estimate,distance_standard_error))
reference_obs=extract_image_observables(refinement_reference,n_fourier_vectors=1); reference_vector=canonical_friedel(reference_obs.fourier_vectors_yx_px[0]); reference_scale=np.sqrt(np.prod(reference_obs.principal_sigma_px))
refinement_observable_errors={"scale":.05*reference_scale,"rotation":2.0,"centroid":np.full(2,.5),"fourier_vector":np.full(2,2.0),"intensity":.05*reference_obs.total_intensity}
refinement_admission=validate_gaussian_refinement_sequence(refinement_snapshots,detector_noise_sigma_yx=refinement_detector_noise,sample_field_noise_sigma=refinement_sample_noise,observable_standard_errors=refinement_observable_errors,required_successive_refinements=2)
last_n,last_image,last_sample,last_distance,last_distance_se=refinement_products[-1]; last_obs=refinement_snapshots[-1].observables
final_observable_residuals={"scale":abs(float(last_obs["scale"])-reference_scale)/refinement_observable_errors["scale"],"rotation":abs((float(last_obs["rotation"])-np.rad2deg(reference_obs.principal_rotation_rad)+180)%360-180)/refinement_observable_errors["rotation"],"centroid":np.max(np.abs(np.asarray(last_obs["centroid"])-np.asarray(reference_obs.centroid_xy_px))/refinement_observable_errors["centroid"]),"fourier_vector":np.max(np.abs(np.asarray(last_obs["fourier_vector"])-reference_vector)/refinement_observable_errors["fourier_vector"]),"intensity":abs(float(last_obs["intensity"])-reference_obs.total_intensity)/refinement_observable_errors["intensity"]}
final_collins_validation=validate_forward_models(last_image,refinement_reference,refinement_detector_noise,candidate_sample_field_yx=last_sample,reference_sample_field_yx=validation_sample_exact,sample_field_noise_sigma=refinement_sample_noise,observable_residuals_in_sigma=final_observable_residuals,parameter_change_in_sigma=abs(last_distance-propagation_distance)/last_distance_se,require_complete=True)
wave_images_admitted=bool(refinement_admission.passed and final_collins_validation.passed)
for step in refinement_admission.steps: print(f"{step.previous_topology_id} → {step.current_topology_id}: detector={step.forward_validation.detector_rms_in_noise_sigma:.3e}σ sample={step.forward_validation.sample_field_rms_in_noise_sigma:.3e}σ parameter={step.forward_validation.parameter_change_in_sigma:.3e}σ pass={step.passed}")
print(f"Final {last_n}×{last_n} vs independent Collins: detector={final_collins_validation.detector_rms_in_noise_sigma:.3e}σ, sample={final_collins_validation.sample_field_rms_in_noise_sigma:.3e}σ, fitted z={last_distance:.9f}±{last_distance_se:.2e} m; admitted={wave_images_admitted}")
assert wave_images_admitted, (refinement_admission.failure_reasons,final_collins_validation.missing_gates,final_observable_residuals)
fig_refine,axes_refine=plt.subplots(2,len(refinement_products)+1,figsize=(12.8,6.0),constrained_layout=True)
for column_index,(n_side,image,sample_field,distance_estimate,distance_se) in enumerate(refinement_products):
    axes_refine[0,column_index].imshow(np.abs(sample_field),origin="lower",cmap="magma"); axes_refine[0,column_index].set_title(f"{n_side}² beamlets\nfixed-grid |field|",fontsize=9)
    axes_refine[1,column_index].imshow(image,origin="lower",cmap="gray",vmin=0,vmax=refinement_reference.max()); axes_refine[1,column_index].set_title(f"ẑ={distance_estimate*1e3:.6f}±{distance_se*1e3:.3f} mm",fontsize=9)
axes_refine[0,-1].imshow(np.abs(last_sample-validation_sample_exact),origin="lower",cmap="viridis"); axes_refine[0,-1].set_title("finest field error",fontsize=9)
difference_limit=max(float(np.max(np.abs(last_image-refinement_reference))),np.finfo(float).tiny); axes_refine[1,-1].imshow(last_image-refinement_reference,origin="lower",cmap="coolwarm",vmin=-difference_limit,vmax=difference_limit); axes_refine[1,-1].set_title("finest − Collins",fontsize=9)
for ax in axes_refine.ravel(): ax.set(xticks=[],yticks=[])
fig_refine.suptitle("Gaussian topology convergence: fields, images, and the inferred optic",fontsize=12.5); plt.show()

## 4. Synthetic/reference compact acquisition: 30 optical settings and 230 detector frames

This 230-frame manifest is a synthetic/reference forward and diagnostic branch, not the primary instrument acquisition contract. It contains five nominal states at 40,000×, 80,000×, 150,000×, 300,000×, and 600,000×; fourteen positive/negative one-lens perturbations for `CL1`, `CL3`, `C-mini`, `IL1`, `IL2`, `IL3`, and `PL1`; eight multivariate information-rich states; and three held-out multivariate states. The nominal raw-DAC rows are solved deterministically against the sample-to-detector Jacobian of the hard-edge column along a declared IL1/IL2/IL3/PL1 control path. `CL3` is retained as the JEOL name for the second condenser lens.

Within this synthetic/reference manifest, every setting receives perforation, calibrated position and angle bases, and an automatically routed certified sample view. These basis frames validate forward conventions and downstream identifiability; they are not required hardware acquisitions for the no-commanded-tilt condenser route. Perforation and basis measurements are explicitly routed through the central vacuum hole. Six settings receive orthogonal/negative bases; a partly different set of six (three nominal and all three held-out settings) receives indexed diffraction, so the held-out diffraction threshold is measurable. Those diffraction rows always use the registered periodic silicon-lattice tile, independently of the real-space image feature selected at that optical state. Nominal settings repeat the complete core from the reverse DAC approach; held-out settings repeat perforation and calibrated bases, retain one forward sample frame, and use the eighth frame for diffraction. Every approach records a preconditioning endpoint, ramp count, and settling time instead of using a bare direction label. The manifest carries all seven synthetic raw-DAC endpoints and approach starts, sample state and covariance, target/reference/predicted magnification, interaction route, hashed topology specification, stage translation, selected region and level, crop, and physical sampling. A fast detector-native surrogate produces all 230 absolute-count frames for pipeline testing, whereas the explicitly labelled Gaussian/Collins comparison above remains the wave-optical validation.

In [ ]:
from temgym_core.critical_line_calibration import circular_gaussian_aperture_transmission, rms_beam_radius
reference_dacs=nominal_dacs[2].copy(); design_parameters=column.nominal_parameters.copy(); configured_wobble_step=float(experiment_cfg["single_lens_raw_dac_step"])
def effective_magnification_for_dac(dac): return float(np.sqrt(abs(np.linalg.det(detector_affine@np.asarray(column.sample_position_jacobian(design_parameters,np.asarray(dac)))))))
reference_effective_magnification=effective_magnification_for_dac(reference_dacs); perturbations=np.empty(len(column.channel_names)); wobble_fractional_band=(.70,1.30)
for channel_index in range(len(column.channel_names)):
    accepted=[]
    for candidate_step in np.geomspace(configured_wobble_step*1e-7,configured_wobble_step,257):
        candidate_magnifications=[]
        for sign in (-1.,1.):
            candidate=reference_dacs.copy(); candidate[channel_index]+=sign*candidate_step; candidate_magnifications.append(effective_magnification_for_dac(candidate))
        if all(wobble_fractional_band[0]*reference_effective_magnification<=value<=wobble_fractional_band[1]*reference_effective_magnification for value in candidate_magnifications): accepted.append(candidate_step)
        else: break  # remain on the connected local branch containing zero wobble
    if not accepted: raise RuntimeError(f"No certified local wobble found for {column.channel_names[channel_index]}.")
    perturbations[channel_index]=accepted[-1]
print("Certified one-lens raw-DAC steps:",dict(zip(column.channel_names,perturbations,strict=True)))
entrance_cfg=beam_cfg["entrance_calibration"]; entrance_mean=np.asarray(entrance_cfg["mean_state_x_thetax_y_thetay"],dtype=float)
sample_sigma_position=np.asarray(entrance_cfg["reference_sample_sigma_position_xy_m"],dtype=float); sample_sigma_tilt=np.asarray(entrance_cfg["reference_sample_sigma_tilt_xy_rad"],dtype=float)
reference_sample_covariance=np.diag((sample_sigma_position[0]**2,sample_sigma_tilt[0]**2,sample_sigma_position[1]**2,sample_sigma_tilt[1]**2)); reference_upstream=np.asarray(column.matrix(design_parameters,reference_dacs,segment="source_to_sample")); entrance_inverse=np.linalg.inv(reference_upstream); entrance_covariance=entrance_inverse@reference_sample_covariance@entrance_inverse.T
entrance_source=CalibratedSourceState(entrance_mean,entrance_covariance,label="frozen_correlated_entrance")
entrance_covariance_chart=EntranceCovarianceChart(entrance_source.covariance); entrance_covariance_reference_coordinates=np.zeros(entrance_covariance_chart.n_coordinates)
recon_cfg=experiment_cfg["condenser_reconnaissance"]; aperture_cfg=beam_cfg["condenser_aperture"]; APERTURE_FIT_MINIMUM_CLEARANCE_M=5e-4; cl1_index=column.channel_names.index(recon_cfg["scan_channel"]); cl3_index=column.channel_names.index(recon_cfg["wobble_channel"])
scan_half_width=float(recon_cfg["scan_half_width_raw_dac"]); critical_scan_values=np.linspace(reference_dacs[cl1_index]-scan_half_width,reference_dacs[cl1_index]+scan_half_width,int(recon_cfg["scan_point_count"])); critical_wobble=float(recon_cfg["wobble_step_raw_dac"]); retrieval_log_radius_sigma=.002; incident_electrons=float(aperture_cfg["incident_electrons_per_frame"])
def make_recon_pair_dacs(scan_values):
    pairs=[]
    for scan_value in np.asarray(scan_values,dtype=float):
        pair=np.repeat(reference_dacs[None,:],2,axis=0); pair[:,cl1_index]=scan_value; pair[0,cl3_index]-=critical_wobble; pair[1,cl3_index]+=critical_wobble; pairs.append(pair)
    return np.stack(pairs)
coarse_recon_pair_dacs=make_recon_pair_dacs(critical_scan_values)
def reconnaissance_frame_observables(parameters,pair_dacs):
    return condenser_reconnaissance_frame_observables(column,parameters,pair_dacs,entrance_source.mean_state,entrance_source.covariance,aperture_z_m=float(aperture_cfg["z_m"]),aperture_radius_m=float(aperture_cfg["radius_m"]),observation_z_m=column.sample_z_m,radial_order=32,azimuthal_order=96)
def reconnaissance_response(parameters,pair_dacs,retain_absolute_log_radius=False):
    return condenser_reconnaissance_response(column,parameters,pair_dacs,entrance_source.mean_state,entrance_source.covariance,wobble_step=critical_wobble,aperture_z_m=float(aperture_cfg["z_m"]),aperture_radius_m=float(aperture_cfg["radius_m"]),observation_z_m=column.sample_z_m,retain_absolute_log_radius=retain_absolute_log_radius,radial_order=32,azimuthal_order=96)
def reconnaissance_response_with_covariance(parameters,source_covariance,pair_dacs,retain_absolute_log_radius=False):
    return condenser_reconnaissance_response(column,parameters,pair_dacs,entrance_source.mean_state,source_covariance,wobble_step=critical_wobble,aperture_z_m=float(aperture_cfg["z_m"]),aperture_radius_m=float(aperture_cfg["radius_m"]),observation_z_m=column.sample_z_m,retain_absolute_log_radius=retain_absolute_log_radius,radial_order=32,azimuthal_order=96)
def independent_reconnaissance_response(parameters,pair_dacs):
    return condenser_independent_reconnaissance_response(column,parameters,pair_dacs,entrance_source.mean_state,entrance_source.covariance,aperture_z_m=float(aperture_cfg["z_m"]),aperture_radius_m=float(aperture_cfg["radius_m"]),observation_z_m=column.sample_z_m,radial_order=32,azimuthal_order=96)
def independent_reconnaissance_response_with_covariance(parameters,source_covariance,pair_dacs):
    return condenser_independent_reconnaissance_response(column,parameters,pair_dacs,entrance_source.mean_state,source_covariance,aperture_z_m=float(aperture_cfg["z_m"]),aperture_radius_m=float(aperture_cfg["radius_m"]),observation_z_m=column.sample_z_m,radial_order=32,azimuthal_order=96)
design_parameter_scales=np.empty(column.n_parameters); design_parameter_scales[:2]=1e-3; design_cursor=2
for lens in column.lenses:
    design_parameter_scales[design_cursor:design_cursor+3]=[5e-4,.08,.08]; design_cursor+=3
    if lens.channel_index is not None: design_parameter_scales[design_cursor]=.08; design_cursor+=1
recon_design_truth=np.asarray(independent_reconnaissance_response(jnp.asarray(design_parameters),jnp.asarray(coarse_recon_pair_dacs))); recon_pair_count=len(critical_scan_values); recon_design_sigma=np.concatenate((np.sqrt(np.maximum(recon_design_truth[:2*recon_pair_count]*incident_electrons,1))/incident_electrons,np.full(2*recon_pair_count,retrieval_log_radius_sigma)))
recon_design_predictor=lambda p: independent_reconnaissance_response(p,jnp.asarray(coarse_recon_pair_dacs)); recon_design_jacobian=np.asarray(jax.jacfwd(recon_design_predictor)(jnp.asarray(design_parameters))); recon_design_whitened=recon_design_jacobian/recon_design_sigma[:,None]*design_parameter_scales[None,:]; recon_design_svd=jacobian_svd(recon_design_whitened,rtol=1e-7)
condenser_indices=np.asarray([index for index,name in enumerate(column.parameter_names) if name.startswith(("CL1.","CL3.","C-mini."))]); condenser_baseline_audit=audit_condenser_identifiability(column,recon_design_predictor,design_parameters,standard_deviations=recon_design_sigma,parameter_scales=design_parameter_scales,parameter_indices=condenser_indices,rank_rtol=1e-7)
condenser_probe_candidates=[]; condenser_probe_candidate_labels=[]
for channel_name in ("CL1","CL3","C-mini"):
    channel_index=column.channel_names.index(channel_name)
    for candidate_value in (-.35,-.175,.175,.35):
        candidate_dac=reference_dacs.copy(); candidate_dac[channel_index]=candidate_value; condenser_probe_candidates.append(candidate_dac); condenser_probe_candidate_labels.append(f"{channel_name}{candidate_value:+.3f}")
condenser_probe_candidates=np.stack(condenser_probe_candidates); condenser_conditional_design=greedy_condenser_probe_design(column,design_parameters,condenser_probe_candidates,entrance_source.mean_state,entrance_source.covariance,baseline_scaled_jacobian=condenser_baseline_audit.scaled_jacobian,parameter_scales=design_parameter_scales,observable_standard_deviations=.002,observable_mode="complex_radial",parameter_indices=condenser_indices,rank_rtol=1e-7,minimum_relative_singular_value=1e-6)
upstream_parameter_indices=np.asarray([index for index,name in enumerate(column.parameter_names) if name=="source_z_m" or name.startswith(("CL1.","CL3.","C-mini.","Obj-prefield."))]); upstream_baseline_audit=audit_condenser_identifiability(column,recon_design_predictor,design_parameters,standard_deviations=recon_design_sigma,parameter_scales=design_parameter_scales,parameter_indices=upstream_parameter_indices,rank_rtol=1e-7)
condenser_multivariate_values=np.asarray(((-.13526724,-.20884910,.24530710),(-.09839488,.12788477,-.34999685),(.05658233,-.27411085,-.17553819),(-.23130277,.28816942,.27868015))); condenser_multivariate_candidates=np.repeat(reference_dacs[None,:],len(condenser_multivariate_values),axis=0); condenser_multivariate_candidates[:,:3]=condenser_multivariate_values
condenser_upstream_design=greedy_condenser_probe_design(column,design_parameters,condenser_multivariate_candidates,entrance_source.mean_state,entrance_source.covariance,baseline_scaled_jacobian=upstream_baseline_audit.scaled_jacobian,parameter_scales=design_parameter_scales,observable_standard_deviations=.002,observable_mode="complex_radial",parameter_indices=upstream_parameter_indices,rank_rtol=1e-7)
condenser_probe_dacs=np.asarray(condenser_upstream_design.selected_dac_values); condenser_probe_truth=np.asarray(sample_plane_probe_observables(column,jnp.asarray(design_parameters),condenser_probe_dacs,entrance_source.mean_state,entrance_source.covariance,mode="complex_radial")); condenser_probe_sigma=np.full(condenser_probe_truth.size,.002); condenser_probe_jacobian=np.asarray(jax.jacfwd(lambda p:sample_plane_probe_observables(column,p,condenser_probe_dacs,entrance_source.mean_state,entrance_source.covariance,mode="complex_radial").reshape(-1))(jnp.asarray(design_parameters))); condenser_probe_whitened=condenser_probe_jacobian/condenser_probe_sigma[:,None]*design_parameter_scales[None,:]; recon_design_fisher=recon_design_whitened.T@recon_design_whitened+condenser_probe_whitened.T@condenser_probe_whitened
basis_position_m=float(entrance_cfg["position_basis_step_m"]); basis_tilt_rad=float(entrance_cfg["tilt_basis_step_rad"])
baseline_plan=compact_30_state_plan(reference_dacs=reference_dacs,perturbations=perturbations,nominal_dacs=nominal_dacs,target_magnifications=magnifications,position_amplitude=basis_position_m,tilt_amplitude=basis_tilt_rad)
baseline_dacs=np.asarray([state.dac_values for state in baseline_plan.states if state.role=="d_optimal"]); downstream_design_sigma=np.array([2e-5]*4+[2e-6]*4)
def design_response(parameters,dac): return response_observable_vector(column.matrix(parameters,dac,segment="sample_to_detector"))
design_rng=np.random.default_rng(824); candidate_patterns=design_rng.choice((-1.,1.),size=(28,7))
candidate_patterns=np.vstack((np.eye(7),-np.eye(7),candidate_patterns)); unfiltered_candidate_dacs=np.vstack((baseline_dacs,reference_dacs+.65*perturbations*candidate_patterns)); candidate_magnifications=np.asarray([effective_magnification_for_dac(dac) for dac in unfiltered_candidate_dacs]); candidate_valid=np.isfinite(candidate_magnifications)&(candidate_magnifications>=magnifications.min())&(candidate_magnifications<=magnifications.max())
if not np.all(candidate_valid[:len(baseline_dacs)]): raise RuntimeError("The deterministic mixed-state baseline left the certified 40k–600k design range.")
candidate_dacs=unfiltered_candidate_dacs[candidate_valid]
candidate_jacobians=np.asarray(jax.vmap(jax.jacfwd(design_response,argnums=0),in_axes=(None,0))(jnp.asarray(design_parameters),jnp.asarray(candidate_dacs)))
candidate_jacobians=candidate_jacobians/downstream_design_sigma[None,:,None]*design_parameter_scales[None,None,:]
design_prior=recon_design_fisher+1e-6*np.eye(column.n_parameters); d_selection=greedy_d_optimal_selection(candidate_jacobians,n_select=8,prior_precision=design_prior)
d_optimal_dacs=candidate_dacs[np.asarray(d_selection.indices)]
baseline_jacobians=candidate_jacobians[:len(baseline_dacs)]
selected_candidate_stack=candidate_jacobians[np.asarray(d_selection.indices)].reshape(-1,column.n_parameters); baseline_candidate_stack=baseline_jacobians.reshape(-1,column.n_parameters); selected_stack=np.vstack((recon_design_whitened,selected_candidate_stack)); baseline_stack=np.vstack((recon_design_whitened,baseline_candidate_stack))
selected_singular=np.linalg.svd(selected_stack,compute_uv=False); baseline_singular=np.linalg.svd(baseline_stack,compute_uv=False)
def weakest_resolved(values):
    resolved=values[values>values[0]*1e-9]; return resolved[-1] if len(resolved) else 0.
selected_logdet=d_optimal_logdet(selected_candidate_stack,prior_precision=design_prior); baseline_logdet=d_optimal_logdet(baseline_candidate_stack,prior_precision=design_prior)
conditional_labels=[condenser_probe_candidate_labels[index] for index in condenser_conditional_design.selected_candidate_indices]; print(f"Condenser design: non-duplicated aperture/radius baseline={condenser_baseline_audit.rank}/12; conditioned local structural rank={condenser_conditional_design.achieved_rank}/12 with {conditional_labels}, weakest relative σ={condenser_conditional_design.achieved_relative_singular_value:.3e}; released source-z/prefield={condenser_upstream_design.achieved_rank}/16 with {len(condenser_probe_dacs)} multivariate states.")
print(f"Non-duplicated measured-reconnaissance + condenser-probe design rank={recon_design_svd.rank}/{column.n_parameters} before probe rows; candidate pool={len(candidate_dacs)} selected={d_selection.indices}; prior-conditioned logdet {baseline_logdet:.2f}→{selected_logdet:.2f}; weakest resolved σ {weakest_resolved(baseline_singular):.3e}→{weakest_resolved(selected_singular):.3e}")
plan=compact_30_state_plan(reference_dacs=reference_dacs,perturbations=perturbations,nominal_dacs=nominal_dacs,target_magnifications=magnifications,d_optimal_dacs=d_optimal_dacs,position_amplitude=basis_position_m,tilt_amplitude=basis_tilt_rad)
nominal_labels=[s.label for s in plan.states if s.role=="nominal"]
view_scale_fraction=.005; view_rotation_rad=np.deg2rad(.25); view_anisotropy_fraction=.0025
def declared_transfer_envelope(central_J):
    rotations=[np.array([[np.cos(a),-np.sin(a)],[np.sin(a),np.cos(a)]]) for a in (view_rotation_rad,-view_rotation_rad)]
    anisotropy=[np.diag([1.+view_anisotropy_fraction,1.-view_anisotropy_fraction]),np.diag([1.-view_anisotropy_fraction,1.+view_anisotropy_fraction])]
    return np.stack((central_J,(1.+view_scale_fraction)*central_J,(1.-view_scale_fraction)*central_J,rotations[0]@central_J,rotations[1]@central_J,anisotropy[0]@central_J,anisotropy[1]@central_J))
print(f"Frozen-view transfer envelope: ±{100*view_scale_fraction:.2f}% isotropic scale, ±{np.rad2deg(view_rotation_rad):.2f}° rotation, and ±{100*view_anisotropy_fraction:.2f}% anisotropy (7 matrices per state).")
view_by_state={}; state_view_records=[]
for state in plan.states:
    state_dac=np.asarray(state.dac_values); central_J=detector_affine@np.asarray(column.sample_position_jacobian(design_parameters,state_dac))
    envelope_J=declared_transfer_envelope(central_J)
    state_view=select_sample_view(atlas,transfer_jacobian_xy=central_J,transfer_jacobian_envelope_xy=envelope_J,detector_shape_yx=detector.shape_yx,detector_pixel_size_yx_m=detector.pixel_size_yx_m,sample_centre_xy_m=sample_origin_xy,sample_rotation_rad=sample_rotation,sample_tilt_xy_rad=sample_tilt_xy,region_name=None,gaussian_radius_min_m=3.4e-9,lens_wobble_margin_m=.15e-9,calibrated_input_margin_m=1.10e-9,gaussian_support_margin_m=.10e-9,pose_uncertainty_margin_m=.10e-9,interpolation_halo_margin_m=.05e-9,boundary=None,z=beam_cfg["sample_z_m"])
    view_by_state[state.label]=state_view; selection=state_view.selection
    state_view_records.append((state.label,selection.effective_magnification,selection.selected_region_name or "global_atlas",selection.feature_family,selection.level_index,selection.local_bounds_xyxy_m,selection.meets_feature_sampling and selection.meets_resolution))
for label,mag,region,family,level,bounds,passed in state_view_records:
    width_nm=(bounds[1]-bounds[0])*1e9; height_nm=(bounds[3]-bounds[2])*1e9
    print(f"  {label:24s} {mag/1000:7.1f}k {region:23s} {family:25s} L{level} union={width_nm:7.2f}×{height_nm:7.2f} nm pass={passed}")
assert all(record[-1] for record in state_view_records), "A per-setting frozen union crop failed its feature/LOD certificate."
family_by_state={label:view.selection.feature_family for label,view in view_by_state.items()}
view_selection_by_state={label:view.selection for label,view in view_by_state.items()}
interaction_route_by_state={label:"sample_plane_reexpansion" for label in view_by_state}
def topology_id_for_view(view,topology_version="reexpand-v1"):
    selection=view.selection; return frozen_topology_fingerprint(specimen_identity=config["sample"]["identity"],atlas_version=atlas.certificate.version,region_name=selection.selected_region_name or "global_atlas",level_index=selection.level_index,crop_bounds_xyxy_m=selection.local_bounds_xyxy_m,sample_translation_xy_m=selection.sample_translation_xy_m,projection_world_from_local=np.asarray(view.projection_world_from_local),route="sample_plane_reexpansion",selected_spacing_m=selection.selected_spacing_m,routed_feature_scale_m=selection.routed_feature_scale_m,centres_per_period=float(experiment_cfg["gaussian_centres_per_shortest_period"]),overlap_factor=float(experiment_cfg["gaussian_overlap_factor"]),interpolation_halo_knots=selection.interpolation_halo_knots,topology_version=topology_version)
frozen_topology_id_by_state={label:topology_id_for_view(view) for label,view in view_by_state.items()}
state_covariance=np.diag([.10e-9**2,.05e-6**2,.10e-9**2,.05e-6**2])
manifest=build_acquisition_manifest(plan,feature_family_by_state=family_by_state,view_selection_by_state=view_selection_by_state,sample_state_covariance=state_covariance,interaction_route_by_state=interaction_route_by_state,frozen_topology_id_by_state=frozen_topology_id_by_state,approach_precondition_raw_dac_step=float(experiment_cfg["approach_precondition_raw_dac_step"]),dac_ramp_steps=int(experiment_cfg["approach_ramp_steps"]),settling_time_s=float(experiment_cfg["settling_time_s"]))
vacuum_measurements={"perforation","position_basis","angle_basis","basis_orthogonal","basis_negative"}; lattice_selection=views[600_000].selection
perforation_projection=np.asarray(views[40_000].projection_world_from_local); perforation_translation=tuple(sample_origin_xy-perforation_projection@np.asarray(atlas.perforation.centre_xy_m))
def with_lattice_selection(frame):
    return replace(frame,feature_family="silicon_lattice_vectors",selected_region_name=lattice_selection.selected_region_name,sample_level_index=lattice_selection.level_index,sample_translation_xy_m=lattice_selection.sample_translation_xy_m,sample_crop_bounds_xyxy_m=lattice_selection.local_bounds_xyxy_m,selected_spacing_m=lattice_selection.selected_spacing_m,interaction_route="sample_plane_reexpansion",frozen_topology_id=topology_id_for_view(views[600_000],"diffraction-reexpand-v1"))
def with_vacuum_selection(frame):
    return replace(frame,feature_family="central_perforation_vacuum",selected_region_name="perforation",sample_level_index=None,sample_translation_xy_m=perforation_translation,sample_crop_bounds_xyxy_m=None,selected_spacing_m=None,interaction_route="vacuum",frozen_topology_id=None)
manifest=tuple(with_vacuum_selection(frame) if frame.measurement in vacuum_measurements else (with_lattice_selection(frame) if frame.measurement=="indexed_diffraction" else frame) for frame in manifest)
slice_envelope_dacs=np.vstack(([frame.raw_dac_values for frame in manifest],[frame.dac_approach_start_values for frame in manifest])); slice_safety_report=assess_gaussian_slice_safety(column,design_parameters,slice_envelope_dacs,slices_per_lens=24); print("Gaussian slice audit:",slice_safety_report.human_summary()); assert slice_safety_report.recommended_slices_per_lens>=1
assert plan.n_states==30 and plan.total_frames==230 and len(manifest)==230
assert all(any(frame.measurement=="indexed_diffraction" for frame in manifest if frame.optical_state_label==state.label) for state in plan.states if state.role=="held_out")
print("states by role:",plan.role_counts()); print("frames by role:",plan.frame_counts()); print("frames by measurement:",dict(Counter(f.measurement for f in manifest)))
for label,target,achieved,dac in zip(nominal_labels,magnifications,nominal_design.achieved_magnifications,nominal_dacs,strict=True):
    print(f"  {label}: target={target:.0f}× achieved={achieved:.3f}× raw DAC={np.array2string(dac,precision=4)}")

manifest_path=GENERATED/"acquisition_manifest.csv"
dac_fields=[f"dac_{name.replace('-', '_')}" for name in plan.channel_names]; approach_fields=[f"approach_start_dac_{name.replace('-', '_')}" for name in plan.channel_names]; covariance_fields=[f"cov_{row}{col}" for row in range(4) for col in range(4)]
fields=["frame_id","optical_state_index","optical_state_label","measurement","interaction_route","frozen_topology_id","target_magnification","reference_mode_magnification","predicted_effective_magnification","dac_approach",*dac_fields,*approach_fields,"dac_ramp_steps","settling_time_s","feature_family","selected_region_name","sample_level_index","stage_x_m","stage_y_m","crop_xmin_m","crop_xmax_m","crop_ymin_m","crop_ymax_m","selected_spacing_m","repeat_index","x_m","theta_x_rad","y_m","theta_y_rad",*covariance_fields]
with manifest_path.open("w",newline="",encoding="utf-8") as stream:
    writer=csv.DictWriter(stream,fieldnames=fields); writer.writeheader()
    for f in manifest:
        s=f.sample_state; crop=f.sample_crop_bounds_xyxy_m or (np.nan,)*4
        row=dict(frame_id=f.frame_id,optical_state_index=f.optical_state_index,optical_state_label=f.optical_state_label,measurement=f.measurement,interaction_route=f.interaction_route,frozen_topology_id=f.frozen_topology_id,target_magnification=f.target_magnification,reference_mode_magnification=f.reference_mode_magnification,predicted_effective_magnification=f.predicted_effective_magnification,dac_approach=f.dac_approach,dac_ramp_steps=f.dac_ramp_steps,settling_time_s=f.settling_time_s,feature_family=f.feature_family,selected_region_name=f.selected_region_name,sample_level_index=f.sample_level_index,stage_x_m=f.sample_translation_xy_m[0],stage_y_m=f.sample_translation_xy_m[1],crop_xmin_m=crop[0],crop_xmax_m=crop[1],crop_ymin_m=crop[2],crop_ymax_m=crop[3],selected_spacing_m=f.selected_spacing_m,repeat_index=f.repeat_index,x_m=s.x,theta_x_rad=s.theta_x,y_m=s.y,theta_y_rad=s.theta_y)
        row.update(dict(zip(dac_fields,f.raw_dac_values,strict=True))); row.update(dict(zip(approach_fields,f.dac_approach_start_values,strict=True))); row.update(dict(zip(covariance_fields,np.asarray(f.calibration_covariance).ravel(),strict=True))); writer.writerow(row)
print("Wrote complete instrument manifest",manifest_path.relative_to(REPO))

# A fast calibrated surrogate supplies one detector-native absolute-count image for every manifest row; these are not Gaussian renders.
manifest_shape=detector.shape_yx  # native 256×256 acquisition remains enabled in smoke mode
surrogate_dose=float(detector_cfg["surrogate_dose_electrons"]); surrogate_read_noise=float(detector_cfg["surrogate_read_noise_e"])
yy_unit,xx_unit=np.meshgrid(np.linspace(-1,1,manifest_shape[0]),np.linspace(-1,1,manifest_shape[1]),indexing="ij")
manifest_y_px,manifest_x_px=np.meshgrid(np.arange(manifest_shape[0]),np.arange(manifest_shape[1]),indexing="ij")
detector_pixel_xy_m=np.asarray(detector.pixel_size_yx_m)[::-1]; detector_pixel_metric=np.diag(1/detector_pixel_xy_m)
manifest_xdet=np.linspace(detector.x_coords_m[0],detector.x_coords_m[-1],manifest_shape[1]); manifest_ydet=np.linspace(detector.y_coords_m[0],detector.y_coords_m[-1],manifest_shape[0])
manifest_xxdet,manifest_yydet=np.meshgrid(manifest_xdet,manifest_ydet,indexing="xy"); manifest_detector_points=np.stack((manifest_xxdet,manifest_yydet),axis=-1)
def state_sample_template(state):
    state_J=detector_affine@np.asarray(column.sample_position_jacobian(design_parameters,np.asarray(state.dac_values)))
    state_world_points=manifest_detector_points@np.linalg.inv(state_J).T+sample_origin_xy
    state_actions=np.asarray(view_by_state[state.label].evaluate_actions(jnp.asarray(state_world_points)))
    transmission=np.exp(state_actions[...,1]+2j*np.pi*state_actions[...,0]/wavelength_m)
    fx_detector=np.fft.fftfreq(manifest_shape[1],d=detector_pixel_xy_m[0]); fy_detector=np.fft.fftfreq(manifest_shape[0],d=detector_pixel_xy_m[1]); qx_detector,qy_detector=np.meshgrid(fx_detector,fy_detector,indexing="xy")
    qx_sample=state_J[0,0]*qx_detector+state_J[1,0]*qy_detector; qy_sample=state_J[0,1]*qx_detector+state_J[1,1]*qy_detector
    fresnel=np.exp(-1j*np.pi*wavelength_m*float(detector_cfg["surrogate_phase_contrast_defocus_m"])*(qx_sample**2+qy_sample**2))
    propagated=np.fft.ifft2(np.fft.fft2(transmission)*fresnel); return ndimage.gaussian_filter(np.abs(propagated)**2,detector.mtf_sigma_px)
sample_template_by_state={state.label:state_sample_template(state) for state in plan.states}  # one noiseless, geometry-aware expectation per optical state
calibrated_spot_radius_m=4e-9
def calibrated_input_template(frame):
    downstream=np.asarray(column.matrix(design_parameters,np.asarray(frame.raw_dac_values),segment="sample_to_detector"))
    propagated=downstream@frame.sample_state.as_array(); centroid_xy_m=detector_affine@propagated[[0,2]]
    centre_xy_px=(np.asarray(manifest_shape)[::-1]-1)/2+detector_pixel_metric@centroid_xy_m
    state_J=detector_affine@np.asarray(column.sample_position_jacobian(design_parameters,np.asarray(frame.raw_dac_values)))
    pixel_J=detector_pixel_metric@state_J; covariance_xy_px2=calibrated_spot_radius_m**2*(pixel_J@pixel_J.T)+detector.mtf_sigma_px**2*np.eye(2)
    offsets=np.stack((manifest_x_px-centre_xy_px[0],manifest_y_px-centre_xy_px[1]),axis=-1); exponent=np.einsum("...i,ij,...j->...",offsets,np.linalg.inv(covariance_xy_px2),offsets)
    spot=np.exp(-.5*exponent); spot*=.18*np.prod(manifest_shape)/spot.sum()  # fixed calibrated vacuum-beam integral
    return .002+spot
perforation_projection=np.asarray(views[40_000].projection_world_from_local); perforation_radius_m=.5*atlas.certificate.perforation_diameter_m
def perforation_template(frame):
    state_J=detector_affine@np.asarray(column.sample_position_jacobian(design_parameters,np.asarray(frame.raw_dac_values)))
    specimen_delta_world=manifest_detector_points@np.linalg.inv(state_J).T; specimen_delta_local=specimen_delta_world@np.linalg.inv(perforation_projection).T
    radius=np.linalg.norm(specimen_delta_local,axis=-1); edge_width=max(2*np.min(np.linalg.svd(np.linalg.inv(state_J),compute_uv=False))*min(detector_pixel_xy_m),.15e-9)
    hole_fraction=np.clip(.5+(perforation_radius_m-radius)/edge_width,0,1); hole_fraction=hole_fraction**2*(3-2*hole_fraction)
    return .04+.96*hole_fraction
diffraction_camera_length_m=float(detector_cfg["diffraction_camera_length_m"]); diffraction_projection=np.asarray(views[600_000].projection_world_from_local)
diffraction_reference_matrix=np.asarray(column.matrix(design_parameters,reference_dacs,segment="sample_to_detector")); diffraction_reference_B=diffraction_reference_matrix[np.ix_([0,2],[1,3])]; diffraction_B_scale=np.sqrt(abs(np.linalg.det(diffraction_reference_B)))
diffraction_spacing_m=float(atlas.silicon_lattice_region.lattice_spacing_m); diffraction_g_local=np.asarray(((1/diffraction_spacing_m,0),(0,1/diffraction_spacing_m)))
def indexed_diffraction_template(frame):
    downstream=np.asarray(column.matrix(design_parameters,np.asarray(frame.raw_dac_values),segment="sample_to_detector")); B=downstream[np.ix_([0,2],[1,3])]
    scattering_world=(diffraction_projection@diffraction_g_local.T).T*wavelength_m; positions_xy_m=(diffraction_camera_length_m/diffraction_B_scale*(detector_affine@B)@scattering_world.T).T; positions_xy_px=(detector_pixel_metric@positions_xy_m.T).T
    expected=np.full(manifest_shape,.002); centre=(np.asarray(manifest_shape)[::-1]-1)/2
    for position in np.vstack((np.zeros((1,2)),positions_xy_px,-positions_xy_px)):
        relative_weight=1.4 if np.allclose(position,0) else .75; expected+=relative_weight*np.exp(-((manifest_x_px-centre[0]-position[0])**2+(manifest_y_px-centre[1]-position[1])**2)/(2*1.8**2))
    expected*=.18*np.prod(manifest_shape)/expected.sum(); return expected
manifest_counts=[]; manifest_dacs=[]; manifest_targets=[]; manifest_measurements=[]; manifest_frame_ids=[]
for frame in manifest:
    target=int(frame.target_magnification) if frame.target_magnification is not None else 150_000
    if frame.measurement in {"sample","sample_repeat"}: expected=sample_template_by_state[frame.optical_state_label].copy()
    elif frame.measurement=="perforation": expected=perforation_template(frame)
    elif frame.measurement=="indexed_diffraction": expected=indexed_diffraction_template(frame)
    else: expected=calibrated_input_template(frame)
    control=np.asarray(frame.raw_dac_values)-reference_dacs; reverse_flux=float(experiment_cfg["reverse_approach_flux_factor"]); expected*=np.exp(-.002*np.dot(control,control))*(reverse_flux if frame.dac_approach=="reverse" else 1.)
    counts=simulate_detector_counts(expected,dose_electrons=surrogate_dose,incident_intensity_sum=float(np.prod(manifest_shape)),read_noise_e=surrogate_read_noise,seed=10_000+frame.frame_id)
    manifest_counts.append(counts.astype(np.float32)); manifest_dacs.append(frame.raw_dac_values); manifest_targets.append(np.nan if frame.target_magnification is None else frame.target_magnification); manifest_measurements.append(frame.measurement); manifest_frame_ids.append(frame.frame_id)
manifest_counts=np.stack(manifest_counts); manifest_frame_ids=np.asarray(manifest_frame_ids,dtype=np.int32)
synthetic_frames_path=GENERATED/"synthetic_230_frame_counts.npz"
np.savez_compressed(synthetic_frames_path,counts=manifest_counts,frame_id=manifest_frame_ids,raw_dac_values=np.asarray(manifest_dacs),target_magnification=np.asarray(manifest_targets),measurement=np.asarray(manifest_measurements),detector_shape_yx=np.asarray(detector.shape_yx),pixel_size_yx_m=np.asarray(detector.pixel_size_yx_m),dose_electrons=surrogate_dose,read_noise_e=surrogate_read_noise,phase_contrast_defocus_m=float(detector_cfg["surrogate_phase_contrast_defocus_m"]),diffraction_camera_length_m=diffraction_camera_length_m)
assert manifest_counts.shape==(230,*manifest_shape) and np.array_equal(manifest_frame_ids,np.arange(230))
print("Wrote detector-native calibrated-surrogate absolute counts",synthetic_frames_path.relative_to(REPO),manifest_counts.shape,f"dose={surrogate_dose:.3g} e-/frame")
image_blocks=[]
for measurement in ("perforation","position_basis","sample","indexed_diffraction"):
    index=next(i for i,frame in enumerate(manifest) if frame.measurement==measurement); obs=extract_image_observables(np.clip(manifest_counts[index],0,None),n_fourier_vectors=3)
    print(f"  frame {manifest[index].frame_id:03d} {measurement:19s} I={obs.total_intensity:8.1f} centroid={obs.centroid_xy_px} covariance={np.array2string(obs.covariance_xy_px2,precision=2)} Fourier(y,x)={obs.fourier_vectors_yx_px}")
    values=[np.log(obs.total_intensity),*obs.centroid_xy_px,*obs.principal_sigma_px]; block_names=["log_intensity","centroid_x","centroid_y","sigma_major","sigma_minor"]
    if measurement in {"sample","indexed_diffraction"}:
        values.extend(obs.fourier_vectors_yx_px.ravel()); block_names.extend(f"fourier_{i}_{axis}" for i in range(len(obs.fourier_vectors_yx_px)) for axis in ("y","x"))
    std=np.array([.03,.2,.2,.2,.2]+[.5]*(len(values)-5)); correlation=.95*np.eye(len(values))+.05*np.ones((len(values),len(values)))
    image_blocks.append(FrameObservationBlock(manifest[index].frame_id,manifest[index].optical_state_index,np.asarray(values),np.outer(std,std)*correlation,tuple(block_names)))
heterogeneous_image_dataset=HeterogeneousInversionDataset(plan,tuple(image_blocks),metadata={"source":"representative absolute-count frames","production_bridge":"fit_heterogeneous_dataset"})
print("Covariance-whitened heterogeneous bridge:",heterogeneous_image_dataset.n_observations,"observables across",len(image_blocks),"variable-length frame blocks")

# Mock edge/Fourier picks from known intervals demonstrate the affine estimator used on real image picks.
known_intervals_nm=np.array([[23.,0.],[0.,46.],[13.,10.],[-23.,23.]])
measured_intervals_px=known_intervals_nm@((known_J(150_000)*1e-9/np.array(detector.pixel_size_yx_m)[::-1,None]).T)
affine=fit_fourier_affine(known_intervals_nm,measured_intervals_px)
print(f"Mock image-pick affine: scale={affine.isotropic_scale:.4f} px/nm, rotation={np.rad2deg(affine.rotation_rad):.3f}°, RMS={affine.rms_residual:.2e} px")

### Synthetic upstream condenser reconnaissance (identifiability proxy)

**Scope warning.** This subsection deliberately uses the inversion configuration's synthetic $z=0.225$ m, 0.5 micrometre aperture as an identifiability proxy. It is not the NEOARM C aperture. The next subsection reads the draft instrument file read-only, uses the actual $z=0.250$ m C-aperture radii and $z=0.290/0.295$ m double-deflector planes, and keeps hardware acceptance fail-closed.

Calibrated position and tilt **at the specimen** are extremely useful downstream, but they erase the source-to-specimen map: their Jacobian with respect to `CL1`, `CL3`, and `C-mini` is exactly zero. The compact 230-frame column experiment therefore needs a short upstream pre-calibration block. We use normalized raw DAC directly and replace current readback by two observables acquired from each paired detector exposure: absolute electrons transmitted through a known condenser aperture, and the sample-plane probe radius recovered from the registered specimen. A nine-position `CL1` scan with `CL3` at $-\delta$ and $+\delta$ costs exactly 18 images. Three measured bisection pairs add six images and refine the critical-line root. Both absolute radii in each pair are retained in the likelihood; they cost no frames and add one identifiable condenser direction. The notebook writes every endpoint and measured quantity to a separate executable manifest.

The synthetic entrance covariance below is constructed once by back-propagating a declared broad, nearly parallel reference illumination. That construction supplies matched synthetic truth, but the inverse problem does **not** freeze it: all ten independent covariance modes are released through a positive-definite exponential chart and shared by every upstream frame. The probe-radius observable is evaluated at the specimen because it comes from known-object probe retrieval; it is not inferred from an overfilled raw detector spot. The 9/10 versus 10/10 covariance-tomography calculation is retained only as a diagnostic of what screen projections could observe; no hypothetical 13-frame tomography stack is repeated at the 30 column settings or included in the fit.

The coarse scan alone is not enough to fit all condenser parameters. A rank-revealing extension uses the **same physical specimen** at safe signed condenser settings. Each selected state supplies (i) a registered perforation/position image for probe radius, (ii) an indexed diffraction or central-disk image for convergence-angle radius, and (iii) one defocus-diverse image pair or equivalent complex-probe retrieval for the position--angle correlation. Retaining the absolute coarse-scan radii gives conditional rank 8/12; the probe extension has local structural rank 12/12 even after the ten entrance-covariance modes are profiled at the declared $10^{-7}$ threshold. This is an identifiability result, not yet a completed nonlinear recovery. Aperture transmission and fitted radius are algebraically non-duplicated, but because they may come from the same images their joint covariance must be estimated by image-level bootstrap for instrument data; the synthetic demonstration currently declares a diagonal noise model. The wider source-to-specimen block is 15/16 after profiling: its remaining family is dominated by objective-prefield strength versus effective length, not by a condenser-lens ambiguity. The optional full-closure protocol therefore replaces the 12 basic probe-extension views with 40 calibrated covariance-tomography views: five independent screen projections at the specimen and at the existing $z=0.225$ m aperture plane, for each of four condenser states. Position-only information at the second plane remains 15/16; a reconstructed full phase-space covariance at both planes gives local structural rank 16/16 after profiling all ten entrance-covariance modes. This is an experimentally conditional design: the five screen maps must first be calibrated, their reconstruction/projection covariance must enter the likelihood, and the aperture plane must be demonstrably imageable. The basic experiment is 230+24+12=266 frames; requesting full upstream closure makes it 230+24+40=294. These are effective raw-DAC-to-integrated-solenoid parameters; without current readback the electrical and magnetic factors cannot be separated.

A second, fail-closed **precision** gate is therefore evaluated after rank. At the declared synthetic noise, the basic route predicts roughly 3.5% uncertainty in the CL1 strength law, 16% in CL3, and 80% in C-mini, while C-mini effective length is the weakest family. Full two-plane covariance closes the exact rank but improves these uncertainties by only about 1–1.5×; it still does not meet 1%. A multistart optimizer can locate basins but cannot manufacture missing Fisher information. The current experiment can fit useful effective combinations and provides an exact-data inversion check, but individual 1% condenser geometry/strength recovery requires either independent bounds on lens centres/thicknesses or stronger, lens-specific upstream caustic/aperture scans. The notebook therefore reports this precision gate separately: it blocks a claim about all twelve named condenser coordinates, while a full-column fit may still be accepted for the stable, explicitly profiled identifiable quotient.

In [ ]:
names=column.parameter_names; smoke_names=("CL1.q_bias","IL2.q_bias","PL1.q_bias"); smoke_idx=np.array([names.index(name) for name in smoke_names]); p_truth=design_parameters.copy(); p_truth[smoke_idx]+=np.array([.018,-.014,.021])
coarse_log_radius_truth,coarse_transmission_truth=map(np.asarray,reconnaissance_frame_observables(jnp.asarray(p_truth),jnp.asarray(coarse_recon_pair_dacs))); coarse_critical_truth=(coarse_log_radius_truth[:,1]-coarse_log_radius_truth[:,0])/(2*critical_wobble)
recon_rng=np.random.default_rng(90210); coarse_log_radius_measured=coarse_log_radius_truth+recon_rng.normal(0,retrieval_log_radius_sigma,coarse_log_radius_truth.shape); coarse_aperture_counts=recon_rng.poisson(incident_electrons*coarse_transmission_truth); coarse_critical_measured=(coarse_log_radius_measured[:,1]-coarse_log_radius_measured[:,0])/(2*critical_wobble); coarse_roots=linear_zero_crossings(critical_scan_values,coarse_critical_measured)
truth_brackets=np.flatnonzero(coarse_critical_truth[:-1]*coarse_critical_truth[1:]<0); measured_brackets=np.flatnonzero(coarse_critical_measured[:-1]*coarse_critical_measured[1:]<0)
if len(truth_brackets)!=1 or len(measured_brackets)!=1: raise RuntimeError("The configured coarse scan must isolate exactly one critical-line branch.")
truth_lo,truth_hi=critical_scan_values[truth_brackets[0]],critical_scan_values[truth_brackets[0]+1]; truth_ylo=coarse_critical_truth[truth_brackets[0]]
for _ in range(48):
    truth_mid=.5*(truth_lo+truth_hi); truth_mid_y=float(np.asarray(reconnaissance_response(jnp.asarray(p_truth),jnp.asarray(make_recon_pair_dacs([truth_mid]))))[0])
    if truth_ylo*truth_mid_y<=0: truth_hi=truth_mid
    else: truth_lo,truth_ylo=truth_mid,truth_mid_y
exact_critical_root=.5*(truth_lo+truth_hi); coarse_root=float(coarse_roots[np.argmin(np.abs(coarse_roots-exact_critical_root))])
bracket_index=int(measured_brackets[0]); refine_lo,refine_hi=critical_scan_values[bracket_index],critical_scan_values[bracket_index+1]; refine_ylo,refine_yhi=coarse_critical_measured[bracket_index],coarse_critical_measured[bracket_index+1]; adaptive_scan_values=[]; adaptive_log_radius_truth=[]; adaptive_log_radius_measured=[]; adaptive_transmission_truth=[]; adaptive_aperture_counts=[]; adaptive_critical_measured=[]
for _ in range(int(recon_cfg["maximum_root_refinement_pairs"])):
    midpoint=.5*(refine_lo+refine_hi); pair=make_recon_pair_dacs([midpoint]); log_truth,transmission_truth=reconnaissance_frame_observables(jnp.asarray(p_truth),jnp.asarray(pair)); log_truth=np.asarray(log_truth)[0]; transmission_truth=np.asarray(transmission_truth)[0]; log_measured=log_truth+recon_rng.normal(0,retrieval_log_radius_sigma,2); counts=recon_rng.poisson(incident_electrons*transmission_truth); response=(log_measured[1]-log_measured[0])/(2*critical_wobble)
    adaptive_scan_values.append(midpoint); adaptive_log_radius_truth.append(log_truth); adaptive_log_radius_measured.append(log_measured); adaptive_transmission_truth.append(transmission_truth); adaptive_aperture_counts.append(counts); adaptive_critical_measured.append(response)
    if refine_ylo*response<=0: refine_hi,refine_yhi=midpoint,response
    else: refine_lo,refine_ylo=midpoint,response
refined_root=float(refine_lo-refine_ylo*(refine_hi-refine_lo)/(refine_yhi-refine_ylo)); adaptive_scan_values=np.asarray(adaptive_scan_values); adaptive_log_radius_truth=np.asarray(adaptive_log_radius_truth); adaptive_log_radius_measured=np.asarray(adaptive_log_radius_measured); adaptive_transmission_truth=np.asarray(adaptive_transmission_truth); adaptive_aperture_counts=np.asarray(adaptive_aperture_counts); adaptive_critical_measured=np.asarray(adaptive_critical_measured)
all_recon_scan_values=np.concatenate((critical_scan_values,adaptive_scan_values)); all_recon_pair_dacs=np.concatenate((coarse_recon_pair_dacs,make_recon_pair_dacs(adaptive_scan_values))); all_recon_log_radius_measured=np.concatenate((coarse_log_radius_measured,adaptive_log_radius_measured)); all_recon_transmission_truth=np.concatenate((coarse_transmission_truth,adaptive_transmission_truth)); all_recon_aperture_counts=np.concatenate((coarse_aperture_counts,adaptive_aperture_counts)); all_recon_critical_measured=np.concatenate((coarse_critical_measured,adaptive_critical_measured))
# The critical-line derivative is used to locate/adapt the scan, but it is an exact linear combination of the same two retrieved log radii. Keep only aperture counts and the two absolute radii in the likelihood to avoid double-counting those frames.
recon_observed=np.concatenate((all_recon_aperture_counts.reshape(-1)/incident_electrons,all_recon_log_radius_measured.reshape(-1))); recon_sigma=np.concatenate((np.sqrt(np.maximum(all_recon_aperture_counts.reshape(-1),1))/incident_electrons,np.full(2*len(all_recon_scan_values),retrieval_log_radius_sigma)))
recon_predictor=lambda p: independent_reconnaissance_response(p,jnp.asarray(all_recon_pair_dacs)); recon_jacobian=np.asarray(jax.jacfwd(recon_predictor)(jnp.asarray(p_truth))); recon_whitened=recon_jacobian/recon_sigma[:,None]*design_parameter_scales[None,:]; recon_svd=jacobian_svd(recon_whitened,rtol=1e-7)
probe_rng=np.random.default_rng(90211); condenser_probe_truth_physical=np.asarray(sample_plane_probe_observables(column,jnp.asarray(p_truth),condenser_probe_dacs,entrance_source.mean_state,entrance_source.covariance,mode="complex_radial")); condenser_probe_observed=condenser_probe_truth_physical+probe_rng.normal(0,.002,condenser_probe_truth_physical.shape); condenser_probe_sigma=np.full(condenser_probe_truth_physical.size,.002)
def complete_condenser_response(parameters): return jnp.concatenate((independent_reconnaissance_response(parameters,jnp.asarray(all_recon_pair_dacs)),sample_plane_probe_observables(column,parameters,jnp.asarray(condenser_probe_dacs),entrance_source.mean_state,entrance_source.covariance,mode="complex_radial").reshape(-1)))
def complete_condenser_response_with_covariance(parameters,covariance_coordinates):
    fitted_covariance=entrance_covariance_chart.covariance(covariance_coordinates)
    return jnp.concatenate((independent_reconnaissance_response_with_covariance(parameters,fitted_covariance,jnp.asarray(all_recon_pair_dacs)),sample_plane_probe_observables_with_entrance_covariance(column,parameters,covariance_coordinates,jnp.asarray(condenser_probe_dacs),entrance_source.mean_state,entrance_covariance_chart,mode="complex_radial").reshape(-1)))
complete_condenser_observed=np.concatenate((recon_observed,condenser_probe_observed.reshape(-1))); complete_condenser_sigma=np.concatenate((recon_sigma,condenser_probe_sigma)); actual_condenser_baseline_audit=audit_condenser_identifiability(column,recon_predictor,p_truth,standard_deviations=recon_sigma,parameter_scales=design_parameter_scales,parameter_indices=condenser_indices,rank_rtol=1e-7); actual_condenser_extended_audit=audit_condenser_identifiability(column,complete_condenser_response,p_truth,standard_deviations=complete_condenser_sigma,parameter_scales=design_parameter_scales,parameter_indices=condenser_indices,rank_rtol=1e-7); actual_upstream_extended_audit=audit_condenser_identifiability(column,complete_condenser_response,p_truth,standard_deviations=complete_condenser_sigma,parameter_scales=design_parameter_scales,parameter_indices=upstream_parameter_indices,rank_rtol=1e-7)
released_covariance_baseline=lambda parameters,covariance: independent_reconnaissance_response_with_covariance(parameters,covariance,jnp.asarray(all_recon_pair_dacs))
condenser_covariance_audit=audit_entrance_covariance_nuisance_identifiability(column,p_truth,condenser_probe_dacs,entrance_source.mean_state,entrance_covariance_chart,probe_standard_deviations=.002,parameter_scales=design_parameter_scales,parameter_indices=condenser_indices,baseline_prediction_function=released_covariance_baseline,baseline_standard_deviations=recon_sigma,rank_rtol=1e-7)
upstream_covariance_audit=audit_entrance_covariance_nuisance_identifiability(column,p_truth,condenser_probe_dacs,entrance_source.mean_state,entrance_covariance_chart,probe_standard_deviations=.002,parameter_scales=design_parameter_scales,parameter_indices=upstream_parameter_indices,baseline_prediction_function=released_covariance_baseline,baseline_standard_deviations=recon_sigma,rank_rtol=1e-7)
ordinary_illumination_projections=standard_illumination_projections(defocus_distance_m=3e-3,diffraction_length_m=.10); complete_illumination_projections=standard_illumination_projections(defocus_distance_m=3e-3,diffraction_length_m=.10,coupled_length_m=5e-3)
ordinary_covariance_rank=np.linalg.matrix_rank(illumination_covariance_design_matrix(ordinary_illumination_projections)); complete_covariance_rank=np.linalg.matrix_rank(illumination_covariance_design_matrix(complete_illumination_projections))
two_plane_baseline=lambda parameters,covariance: independent_reconnaissance_response_with_covariance(parameters,covariance,jnp.asarray(all_recon_pair_dacs))
sample_full_covariance_block=AxialPlaneCovarianceBlock("sample_phase_space_covariance",column.sample_z_m,condenser_probe_dacs,.002,mode="full"); aperture_position_covariance_block=AxialPlaneCovarianceBlock("aperture_position_covariance",float(aperture_cfg["z_m"]),condenser_probe_dacs,.002,mode="position"); aperture_full_covariance_block=AxialPlaneCovarianceBlock("aperture_phase_space_covariance",float(aperture_cfg["z_m"]),condenser_probe_dacs,.002,mode="full")
upstream_two_plane_position_audit=audit_axial_plane_covariance_nuisance_identifiability(column,p_truth,entrance_covariance_chart,(sample_full_covariance_block,aperture_position_covariance_block),parameter_scales=design_parameter_scales,parameter_indices=upstream_parameter_indices,baseline_prediction_function=two_plane_baseline,baseline_standard_deviations=recon_sigma,baseline_block_name="independent_reconnaissance",rank_rtol=1e-7)
upstream_two_plane_full_audit=audit_axial_plane_covariance_nuisance_identifiability(column,p_truth,entrance_covariance_chart,(sample_full_covariance_block,aperture_full_covariance_block),parameter_scales=design_parameter_scales,parameter_indices=upstream_parameter_indices,baseline_prediction_function=two_plane_baseline,baseline_standard_deviations=recon_sigma,baseline_block_name="independent_reconnaissance",rank_rtol=1e-7); condenser_two_plane_full_audit=audit_axial_plane_covariance_nuisance_identifiability(column,p_truth,entrance_covariance_chart,(sample_full_covariance_block,aperture_full_covariance_block),parameter_scales=design_parameter_scales,parameter_indices=condenser_indices,baseline_prediction_function=two_plane_baseline,baseline_standard_deviations=recon_sigma,baseline_block_name="independent_reconnaissance",rank_rtol=1e-7)
two_plane_profiled_singular=upstream_two_plane_full_audit.diagnostics.combined.column_equilibrated_profiled_optical.singular_values; two_plane_weakest_relative=float(two_plane_profiled_singular[-1]/two_plane_profiled_singular[0])
def profiled_parameter_precision(audit):
    precision=np.asarray(audit.diagnostics.combined.profiled_optical_precision,float); eigenvalues,eigenvectors=np.linalg.eigh(precision); tolerance=np.finfo(float).eps*max(precision.shape)*max(float(eigenvalues[-1]),1.); full_rank=bool(eigenvalues[0]>tolerance)
    if not full_rank: return np.full(len(audit.parameter_indices),np.inf),np.full(len(audit.parameter_indices),np.nan),eigenvalues,eigenvectors[:,0]
    covariance_scaled=(eigenvectors*(1/eigenvalues)[None,:])@eigenvectors.T; sigma_scaled=np.sqrt(np.maximum(np.diag(covariance_scaled),0)); sigma_physical=sigma_scaled*design_parameter_scales[audit.parameter_indices]; truth=np.asarray(p_truth)[audit.parameter_indices]; precision_metric=np.asarray([sigma if name.endswith(".log_length_m") else sigma/max(abs(value),np.finfo(float).tiny) for name,sigma,value in zip(audit.parameter_names,sigma_physical,truth,strict=True)]); return sigma_physical,precision_metric,eigenvalues,eigenvectors[:,0]
condenser_basic_sigma,condenser_basic_precision_metric,condenser_basic_precision_eigenvalues,condenser_basic_weak_vector=profiled_parameter_precision(condenser_covariance_audit); condenser_two_plane_sigma,condenser_two_plane_precision_metric,condenser_two_plane_precision_eigenvalues,condenser_two_plane_weak_vector=profiled_parameter_precision(condenser_two_plane_full_audit); condenser_basic_one_percent_ready=bool(np.all(condenser_basic_precision_metric<.01)); condenser_two_plane_one_percent_ready=bool(np.all(condenser_two_plane_precision_metric<.01))
condenser_indices=np.asarray([index for index,name in enumerate(column.parameter_names) if name.startswith(("CL1.","CL3.","C-mini."))]); sample_input_jacobian=np.asarray(jax.jacfwd(lambda p: response_observable_vector(column.matrix(p,jnp.asarray(reference_dacs),segment="sample_to_detector")))(jnp.asarray(design_parameters)))[:,condenser_indices]
recon_manifest_path=GENERATED/"condenser_reconnaissance_manifest.csv"; recon_dac_fields=[f"dac_{name.replace('-', '_')}" for name in column.channel_names]; recon_approach_dac_fields=[f"approach_start_dac_{name.replace('-', '_')}" for name in column.channel_names]; upstream_approach_step=float(experiment_cfg["approach_precondition_raw_dac_step"]); upstream_ramp_steps=int(experiment_cfg["approach_ramp_steps"]); upstream_settling_time_s=float(experiment_cfg["settling_time_s"]); upstream_approach_delta=np.full(len(column.channel_names),upstream_approach_step)
with recon_manifest_path.open("w",newline="",encoding="utf-8") as stream:
    fieldnames=["frame_id","phase","pair_index","scan_channel","scan_value","wobble_channel","wobble_sign","sample_probe_retrieval","absolute_aperture_counts","incident_electrons","observed_log_probe_radius","observed_aperture_electrons","dac_approach","dac_ramp_steps","settling_time_s",*recon_dac_fields,*recon_approach_dac_fields]; writer=csv.DictWriter(stream,fieldnames=fieldnames); writer.writeheader()
    for pair_index,(scan_value,pair_dacs,log_pair,count_pair) in enumerate(zip(all_recon_scan_values,all_recon_pair_dacs,all_recon_log_radius_measured,all_recon_aperture_counts,strict=True)):
        phase="coarse" if pair_index<len(critical_scan_values) else "adaptive_bisection"
        for sign_index,wobble_sign in enumerate((-1,1)):
            endpoint=np.asarray(pair_dacs[sign_index],dtype=float); approach_start=endpoint-upstream_approach_delta; row=dict(frame_id=2*pair_index+sign_index,phase=phase,pair_index=pair_index,scan_channel=recon_cfg["scan_channel"],scan_value=scan_value,wobble_channel=recon_cfg["wobble_channel"],wobble_sign=wobble_sign,sample_probe_retrieval=True,absolute_aperture_counts=True,incident_electrons=incident_electrons,observed_log_probe_radius=log_pair[sign_index],observed_aperture_electrons=int(count_pair[sign_index]),dac_approach="forward_from_lower_raw_dac",dac_ramp_steps=upstream_ramp_steps,settling_time_s=upstream_settling_time_s); row.update(dict(zip(recon_dac_fields,endpoint,strict=True))); row.update(dict(zip(recon_approach_dac_fields,approach_start,strict=True))); writer.writerow(row)
condenser_probe_manifest_path=GENERATED/"condenser_probe_extension_manifest.csv"; condenser_probe_measurements=(("perforation_position",0,"log_sample_rms_radius_m"),("indexed_diffraction_central_disk",1,"log_sample_rms_angle_rad"),("defocus_diversity",2,"sample_position_angle_correlation"))
with condenser_probe_manifest_path.open("w",newline="",encoding="utf-8") as stream:
    fieldnames=["frame_id","probe_state_index","measurement","observable_name","observed_value","standard_deviation","same_physical_sample","safe_range_requires_instrument_certificate","dac_approach","dac_ramp_steps","settling_time_s",*recon_dac_fields,*recon_approach_dac_fields]; writer=csv.DictWriter(stream,fieldnames=fieldnames); writer.writeheader(); probe_frame_id=2*len(all_recon_scan_values)
    for probe_state_index,(probe_dac,probe_values) in enumerate(zip(condenser_probe_dacs,condenser_probe_observed,strict=True)):
        for measurement,component_index,observable_name in condenser_probe_measurements:
            endpoint=np.asarray(probe_dac,dtype=float); approach_start=endpoint-upstream_approach_delta; row=dict(frame_id=probe_frame_id,probe_state_index=probe_state_index,measurement=measurement,observable_name=observable_name,observed_value=probe_values[component_index],standard_deviation=.002,same_physical_sample=True,safe_range_requires_instrument_certificate=True,dac_approach="forward_from_lower_raw_dac",dac_ramp_steps=upstream_ramp_steps,settling_time_s=upstream_settling_time_s); row.update(dict(zip(recon_dac_fields,endpoint,strict=True))); row.update(dict(zip(recon_approach_dac_fields,approach_start,strict=True))); writer.writerow(row); probe_frame_id+=1
two_plane_tomography_manifest_path=GENERATED/"condenser_two_plane_tomography_manifest.csv"; tomography_planes=(("sample",float(column.sample_z_m)),("condenser_aperture",float(aperture_cfg["z_m"])))
with two_plane_tomography_manifest_path.open("w",newline="",encoding="utf-8") as stream:
    fieldnames=["frame_id","probe_state_index","reference_plane","plane_z_m","projection_label","coordinate_unit","position_map_json","predicted_centroid_u","predicted_centroid_v","predicted_covariance_uu","predicted_covariance_uv","predicted_covariance_vv","fit_summary_standard_deviation","measurement_contract","same_registered_specimen","replaces_basic_probe_extension","requires_calibrated_screen_map","safe_range_requires_instrument_certificate","dac_approach","dac_ramp_steps","settling_time_s",*recon_dac_fields,*recon_approach_dac_fields]; writer=csv.DictWriter(stream,fieldnames=fieldnames); writer.writeheader(); tomography_frame_id=0
    for probe_state_index,probe_dac in enumerate(condenser_probe_dacs):
        endpoint=np.asarray(probe_dac,dtype=float); approach_start=endpoint-upstream_approach_delta
        for reference_plane,plane_z_m in tomography_planes:
            plane_matrix=np.asarray(hard_edge_matrix_to_plane(column,jnp.asarray(p_truth),jnp.asarray(endpoint),plane_z_m)); plane_mean=plane_matrix@np.asarray(entrance_source.mean_state); plane_covariance=plane_matrix@np.asarray(entrance_source.covariance)@plane_matrix.T
            for projection in complete_illumination_projections:
                position_map=np.asarray(projection.position_map); screen_mean=position_map@plane_mean; screen_covariance=position_map@plane_covariance@position_map.T; row=dict(frame_id=tomography_frame_id,probe_state_index=probe_state_index,reference_plane=reference_plane,plane_z_m=plane_z_m,projection_label=projection.label,coordinate_unit=projection.coordinate_unit,position_map_json=json.dumps(position_map.tolist(),separators=(",",":")),predicted_centroid_u=screen_mean[0],predicted_centroid_v=screen_mean[1],predicted_covariance_uu=screen_covariance[0,0],predicted_covariance_uv=screen_covariance[0,1],predicted_covariance_vv=screen_covariance[1,1],fit_summary_standard_deviation=.002,measurement_contract="registered 2D intensity image; jointly reconstruct 10 log-RMS/correlation summaries per plane and state",same_registered_specimen=True,replaces_basic_probe_extension=True,requires_calibrated_screen_map=True,safe_range_requires_instrument_certificate=True,dac_approach="forward_from_lower_raw_dac",dac_ramp_steps=upstream_ramp_steps,settling_time_s=upstream_settling_time_s); row.update(dict(zip(recon_dac_fields,endpoint,strict=True))); row.update(dict(zip(recon_approach_dac_fields,approach_start,strict=True))); writer.writerow(row); tomography_frame_id+=1
condenser_precision_path=GENERATED/"condenser_profiled_precision_audit.csv"
with condenser_precision_path.open("w",newline="",encoding="utf-8") as stream:
    fieldnames=["parameter_name","truth_native_parameter","declared_parameter_scale","basic_profiled_sigma_native","basic_fractional_or_log_length_sigma","basic_meets_one_percent","two_plane_profiled_sigma_native","two_plane_fractional_or_log_length_sigma","two_plane_meets_one_percent"]; writer=csv.DictWriter(stream,fieldnames=fieldnames); writer.writeheader()
    for local_index,(parameter_index,parameter_name) in enumerate(zip(condenser_indices,condenser_covariance_audit.parameter_names,strict=True)): writer.writerow(dict(parameter_name=parameter_name,truth_native_parameter=p_truth[parameter_index],declared_parameter_scale=design_parameter_scales[parameter_index],basic_profiled_sigma_native=condenser_basic_sigma[local_index],basic_fractional_or_log_length_sigma=condenser_basic_precision_metric[local_index],basic_meets_one_percent=condenser_basic_precision_metric[local_index]<.01,two_plane_profiled_sigma_native=condenser_two_plane_sigma[local_index],two_plane_fractional_or_log_length_sigma=condenser_two_plane_precision_metric[local_index],two_plane_meets_one_percent=condenser_two_plane_precision_metric[local_index]<.01))
print(f"Condenser reconnaissance: {2*len(critical_scan_values)} coarse frames + {2*len(adaptive_scan_values)} adaptive frames; coarse root={coarse_root:+.6f}, refined={refined_root:+.6f}, dense truth={exact_critical_root:+.6f} raw DAC")
print(f"Absolute aperture transmission={all_recon_transmission_truth.min():.3f}–{all_recon_transmission_truth.max():.3f}; non-duplicated aperture/radius global rank={recon_svd.rank}/{column.n_parameters}; fixed-covariance condenser local rank {actual_condenser_baseline_audit.rank}/12→{actual_condenser_extended_audit.rank}/12; expanded upstream local rank={actual_upstream_extended_audit.rank}/16.")
print(f"Released entrance covariance (local structural audit): condenser conditional/profiled={condenser_covariance_audit.conditional_rank}/{condenser_covariance_audit.nuisance_profiled_rank} of 12; full upstream conditional/profiled={upstream_covariance_audit.conditional_rank}/{upstream_covariance_audit.nuisance_profiled_rank} of 16; covariance nuisance raw/column-equilibrated rank={condenser_covariance_audit.diagnostics.combined.nuisance.rank}/{condenser_covariance_audit.nuisance_rank} of 10 (only the raw rank is a practical visibility claim).")
print(f"Optional two-plane local closure audit: position-only second plane={upstream_two_plane_position_audit.nuisance_profiled_rank}/16; full sample+aperture phase space={upstream_two_plane_full_audit.nuisance_profiled_rank}/16 after profiling {upstream_two_plane_full_audit.nuisance_rank}/10 covariance modes; weakest relative singular value={two_plane_weakest_relative:.3e}.")
basic_precision_by_name=dict(zip(condenser_covariance_audit.parameter_names,condenser_basic_precision_metric,strict=True)); two_plane_precision_by_name=dict(zip(condenser_two_plane_full_audit.parameter_names,condenser_two_plane_precision_metric,strict=True))
for lens_name in ("CL1","CL3","C-mini"): print(f"  {lens_name:6s} basic 1σ: z={condenser_basic_sigma[condenser_covariance_audit.parameter_names.index(lens_name+'.z_center_m')]*1e6:.0f} µm, length≈{100*basic_precision_by_name[lens_name+'.log_length_m']:.1f}%, q_bias={100*basic_precision_by_name[lens_name+'.q_bias']:.1f}%, q_slope={100*basic_precision_by_name[lens_name+'.q_slope_per_dac']:.1f}%; two-plane length≈{100*two_plane_precision_by_name[lens_name+'.log_length_m']:.1f}%, q_bias={100*two_plane_precision_by_name[lens_name+'.q_bias']:.1f}%, q_slope={100*two_plane_precision_by_name[lens_name+'.q_slope_per_dac']:.1f}%")
print(f"One-percent individual-condenser precision gate: basic={condenser_basic_one_percent_ready}; optional two-plane={condenser_two_plane_one_percent_ready}. A 12/12 rank is not promoted to recovery while this gate is false.")
print(f"Manifests: {recon_manifest_path.relative_to(REPO)}, {condenser_probe_manifest_path.relative_to(REPO)}, optional {two_plane_tomography_manifest_path.relative_to(REPO)}, and precision audit {condenser_precision_path.relative_to(REPO)}; basic extension={3*len(condenser_probe_dacs)} views, optional closure={tomography_frame_id} views.")
print(f"Calibrated sample-plane inputs alone: condenser rank={np.linalg.matrix_rank(sample_input_jacobian)}, max |J|={np.max(np.abs(sample_input_jacobian)):.1e}. Covariance tomography basis: ordinary={ordinary_covariance_rank}/10, coupled={complete_covariance_rank}/10; the 40-view two-plane option is an alternative to, not an extra copy of, the basic 12-view probe likelihood.")
assert np.max(np.abs(sample_input_jacobian))==0 and ordinary_covariance_rank==9 and complete_covariance_rank==10 and abs(refined_root-exact_critical_root)<abs(coarse_root-exact_critical_root) and actual_condenser_baseline_audit.rank==8 and actual_condenser_extended_audit.rank==12 and actual_upstream_extended_audit.rank==16 and condenser_covariance_audit.conditional_rank==12 and condenser_covariance_audit.nuisance_profiled_rank==12 and upstream_covariance_audit.conditional_rank==16 and upstream_covariance_audit.nuisance_profiled_rank==15 and upstream_two_plane_position_audit.nuisance_profiled_rank==15 and upstream_two_plane_full_audit.conditional_rank==16 and upstream_two_plane_full_audit.nuisance_profiled_rank==16 and upstream_two_plane_full_audit.nuisance_rank==10 and condenser_two_plane_full_audit.nuisance_profiled_rank==12 and condenser_two_plane_full_audit.diagnostics.combined.joint.rank==22 and two_plane_weakest_relative>1e-5 and tomography_frame_id==40 and np.all(np.isfinite(condenser_basic_sigma)) and np.all(np.isfinite(condenser_two_plane_sigma))
fig_upstream,upstream_axes=plt.subplots(1,4,figsize=(17.2,4.1),constrained_layout=True); coarse_fraction=coarse_aperture_counts/incident_electrons; coarse_fraction_sigma=np.sqrt(np.maximum(coarse_aperture_counts,1))/incident_electrons
upstream_axes[0].plot(critical_scan_values,coarse_transmission_truth.mean(axis=1),color="#0072B2",lw=2,label="model"); [upstream_axes[0].errorbar(critical_scan_values,coarse_fraction[:,sign_index],yerr=coarse_fraction_sigma[:,sign_index],fmt="o",ms=3.5,color=color,label=label) for sign_index,(color,label) in enumerate((("#D55E00","−CL3 counted image"),("#009E73","+CL3 counted image")))]; upstream_axes[0].set(xlabel="CL1 normalized raw DAC",ylabel="transmitted incident fraction",title="Known-aperture absolute counts"); upstream_axes[0].legend(frameon=False,fontsize=8)
upstream_axes[1].plot(critical_scan_values,coarse_critical_truth,color="#0072B2",lw=2,label="exact model"); upstream_axes[1].plot(critical_scan_values,coarse_critical_measured,"o",color="#D55E00",ms=4,label="coarse pairs"); upstream_axes[1].plot(adaptive_scan_values,adaptive_critical_measured,"D",color="#009E73",ms=4,label="adaptive pairs"); upstream_axes[1].axhline(0,color="0.45",lw=1); upstream_axes[1].axvline(coarse_root,color="#D55E00",ls=":",label="coarse root"); upstream_axes[1].axvline(refined_root,color="#009E73",ls="--",label="refined root"); upstream_axes[1].set(xlabel="CL1 normalized raw DAC",ylabel="∂ log probe radius / ∂ CL3 DAC",title="Critical line: measured bisection"); upstream_axes[1].legend(frameon=False,fontsize=8)
baseline_sv=actual_condenser_baseline_audit.relative_singular_values; extended_sv=actual_condenser_extended_audit.relative_singular_values; profiled_singular=condenser_covariance_audit.diagnostics.combined.profiled_optical.singular_values; profiled_sv=profiled_singular/profiled_singular[0]; upstream_axes[2].semilogy(np.arange(1,len(baseline_sv)+1),np.maximum(baseline_sv,1e-18),"o-",color="#0072B2",label=f"non-duplicated scan {actual_condenser_baseline_audit.rank}/12"); upstream_axes[2].semilogy(np.arange(1,len(extended_sv)+1),np.maximum(extended_sv,1e-18),"s-",color="#D55E00",label=f"+ probes, covariance fixed {actual_condenser_extended_audit.rank}/12"); upstream_axes[2].semilogy(np.arange(1,len(profiled_sv)+1),np.maximum(profiled_sv,1e-18),"^-",color="#009E73",label=f"entrance covariance profiled {condenser_covariance_audit.nuisance_profiled_rank}/12"); upstream_axes[2].axhline(1e-7,color="0.35",ls="--",lw=1); upstream_axes[2].set(xlabel="condenser singular direction",ylabel="relative whitened singular value",title="Condenser information survives nuisance profiling"); upstream_axes[2].legend(frameon=False,fontsize=7.5)
precision_labels=[name.replace(".z_center_m"," z").replace(".log_length_m"," L").replace(".q_bias"," q₀").replace(".q_slope_per_dac"," q′") for name in condenser_covariance_audit.parameter_names]; precision_x=np.arange(len(precision_labels)); upstream_axes[3].bar(precision_x-.19,condenser_basic_precision_metric,width=.38,color="#0072B2",label="basic 266 frames"); upstream_axes[3].bar(precision_x+.19,condenser_two_plane_precision_metric,width=.38,color="#D55E00",label="two-plane 294 frames"); upstream_axes[3].axhline(.01,color="#009E73",ls="--",lw=1.4,label="1% gate"); upstream_axes[3].set_yscale("log"); upstream_axes[3].set(xticks=precision_x,xticklabels=precision_labels,ylabel="profiled 1σ / parameter\n(log-length σ for L)",title="Rank is full; individual precision is not"); upstream_axes[3].tick_params(axis="x",labelrotation=90,labelsize=6.5); upstream_axes[3].legend(frameon=False,fontsize=7)
for ax in upstream_axes: ax.grid(alpha=.2)
fig_upstream.suptitle("Raw-DAC condenser anchors: no electrical-current readback required",fontsize=12.5); upstream_exports=export_ray_figure(fig_upstream,GENERATED/"neoarm_condenser_reconnaissance",metadata={"title":"NEOARM raw-DAC condenser reconnaissance","scan_channel":recon_cfg["scan_channel"],"wobble_channel":recon_cfg["wobble_channel"],"coarse_scan_values":critical_scan_values,"adaptive_scan_values":adaptive_scan_values,"wobble_step":critical_wobble,"observation_plane_z_m":column.sample_z_m,"aperture_z_m":aperture_cfg["z_m"],"aperture_radius_m":aperture_cfg["radius_m"],"reconnaissance_frames":2*len(all_recon_scan_values),"probe_extension_frames":3*len(condenser_probe_dacs),"optional_two_plane_frames":tomography_frame_id,"optional_two_plane_replaces_basic_probe_extension":True,"dac_approach":"forward_from_lower_raw_dac","dac_approach_precondition_step":upstream_approach_step,"dac_ramp_steps":upstream_ramp_steps,"settling_time_s":upstream_settling_time_s,"critical_line_derivative_used_for_adaptive_scan_only":True,"likelihood_observables":"algebraically non-duplicated per-frame aperture transmissions and absolute log radii","synthetic_reconnaissance_noise_covariance":"diagonal; replace by image-bootstrap joint covariance for instrument data","coarse_root":coarse_root,"refined_root":refined_root,"truth_root":exact_critical_root,"retained_scan_condenser_rank":actual_condenser_baseline_audit.rank,"extended_condenser_rank_fixed_covariance":actual_condenser_extended_audit.rank,"extended_upstream_rank_fixed_covariance":actual_upstream_extended_audit.rank,"rank_interpretation":"local structural identifiability, not completed nonlinear recovery","entrance_covariance_released_in_fit":True,"condenser_conditional_rank":condenser_covariance_audit.conditional_rank,"condenser_covariance_profiled_rank":condenser_covariance_audit.nuisance_profiled_rank,"entrance_covariance_raw_rank":condenser_covariance_audit.diagnostics.combined.nuisance.rank,"entrance_covariance_column_equilibrated_rank":condenser_covariance_audit.nuisance_rank,"upstream_conditional_rank":upstream_covariance_audit.conditional_rank,"upstream_covariance_profiled_rank":upstream_covariance_audit.nuisance_profiled_rank,"ordinary_covariance_rank":ordinary_covariance_rank,"complete_covariance_rank":complete_covariance_rank,"two_plane_position_profiled_rank":upstream_two_plane_position_audit.nuisance_profiled_rank,"two_plane_full_profiled_rank":upstream_two_plane_full_audit.nuisance_profiled_rank,"two_plane_covariance_nuisance_rank":upstream_two_plane_full_audit.nuisance_rank,"two_plane_weakest_relative_singular_value":two_plane_weakest_relative,"basic_condenser_profiled_sigma_native":condenser_basic_sigma,"basic_condenser_precision_metric":condenser_basic_precision_metric,"two_plane_condenser_profiled_sigma_native":condenser_two_plane_sigma,"two_plane_condenser_precision_metric":condenser_two_plane_precision_metric,"basic_one_percent_precision_ready":condenser_basic_one_percent_ready,"two_plane_one_percent_precision_ready":condenser_two_plane_one_percent_ready,"basic_weakest_scaled_fisher_direction":condenser_basic_weak_vector,"precision_audit_csv":str(condenser_precision_path.relative_to(REPO)),"two_plane_manifest":str(two_plane_tomography_manifest_path.relative_to(REPO)),"two_plane_requires_calibrated_screen_maps":True,"two_plane_requires_reconstruction_covariance":True}); print("Condenser reconnaissance exports:",upstream_exports); plt.show()

### Actual NEOARM condenser route: two experimentally distinct branches

This section reads the draft instrument geometry **read-only**. The physical C aperture is at 0.250 m with 10–150 micrometre radii, while the double-deflector planes are at 0.290 and 0.295 m. Consequently, the deflector is downstream of CL1 and CL3: its central differences cancel the source and directly constrain C-mini/Obj-prefield transport, but carry exactly zero CL1 or CL3 derivative. It also cannot translate the beam across the upstream C-aperture edge. More fundamentally, once illumination is parameterized by its state at the sample plane, downstream sample images are conditionally independent of the upstream condenser and have exactly zero upstream sensitivity. The sample-plane tilt is not assumed calibrated; it must be inferred as a nuisance from registered known-pattern image/diffraction data, but that inference alone still cannot identify the condenser. The condenser fit is therefore conditional: Branch A requires an independent Obj-prefield anchor, while the unconditional actual-hardware route requires a calibrated upstream traversal of the C-aperture edge. Centred transmission through the larger physical holes is usually saturated, so an edge traversal—not another centred count—is the informative experiment.

**Branch A (108 frames, conditional).** If Obj-prefield has an independent effective anchor, use a stable nonround illumination tag at twelve frozen safe CL1/CL3/C-mini settings. Five calibrated two-dimensional screen projections per state jointly reconstruct the ten full phase-space covariance summaries (60 images total; the ten correlated summaries are not ten images). At three of those settings, four deflector commands are acquired at both signs in real and diffraction space (48 images). The live Jacobian releases source $z$, all ten entrance-covariance coordinates, and all sixteen command-to-kick gains. It has one explicitly reported source/covariance nuisance gauge while retaining 12/12 profiled condenser rank. A deterministic noisy eight-start nonlinear recovery is run in the remaining identifiable subspace; it is matched-model evidence for this conditional route, not an instrument calibration or proof of global uniqueness. The Branch-A block currently connected to the final global demonstration uses the real 0.250/0.290/0.295 m plane geometry but deterministic matched synthetic observations. Instrument readiness stays false until a data loader supplies the 168 instrument observables and their full covariance Cholesky factor and the Obj-prefield anchor is independently certified.

**Branch B (296 frames, Obj-prefield free).** Without that anchor, the same sample-plane tomography plus downstream deflector block leaves a two-dimensional condenser family. The added measurement is a calibrated traversal of the same 10 micrometre C-aperture edge at four states: three non-collinear normals, seven offsets per normal, and one aperture-out dose reference, or 22 raw count images per state. Thirty-two five-image sample phase-space tomography states and three sixteen-image deflector states make the preferred 296-frame cross-check. The checksum-bound reusable likelihood preserves an exact 56-parameter order (12 condenser targets, 44 nuisances) and 458-residual order (88 Poisson edge, 320 dense-whitened tomography, 48 dense-whitened deflector, two shared priors). Its explicit matched synthetic expectation reproduces 12/12 profiled target rank after nuisance rank 43 and passes a bounded target-only nonlinear smoke recovery. That smoke fixes the nuisances and is an execution test, not a joint multistart or instrument recovery. The preferred local design has a worst nominal one-sigma metric of 0.9495%; even the full candidate pool has only a 0.9069% worst metric, so there is no 0.8% safety margin and no held-out hardware recovery has passed.

The edge-state beam is micrometre-scale and cannot pass through the 40 nm sample-atlas perforation. These frames therefore require the specimen retracted or a covariance-certified conservative clear region/current monitor at least 60 micrometres across. Sample-tag frames require the C aperture removed unless truncation is modeled explicitly. These condenser calibrations are additions whose selected optical settings do not duplicate the main 230-frame acquisition: branch A makes 338 total frames, branch B makes 526, and the preferred no-commanded-tilt astigmatic route makes 462 under conservative independent-frame accounting (414 if the seven summaries truly share one image with a measured joint covariance). `NEOARM_CONDENSER_ROUTE` accepts synthetic/instrument variants of `astigmatic_no_tilt`, `branch_a_conditional`, and `branch_b_unconditional`. Synthetic routes can never produce final instrument acceptance. Instrument routes are recognized but fail closed until registered observations, full covariance factors, and route-specific hardware evidence are loaded. The synthetic 0.225 m proxy above is not an additional hardware prescription or a final-fit bound. The fitted magnetic law is raw DAC to effective integrated solenoid strength; without current readback it cannot be factorized into DAC-to-current and current-to-field calibrations.

In [ ]:
INSTRUMENT_DRAFT_PATH=REPO/"examples/microscope_models/microscope.toml"
with INSTRUMENT_DRAFT_PATH.open("rb") as stream: instrument_draft=tomllib.load(stream)
actual_c_aperture_cfg=instrument_draft["apertures"]["C_aperture"]; actual_deflector_cfg=instrument_draft["deflectors"]["CL12_deflector"]
actual_c_aperture_z_m=float(actual_c_aperture_cfg["z_m"]); actual_c_aperture_radii_m=np.asarray(actual_c_aperture_cfg["radii_m"],float)
actual_deflector_planes_m=np.asarray((actual_deflector_cfg["z_m"],actual_deflector_cfg["z_m"]+actual_deflector_cfg["spacing_m"]),float); actual_sample_z_m=float(instrument_draft["sample"]["z_m"])
assert np.isclose(actual_c_aperture_z_m,.250) and np.allclose(actual_deflector_planes_m,(.290,.295)) and np.isclose(actual_sample_z_m,column.sample_z_m)

# First use the illumination already declared by the main notebook. Its centred C-aperture counts are almost completely saturated, so moving across a certified edge is the useful measurement.
actual_reference_aperture_matrix=hard_edge_matrix_to_plane(column,jnp.asarray(p_truth),jnp.asarray(reference_dacs),actual_c_aperture_z_m)
actual_reference_aperture_mean,actual_reference_aperture_covariance=map(np.asarray,position_moments(actual_reference_aperture_matrix,entrance_source.mean_state,entrance_source.covariance))
actual_reference_aperture_rms=float(np.asarray(rms_beam_radius(actual_reference_aperture_covariance)))
actual_centered_transmission=np.asarray([circular_gaussian_aperture_transmission(actual_reference_aperture_mean,actual_reference_aperture_covariance,radius,radial_order=256,azimuthal_order=256) for radius in actual_c_aperture_radii_m],float)
actual_centered_clearance_sigma=(actual_c_aperture_radii_m-np.linalg.norm(actual_reference_aperture_mean))/np.sqrt(np.linalg.eigvalsh(actual_reference_aperture_covariance).max())
actual_smallest_radius_m=float(actual_c_aperture_radii_m[0]); actual_edge_demo_step_m=min(max(.5*actual_reference_aperture_rms,.05e-6),.30*actual_smallest_radius_m)
actual_edge_demo_design=three_normal_aperture_edge_design(actual_smallest_radius_m,actual_edge_demo_step_m)
actual_edge_demo_prediction=condenser_aperture_edge_scan(column,jnp.asarray(p_truth),jnp.asarray(reference_dacs),entrance_source.mean_state,entrance_source.covariance,aperture_z_m=actual_c_aperture_z_m,aperture_centre_xy_m=np.zeros(2),aperture_radius_m=actual_smallest_radius_m,incident_dose_electrons=2e7,aperture_centre_offsets_xy_m=actual_edge_demo_design.aperture_centre_offsets_xy_m,radial_order=48,azimuthal_order=128)
actual_edge_demo_counts=np.asarray(actual_edge_demo_design.raw_expected_counts(actual_edge_demo_prediction))[0]
assert actual_edge_demo_design.n_frames==22 and actual_edge_demo_counts.shape==(22,) and np.all(np.isfinite(actual_edge_demo_counts))
print(f"Draft hardware (read-only): C aperture z={actual_c_aperture_z_m:.3f} m, radii={1e6*actual_c_aperture_radii_m} µm; double deflector z={actual_deflector_planes_m} m.")
print(f"Current centred beam at the real aperture: rms={1e6*actual_reference_aperture_rms:.4f} µm; edge clearances={np.array2string(actual_centered_clearance_sigma,precision=1)} largest-axis σ; transmissions={np.array2string(actual_centered_transmission,precision=8)}. Near-unity entries are saturated, not useful independent constraints.")

# Branch A: 12 frozen raw-DAC states, each reconstructed from five registered screen images. The ten covariance summaries are correlated outputs of those five images, not ten extra images.
hardware_conditional_dacs_3=np.asarray(((.344334711,.300923170,.034372425),(-.25,.35,.35),(.35,-.175,-.175),(.175,.35,.35),(.35,.35,-.175),(.35,.175,.35),(.175,.35,-.175),(-.25,-.175,.35),(.35,-.175,.35),(.35,.35,.35),(.294307546,.338190740,.170964461),(-.303560131,.340665327,-.088153601)))
hardware_conditional_dacs=np.zeros((len(hardware_conditional_dacs_3),len(column.channel_names))); hardware_conditional_dacs[:,:3]=hardware_conditional_dacs_3
hardware_conditional_deflector_indices=np.asarray((1,2,4)); hardware_conditional_deflector_dacs=hardware_conditional_dacs[hardware_conditional_deflector_indices]
hardware_conditional_covariance=np.diag((3.75e-17,3.75e-9,1.6666666666666667e-17,1.6666666666666667e-9))
hardware_conditional_chart=EntranceCovarianceChart(hardware_conditional_covariance); hardware_zero_covariance_coordinates=np.zeros(10)
hardware_parameters=np.asarray(column.nominal_parameters); hardware_parameter_names=column.parameter_names
hardware_condenser_indices=np.asarray([index for index,name in enumerate(hardware_parameter_names) if name.startswith(("CL1.","CL3.","C-mini."))]); hardware_source_z_index=hardware_parameter_names.index("source_z_m")
hardware_objective_nuisance_indices=np.asarray([index for index,name in enumerate(hardware_parameter_names) if name=="source_z_m" or name.startswith("Obj-prefield.")])
hardware_condenser_names=tuple(hardware_parameter_names[index] for index in hardware_condenser_indices)
hardware_condenser_scales=np.asarray([1. if name.endswith(".log_length_m") else abs(hardware_parameters[index]) for name,index in zip(hardware_condenser_names,hardware_condenser_indices,strict=True)])
hardware_condenser_truth=hardware_parameters[hardware_condenser_indices]; hardware_objective_truth=hardware_parameters[hardware_objective_nuisance_indices]
def hardware_with_condenser(target):
    return jnp.asarray(hardware_parameters).at[jnp.asarray(hardware_condenser_indices)].set(target)
def hardware_with_condenser_and_source(target,source_z_m):
    return hardware_with_condenser(target).at[hardware_source_z_index].set(source_z_m)
def hardware_with_condenser_and_objective(target,objective):
    return hardware_with_condenser(target).at[jnp.asarray(hardware_objective_nuisance_indices)].set(objective)
def hardware_conditional_tomography(target,covariance_coordinates):
    return axial_plane_covariance_observables_with_entrance_covariance(column,hardware_with_condenser(target),covariance_coordinates,jnp.asarray(hardware_conditional_dacs),hardware_conditional_chart,plane_z_m=actual_sample_z_m,mode="full").reshape(-1)
def hardware_conditional_tomography_with_source(target,source_z_m,covariance_coordinates):
    return axial_plane_covariance_observables_with_entrance_covariance(column,hardware_with_condenser_and_source(target,source_z_m),covariance_coordinates,jnp.asarray(hardware_conditional_dacs),hardware_conditional_chart,plane_z_m=actual_sample_z_m,mode="full").reshape(-1)

# Four independent shift/tilt command axes feed two physical kick planes. Bias cancels from the central difference; all 16 gains remain calibration nuisances.
hardware_kick_gain=np.zeros((2,2,4)); hardware_kick_gain[0,0,(0,2)]=1.; hardware_kick_gain[0,1,(1,3)]=1.; hardware_kick_gain[1,0,(0,2)]=(1.,-1.); hardware_kick_gain[1,1,(1,3)]=(1.,-1.)
hardware_deflector_map=DeflectorCommandKickMap(plane_z_m=actual_deflector_planes_m,plane_names=("upper","lower"),command_names=("shift_x","shift_y","tilt_x","tilt_y"),reference_command=np.zeros(4),kick_bias_xy_rad=np.zeros((2,2)),gain_xy_rad_per_command=hardware_kick_gain)
hardware_deflector_drive=10e-6; hardware_deflector_command_deltas=jnp.eye(4)*hardware_deflector_drive; hardware_deflector_gain_truth=hardware_deflector_map.nominal_parameters[4:]
def hardware_conditional_deflector_response(target,gain_parameters,objective=None):
    parameters=hardware_with_condenser(target) if objective is None else hardware_with_condenser_and_objective(target,objective)
    calibration_parameters=jnp.concatenate((jnp.zeros(4),gain_parameters)); responses=[]
    for dacs in hardware_conditional_deflector_dacs:
        for command_delta in hardware_deflector_command_deltas:
            responses.append(central_deflector_command_difference(column,parameters,jnp.asarray(dacs),np.zeros(4),hardware_conditional_covariance,np.zeros(4),command_delta,hardware_deflector_map,calibration_parameters=calibration_parameters,mode="full").mean_half_difference)
    return jnp.stack(responses).reshape(-1)

hardware_tomography_jacobian_target=np.asarray(jax.jacfwd(hardware_conditional_tomography,0)(jnp.asarray(hardware_condenser_truth),jnp.asarray(hardware_zero_covariance_coordinates)))/.002
hardware_tomography_jacobian_covariance=np.asarray(jax.jacfwd(hardware_conditional_tomography,1)(jnp.asarray(hardware_condenser_truth),jnp.asarray(hardware_zero_covariance_coordinates)))/.002
hardware_tomography_jacobian_source=np.asarray(jax.jacfwd(hardware_conditional_tomography_with_source,1)(jnp.asarray(hardware_condenser_truth),hardware_parameters[hardware_source_z_index],jnp.asarray(hardware_zero_covariance_coordinates))).reshape(-1,1)/.002
hardware_deflector_jacobian_target=np.asarray(jax.jacfwd(hardware_conditional_deflector_response,0)(jnp.asarray(hardware_condenser_truth),jnp.asarray(hardware_deflector_gain_truth)))
hardware_deflector_jacobian_gain=np.asarray(jax.jacfwd(hardware_conditional_deflector_response,1)(jnp.asarray(hardware_condenser_truth),jnp.asarray(hardware_deflector_gain_truth)))
hardware_deflector_sigma=np.tile(np.asarray((.1e-9,.02e-6,.1e-9,.02e-6))/np.sqrt(2),len(hardware_conditional_deflector_dacs)*4)
hardware_deflector_jacobian_target=hardware_deflector_jacobian_target/hardware_deflector_sigma[:,None]; hardware_deflector_jacobian_gain=hardware_deflector_jacobian_gain/hardware_deflector_sigma[:,None]
hardware_conditional_blocks={
    "five-image_sample_tomography":np.hstack((hardware_tomography_jacobian_target,hardware_tomography_jacobian_source,hardware_tomography_jacobian_covariance,np.zeros((hardware_tomography_jacobian_target.shape[0],16)))),
    "double_deflector_central_differences":np.hstack((hardware_deflector_jacobian_target,np.zeros((hardware_deflector_jacobian_target.shape[0],11)),hardware_deflector_jacobian_gain)),
}
hardware_conditional_audit=diagnose_blockwise_identifiability(hardware_conditional_blocks,optical_parameter_count=12,parameter_scales=np.concatenate((hardware_condenser_scales,design_parameter_scales[[hardware_source_z_index]],np.ones(26))),rank_rtol=1e-7)
hardware_conditional_precision=hardware_conditional_audit.combined.profiled_optical_precision; hardware_conditional_eigenvalues,hardware_conditional_eigenvectors=np.linalg.eigh(hardware_conditional_precision)
hardware_conditional_metrics=np.sqrt(np.maximum(np.diag((hardware_conditional_eigenvectors*(1/hardware_conditional_eigenvalues)[None,:])@hardware_conditional_eigenvectors.T),0))
assert hardware_conditional_audit.combined.profiled_optical.rank==12 and hardware_conditional_audit.combined.joint.rank==38 and np.max(hardware_conditional_metrics)<.01

# A deflector placed after CL3 cannot identify CL1 or CL3. This derivative audit is structural; its nonzero C-mini block is why three deflector states are retained as an independent cross-check.
def hardware_one_deflector_difference(parameters):
    return central_deflector_command_difference(column,parameters,jnp.asarray(hardware_conditional_deflector_dacs[0]),np.zeros(4),hardware_conditional_covariance,np.zeros(4),np.asarray((hardware_deflector_drive,0.,0.,0.)),hardware_deflector_map,mode="full").mean_half_difference
hardware_deflector_parameter_jacobian=np.asarray(jax.jacfwd(hardware_one_deflector_difference)(jnp.asarray(hardware_parameters)))
hardware_deflector_sensitivity={lens:max(float(np.max(np.abs(hardware_deflector_parameter_jacobian[:,[index for index,name in enumerate(hardware_parameter_names) if name.startswith(lens+".")]]))),0.) for lens in ("CL1","CL3","C-mini")}
assert hardware_deflector_sensitivity["CL1"]<1e-12 and hardware_deflector_sensitivity["CL3"]<1e-12 and hardware_deflector_sensitivity["C-mini"]>1e-12

# Releasing source-z and the complete Obj-prefield hard-edge section exposes the two-dimensional family that branch B must close.
def hardware_free_tomography(target,covariance_coordinates,objective):
    return axial_plane_covariance_observables_with_entrance_covariance(column,hardware_with_condenser_and_objective(target,objective),covariance_coordinates,jnp.asarray(hardware_conditional_dacs),hardware_conditional_chart,plane_z_m=actual_sample_z_m,mode="full").reshape(-1)
def hardware_free_deflector(target,gain_parameters,objective):
    return hardware_conditional_deflector_response(target,gain_parameters,objective)
hardware_free_tomography_jacobian_objective=np.asarray(jax.jacfwd(hardware_free_tomography,2)(jnp.asarray(hardware_condenser_truth),jnp.asarray(hardware_zero_covariance_coordinates),jnp.asarray(hardware_objective_truth)))/.002
hardware_free_deflector_jacobian_objective=np.asarray(jax.jacfwd(hardware_free_deflector,2)(jnp.asarray(hardware_condenser_truth),jnp.asarray(hardware_deflector_gain_truth),jnp.asarray(hardware_objective_truth)))/hardware_deflector_sigma[:,None]
hardware_free_blocks={
    "five-image_sample_tomography":np.hstack((hardware_tomography_jacobian_target,hardware_tomography_jacobian_covariance,hardware_free_tomography_jacobian_objective,np.zeros((hardware_tomography_jacobian_target.shape[0],16)))),
    "double_deflector_central_differences":np.hstack((hardware_deflector_jacobian_target,np.zeros((hardware_deflector_jacobian_target.shape[0],10)),hardware_free_deflector_jacobian_objective,hardware_deflector_jacobian_gain)),
}
hardware_free_objective_audit=diagnose_blockwise_identifiability(hardware_free_blocks,optical_parameter_count=12,parameter_scales=np.concatenate((hardware_condenser_scales,np.ones(10),design_parameter_scales[hardware_objective_nuisance_indices],np.ones(16))),rank_rtol=1e-7)
assert hardware_free_objective_audit.combined.profiled_optical.rank==10

# Frozen, corrected raw-likelihood D-optimal design for branch B. It was searched offline over the declared safe model envelope; the notebook verifies its geometry and emits the executable frame manifest, but does not mislabel the local Fisher result as a nonlinear recovery.
hardware_free_edge_dacs=np.asarray(((.32129185569563756,.28789468334456914,-.0031792407020097557),(.3434091160605709,.19049255593340686,.14113707358106006),(-.30311942339461456,.3043391652107576,.06846903857116954),(-.3024702518915204,-.1702431129912892,-.11896712085262534)))
hardware_free_tomography_dacs=np.asarray(((.05390728507813741,-.055578895420375696,.011001175079379227),(-.3229228893933024,.17682114187694115,.2201414251569114),(-.3045082387943909,.15950252718846486,.04799249139326217),(-.3127860588832398,.20619352012056363,.02663145452641047),(.027536957857594913,.015102697190588932,-.13599768836057197),(.050965604488568295,-.1983389748648254,-.07285763264307156),(-.3228822003287005,.15254399937527297,.30145773754997374),(.08027355909151379,-.17932584439722124,.12392817857007224),(.05229236931221548,-.04396647169457085,-.014285365002947553),(.06511249998756091,-.1463884840555505,-.030379832047153055),(.029889389150115575,-.031787428866532186,.11515660791209217),(.05023315975650339,-.11852888967527467,.13496163424186888),(-.3087943154146932,.11634082222975345,.2508981419563352),(.05755931978108064,-.06837499650112086,.016395391096466067),(-.3204863918828997,.20618389084972422,-.03918315597089664),(.02997663878394785,-.19918477073121219,.1957745731149146),(.043413063420047526,-.18080117887616176,-.05624933877841294),(.04541208869383151,-.1446553817283501,-.046161154754916695),(-.32652744598928457,.18705684961869973,-.02838622212836417),(.07861291219270622,-.15425802837067928,.10343491909349953),(.019973202970508086,-.15817125041242674,.13181710821678702),(-.2983563963262108,.14379535390724524,.13737257824447333),(-.26440303566083895,.12246402891247382,.21745424583458137),(.03412924670816769,-.04613163592540298,-.14750660289365192),(.04898871249212511,-.18938133467590682,-.12050311571797714),(-.11963524954935381,.11982630010153505,-.08567368954230754),(-.24291602366661547,.12117850722529383,.14327255920586238),(.00234940932459482,-.06981485086472214,.012365891755216268),(-.30902695035691763,.193677974475909,.06974155874766211),(-.27612235380048045,.15952332114218992,-.07354762529792196),(.08280990334343663,-.11483586727513764,-.15686817208320245),(-.3252481412029846,.17435618095714528,-.13730438770388964)))
hardware_free_deflector_dacs=np.asarray(((-.28745443142472027,.13101293200189423,.3153155551756718),(-.28376611094571225,.12197162255971189,.3488748362751629),(-.32058494124275916,.18748719841911737,-.14843379472992696)))
def hardware_expand_condenser_dacs(values):
    expanded=np.zeros((len(values),len(column.channel_names))); expanded[:,:3]=np.asarray(values); return expanded
hardware_free_edge_dacs=hardware_expand_condenser_dacs(hardware_free_edge_dacs); hardware_free_tomography_dacs=hardware_expand_condenser_dacs(hardware_free_tomography_dacs); hardware_free_deflector_dacs=hardware_expand_condenser_dacs(hardware_free_deflector_dacs)
hardware_free_safe_dac_bounds=np.asarray(((-.32727272727272727,-.2,-.17777777777777778),(.35,.35,.35)))
hardware_free_declared_q_bounds=np.asarray((.02,1.5)); hardware_free_actual_safe_q_ranges=np.asarray(((.02,.3925),(.02,.3775),(.02,.2575))); hardware_free_selected_q_ranges=np.asarray(((.02040990470589349,.388875013833314),(.020529899024712056,.3478204573869924),(.029409322562558904,.2569936763238233)))
hardware_free_local_metrics=np.asarray((.00159754873692817,.009495143316064191,.001413495844670551,.001381273763274076,.00019848838289290442,.006163368561354176,.0019428419336317668,.0019342306768173122,.00025033681850412666,.004211595169629972,.0029123572901736186,.0007359796378181136))
hardware_free_minimum_248_metrics=np.asarray((.0015974052885393605,.009495900821723961,.001413221080447653,.0013835599700495623,.00019826823337041553,.006112384276858299,.001911652819820262,.0018938252490682274,.00031213428481490396,.009053427630236421,.004857901321159543,.0040041204031179495))
hardware_free_local_rank=12; hardware_free_local_nuisance_rank=31; hardware_free_full_pool_max_metric=.00906869166438818
hardware_free_all_dacs=np.vstack((hardware_free_edge_dacs,hardware_free_tomography_dacs,hardware_free_deflector_dacs))
assert hardware_free_all_dacs.shape==(39,len(column.channel_names)) and np.all(hardware_free_all_dacs[:,:3]>=hardware_free_safe_dac_bounds[0]) and np.all(hardware_free_all_dacs[:,:3]<=hardware_free_safe_dac_bounds[1])
main_state_dacs=np.asarray([state.dac_values for state in plan.states]); hardware_design_has_main_state_overlap=any(np.any(np.all(np.isclose(main_state_dacs,dacs,rtol=0.,atol=1e-12),axis=1)) for dacs in np.vstack((hardware_conditional_dacs,hardware_free_all_dacs)))
assert not hardware_design_has_main_state_overlap and np.all(hardware_free_selected_q_ranges[:,0]>=hardware_free_declared_q_bounds[0]) and np.all(hardware_free_selected_q_ranges[:,1]<=hardware_free_declared_q_bounds[1])
assert hardware_free_local_rank==12 and np.max(hardware_free_local_metrics)<.01 and np.max(hardware_free_minimum_248_metrics)<.01 and hardware_free_full_pool_max_metric>.008

# The edge-state illumination is intentionally micrometre-scale at the aperture. It cannot use the 40 nm atlas perforation: retract the specimen or use a covariance-certified conservative >=60 µm clear blank/current monitor.
hardware_free_entrance_covariance=np.asarray(((3.2996222470133553e-10,-7.902707288210568e-10,1.1567235947574048e-10,-2.755108387142808e-10),(-7.90270728821057e-10,1.893784930950994e-9,-2.7551083871428074e-10,6.559955939505998e-10),(1.1567235947574047e-10,-2.7551083871428074e-10,5.825180216572556e-10,-1.3918133943017606e-9),(-2.755108387142807e-10,6.559955939505998e-10,-1.3918133943017606e-9,3.3260673775151802e-9)))
hardware_edge_covariances=[]
for dacs in hardware_free_edge_dacs:
    edge_matrix=hard_edge_matrix_to_plane(column,jnp.asarray(hardware_parameters),jnp.asarray(dacs),actual_c_aperture_z_m); _,edge_covariance=position_moments(edge_matrix,np.zeros(4),hardware_free_entrance_covariance); hardware_edge_covariances.append(np.asarray(edge_covariance))
hardware_edge_covariances=np.stack(hardware_edge_covariances); hardware_edge_normal_angles=np.deg2rad((0.,60.,120.)); hardware_edge_normals=np.stack((np.cos(hardware_edge_normal_angles),np.sin(hardware_edge_normal_angles)),axis=-1); hardware_edge_sigma_offsets=np.asarray((-2.,-1.25,-.6,0.,.6,1.25,2.))
hardware_edge_commands=np.stack([np.stack([normal*(actual_smallest_radius_m+offset*np.sqrt(normal@covariance@normal)) for normal in hardware_edge_normals for offset in hardware_edge_sigma_offsets]) for covariance in hardware_edge_covariances])
hardware_free_edge_prediction=condenser_aperture_edge_scan(column,jnp.asarray(hardware_parameters),jnp.asarray(hardware_free_edge_dacs),np.zeros(4),hardware_free_entrance_covariance,aperture_z_m=actual_c_aperture_z_m,aperture_centre_xy_m=np.zeros(2),aperture_radius_m=actual_smallest_radius_m,incident_dose_electrons=2e7,aperture_centre_offsets_xy_m=hardware_edge_commands,radial_order=48,azimuthal_order=128)
hardware_free_edge_expected_counts=np.asarray(hardware_free_edge_prediction.expected_transmitted_electrons); hardware_free_tomography_truth=np.asarray(axial_plane_covariance_observables(column,jnp.asarray(hardware_parameters),jnp.asarray(hardware_free_tomography_dacs),hardware_free_entrance_covariance,plane_z_m=actual_sample_z_m,mode="full"))
assert hardware_free_edge_expected_counts.shape==(4,21) and hardware_free_tomography_truth.shape==(32,10) and np.all(np.isfinite(hardware_free_edge_expected_counts))

# Reusable Branch-B likelihood: checksum-bound design_296, exact 56-parameter/458-residual ordering, and explicitly synthetic observations. This route does not require an Obj-prefield anchor, but its normalized controls are not certified instrument DACs.
BRANCH_B_PAYLOAD_PATH=REPO/"examples/lens_inversion/neoarm_condenser_branch_b_design.json"; reusable_branch_B_experiment=load_branch_b_experiment(BRANCH_B_PAYLOAD_PATH,repository_root=REPO,design_name="design_296")
assert reusable_branch_B_experiment.payload_sha256==EXPECTED_BRANCH_B_PAYLOAD_SHA256 and reusable_branch_B_experiment.coordinate_status==NORMALIZED_CONTROL_WARNING and reusable_branch_B_experiment.raw_frame_count==296
reusable_branch_B_truth_parameters=np.asarray(reusable_branch_B_experiment.nominal_parameters); reusable_branch_B_synthetic_observations=make_synthetic_branch_b_observations(reusable_branch_B_experiment,reusable_branch_B_truth_parameters,sample_noise=False,seed=20260909); reusable_branch_B_synthetic_likelihood=BranchBLikelihood(reusable_branch_B_experiment,reusable_branch_B_synthetic_observations)
reusable_branch_B_deflector_coordinate_contract={"stored_name":"response_per_rad","synthetic_meaning":"central difference divided by the frozen synthetic drive convention","instrument_requirement":"retain central differences per raw command unit; infer the shared 16-coordinate K16 raw-command-to-kick map and sample tilt from registered known-pattern image/diffraction data","calibrated_physical_deflection_angle_assumed":False}
assert reusable_branch_B_synthetic_observations.kind=="synthetic_expectation" and "synthetic" in reusable_branch_B_synthetic_observations.provenance.lower() and len(reusable_branch_B_synthetic_likelihood.parameter_names)==56 and len(reusable_branch_B_synthetic_likelihood.residual_names)==458
assert reusable_branch_B_synthetic_likelihood.parameter_slices=={"target":slice(0,12),"entrance_covariance":slice(12,22),"source_and_obj_prefield":slice(22,26),"K16":slice(26,42),"shared_log_hole_radius":slice(42,43),"shared_log_stage_xy_scale":slice(43,44),"edge_local":slice(44,56)} and reusable_branch_B_synthetic_likelihood.residual_slices=={"edge_poisson":slice(0,88),"tomography_whitened":slice(88,408),"deflector_whitened":slice(408,456),"shared_priors":slice(456,458)}
reusable_branch_B_reference_J=np.asarray(reusable_branch_B_synthetic_likelihood.jacobian(jnp.asarray(reusable_branch_B_truth_parameters))); reusable_branch_B_profiled_diagnostics=reusable_branch_B_synthetic_likelihood.local_profiled_target_diagnostics(reusable_branch_B_truth_parameters,jacobian=reusable_branch_B_reference_J)
assert reusable_branch_B_reference_J.shape==(458,56) and reusable_branch_B_profiled_diagnostics.target_rank==12 and reusable_branch_B_profiled_diagnostics.nuisance_rank==43
def reusable_branch_B_target_smoke_residual(target_coordinates): return reusable_branch_B_synthetic_likelihood.residual(jnp.asarray(reusable_branch_B_truth_parameters).at[:12].set(target_coordinates))
reusable_branch_B_target_smoke_residual_jit=jax.jit(reusable_branch_B_target_smoke_residual); reusable_branch_B_target_smoke_jacobian_jit=jax.jit(jax.jacfwd(reusable_branch_B_target_smoke_residual)); reusable_branch_B_target_smoke_initial=np.asarray((4e-4,-6e-4,5e-4,-4e-4,-3e-4,5e-4,-6e-4,4e-4,2e-4,-5e-4,4e-4,6e-4)); reusable_branch_B_target_smoke_residual_jit(jnp.asarray(reusable_branch_B_target_smoke_initial)).block_until_ready(); reusable_branch_B_target_smoke_jacobian_jit(jnp.asarray(reusable_branch_B_target_smoke_initial)).block_until_ready()
reusable_branch_B_target_smoke_fit=least_squares(lambda values:np.asarray(reusable_branch_B_target_smoke_residual_jit(jnp.asarray(values))),reusable_branch_B_target_smoke_initial,jac=lambda values:np.asarray(reusable_branch_B_target_smoke_jacobian_jit(jnp.asarray(values))),bounds=(-2e-3,2e-3),x_scale="jac",max_nfev=30,ftol=1e-12,xtol=1e-12,gtol=1e-8); reusable_branch_B_target_smoke_max_error=float(np.max(np.abs(reusable_branch_B_target_smoke_fit.x))); reusable_branch_B_target_smoke_passed=bool(reusable_branch_B_target_smoke_fit.success and reusable_branch_B_target_smoke_fit.cost<1e-14 and reusable_branch_B_target_smoke_max_error<1e-8); reusable_branch_B_synthetic_demo_ready=bool(reusable_branch_B_profiled_diagnostics.target_rank==12 and reusable_branch_B_target_smoke_passed)
assert reusable_branch_B_target_smoke_passed and reusable_branch_B_synthetic_demo_ready
print(f"Reusable Branch B synthetic expectation: payload={reusable_branch_B_experiment.payload_sha256}, raw frames={reusable_branch_B_experiment.raw_frame_count}, parameter/residual order={len(reusable_branch_B_synthetic_likelihood.parameter_names)}/{len(reusable_branch_B_synthetic_likelihood.residual_names)}, profiled target rank={reusable_branch_B_profiled_diagnostics.target_rank}/12 after nuisance rank {reusable_branch_B_profiled_diagnostics.nuisance_rank}, bounded target-only nonlinear smoke success={reusable_branch_B_target_smoke_fit.success}, nfev={reusable_branch_B_target_smoke_fit.nfev}, cost={reusable_branch_B_target_smoke_fit.cost:.3e}, max target error={reusable_branch_B_target_smoke_max_error:.3e}. This is synthetic route execution, not hardware admission; `response_per_rad` is a synthetic normalization and K16 remains unknown in raw-command instrument data.")

# Branch A matched-model recovery. All 12 condenser coordinates, source z, ten entrance-covariance coordinates, and all 16 deflector gains are released. The one data-null source/covariance gauge is removed by an SVD coordinate chart, not by fixing source z.
hardware_branch_A_optical_scales=np.asarray([1e-4 if name.endswith(".z_center_m") else (.02 if name.endswith((".log_length_m",".q_bias")) else .03) for name in hardware_condenser_names]); hardware_branch_A_source_scale=2e-4; hardware_branch_A_covariance_scales=np.full(10,.10); hardware_branch_A_gain_scales=np.full(16,.05)
hardware_branch_A_joint_names=tuple([f"scaled:{name}" for name in hardware_condenser_names]+["scaled:source_z_m"]+[f"scaled:{name}" for name in hardware_conditional_chart.coordinate_names]+[f"scaled:{name}" for name in hardware_deflector_map.parameter_names[4:]])
def decode_hardware_branch_A_joint(scaled_joint):
    target=jnp.asarray(hardware_condenser_truth)+jnp.asarray(hardware_branch_A_optical_scales)*scaled_joint[:12]; source_z_m=hardware_parameters[hardware_source_z_index]+hardware_branch_A_source_scale*scaled_joint[12]; covariance_coordinates=jnp.asarray(hardware_branch_A_covariance_scales)*scaled_joint[13:23]; gain_parameters=jnp.asarray(hardware_deflector_gain_truth)+jnp.asarray(hardware_branch_A_gain_scales)*scaled_joint[23:]; parameters=hardware_with_condenser_and_source(target,source_z_m); return parameters,covariance_coordinates,gain_parameters
def actual_hardware_branch_A_response(parameters,covariance_coordinates,gain_parameters):
    tomography=axial_plane_covariance_observables_with_entrance_covariance(column,parameters,covariance_coordinates,jnp.asarray(hardware_conditional_dacs),hardware_conditional_chart,plane_z_m=actual_sample_z_m,mode="full").reshape(-1); calibration_parameters=jnp.concatenate((jnp.zeros(4),gain_parameters)); deflector=[]
    for dacs in hardware_conditional_deflector_dacs:
        for command_delta in hardware_deflector_command_deltas: deflector.append(central_deflector_command_difference(column,parameters,jnp.asarray(dacs),np.zeros(4),hardware_conditional_covariance,np.zeros(4),command_delta,hardware_deflector_map,calibration_parameters=calibration_parameters,mode="full").mean_half_difference)
    return tomography,jnp.stack(deflector).reshape(-1)
def hardware_branch_A_prediction(scaled_joint):
    return actual_hardware_branch_A_response(*decode_hardware_branch_A_joint(scaled_joint))
def hardware_branch_A_whitened_prediction(scaled_joint):
    tomography,deflector=hardware_branch_A_prediction(scaled_joint); return jnp.concatenate((tomography/.002,deflector/jnp.asarray(hardware_deflector_sigma)))
hardware_branch_A_reference=np.zeros(39); hardware_branch_A_reference_J=np.asarray(jax.jacfwd(hardware_branch_A_whitened_prediction)(jnp.asarray(hardware_branch_A_reference))); hardware_branch_A_reference_subspace=scaled_identifiable_subspace(hardware_branch_A_reference_J,hardware_branch_A_reference,np.ones(39),parameter_names=hardware_branch_A_joint_names,rank_rtol=1e-7)
hardware_branch_A_null_vector=hardware_branch_A_reference_subspace.null_basis_scaled[:,0]; hardware_branch_A_null_optical_norm=float(np.linalg.norm(hardware_branch_A_null_vector[:12])); hardware_branch_A_null_source_covariance_norm=float(np.linalg.norm(hardware_branch_A_null_vector[12:23])); hardware_branch_A_null_gain_norm=float(np.linalg.norm(hardware_branch_A_null_vector[23:])); assert hardware_branch_A_reference_subspace.rank==38 and hardware_branch_A_reference_subspace.nullity==1 and hardware_branch_A_null_optical_norm<1e-10 and hardware_branch_A_null_gain_norm<1e-10
hardware_branch_A_truth_scaled=np.zeros(39); hardware_branch_A_truth_scaled[:12]=.25*np.sin((np.arange(12)+1)*np.sqrt(2.)); hardware_branch_A_truth_scaled[23:]=.20*np.sin((np.arange(16)+1)*np.sqrt(3.)); hardware_branch_A_truth_active=hardware_branch_A_reference_subspace.active_basis_scaled.T@hardware_branch_A_truth_scaled; hardware_branch_A_truth_projection=np.asarray(hardware_branch_A_reference_subspace.parameters_from_active(jnp.asarray(hardware_branch_A_truth_active))); assert np.max(np.abs(hardware_branch_A_truth_projection-hardware_branch_A_truth_scaled))<1e-10
hardware_branch_A_truth_tomography,hardware_branch_A_truth_deflector=map(np.asarray,hardware_branch_A_prediction(jnp.asarray(hardware_branch_A_truth_scaled))); hardware_branch_A_rng=np.random.default_rng(20260906); hardware_branch_A_observed_tomography=hardware_branch_A_truth_tomography+hardware_branch_A_rng.normal(0,.002,hardware_branch_A_truth_tomography.shape); hardware_branch_A_observed_deflector=hardware_branch_A_truth_deflector+hardware_branch_A_rng.normal(0,hardware_deflector_sigma,hardware_branch_A_truth_deflector.shape)
hardware_branch_A_synthetic_global_truth_tomography,hardware_branch_A_synthetic_global_truth_deflector=map(np.asarray,actual_hardware_branch_A_response(jnp.asarray(hardware_parameters),jnp.asarray(hardware_zero_covariance_coordinates),jnp.asarray(hardware_deflector_gain_truth))); hardware_branch_A_global_rng=np.random.default_rng(20260908); hardware_branch_A_synthetic_global_observed_tomography=hardware_branch_A_synthetic_global_truth_tomography+hardware_branch_A_global_rng.normal(0,.002,hardware_branch_A_synthetic_global_truth_tomography.shape); hardware_branch_A_synthetic_global_observed_deflector=hardware_branch_A_synthetic_global_truth_deflector+hardware_branch_A_global_rng.normal(0,hardware_deflector_sigma,hardware_branch_A_synthetic_global_truth_deflector.shape)
hardware_branch_A_synthetic_global_observed_vector=np.concatenate((hardware_branch_A_synthetic_global_observed_tomography,hardware_branch_A_synthetic_global_observed_deflector)); hardware_branch_A_synthetic_global_sigma_vector=np.concatenate((np.full(hardware_branch_A_synthetic_global_observed_tomography.size,.002),hardware_deflector_sigma)); hardware_branch_A_synthetic_global_observation_source="deterministic_matched_synthetic_seed_20260908"
def hardware_branch_A_dataset_residual(parameters,covariance_coordinates,gain_parameters,observed_vector,observation_covariance_cholesky):
    tomography,deflector=actual_hardware_branch_A_response(parameters,covariance_coordinates,gain_parameters); prediction=jnp.concatenate((tomography,deflector)); return jnp.linalg.solve(jnp.asarray(observation_covariance_cholesky),prediction-jnp.asarray(observed_vector))
def hardware_branch_A_synthetic_global_residual(parameters,covariance_coordinates,gain_parameters):
    tomography,deflector=actual_hardware_branch_A_response(parameters,covariance_coordinates,gain_parameters); prediction=jnp.concatenate((tomography,deflector)); return (prediction-jnp.asarray(hardware_branch_A_synthetic_global_observed_vector))/jnp.asarray(hardware_branch_A_synthetic_global_sigma_vector)
hardware_branch_A_actual_geometry_residual_bound_to_synthetic_global_demo=True; hardware_branch_A_global_uses_synthetic_observations=True; hardware_branch_A_instrument_observed_vector=None; hardware_branch_A_instrument_covariance_cholesky=None; hardware_branch_A_instrument_likelihood_bound_to_global=False
reusable_branch_B_target_indices=np.asarray(reusable_branch_B_experiment.target_parameter_indices); reusable_branch_B_upstream_indices=np.asarray(reusable_branch_B_experiment.upstream_parameter_indices); reusable_branch_B_extra_nuisance_names=tuple(reusable_branch_B_synthetic_likelihood.parameter_names[42:]); reusable_branch_B_extra_nuisance_reference=np.asarray(reusable_branch_B_truth_parameters[42:])
assert tuple(reusable_branch_B_experiment.column.parameter_names)==tuple(column.parameter_names) and np.allclose(reusable_branch_B_experiment.column.nominal_parameters,column.nominal_parameters,rtol=0.,atol=0.) and np.array_equal(reusable_branch_B_target_indices,hardware_condenser_indices) and np.array_equal(reusable_branch_B_upstream_indices,hardware_objective_nuisance_indices) and np.allclose(reusable_branch_B_truth_parameters[26:42],hardware_deflector_gain_truth) and len(reusable_branch_B_extra_nuisance_names)==14
def hardware_branch_B_parameters_from_shared_physical(parameters,covariance_coordinates,gain_parameters,extra_nuisance_parameters):
    physical=jnp.asarray(parameters); covariance=jnp.asarray(covariance_coordinates); gains=jnp.asarray(gain_parameters); extra=jnp.asarray(extra_nuisance_parameters)
    if physical.shape!=(column.n_parameters,) or covariance.shape!=(10,) or gains.shape!=(16,) or extra.shape!=(14,): raise ValueError("Branch-B global adapter received a wrong parameter shape.")
    values=jnp.asarray(reusable_branch_B_truth_parameters); values=values.at[:12].set((physical[jnp.asarray(reusable_branch_B_target_indices)]-jnp.asarray(reusable_branch_B_experiment.column.nominal_parameters)[jnp.asarray(reusable_branch_B_target_indices)])/jnp.asarray(reusable_branch_B_experiment.target_parameter_scales)); values=values.at[12:22].set(covariance); values=values.at[22:26].set((physical[jnp.asarray(reusable_branch_B_upstream_indices)]-jnp.asarray(reusable_branch_B_experiment.column.nominal_parameters)[jnp.asarray(reusable_branch_B_upstream_indices)])/jnp.asarray(reusable_branch_B_experiment.upstream_parameter_scales)); values=values.at[26:42].set(gains); return values.at[42:56].set(extra)
def hardware_branch_B_synthetic_global_residual(parameters,covariance_coordinates,gain_parameters,extra_nuisance_parameters): return reusable_branch_B_synthetic_likelihood.residual(hardware_branch_B_parameters_from_shared_physical(parameters,covariance_coordinates,gain_parameters,extra_nuisance_parameters))
hardware_branch_B_instrument_observations=None; hardware_branch_B_instrument_likelihood=None; hardware_branch_B_instrument_likelihood_bound_to_global=False
def hardware_branch_A_residual(scaled_joint):
    tomography,deflector=hardware_branch_A_prediction(scaled_joint); return jnp.concatenate(((tomography-jnp.asarray(hardware_branch_A_observed_tomography))/.002,(deflector-jnp.asarray(hardware_branch_A_observed_deflector))/jnp.asarray(hardware_deflector_sigma)))
hardware_branch_A_residual=jax.jit(hardware_branch_A_residual)
hardware_branch_A_unrestricted_bounds=column.default_bounds(position_half_width_m=1e-3,length_ratio=(.75,1.35),q_half_width=.30,slope_half_width=.30); hardware_branch_A_plane_constraints=tuple(HardEdgeInternalPlaneConstraint(float(z_m),"CL3","C-mini",5e-4) for z_m in (actual_c_aperture_z_m,*actual_deflector_planes_m)); hardware_branch_A_plane_certificates=restrict_hard_edge_internal_planes_bounds(column,hardware_branch_A_unrestricted_bounds,hardware_branch_A_plane_constraints,reference_parameters=hardware_parameters); hardware_branch_A_physical_lower,hardware_branch_A_physical_upper=hardware_branch_A_plane_certificates[0].bounds; hardware_branch_A_worst_plane_clearance_m=min(certificate.worst_case_clearance_m for certificate in hardware_branch_A_plane_certificates)
hardware_branch_A_lower=np.concatenate(((hardware_branch_A_physical_lower[hardware_condenser_indices]-hardware_condenser_truth)/hardware_branch_A_optical_scales,[(hardware_branch_A_physical_lower[hardware_source_z_index]-hardware_parameters[hardware_source_z_index])/hardware_branch_A_source_scale],np.full(10,-.35)/hardware_branch_A_covariance_scales,np.full(16,-.25)/hardware_branch_A_gain_scales)); hardware_branch_A_upper=np.concatenate(((hardware_branch_A_physical_upper[hardware_condenser_indices]-hardware_condenser_truth)/hardware_branch_A_optical_scales,[(hardware_branch_A_physical_upper[hardware_source_z_index]-hardware_parameters[hardware_source_z_index])/hardware_branch_A_source_scale],np.full(10,.35)/hardware_branch_A_covariance_scales,np.full(16,.25)/hardware_branch_A_gain_scales))
assert np.all(hardware_branch_A_truth_scaled>hardware_branch_A_lower) and np.all(hardware_branch_A_truth_scaled<hardware_branch_A_upper) and hardware_branch_A_worst_plane_clearance_m>=5e-4
hardware_branch_A_fit=fit_identifiable_subspace(hardware_branch_A_residual,hardware_branch_A_reference,np.ones(39),rank_rtol=1e-7,bounds=(hardware_branch_A_lower,hardware_branch_A_upper),parameter_names=hardware_branch_A_joint_names,loss="linear",multistart=8,seed=20260907,jitter_scale=.12,max_nfev=500); hardware_branch_A_basins=diagnose_multistart_basins(hardware_branch_A_fit.active_fit,delta_cost=1.,scaled_distance_threshold=.25,parameter_scales=np.ones(hardware_branch_A_fit.subspace.rank),bounds=hardware_branch_A_fit.active_coordinate_bounds)
hardware_branch_A_truth_active=hardware_branch_A_fit.subspace.active_basis_scaled.T@hardware_branch_A_truth_scaled; hardware_branch_A_truth_inside_conservative_active_box=bool(np.all(hardware_branch_A_truth_active>hardware_branch_A_fit.active_coordinate_bounds[0]) and np.all(hardware_branch_A_truth_active<hardware_branch_A_fit.active_coordinate_bounds[1]))
hardware_branch_A_estimated_condenser=hardware_condenser_truth+hardware_branch_A_optical_scales*hardware_branch_A_fit.parameters[:12]; hardware_branch_A_true_condenser=hardware_condenser_truth+hardware_branch_A_optical_scales*hardware_branch_A_truth_scaled[:12]; hardware_branch_A_condenser_error=hardware_branch_A_estimated_condenser-hardware_branch_A_true_condenser; hardware_branch_A_condenser_sigma=hardware_branch_A_optical_scales*hardware_branch_A_fit.projected_parameter_standard_errors[:12]; hardware_branch_A_condenser_zscore=np.divide(hardware_branch_A_condenser_error,hardware_branch_A_condenser_sigma,out=np.full(12,np.inf),where=hardware_branch_A_condenser_sigma>0)
hardware_branch_A_best_basin=hardware_branch_A_basins.basins[0] if hardware_branch_A_basins.basins else None; hardware_branch_A_nonlinear_execution_complete=bool(hardware_branch_A_fit.success and hardware_branch_A_fit.subspace.rank==38 and hardware_branch_A_fit.fit_rank==38 and np.all(np.isfinite(hardware_branch_A_fit.parameters)) and np.isfinite(hardware_branch_A_fit.weighted_rmse)); hardware_branch_A_optical_recovery_within_uncertainty=bool(hardware_branch_A_fit.weighted_rmse<1.2 and np.max(np.abs(hardware_branch_A_condenser_zscore))<3.5); hardware_branch_A_synthetic_recovery_passed=bool(hardware_branch_A_nonlinear_execution_complete and hardware_branch_A_optical_recovery_within_uncertainty and hardware_branch_A_truth_inside_conservative_active_box and len(hardware_branch_A_basins.near_best_start_indices)>=2 and hardware_branch_A_basins.single_sampled_near_best_basin and hardware_branch_A_best_basin is not None and not hardware_branch_A_best_basin.representative_bound_hits)
assert hardware_branch_A_nonlinear_execution_complete and hardware_branch_A_optical_recovery_within_uncertainty and hardware_branch_A_best_basin is not None and hardware_branch_A_best_basin.representative_bound_hits and not hardware_branch_A_synthetic_recovery_passed
hardware_branch_A_recovery_path=GENERATED/"actual_hardware_condenser_branch_A_synthetic_recovery.csv"
with hardware_branch_A_recovery_path.open("w",newline="",encoding="utf-8") as stream:
    fields=("parameter_name","truth_native","estimate_native","error_native","one_sigma_native","standardized_error","z_error_um","z_one_sigma_um","fractional_or_log_error","fractional_or_log_one_sigma","truth_inside_conservative_active_box","joint_synthetic_gate","representative_active_bound_hits","evidence_scope"); writer=csv.DictWriter(stream,fieldnames=fields); writer.writeheader()
    for name,truth,estimate,error,sigma,zscore in zip(hardware_condenser_names,hardware_branch_A_true_condenser,hardware_branch_A_estimated_condenser,hardware_branch_A_condenser_error,hardware_branch_A_condenser_sigma,hardware_branch_A_condenser_zscore,strict=True):
        is_z=name.endswith(".z_center_m"); denominator=1. if name.endswith(".log_length_m") else max(abs(truth),np.finfo(float).tiny); writer.writerow(dict(parameter_name=name,truth_native=truth,estimate_native=estimate,error_native=error,one_sigma_native=sigma,standardized_error=zscore,z_error_um=1e6*error if is_z else np.nan,z_one_sigma_um=1e6*sigma if is_z else np.nan,fractional_or_log_error=error/denominator,fractional_or_log_one_sigma=sigma/denominator,truth_inside_conservative_active_box=hardware_branch_A_truth_inside_conservative_active_box,joint_synthetic_gate=hardware_branch_A_synthetic_recovery_passed,representative_active_bound_hits="|".join(hardware_branch_A_best_basin.representative_bound_hits),evidence_scope="deterministic noisy matched-model Branch A audit; optical recovery is within uncertainty, but bound-limited nuisance directions keep the joint gate false; Obj-prefield anchored; not instrument acceptance"))
hardware_branch_A_z_recovery_um={name.removesuffix(".z_center_m"):{"error_um":float(1e6*error),"one_sigma_um":float(1e6*sigma)} for name,error,sigma in zip(hardware_condenser_names,hardware_branch_A_condenser_error,hardware_branch_A_condenser_sigma,strict=True) if name.endswith(".z_center_m")}

hardware_dac_fields=[f"dac_{name.replace('-','_')}" for name in column.channel_names]
hardware_conditional_manifest_path=GENERATED/"actual_hardware_condenser_conditional_108_manifest.csv"
with hardware_conditional_manifest_path.open("w",newline="",encoding="utf-8") as stream:
    fields=("frame_id","branch","state_index","measurement","projection_or_command","command_sign","observable_contract","objective_prefield_anchor_required",*hardware_dac_fields); writer=csv.DictWriter(stream,fieldnames=fields); writer.writeheader(); frame_id=0
    for state_index,dacs in enumerate(hardware_conditional_dacs):
        for projection in complete_illumination_projections:
            row=dict(frame_id=frame_id,branch="A_conditional",state_index=state_index,measurement="registered_2d_screen",projection_or_command=projection.label,command_sign=0,observable_contract="five images jointly reconstruct one correlated ten-component phase-space covariance; use its full covariance",objective_prefield_anchor_required=True); row.update(dict(zip(hardware_dac_fields,dacs,strict=True))); writer.writerow(row); frame_id+=1
    for state_index,dacs in zip(hardware_conditional_deflector_indices,hardware_conditional_deflector_dacs,strict=True):
        for command_name in hardware_deflector_map.command_names:
            for command_sign in (-1,1):
                for domain in ("real_space","diffraction"):
                    row=dict(frame_id=frame_id,branch="A_conditional",state_index=int(state_index),measurement=f"deflector_{domain}",projection_or_command=command_name,command_sign=command_sign,observable_contract="paired +/- images form one source-cancelling position/tilt response; image and diffraction supply position and angle",objective_prefield_anchor_required=True); row.update(dict(zip(hardware_dac_fields,dacs,strict=True))); writer.writerow(row); frame_id+=1
assert frame_id==108

hardware_free_manifest_path=GENERATED/"actual_hardware_condenser_free_objective_296_manifest.csv"
with hardware_free_manifest_path.open("w",newline="",encoding="utf-8") as stream:
    fields=("frame_id","branch","state_index","measurement","projection_or_command","command_sign","offset_x_m","offset_y_m","expected_electrons","observable_contract","sample_requirement","aperture_motion_requirement",*hardware_dac_fields); writer=csv.DictWriter(stream,fieldnames=fields); writer.writeheader(); frame_id=0
    for state_index,dacs in enumerate(hardware_free_edge_dacs):
        for command_index,(offset_xy,expected) in enumerate(zip(hardware_edge_commands[state_index],hardware_free_edge_expected_counts[state_index],strict=True)):
            normal_index=command_index//len(hardware_edge_sigma_offsets); sigma_index=command_index%len(hardware_edge_sigma_offsets); row=dict(frame_id=frame_id,branch="B_free_objective",state_index=state_index,measurement="translated_10um_C_aperture_edge",projection_or_command=f"normal_{np.rad2deg(hardware_edge_normal_angles[normal_index]):.0f}deg_sigma_{hardware_edge_sigma_offsets[sigma_index]:+.2f}",command_sign=0,offset_x_m=offset_xy[0],offset_y_m=offset_xy[1],expected_electrons=expected,observable_contract="raw Poisson count; jointly fit shared radius/stage scale and per-state relative centre/log dose",sample_requirement="sample_retracted_or_covariance-certified_conservative_>=60um_clear_blank/current_monitor",aperture_motion_requirement="calibrated translated aperture or pre-0.250m scan; the 0.290/0.295m deflector cannot perform this scan"); row.update(dict(zip(hardware_dac_fields,dacs,strict=True))); writer.writerow(row); frame_id+=1
        row=dict(frame_id=frame_id,branch="B_free_objective",state_index=state_index,measurement="aperture_out_dose_reference",projection_or_command="aperture_out",command_sign=0,offset_x_m=np.nan,offset_y_m=np.nan,expected_electrons=2e7,observable_contract="absolute incident-dose reference; never normalise each inserted-aperture frame independently",sample_requirement="sample_retracted_or_covariance-certified_conservative_>=60um_clear_blank/current_monitor",aperture_motion_requirement="remove aperture reproducibly"); row.update(dict(zip(hardware_dac_fields,dacs,strict=True))); writer.writerow(row); frame_id+=1
    for state_index,(dacs,truth) in enumerate(zip(hardware_free_tomography_dacs,hardware_free_tomography_truth,strict=True)):
        for projection in complete_illumination_projections:
            row=dict(frame_id=frame_id,branch="B_free_objective",state_index=state_index,measurement="registered_2d_screen",projection_or_command=projection.label,command_sign=0,offset_x_m=np.nan,offset_y_m=np.nan,expected_electrons=np.nan,observable_contract="five images jointly reconstruct one correlated ten-component phase-space covariance "+json.dumps(truth.tolist(),separators=(',',':')),sample_requirement="stable calibrated nonround illumination tag",aperture_motion_requirement="C aperture removed, or truncation included explicitly"); row.update(dict(zip(hardware_dac_fields,dacs,strict=True))); writer.writerow(row); frame_id+=1
    for state_index,dacs in enumerate(hardware_free_deflector_dacs):
        for command_name in hardware_deflector_map.command_names:
            for command_sign in (-1,1):
                for domain in ("real_space","diffraction"):
                    row=dict(frame_id=frame_id,branch="B_free_objective",state_index=state_index,measurement=f"deflector_{domain}",projection_or_command=command_name,command_sign=command_sign,offset_x_m=np.nan,offset_y_m=np.nan,expected_electrons=np.nan,observable_contract="independent C-mini cross-check; profile all 16 command-to-kick gains",sample_requirement="registered clear/sample state suitable for centroid or diffraction retrieval",aperture_motion_requirement="C aperture removed"); row.update(dict(zip(hardware_dac_fields,dacs,strict=True))); writer.writerow(row); frame_id+=1
assert frame_id==296

hardware_precision_path=GENERATED/"actual_hardware_condenser_precision_audit.csv"
with hardware_precision_path.open("w",newline="",encoding="utf-8") as stream:
    fields=("parameter_name","branch_A_live_profiled_fraction_or_log_sigma","branch_B_offline_248_minimum_fraction_or_log_sigma","branch_B_offline_296_preferred_fraction_or_log_sigma","branch_B_296_meets_local_one_percent","evidence_scope"); writer=csv.DictWriter(stream,fieldnames=fields); writer.writeheader()
    for name,conditional_metric,minimum_metric,preferred_metric in zip(hardware_condenser_names,hardware_conditional_metrics,hardware_free_minimum_248_metrics,hardware_free_local_metrics,strict=True):
        writer.writerow(dict(parameter_name=name,branch_A_live_profiled_fraction_or_log_sigma=conditional_metric,branch_B_offline_248_minimum_fraction_or_log_sigma=minimum_metric,branch_B_offline_296_preferred_fraction_or_log_sigma=preferred_metric,branch_B_296_meets_local_one_percent=preferred_metric<.01,evidence_scope="local matched-model Fisher audit only; not nonlinear or held-out certification"))

hardware_branch_A_synthetic_demo_status={"actual_geometry_planes_m":(actual_c_aperture_z_m,*actual_deflector_planes_m),"observation_source":hardware_branch_A_synthetic_global_observation_source,"joint_rank":hardware_branch_A_reference_subspace.rank,"joint_parameter_count":39,"source_covariance_gauge_count":hardware_branch_A_reference_subspace.nullity,"nonlinear_execution_complete":hardware_branch_A_nonlinear_execution_complete,"optical_recovery_within_uncertainty":hardware_branch_A_optical_recovery_within_uncertainty,"joint_synthetic_recovery_passed":hardware_branch_A_synthetic_recovery_passed}
hardware_branch_A_instrument_fit_prerequisites={
    "actual_safe_DAC_envelope_certified_on_instrument":False,
    "stable_nonround_illumination_and_screen_maps_calibrated":False,
    "instrument_observation_vector_loaded":False,
    "instrument_full_covariance_cholesky_loaded":False,
    "sample_tilt_extracted_from_registered_known_pattern_image_and_diffraction":False,
    "raw_deflector_commands_retained_and_K16_profiled":False,
    "Obj_prefield_independently_anchored":False,
}
hardware_branch_A_instrument_postfit_prerequisites={"instrument_64_start_nonlinear_recovery_passed":False,"held_out_image_and_diffraction_metrics_passed":False}
hardware_branch_A_instrument_fit_ready=bool(hardware_branch_A_instrument_likelihood_bound_to_global and all(hardware_branch_A_instrument_fit_prerequisites.values())); hardware_branch_A_instrument_ready=bool(hardware_branch_A_instrument_fit_ready and all(hardware_branch_A_instrument_postfit_prerequisites.values()))
hardware_branch_B_local_fisher_audit_passed=bool(reusable_branch_B_profiled_diagnostics.target_rank==12 and hardware_free_local_rank==12 and np.max(hardware_free_local_metrics)<.01); hardware_branch_B_design_margin_passed=bool(hardware_free_full_pool_max_metric<.008)
hardware_branch_B_synthetic_demo_status={"observation_kind":reusable_branch_B_synthetic_observations.kind,"observation_provenance":reusable_branch_B_synthetic_observations.provenance,"payload_sha256":reusable_branch_B_experiment.payload_sha256,"raw_frame_count":reusable_branch_B_experiment.raw_frame_count,"parameter_count":len(reusable_branch_B_synthetic_likelihood.parameter_names),"residual_count":len(reusable_branch_B_synthetic_likelihood.residual_names),"profiled_target_rank":reusable_branch_B_profiled_diagnostics.target_rank,"profiled_nuisance_rank":reusable_branch_B_profiled_diagnostics.nuisance_rank,"bounded_target_smoke_passed":reusable_branch_B_target_smoke_passed,"synthetic_demo_ready":reusable_branch_B_synthetic_demo_ready}
hardware_branch_B_design_prerequisites={
    "actual_safe_DAC_envelope_certified_on_instrument":False,
    "translated_C_aperture_edge_scan_calibrated":False,
    "specimen_retracted_or_covariance_certified_clear_monitor":False,
    "joint_image_and_count_covariances_measured":False,
    "branch_B_nonlinear_multistart_recovery_passed":False,
    "branch_B_held_out_metrics_passed":False,
}
hardware_branch_B_design_ready=bool(hardware_branch_B_local_fisher_audit_passed and hardware_branch_B_design_margin_passed and all(hardware_branch_B_design_prerequisites.values()))
hardware_branch_B_instrument_fit_prerequisites={"normalized_controls_mapped_to_certified_instrument_DACs":False,"instrument_B_observations_loaded_and_payload_registered":False,"translated_C_aperture_scan_certified":False,"instrument_dense_covariance_factors_loaded":False,"sample_tilt_extracted_from_registered_known_pattern_image_and_diffraction":False,"raw_deflector_commands_retained_and_K16_profiled":False}; hardware_branch_B_instrument_postfit_prerequisites={"instrument_multistart_and_held_out_acceptance_passed":False}; hardware_branch_B_instrument_fit_ready=bool(hardware_branch_B_instrument_likelihood_bound_to_global and hardware_branch_B_instrument_likelihood is not None and all(hardware_branch_B_instrument_fit_prerequisites.values())); hardware_branch_B_instrument_ready=bool(hardware_branch_B_instrument_fit_ready and all(hardware_branch_B_instrument_postfit_prerequisites.values()))
ASTIGMATIC_DESIGN_PATH=REPO/"examples/lens_inversion/neoarm_condenser_astigmatic_design.json"; fullcolumn_astigmatic_design=load_astigmatic_condenser_design(ASTIGMATIC_DESIGN_PATH,repository_root=REPO); fullcolumn_astigmatic_design_audit=recompute_astigmatic_design_audit(fullcolumn_astigmatic_design)
fullcolumn_astigmatic_template_likelihood=make_astigmatic_condenser_likelihood(ASTIGMATIC_DESIGN_PATH,repository_root=REPO); fullcolumn_astigmatic_template_adapter=make_astigmatic_full_column_adapter(fullcolumn_astigmatic_template_likelihood,column); fullcolumn_astigmatic_outer_truth=np.asarray(fullcolumn_astigmatic_template_adapter.outer_contract.reference); fullcolumn_astigmatic_native_truth=np.asarray(fullcolumn_astigmatic_template_adapter.likelihood_parameters_from_parts(hardware_parameters,fullcolumn_astigmatic_outer_truth))
fullcolumn_astigmatic_synthetic_observations=make_synthetic_astigmatic_observations(fullcolumn_astigmatic_design,parameters=fullcolumn_astigmatic_native_truth,sample_noise=True,seed=26090419); fullcolumn_astigmatic_synthetic_likelihood=make_astigmatic_condenser_likelihood(ASTIGMATIC_DESIGN_PATH,observations=fullcolumn_astigmatic_synthetic_observations,repository_root=REPO); fullcolumn_astigmatic_adapter=make_astigmatic_full_column_adapter(fullcolumn_astigmatic_synthetic_likelihood,column); fullcolumn_astigmatic_adapter_rank=fullcolumn_astigmatic_adapter.local_rank_diagnostics()
fullcolumn_astigmatic_held_out_split=make_astigmatic_held_out_split(fullcolumn_astigmatic_synthetic_likelihood); fullcolumn_astigmatic_training_audit=audit_astigmatic_held_out_training(fullcolumn_astigmatic_synthetic_likelihood,fullcolumn_astigmatic_held_out_split)
fullcolumn_astigmatic_synthetic_demo_ready=bool(fullcolumn_astigmatic_design_audit.target_rank==12 and fullcolumn_astigmatic_design_audit.conditional_precision_pass and fullcolumn_astigmatic_adapter_rank.profiled_condenser_target_rank==12 and fullcolumn_astigmatic_training_audit.profiled_target_rank==12 and fullcolumn_astigmatic_training_audit.conditional_precision_pass and fullcolumn_astigmatic_training_audit.checksum_contract_verified and fullcolumn_astigmatic_synthetic_likelihood.observations.kind=="synthetic_draw"); fullcolumn_astigmatic_instrument_likelihood=None; fullcolumn_astigmatic_instrument_adapter=None; fullcolumn_astigmatic_instrument_held_out_split=None; fullcolumn_astigmatic_instrument_training_audit=None; fullcolumn_astigmatic_instrument_fit_prerequisites={"upstream_of_CL1_tag_plane_verified":False,"tag_repeatability_verified":False,"translated_C_aperture_traversal_calibrated":False,"realspace_and_angular_scale_priors_demonstrated":False,"detector_MTF_SNR_saturation_and_deblending_verified":False,"instrument_observations_and_dense_covariances_loaded":False}; fullcolumn_astigmatic_instrument_postfit_prerequisites={"instrument_multistart_profiles_and_held_out_passed":False}; fullcolumn_astigmatic_instrument_fit_ready=bool(fullcolumn_astigmatic_instrument_likelihood is not None and fullcolumn_astigmatic_instrument_adapter is not None and fullcolumn_astigmatic_instrument_held_out_split is not None and fullcolumn_astigmatic_instrument_training_audit is not None and fullcolumn_astigmatic_instrument_training_audit.checksum_contract_verified and all(fullcolumn_astigmatic_instrument_fit_prerequisites.values())); fullcolumn_astigmatic_instrument_ready=bool(fullcolumn_astigmatic_instrument_fit_ready and all(fullcolumn_astigmatic_instrument_postfit_prerequisites.values()))
selected_astigmatic_likelihood=fullcolumn_astigmatic_synthetic_likelihood if CONDENSER_ROUTE=="synthetic_astigmatic_no_tilt" else fullcolumn_astigmatic_instrument_likelihood if CONDENSER_ROUTE=="instrument_astigmatic_no_tilt" else None; selected_astigmatic_adapter=fullcolumn_astigmatic_adapter if CONDENSER_ROUTE=="synthetic_astigmatic_no_tilt" else fullcolumn_astigmatic_instrument_adapter if CONDENSER_ROUTE=="instrument_astigmatic_no_tilt" else None; selected_astigmatic_held_out_split=fullcolumn_astigmatic_held_out_split if CONDENSER_ROUTE=="synthetic_astigmatic_no_tilt" else fullcolumn_astigmatic_instrument_held_out_split if CONDENSER_ROUTE=="instrument_astigmatic_no_tilt" else None; selected_astigmatic_observation_kind=(None if selected_astigmatic_likelihood is None else selected_astigmatic_likelihood.observations.kind); selected_astigmatic_route_ready=(fullcolumn_astigmatic_synthetic_demo_ready if CONDENSER_ROUTE=="synthetic_astigmatic_no_tilt" else fullcolumn_astigmatic_instrument_ready if CONDENSER_ROUTE=="instrument_astigmatic_no_tilt" else None)
actual_hardware_condenser_global_fit_gate_ready=bool(hardware_branch_A_instrument_fit_ready); actual_hardware_condenser_global_gate_ready=bool(hardware_branch_A_instrument_ready)
condenser_route_status_by_name={
    "synthetic_astigmatic_no_tilt":{"branch":"ASTIGMATIC","data_kind":"synthetic","requires_Obj_prefield_anchor":False,"obj_prefield_anchor_bound":False,"requires_commanded_tilt":False,"global_fit_gate":fullcolumn_astigmatic_synthetic_demo_ready,"instrument_acceptance_gate":False},
    "synthetic_branch_a_conditional":{"branch":"A","data_kind":"synthetic","requires_Obj_prefield_anchor":True,"obj_prefield_anchor_bound":True,"global_fit_gate":False,"instrument_acceptance_gate":False},
    "synthetic_branch_b_unconditional":{"branch":"B","data_kind":"synthetic","requires_Obj_prefield_anchor":False,"obj_prefield_anchor_bound":False,"global_fit_gate":reusable_branch_B_synthetic_demo_ready,"instrument_acceptance_gate":False},
    "instrument_astigmatic_no_tilt":{"branch":"ASTIGMATIC","data_kind":"instrument","requires_Obj_prefield_anchor":False,"obj_prefield_anchor_bound":False,"requires_commanded_tilt":False,"global_fit_gate":fullcolumn_astigmatic_instrument_fit_ready,"instrument_acceptance_gate":fullcolumn_astigmatic_instrument_ready},
    "instrument_branch_a_conditional":{"branch":"A","data_kind":"instrument","requires_Obj_prefield_anchor":True,"obj_prefield_anchor_bound":hardware_branch_A_instrument_fit_prerequisites["Obj_prefield_independently_anchored"],"global_fit_gate":actual_hardware_condenser_global_fit_gate_ready,"instrument_acceptance_gate":actual_hardware_condenser_global_gate_ready},
    "instrument_branch_b_unconditional":{"branch":"B","data_kind":"instrument","requires_Obj_prefield_anchor":False,"obj_prefield_anchor_bound":False,"global_fit_gate":hardware_branch_B_instrument_fit_ready,"instrument_acceptance_gate":hardware_branch_B_instrument_ready},
}
selected_condenser_route_status=condenser_route_status_by_name[CONDENSER_ROUTE]; selected_condenser_route_global_gate_ready=bool(selected_condenser_route_status["global_fit_gate"]); selected_condenser_route_declared_prefit_instrument_status=bool(selected_condenser_route_status["instrument_acceptance_gate"]); selected_obj_prefield_anchor_bound=bool(selected_condenser_route_status["obj_prefield_anchor_bound"])
assert hardware_branch_A_actual_geometry_residual_bound_to_synthetic_global_demo and hardware_branch_A_global_uses_synthetic_observations and not hardware_branch_A_instrument_likelihood_bound_to_global and not hardware_branch_A_instrument_ready and reusable_branch_B_synthetic_demo_ready and not hardware_branch_B_design_ready and not hardware_branch_B_instrument_likelihood_bound_to_global and not hardware_branch_B_instrument_ready and not actual_hardware_condenser_global_gate_ready
print(f"Branch A: {len(hardware_conditional_dacs)}×5 tomography + {len(hardware_conditional_deflector_dacs)}×16 deflector = 108 frames; joint data rank={hardware_conditional_audit.combined.joint.rank}/39 with one source/covariance nuisance gauge; profiled condenser rank={hardware_conditional_audit.combined.profiled_optical.rank}/12, max local 1σ metric={100*np.max(hardware_conditional_metrics):.3f}% conditional on an independent Obj-prefield anchor.")
print(f"Branch A deterministic noisy eight-start recovery: success={hardware_branch_A_fit.success}, retained rank={hardware_branch_A_fit.fit_rank}/{hardware_branch_A_fit.subspace.rank}, WRMS={hardware_branch_A_fit.weighted_rmse:.3f} σ, near-best={len(hardware_branch_A_basins.near_best_start_indices)}/8 in {hardware_branch_A_basins.n_near_best_basins} sampled basin(s), high-cost={len(hardware_branch_A_basins.high_cost_start_indices)}, max condenser |error|={np.max(np.abs(hardware_branch_A_condenser_zscore)):.2f} local σ, z-centre error/σ (µm)={hardware_branch_A_z_recovery_um}, active bound hits={hardware_branch_A_best_basin.representative_bound_hits}, joint synthetic pass={hardware_branch_A_synthetic_recovery_passed}. The optical recovery is within uncertainty, but bound-limited nuisance directions keep the nonlinear gate false; this is not an instrument acceptance test.")
print(f"Double-deflector parameter sensitivities: {hardware_deflector_sensitivity}. It identifies C-mini transport, not CL1/CL3.")
print(f"Free source-z/Obj-prefield without an edge scan: profiled condenser rank={hardware_free_objective_audit.combined.profiled_optical.rank}/12. The frozen raw-likelihood minimum is 248 frames; the preferred 296-frame cross-check adds three deflector states and has local rank={hardware_free_local_rank}/12, max metric={100*np.max(hardware_free_local_metrics):.4f}%. Full-pool best={100*hardware_free_full_pool_max_metric:.4f}%, so there is no 0.8% design margin. Selected q ranges={hardware_free_selected_q_ranges.tolist()}.")
selected_astigmatic_split_summary=(None if selected_astigmatic_held_out_split is None else (len(selected_astigmatic_held_out_split.training_acquisition_indices),len(selected_astigmatic_held_out_split.held_out_acquisition_indices))); selected_astigmatic_rank_summary=(None if selected_astigmatic_likelihood is None else selected_astigmatic_likelihood.local_rank_diagnostics().profiled_target_rank); print(f"Route status — selected={CONDENSER_ROUTE}: {selected_condenser_route_status}; selected astigmatic observation kind={selected_astigmatic_observation_kind}, training/held-out acquisitions={selected_astigmatic_split_summary}, local target rank={selected_astigmatic_rank_summary}; astigmatic synthetic ready={fullcolumn_astigmatic_synthetic_demo_ready}; astigmatic instrument ready={fullcolumn_astigmatic_instrument_ready}; Branch-A synthetic pass={hardware_branch_A_synthetic_recovery_passed}; Branch-A instrument ready={hardware_branch_A_instrument_ready}; Branch-B synthetic ready={reusable_branch_B_synthetic_demo_ready}; Branch-B design ready={hardware_branch_B_design_ready}; Branch-B instrument ready={hardware_branch_B_instrument_ready}. Synthetic gates admit only synthetic demonstrations; every instrument route remains fail-closed.")
print("Condenser conclusion: downstream sample images have zero upstream sensitivity when conditioned on a sample-plane state. The preferred no-commanded-tilt route combines translated C-aperture counts with state-relative orientation/covariance of a deliberately astigmatic tag created upstream of CL1; its command need not be angle-calibrated. Branch A remains conditional on an Obj-prefield anchor, and Branch B retains raw-command K16 as a comparison route.")
print(f"Manifests: {hardware_conditional_manifest_path.relative_to(REPO)}, {hardware_free_manifest_path.relative_to(REPO)}; precision: {hardware_precision_path.relative_to(REPO)}")

fig_hardware,hardware_axes=plt.subplots(1,3,figsize=(15.8,4.2),constrained_layout=True)
hardware_axes[0].plot(actual_c_aperture_radii_m*1e6,actual_centered_transmission,"o-",color="#0072B2",lw=2); hardware_axes[0].axhline(.999,color=".4",ls="--",lw=1); hardware_axes[0].set(xlabel="physical C-aperture radius (µm)",ylabel="centred transmitted fraction",title="Centred count saturation")
edge_demo_transmission=np.asarray(actual_edge_demo_prediction.transmission)[0].reshape(actual_edge_demo_design.n_normals,actual_edge_demo_design.n_offsets)
for normal_index,(values,color) in enumerate(zip(edge_demo_transmission,("#0072B2","#D55E00","#009E73"),strict=True)): hardware_axes[1].plot(actual_edge_demo_design.signed_edge_offsets_m/actual_edge_demo_step_m,values,"o-",color=color,label=f"normal {normal_index+1}")
hardware_axes[1].set(xlabel="edge-step index",ylabel="transmitted fraction",title="Executable 3×7 edge traversal"); hardware_axes[1].legend(frameon=False,fontsize=8)
metric_x=np.arange(len(hardware_condenser_names)); hardware_axes[2].plot(metric_x,100*hardware_conditional_metrics,"o",color="#0072B2",label="A: Obj-prefield anchored"); hardware_axes[2].plot(metric_x,100*hardware_free_local_metrics,"s",color="#D55E00",label="B: free, 296-frame offline design"); hardware_axes[2].axhline(1.,color="#009E73",ls="--",lw=1.4,label="local 1% line"); hardware_axes[2].set_yscale("log"); hardware_axes[2].set(xticks=metric_x,xticklabels=[name.replace(".z_center_m"," z").replace(".log_length_m"," L").replace(".q_bias"," q₀").replace(".q_slope_per_dac"," q′") for name in hardware_condenser_names],ylabel="profiled local 1σ metric (%)",title="Local Fisher evidence—not acceptance"); hardware_axes[2].tick_params(axis="x",labelrotation=90,labelsize=6.5); hardware_axes[2].legend(frameon=False,fontsize=7)
for axis in hardware_axes: axis.grid(alpha=.2)
hardware_exports=export_ray_figure(fig_hardware,GENERATED/"neoarm_actual_hardware_condenser_audit",metadata={"title":"Actual NEOARM condenser hardware identifiability audit","instrument_draft":str(INSTRUMENT_DRAFT_PATH.relative_to(REPO)),"instrument_file_read_only":True,"synthetic_proxy_aperture_z_m":float(aperture_cfg["z_m"]),"actual_C_aperture_z_m":actual_c_aperture_z_m,"actual_C_aperture_radii_m":actual_c_aperture_radii_m,"double_deflector_planes_m":actual_deflector_planes_m,"branch_A_frames":108,"branch_A_total_with_main":338,"branch_A_joint_rank":hardware_conditional_audit.combined.joint.rank,"branch_A_joint_parameter_count":39,"branch_A_profiled_rank":hardware_conditional_audit.combined.profiled_optical.rank,"branch_A_source_covariance_gauge_count":hardware_branch_A_reference_subspace.nullity,"branch_A_synthetic_observation_source":hardware_branch_A_synthetic_global_observation_source,"branch_A_synthetic_nonlinear_execution_complete":hardware_branch_A_nonlinear_execution_complete,"branch_A_synthetic_optical_recovery_within_uncertainty":hardware_branch_A_optical_recovery_within_uncertainty,"branch_A_synthetic_nonlinear_recovery_passed":hardware_branch_A_synthetic_recovery_passed,"branch_A_synthetic_nonlinear_WRMS_sigma":hardware_branch_A_fit.weighted_rmse,"branch_A_synthetic_nonlinear_max_condenser_error_sigma":float(np.max(np.abs(hardware_branch_A_condenser_zscore))),"branch_A_synthetic_nonlinear_near_best_starts":len(hardware_branch_A_basins.near_best_start_indices),"branch_A_synthetic_nonlinear_high_cost_starts":len(hardware_branch_A_basins.high_cost_start_indices),"branch_A_synthetic_nonlinear_representative_bound_hits":hardware_branch_A_best_basin.representative_bound_hits,"branch_A_synthetic_recovery_csv":str(hardware_branch_A_recovery_path.relative_to(REPO)),"branch_A_actual_geometry_bound_to_synthetic_global_demo":hardware_branch_A_actual_geometry_residual_bound_to_synthetic_global_demo,"branch_A_instrument_likelihood_bound_to_global":hardware_branch_A_instrument_likelihood_bound_to_global,"branch_A_instrument_fit_ready":hardware_branch_A_instrument_fit_ready,"branch_A_instrument_ready":hardware_branch_A_instrument_ready,"branch_A_instrument_fit_prerequisites":hardware_branch_A_instrument_fit_prerequisites,"branch_A_instrument_postfit_prerequisites":hardware_branch_A_instrument_postfit_prerequisites,"branch_A_requires_independent_Obj_prefield_anchor":True,"branch_B_mathematical_minimum_frames":248,"branch_B_preferred_frames":296,"branch_B_total_with_main":526,"branch_B_offline_local_rank":hardware_free_local_rank,"branch_B_offline_max_metric":float(np.max(hardware_free_local_metrics)),"branch_B_local_fisher_audit_passed":hardware_branch_B_local_fisher_audit_passed,"branch_B_design_margin_passed":hardware_branch_B_design_margin_passed,"branch_B_design_ready":hardware_branch_B_design_ready,"branch_B_design_prerequisites":hardware_branch_B_design_prerequisites,"branch_B_evidence_scope":"local Fisher design audit only; never gates the Branch-A residual","full_pool_max_metric":hardware_free_full_pool_max_metric,"declared_q_bounds":hardware_free_declared_q_bounds,"actual_safe_q_ranges":hardware_free_actual_safe_q_ranges,"selected_q_ranges":hardware_free_selected_q_ranges,"duplicates_main_230_state":hardware_design_has_main_state_overlap,"final_instrument_condenser_acceptance":actual_hardware_condenser_global_gate_ready}); plt.show(); plt.close(fig_hardware)


### A coherent mini-acquisition at one magnification

This is a synthetic/reference diagnostic, not a required hardware acquisition. `DEMO_MAGNIFICATION` isolates an image set for checking together: certified sample, central-hole vacuum, calibrated position and tilt bases, their orthogonal/negative checks, and a representative negative/positive lens wobble. The complete surrogate and these panels use the native 256×256 detector geometry with one fixed incident dose and one shared display range; they are not separately normalized. Calibrated sample-plane input centroids and covariances are propagated through each setting's actual downstream matrix. These fast absolute-count surrogate images exercise frame identity, controls, flux preservation, and observable extraction, but do not enter a physical inversion until the Gaussian/Collins gates above pass. Scale and rotation come from registered Fourier or layer vectors, position/tilt frames constrain only the synthetic downstream response columns, vacuum constrains illumination and absolute intensity, and the signed difference panel estimates a finite-difference control derivative in noise-standard-deviation units.

In [ ]:
DEMO_MAGNIFICATION=150_000
demo_nominal_label=next(state.label for state in plan.states if state.target_magnification==DEMO_MAGNIFICATION)
def demo_request(state_label,measurement):
    return next(frame for frame in manifest if frame.optical_state_label==state_label and frame.measurement==measurement and frame.dac_approach=="forward" and frame.repeat_index==0)
demo_specs=(("certified sample",demo_nominal_label,"sample"),("central-hole vacuum",demo_nominal_label,"perforation"),("calibrated +x",demo_nominal_label,"position_basis"),("calibrated +tilt",demo_nominal_label,"angle_basis"),("orthogonal +y",demo_nominal_label,"basis_orthogonal"),("negative tilt",demo_nominal_label,"basis_negative"),("IL2 − wobble","single_IL2_minus","sample"),("IL2 + wobble","single_IL2_plus","sample"))
demo_requests=[demo_request(state_label,measurement) for _,state_label,measurement in demo_specs]
demo_images=[np.clip(manifest_counts[request.frame_id],0,None) for request in demo_requests]
demo_observables=[extract_image_observables(image,n_fourier_vectors=6) for image in demo_images]
demo_sample_reference=demo_observables[0]; demo_vacuum_reference=demo_observables[1]; demo_vmax=float(np.percentile(np.stack(demo_images),99.8))
detector_fov_xy_m=detector_pixel_xy_m*np.asarray(detector.shape_yx)[::-1]
multilayer_g_local_xy=np.asarray([1/atlas.certificate.multilayer_period_m,0.])
def demo_geometry(state_label):
    state=next(state for state in plan.states if state.label==state_label); state_J=detector_affine@np.asarray(column.sample_position_jacobian(design_parameters,np.asarray(state.dac_values)))
    pixel_map=np.diag(1/detector_pixel_xy_m)@state_J; affine=fit_fourier_affine(np.eye(2),pixel_map.T)
    projection=np.asarray(view_by_state[state_label].projection_world_from_local); g_world_xy=np.linalg.inv(projection).T@multilayer_g_local_xy
    g_detector_xy=np.linalg.inv(state_J).T@g_world_xy; g_detector_yx_bins=(g_detector_xy*detector_fov_xy_m)[::-1]
    return affine,g_detector_yx_bins
demo_reference_affine,demo_reference_g=demo_geometry(demo_nominal_label)
def matched_friedel_vector(vectors,target):
    vectors=np.asarray(vectors); candidate=min(vectors,key=lambda vector:min(np.linalg.norm(vector-target),np.linalg.norm(vector+target)))
    return candidate if np.dot(candidate,target)>=0 else -candidate
fig,axes=plt.subplots(2,4,figsize=(13.8,7.0),constrained_layout=True)
print(f"{DEMO_MAGNIFICATION:,}× mini-acquisition; DAC order={plan.channel_names}")
for ax,(caption,_,_),request,image,obs in zip(axes.flat,demo_specs,demo_requests,demo_images,demo_observables,strict=True):
    delta=np.asarray(request.raw_dac_values)-nominal_dacs[2]; centroid_reference=demo_sample_reference if request.measurement=="sample" else demo_vacuum_reference
    centroid_change=np.asarray(obs.centroid_xy_px)-np.asarray(centroid_reference.centroid_xy_px); detected_fraction=obs.total_intensity/surrogate_dose
    if request.measurement=="sample":
        state_affine,predicted_g=demo_geometry(request.optical_state_label); relative_scale=state_affine.isotropic_scale/demo_reference_affine.isotropic_scale; rotation_change=np.rad2deg(np.angle(np.exp(1j*(state_affine.rotation_rad-demo_reference_affine.rotation_rad)))); geometry_text=f"J model: M/M₀={relative_scale:.3f}; Δφ={rotation_change:+.2f}°"
    else: predicted_g=np.array([np.nan,np.nan]); relative_scale=np.nan; rotation_change=np.nan; geometry_text="calibrated vacuum input"
    ax.imshow(image,origin="lower",cmap="gray",vmin=0,vmax=demo_vmax); ax.set_title(f"{caption} · frame {request.frame_id}\nI/I_inc={detected_fraction:.3f}; Δc=({centroid_change[0]:+.2f},{centroid_change[1]:+.2f}) px\n{geometry_text}; max|ΔDAC|={np.max(np.abs(delta)):.3f}",fontsize=8.2); ax.set(xticks=[],yticks=[])
    first_vector=obs.fourier_vectors_yx_px[0] if len(obs.fourier_vectors_yx_px) else np.array([np.nan,np.nan])
    matched_g=matched_friedel_vector(obs.fourier_vectors_yx_px,predicted_g) if request.measurement=="sample" and len(obs.fourier_vectors_yx_px) else first_vector
    print(f"  frame {request.frame_id:03d} {caption:21s} r_s={np.array2string(request.sample_state.as_array(),precision=3)} raw={np.array2string(np.asarray(request.raw_dac_values),precision=4)} Δ={np.array2string(delta,precision=4)} I/Iinc={detected_fraction:.4f} Δc={np.array2string(centroid_change,precision=3)} σ={np.array2string(np.asarray(obs.principal_sigma_px),precision=3)} model_M/M0={relative_scale:.5f} model_Δφ={rotation_change:+.3f}° g_obs(y,x)={np.array2string(matched_g,precision=2)} g_model(y,x)={np.array2string(predicted_g,precision=2)}")
minus_affine,minus_g=demo_geometry("single_IL2_minus"); plus_affine,plus_g=demo_geometry("single_IL2_plus")
signed_template_rms=np.sqrt(np.mean((sample_template_by_state["single_IL2_plus"]-sample_template_by_state["single_IL2_minus"])**2))/np.mean(sample_template_by_state[demo_nominal_label])
print(f"Signed IL2 geometry: scale {minus_affine.isotropic_scale/demo_reference_affine.isotropic_scale:.5f}→{plus_affine.isotropic_scale/demo_reference_affine.isotropic_scale:.5f}; rotation {np.rad2deg(minus_affine.rotation_rad):+.3f}°→{np.rad2deg(plus_affine.rotation_rad):+.3f}°; Fourier {minus_g}→{plus_g}; noiseless image RMS change={signed_template_rms:.3e}")
assert signed_template_rms>1e-5 and np.linalg.norm(plus_g-minus_g)>1e-3
fig.suptitle(f"Geometry-aware calibrated surrogate — shared count scale; inputs ±{basis_position_m*1e9:.1f} nm and ±{basis_tilt_rad*1e6:.2f} µrad",fontsize=12.5)
demo_exports=export_ray_figure(fig,GENERATED/f"neoarm_{DEMO_MAGNIFICATION//1000}k_mini_acquisition",metadata={"title":f"NEOARM {DEMO_MAGNIFICATION:,}× mini-acquisition","magnification":DEMO_MAGNIFICATION,"detector_shape_yx":detector.shape_yx,"pixel_size_yx_m":detector.pixel_size_yx_m,"dose_electrons":surrogate_dose,"raw_dac_order":plan.channel_names,"raw_dac_values":[request.raw_dac_values for request in demo_requests],"sample_states":[request.sample_state.as_array() for request in demo_requests],"frame_ids":[request.frame_id for request in demo_requests],"shared_display_range_counts":[0,demo_vmax],"scope":"detector-native calibrated surrogate; physical Gaussian subset is validated separately"})
print("Mini-acquisition exports:",demo_exports); plt.show()
minus_mean=sample_template_by_state["single_IL2_minus"]*surrogate_dose/np.prod(manifest_shape); nominal_mean=sample_template_by_state[demo_nominal_label]*surrogate_dose/np.prod(manifest_shape); plus_mean=sample_template_by_state["single_IL2_plus"]*surrogate_dose/np.prod(manifest_shape)
signed_response=(plus_mean-minus_mean)/np.sqrt(plus_mean+minus_mean+2*surrogate_read_noise**2); response_limit=float(np.percentile(np.abs(signed_response),99.5))
fig_response,response_axes=plt.subplots(1,4,figsize=(12.8,3.25),constrained_layout=True); common_max=float(np.percentile(np.stack((minus_mean,nominal_mean,plus_mean)),99.8))
for ax,image,title in zip(response_axes[:3],(minus_mean,nominal_mean,plus_mean),("IL2 − expected counts","nominal expected counts","IL2 + expected counts"),strict=True): ax.imshow(image,origin="lower",cmap="gray",vmin=0,vmax=common_max); ax.set_title(title,fontsize=9); ax.set(xticks=[],yticks=[])
response_art=response_axes[3].imshow(signed_response,origin="lower",cmap="coolwarm",vmin=-response_limit,vmax=response_limit); response_axes[3].set_title("signed (+) − (−) response [noise σ]",fontsize=9); response_axes[3].set(xticks=[],yticks=[]); fig_response.colorbar(response_art,ax=response_axes[3],shrink=.72)
fig_response.suptitle(f"Why the image pair matters at {DEMO_MAGNIFICATION:,}×: a finite-difference control derivative",fontsize=12)
response_exports=export_ray_figure(fig_response,GENERATED/f"neoarm_{DEMO_MAGNIFICATION//1000}k_il2_signed_response",metadata={"title":"NEOARM signed IL2 image response","magnification":DEMO_MAGNIFICATION,"wobble_raw_dac":float(perturbations[4]),"response_units":"expected Poisson-plus-read-noise standard deviations"})
print("Signed-response exports:",response_exports); plt.show()

### Image-space inversion bridge: pixels to hard-edge parameters

This is a deliberately small synthetic/reference end-to-end recovery test, not a required hardware acquisition. Its values are extracted from the noisy detector frames above, rather than copied from ideal transfer-matrix entries. Each selected optical state contributes the registered specimen Fourier vector and the centroid plus full second-moment tensor of the calibrated position and tilt images. A parametric Poisson/read-noise bootstrap supplies a full covariance for every image block; the synthetic sample-state calibration covariance is propagated into the centroid block. The prediction remains differentiable because those observables are recomputed from the complete JAX hard-edge sample-to-detector matrix for every trial parameter vector.

Only downstream lens biases are examined here. Once position and tilt are calibrated at the sample, these images contain no source-to-sample information and therefore cannot identify `CL1`, `CL3`, or `C-mini`; condenser recovery requires the upstream illumination and vacuum-beam measurements in the complete likelihood. We report the image-only singular spectrum before fitting. If it resolves only one direction, the demonstration releases only the physical downstream q-bias with the largest whitened column norm and leaves the coupled directions explicit. This bridge remains a surrogate-data demonstration until the Gaussian/Collins validation gates admit physical images.

In [ ]:
bridge_state_labels=(demo_nominal_label,"single_IL1_minus","single_IL1_plus","single_IL2_minus","single_IL2_plus","single_IL3_minus","single_IL3_plus","single_PL1_minus","single_PL1_plus")
bridge_position_indices=jnp.asarray([0,2]); bridge_detector_centre=jnp.asarray((np.asarray(manifest_shape)[::-1]-1)/2); bridge_pixel_metric=jnp.asarray(detector_pixel_metric); bridge_detector_affine=jnp.asarray(detector_affine); bridge_fov_xy_m=jnp.asarray(detector_fov_xy_m)
bridge_x_px,bridge_y_px=np.meshgrid(np.arange(manifest_shape[1],dtype=float),np.arange(manifest_shape[0],dtype=float),indexing="xy"); bridge_coordinates_xy=np.stack((bridge_x_px,bridge_y_px),axis=-1); bridge_coordinates_flat_xy=jnp.asarray(bridge_coordinates_xy.reshape(-1,2))
bridge_background_counts=.002*surrogate_dose/np.prod(manifest_shape)
def bridge_request(state_label,measurement):
    return next(frame for frame in manifest if frame.optical_state_label==state_label and frame.measurement==measurement and frame.dac_approach=="forward" and frame.repeat_index==0)
def bridge_spot_geometry(parameters,frame):
    matrix=column.matrix(parameters,jnp.asarray(frame.raw_dac_values),segment="sample_to_detector"); position_map=matrix[jnp.ix_(bridge_position_indices,jnp.arange(4))]
    detector_response=bridge_pixel_metric@bridge_detector_affine@position_map; centre=bridge_detector_centre+detector_response@jnp.asarray(frame.sample_state.as_array())
    pixel_J=detector_response[:,bridge_position_indices]; covariance=calibrated_spot_radius_m**2*(pixel_J@pixel_J.T)+detector.mtf_sigma_px**2*jnp.eye(2)
    return centre,covariance,detector_response
def bridge_spot_model_vector(parameters,frame,coordinate_mask):
    centre,covariance,_=bridge_spot_geometry(parameters,frame); coordinates=bridge_coordinates_flat_xy; mask=jnp.asarray(coordinate_mask,dtype=bool); offsets=coordinates-centre
    weights=jnp.where(mask,jnp.exp(-.5*jnp.einsum("...i,ij,...j->...",offsets,jnp.linalg.inv(covariance),offsets)),0.); total=jnp.sum(weights); centroid=jnp.sum(weights[:,None]*coordinates,axis=0)/total
    centred=coordinates-centroid; moments=jnp.einsum("n,ni,nj->ij",weights,centred,centred)/total
    return jnp.stack((centroid[0],centroid[1],moments[0,0],moments[0,1],moments[1,1]))
def bridge_spot_observation(image_yx,coordinates_xy):
    yi=coordinates_xy[:,1].astype(int); xi=coordinates_xy[:,0].astype(int); weights=np.asarray(image_yx)[yi,xi]-bridge_background_counts
    total=weights.sum(); centroid=(weights[:,None]*coordinates_xy).sum(axis=0)/total; centred=coordinates_xy-centroid; moments=np.einsum("n,ni,nj->ij",weights,centred,centred)/total
    return np.asarray((centroid[0],centroid[1],moments[0,0],moments[0,1],moments[1,1]))
def bridge_fourier_prediction(parameters,frame,g_world_xy):
    matrix=column.matrix(parameters,jnp.asarray(frame.raw_dac_values),segment="sample_to_detector"); real_map=bridge_detector_affine@matrix[jnp.ix_(bridge_position_indices,bridge_position_indices)]
    detector_frequency_xy=jnp.linalg.solve(real_map.T,jnp.asarray(g_world_xy)); return (detector_frequency_xy*bridge_fov_xy_m)[::-1]
def bridge_fourier_observation(image_yx,target_yx):
    vectors=extract_image_observables(np.clip(image_yx,0,None),n_fourier_vectors=12,dc_exclusion_radius_px=2.25).fourier_vectors_yx_px
    if not len(vectors): raise RuntimeError("No non-DC Fourier peak was found in a selected sample frame.")
    return matched_friedel_vector(vectors,target_yx)
def bridge_bootstrap_covariance(image_yx,extractor,floors,seed,n_draws=24):
    rng=np.random.default_rng(seed); mean=np.clip(np.asarray(image_yx,float),0,None); draws=[]
    for _ in range(n_draws):
        realization=rng.poisson(mean).astype(float)+rng.normal(0,surrogate_read_noise,size=mean.shape); draws.append(extractor(realization))
    covariance=np.cov(np.asarray(draws),rowvar=False,ddof=1)+np.diag(np.asarray(floors,float)**2)
    eigenvalues,eigenvectors=np.linalg.eigh(covariance); floor=np.finfo(float).eps*max(1.,float(np.trace(covariance))); return (eigenvectors*np.maximum(eigenvalues,floor))@eigenvectors.T
bridge_blocks=[]; bridge_specs=[]; bridge_nominal_parameters=jnp.asarray(design_parameters)
for state_label in bridge_state_labels:
    sample_frame=bridge_request(state_label,"sample"); projection=np.asarray(view_by_state[state_label].projection_world_from_local); g_world=np.linalg.inv(projection).T@multilayer_g_local_xy
    target=np.asarray(bridge_fourier_prediction(bridge_nominal_parameters,sample_frame,g_world)); image=manifest_counts[sample_frame.frame_id]; observed=bridge_fourier_observation(image,target)
    covariance=bridge_bootstrap_covariance(image,lambda draw,target=target:bridge_fourier_observation(draw,target),(.035,.035),31_000+sample_frame.frame_id)
    bridge_blocks.append(FrameObservationBlock(sample_frame.frame_id,sample_frame.optical_state_index,observed,covariance,("fourier_y_bin","fourier_x_bin"))); bridge_specs.append(("fourier",sample_frame,jnp.asarray(g_world)))
    for measurement in ("position_basis","angle_basis"):
        frame=bridge_request(state_label,measurement); centre,covariance0,response=np.asarray(bridge_spot_geometry(bridge_nominal_parameters,frame)[0]),np.asarray(bridge_spot_geometry(bridge_nominal_parameters,frame)[1]),np.asarray(bridge_spot_geometry(bridge_nominal_parameters,frame)[2])
        offsets=bridge_coordinates_xy-centre; mask=np.einsum("...i,ij,...j->...",offsets,np.linalg.inv(covariance0),offsets)<=25.; coordinates=bridge_coordinates_xy[mask]
        assert mask.shape==tuple(manifest_shape) and np.any(mask) and np.all(np.isfinite(coordinates)), f"Invalid bridge support mask for frame {frame.frame_id}."
        image=manifest_counts[frame.frame_id]; extractor=lambda draw,coordinates=coordinates:bridge_spot_observation(draw,coordinates); observed=extractor(image)
        covariance=bridge_bootstrap_covariance(image,extractor,(.025,.025,.12,.08,.12),32_000+frame.frame_id); covariance[:2,:2]+=response@np.asarray(frame.calibration_covariance)@response.T
        names=("centroid_x_px","centroid_y_px","covariance_xx_px2","covariance_xy_px2","covariance_yy_px2")
        bridge_blocks.append(FrameObservationBlock(frame.frame_id,frame.optical_state_index,observed,covariance,names)); bridge_specs.append(("spot",frame,jnp.asarray(mask.reshape(-1))))
bridge_dataset=HeterogeneousInversionDataset(plan,tuple(bridge_blocks),metadata={"source":"noisy detector frames","covariance":"Poisson/read bootstrap plus propagated calibrated-state covariance","scope":"downstream sample-to-detector optics only"})
def bridge_predict_blocks(parameters):
    predictions=[]
    for kind,frame,auxiliary in bridge_specs:
        predictions.append(bridge_fourier_prediction(parameters,frame,auxiliary) if kind=="fourier" else bridge_spot_model_vector(parameters,frame,auxiliary))
    return tuple(predictions)
diagnostic_names=("IL1.q_bias","IL2.q_bias","IL3.q_bias","PL1.q_bias"); diagnostic_indices=np.asarray([column.parameter_names.index(name) for name in diagnostic_names]); diagnostic_scale=np.full(len(diagnostic_indices),.02)
def bridge_diagnostic_model(u): return bridge_predict_blocks(bridge_nominal_parameters.at[jnp.asarray(diagnostic_indices)].add(jnp.asarray(diagnostic_scale)*u))
def bridge_diagnostic_residual(u): return bridge_dataset.jax_whitened_residuals(bridge_diagnostic_model(u))
bridge_J=np.asarray(jax.jacfwd(bridge_diagnostic_residual)(jnp.zeros(len(diagnostic_indices)))); bridge_svd=jacobian_svd(bridge_J,rtol=1e-2)
print(f"Image-only bridge: {len(bridge_blocks)} measured frame blocks, {bridge_dataset.n_observations} observables; effective rank={bridge_svd.rank}/{len(diagnostic_names)} at rtol=1e-2.")
for order in range(min(2,len(diagnostic_names))):
    direction=bridge_svd.right_singular_vectors[-1-order]; print(f"  weak direction {order+1} (σ={bridge_svd.singular_values[-1-order]:.3g}): "+", ".join(f"{name}={value:+.3f}" for name,value in zip(diagnostic_names,direction,strict=True)))
bridge_column_norms=np.linalg.norm(bridge_J,axis=0); bridge_resolved_local=int(np.argmax(bridge_column_norms)); bridge_fit_names=(diagnostic_names[bridge_resolved_local],); bridge_fit_indices=np.asarray((diagnostic_indices[bridge_resolved_local],)); bridge_fit_scale=np.asarray((diagnostic_scale[bridge_resolved_local],)); bridge_initial=np.asarray((.70,))
print("  physical one-parameter release by whitened column norm: "+", ".join(f"{name}={norm:.3g}" for name,norm in zip(diagnostic_names,bridge_column_norms,strict=True))+f"; selected {bridge_fit_names[0]}")
def bridge_fit_model(u): return bridge_predict_blocks(bridge_nominal_parameters.at[jnp.asarray(bridge_fit_indices)].add(jnp.asarray(bridge_fit_scale)*u))
initial_wrms=float(np.sqrt(np.mean(np.asarray(bridge_dataset.jax_whitened_residuals(bridge_fit_model(jnp.asarray(bridge_initial))))**2)))
bridge_fit=fit_heterogeneous_dataset(bridge_fit_model,bridge_dataset,bridge_initial,bounds=(np.full(1,-2.5),np.full(1,2.5)),parameter_names=tuple(f"scaled:{name}" for name in bridge_fit_names),loss="soft_l1",f_scale=1.,multistart=4,seed=404,max_nfev=250)
bridge_truth_physical=float(np.asarray(design_parameters)[bridge_fit_indices[0]]); bridge_initial_physical=bridge_truth_physical+float(bridge_fit_scale[0]*bridge_initial[0]); bridge_fitted_physical=bridge_truth_physical+float(bridge_fit_scale[0]*bridge_fit.parameters[0]); bridge_standard_error=float(bridge_fit_scale[0]*bridge_fit.parameter_standard_errors[0]); bridge_recovery_z=(bridge_fitted_physical-bridge_truth_physical)/bridge_standard_error
print(f"Offset-start image fit: weighted RMS {initial_wrms:.3f} → {bridge_fit.weighted_rmse:.3f}; local fit rank={bridge_fit.jacobian_rank}/1")
print(f"  {bridge_fit_names[0]:14s} truth={bridge_truth_physical:+.6f} initial={bridge_initial_physical:+.6f} fit={bridge_fitted_physical:+.6f} σ={bridge_standard_error:.2e} z={bridge_recovery_z:+.2f}")
fig,axes=plt.subplots(1,2,figsize=(9.6,3.6),constrained_layout=True); axes[0].semilogy(np.arange(1,len(bridge_svd.singular_values)+1),bridge_svd.singular_values,"o-",color="#0072B2"); axes[0].axhline(bridge_svd.threshold,color="#D55E00",ls="--",label="effective-rank threshold"); axes[0].set(xlabel="singular-value index",ylabel="whitened singular value",title="Image-observable identifiability"); axes[0].legend(frameon=False)
axes[1].axhline(0,color="0.7",lw=1); axes[1].errorbar((0,),(bridge_fitted_physical-bridge_truth_physical,),yerr=(bridge_standard_error,),fmt="o",capsize=4,color="#0072B2",label="fit ±1σ"); axes[1].plot((0,),(bridge_initial_physical-bridge_truth_physical,),"o",color="#D55E00",label="initial error"); axes[1].set(xlim=(-.65,.65),xticks=(0,),xticklabels=(bridge_fit_names[0].replace(".q_bias"," bias"),),ylabel="integrated Larmor-strength error q [dimensionless]",title="Resolved recovery from image observables"); axes[1].text(.03,.06,f"fit error={bridge_fitted_physical-bridge_truth_physical:+.2e}\n1σ={bridge_standard_error:.2e}",transform=axes[1].transAxes,fontsize=8,va="bottom"); axes[1].legend(frameon=False)
bridge_exports=export_ray_figure(fig,GENERATED/"neoarm_image_observable_inversion",metadata={"title":"NEOARM image-observable inversion bridge","frame_ids":[block.frame_id for block in bridge_blocks],"diagnostic_parameter_names":diagnostic_names,"effective_rank":bridge_svd.rank,"rank_rtol":1e-2,"released_parameter":bridge_fit_names[0],"truth":bridge_truth_physical,"estimate":bridge_fitted_physical,"standard_error":bridge_standard_error,"recovery_z":bridge_recovery_z,"scope":"downstream calibrated sample-to-detector surrogate images"}); print("Image-fit exports:",bridge_exports); plt.show()
assert bridge_fit.success and bridge_fit.jacobian_rank==1 and bridge_fit.weighted_rmse<initial_wrms and np.isfinite(bridge_recovery_z) and abs(bridge_recovery_z)<=2


### Synthetic physical-forward Gaussian validation at a chosen magnification

This block is synthetic forward/diagnostic validation; its calibrated sample-plane bases are not required hardware acquisitions. `SIMULATION_MAGNIFICATION` must match one of the configured focused nominal modes (40k, 80k, 150k, 300k, or 600k), and the block re-asserts the stored specimen-conjugacy tolerance `||B||` before any render is resumed. It assembles six physical-model frames at that mode: the registered specimen, a narrow calibrated probe wholly contained in the central perforation, synthetic calibrated sample-plane position and angle inputs, and an IL2 negative/positive wobble constructed around the chosen nominal DAC state. A wider specimen probe is back-propagated to the source, advanced through sliced finite hard-edge lenses, multiplied exactly by the selected cubic specimen, re-expanded on one frozen Gaussian topology, and propagated to the native 256×256 Medipix detector. Both source probes are normalized to the same incident current before one common dose/gain model produces raw noisy counts.

The material grid is derived in world coordinates from the selected `SampleView` crop, its declared support margin, and the inverse-mapped detector footprints over the displayed wobble. Sampling is limited by detector back-projection and the retained certified feature scale (plus interface width when individual interfaces are routed). The accepted centre lattice, Gaussian waist/radius surrogate, spacing, and resulting concrete topology fingerprint are attached to the material frame requests and checkpoint metadata; an environment override may refine this grid but may not coarsen it past the certificate.

The first figure row preserves absolute counts on one shared scale; the second shows explicitly labelled noiseless local contrast for display only; the third supplies certificate-targeted Fourier evidence for material frames and the simulated vacuum/input spectrum for the through-hole probe. Registered multilayer boundaries are overlaid on material images. Scale bars are recomputed from each frame's own transfer Jacobian, and titles report the changed raw-DAC channel and signed `ΔDAC` rather than only a worst-case norm. Finally, extracted physical image observables from the calibrated position/angle frames are fed into a matched one-parameter downstream hard-edge recovery only when the local sensitivity is finite. That is a deliberately narrow preflight check; it is never substituted for the later full-column global fit. Rendering is GPU-required, atomic, and resumable. The complete native imaging and diffraction manifests retain separate opt-in checkpoints.

The six production diffraction frames use solved **native hard-edge** projector modes, never an ideal Fourier relay. For each original request the binder preserves CL1, CL3, C-mini, IL2, and IL3 exactly, adjusts only IL1 and PL1 to enforce A approximately zero and B approximately L R, and carries the original approach-start displacement onto the new endpoint. Camera lengths from 0.30--0.40 m put first-order Si {111} spots at roughly 44--58 pixels on this detector while leaving at least 15% angular-Nyquist margin for every detector pixel and arbitrary Larmor rotation. The old ideal-relay files, if present, are visibility-only and are excluded from every readiness or acceptance decision.

In [ ]:
SIMULATION_MAGNIFICATION=150_000; physical_parameters=column.nominal_parameters.copy()
physical_allowed_magnifications=tuple(int(value) for value in magnifications); physical_b_norm_atol=float(experiment_cfg["nominal_conjugacy_b_atol_m"])
if isinstance(SIMULATION_MAGNIFICATION,(bool,np.bool_)) or not np.isfinite(SIMULATION_MAGNIFICATION) or int(SIMULATION_MAGNIFICATION)!=SIMULATION_MAGNIFICATION or int(SIMULATION_MAGNIFICATION) not in physical_allowed_magnifications: raise ValueError(f"SIMULATION_MAGNIFICATION must be one of {physical_allowed_magnifications}, got {SIMULATION_MAGNIFICATION!r}.")
SIMULATION_MAGNIFICATION=int(SIMULATION_MAGNIFICATION); physical_mode_index=physical_allowed_magnifications.index(SIMULATION_MAGNIFICATION)
assert nominal_design.all_targets_feasible and nominal_design.maximum_specimen_conjugacy_b_norm<=physical_b_norm_atol*(1+1e-12)
physical_b_norm=float(nominal_design.specimen_conjugacy_b_norms[physical_mode_index])
if not np.isfinite(physical_b_norm) or physical_b_norm>physical_b_norm_atol*(1+1e-12): raise RuntimeError(f"Chosen mode {SIMULATION_MAGNIFICATION:,}× violates the stored specimen-conjugacy tolerance: ||B||={physical_b_norm:.3e} m > {physical_b_norm_atol:.3e} m.")
physical_nominal_label=next(state.label for state in plan.states if state.role=="nominal" and state.target_magnification==SIMULATION_MAGNIFICATION)
def physical_nominal_request(measurement): return next(frame for frame in manifest if frame.optical_state_label==physical_nominal_label and frame.measurement==measurement and frame.dac_approach=="forward" and frame.repeat_index==0)
physical_sample_request=physical_nominal_request("sample"); physical_vacuum_request=physical_nominal_request("perforation"); physical_position_request=physical_nominal_request("position_basis"); physical_angle_request=physical_nominal_request("angle_basis"); physical_reference_dacs=np.asarray(physical_sample_request.raw_dac_values); physical_nominal_view=view_by_state[physical_nominal_label]
physical_wobble_channel=plan.channel_names.index("IL2"); physical_wobble_step=float(perturbations[physical_wobble_channel]); physical_virtual_frame_base=1_000_000+100*physical_mode_index
def chosen_mode_wobble_request(sign):
    direction="minus" if sign<0 else "plus"; dacs=physical_reference_dacs.copy(); dacs[physical_wobble_channel]+=sign*physical_wobble_step; approach=np.asarray(physical_sample_request.dac_approach_start_values).copy(); approach[physical_wobble_channel]+=sign*physical_wobble_step
    effective=float(np.sqrt(abs(np.linalg.det(detector_affine@np.asarray(column.sample_position_jacobian(physical_parameters,dacs))))))
    return replace(physical_sample_request,frame_id=physical_virtual_frame_base+(0 if sign<0 else 1),optical_state_label=f"physical_{SIMULATION_MAGNIFICATION}_IL2_{direction}",raw_dac_values=tuple(dacs),predicted_effective_magnification=effective,dac_approach_start_values=tuple(approach),repeat_index=0)
physical_wobble_minus=chosen_mode_wobble_request(-1.); physical_wobble_plus=chosen_mode_wobble_request(1.)
physical_detector=detector; physical_width_over=(1.25,1.25)  # native 256×256 Medipix contract
physical_pixel_xy=np.asarray(physical_detector.pixel_size_yx_m)[::-1]; physical_detector_half_xy=.5*np.asarray(physical_detector.shape_yx)[::-1]*physical_pixel_xy; physical_detector_corners=np.asarray(((-physical_detector_half_xy[0],-physical_detector_half_xy[1]),(physical_detector_half_xy[0],-physical_detector_half_xy[1]),(physical_detector_half_xy[0],physical_detector_half_xy[1]),(-physical_detector_half_xy[0],physical_detector_half_xy[1])))
physical_seed_view=physical_nominal_view; physical_seed_scale_m=float(physical_seed_view.selection.routed_feature_scale_m or 8*physical_seed_view.selection.selected_spacing_m); physical_preliminary_step_m=min(physical_seed_view.selection.specimen_pixel_min_m/2,physical_seed_scale_m/10,atlas.certificate.interface_width_m/3 if physical_seed_view.selection.feature_family=="individual_layer_widths" else np.inf); physical_exact_material_requests=(physical_sample_request,physical_wobble_minus,physical_wobble_plus); physical_exact_jacobians=np.stack([detector_affine@np.asarray(column.sample_position_jacobian(physical_parameters,np.asarray(request.raw_dac_values))) for request in physical_exact_material_requests]); physical_preliminary_footprints=np.concatenate([physical_detector_corners@np.linalg.inv(request_J).T+sample_origin_xy for request_J in physical_exact_jacobians]); physical_preliminary_span=physical_preliminary_footprints.max(axis=0)-physical_preliminary_footprints.min(axis=0); sample_illumination_radius_m=max(2*physical_seed_scale_m,.22*float(np.min(physical_preliminary_span)),12*physical_preliminary_step_m)
physical_gaussian_support_radius_m=2.5*sample_illumination_radius_m; physical_detector_half_specimen_span=.5*float(np.min(physical_preliminary_span)); physical_seed_world_to_local=np.linalg.inv(np.asarray(physical_seed_view.projection_world_from_local)); physical_footprint_local_offsets=(physical_preliminary_footprints-sample_origin_xy)@physical_seed_world_to_local.T; physical_support_world_offsets=np.asarray(((-physical_gaussian_support_radius_m,-physical_gaussian_support_radius_m),(physical_gaussian_support_radius_m,-physical_gaussian_support_radius_m),(physical_gaussian_support_radius_m,physical_gaussian_support_radius_m),(-physical_gaussian_support_radius_m,physical_gaussian_support_radius_m))); physical_support_local_offsets=physical_support_world_offsets@physical_seed_world_to_local.T; physical_view_gaussian_margin_m=max(.50e-9,float(np.max(np.max(np.abs(physical_support_local_offsets),axis=0)-np.max(np.abs(physical_footprint_local_offsets),axis=0)))+2*physical_preliminary_step_m)
physical_nominal_view=select_sample_view(atlas,transfer_jacobian_xy=physical_exact_jacobians[0],transfer_jacobian_envelope_xy=physical_exact_jacobians[1:],detector_shape_yx=physical_detector.shape_yx,detector_pixel_size_yx_m=physical_detector.pixel_size_yx_m,sample_centre_xy_m=sample_origin_xy,sample_rotation_rad=sample_rotation,sample_tilt_xy_rad=sample_tilt_xy,region_name=physical_seed_view.selection.selected_region_name,gaussian_radius_min_m=3.4e-9,lens_wobble_margin_m=.15e-9,calibrated_input_margin_m=1.10e-9,gaussian_support_margin_m=physical_view_gaussian_margin_m,pose_uncertainty_margin_m=.10e-9,interpolation_halo_margin_m=.05e-9,boundary=None,z=beam_cfg["sample_z_m"])
if physical_nominal_view.selection.selected_region_name!=physical_seed_view.selection.selected_region_name or not physical_nominal_view.selection.meets_resolution: raise RuntimeError("The exact chosen-mode/wobble union has no covering certified SampleView at the required LOD.")
selection_bounds=np.asarray(physical_nominal_view.selection.local_bounds_xyxy_m); selection_margin_m=float(physical_nominal_view.selection.total_margin_m); selection_support_bounds=selection_bounds+np.asarray((-selection_margin_m,selection_margin_m,-selection_margin_m,selection_margin_m))
projection=np.asarray(physical_nominal_view.projection_world_from_local); view_centre=np.asarray(physical_nominal_view.centre_xy_m)
selection_local_corners=np.asarray(((selection_bounds[0],selection_bounds[2]),(selection_bounds[1],selection_bounds[2]),(selection_bounds[1],selection_bounds[3]),(selection_bounds[0],selection_bounds[3]))); selection_world_corners=selection_local_corners@projection.T+view_centre
selection_support_local_corners=np.asarray(((selection_support_bounds[0],selection_support_bounds[2]),(selection_support_bounds[1],selection_support_bounds[2]),(selection_support_bounds[1],selection_support_bounds[3]),(selection_support_bounds[0],selection_support_bounds[3]))); selection_support_world_corners=selection_support_local_corners@projection.T+view_centre
physical_selection_world_bounds_xyxy_m=(float(selection_world_corners[:,0].min()),float(selection_world_corners[:,0].max()),float(selection_world_corners[:,1].min()),float(selection_world_corners[:,1].max())); physical_support_world_bounds_xyxy_m=(float(selection_support_world_corners[:,0].min()),float(selection_support_world_corners[:,0].max()),float(selection_support_world_corners[:,1].min()),float(selection_support_world_corners[:,1].max()))
physical_request_J_by_frame_id={}
def physical_request_J(request):
    request_J=physical_request_J_by_frame_id.get(request.frame_id)
    if request_J is None:
        request_J=detector_affine@np.asarray(column.sample_position_jacobian(physical_parameters,np.asarray(request.raw_dac_values))); physical_request_J_by_frame_id[request.frame_id]=request_J
    return request_J

physical_material_requests=(physical_sample_request,physical_wobble_minus,physical_wobble_plus); physical_footprint_points=[]; physical_backprojected_pixels=[]
for request in physical_material_requests:
    request_J=physical_request_J(request); physical_footprint_points.append(physical_detector_corners@np.linalg.inv(request_J).T+sample_origin_xy)
    local_pixel_steps=np.linalg.inv(request_J)@np.diag(physical_pixel_xy); physical_backprojected_pixels.append(np.linalg.svd(local_pixel_steps,compute_uv=False).min())
physical_footprint_points=np.concatenate(physical_footprint_points); physical_footprint_min=physical_footprint_points.min(axis=0); physical_footprint_max=physical_footprint_points.max(axis=0); physical_footprint_world_bounds_xyxy_m=(float(physical_footprint_min[0]),float(physical_footprint_max[0]),float(physical_footprint_min[1]),float(physical_footprint_max[1]))
physical_retained_scale_m=float(physical_nominal_view.selection.routed_feature_scale_m or 8*physical_nominal_view.selection.selected_spacing_m); physical_spacing_requirements=[min(physical_backprojected_pixels)/2,physical_retained_scale_m/10]
if physical_nominal_view.selection.feature_family=="individual_layer_widths": physical_spacing_requirements.append(atlas.certificate.interface_width_m/3)
physical_required_step_m=min(physical_spacing_requirements); physical_crop_support_radius_m=float(np.max(np.linalg.norm(selection_support_world_corners-sample_origin_xy,axis=1))); physical_footprint_radius_m=float(np.max(np.linalg.norm(physical_footprint_points-sample_origin_xy,axis=1))); physical_footprint_span_xy=physical_footprint_max-physical_footprint_min
sample_illumination_radius_m=max(sample_illumination_radius_m,12*physical_required_step_m); physical_gaussian_support_radius_m=2.5*sample_illumination_radius_m; physical_support_margin_m=0.0  # selection already carries the exact footprint/support/pose/halo margins
physical_gaussian_support_min=sample_origin_xy-physical_gaussian_support_radius_m; physical_gaussian_support_max=sample_origin_xy+physical_gaussian_support_radius_m; physical_grid_min=np.minimum(physical_footprint_min,physical_gaussian_support_min)-physical_support_margin_m; physical_grid_max=np.maximum(physical_footprint_max,physical_gaussian_support_max)+physical_support_margin_m; physical_longest_grid_span=float(np.max(physical_grid_max-physical_grid_min)); physical_grid_step_target_m=physical_required_step_m
physical_grid_override=os.environ.get("NEOARM_PHYSICAL_GRID_SIDE","").strip()
if physical_grid_override:
    override_side=int(physical_grid_override)
    if override_side<2: raise ValueError("NEOARM_PHYSICAL_GRID_SIDE must be at least 2.")
    override_step=physical_longest_grid_span/(override_side-1)
    if override_step>physical_required_step_m*(1+1e-12): raise ValueError(f"NEOARM_PHYSICAL_GRID_SIDE={override_side} would coarsen the certified {physical_required_step_m*1e9:.4f} nm grid; it may only refine it.")
    physical_grid_step_target_m=override_step
physical_grid_indices_min=np.floor((physical_grid_min-sample_origin_xy)/physical_grid_step_target_m).astype(int); physical_grid_indices_max=np.ceil((physical_grid_max-sample_origin_xy)/physical_grid_step_target_m).astype(int); physical_sample_x=sample_origin_xy[0]+np.arange(physical_grid_indices_min[0],physical_grid_indices_max[0]+1)*physical_grid_step_target_m; physical_sample_y=sample_origin_xy[1]+np.arange(physical_grid_indices_min[1],physical_grid_indices_max[1]+1)*physical_grid_step_target_m
physical_grid_step_m=float(np.diff(physical_sample_x).mean()); physical_grid_shape_yx=(len(physical_sample_y),len(physical_sample_x)); physical_topology_certified=physical_grid_step_m<=physical_required_step_m*(1+1e-12); physical_beamlet_count=int(np.prod(physical_grid_shape_yx))
if not physical_topology_certified: raise RuntimeError("The chosen physical Gaussian grid failed its sampling certificate.")
physical_grid_world_corners=np.asarray(((physical_sample_x[0],physical_sample_y[0]),(physical_sample_x[-1],physical_sample_y[0]),(physical_sample_x[-1],physical_sample_y[-1]),(physical_sample_x[0],physical_sample_y[-1]))); physical_grid_local_corners=np.asarray(physical_nominal_view.world_to_local(jnp.asarray(physical_grid_world_corners))); physical_view_domain=np.asarray(physical_nominal_view.domain_bounds_xyxy_m); physical_view_covers_grid=bool(physical_nominal_view.boundary=="periodic" or (physical_grid_local_corners[:,0].min()>=physical_view_domain[0]-1e-15 and physical_grid_local_corners[:,0].max()<=physical_view_domain[1]+1e-15 and physical_grid_local_corners[:,1].min()>=physical_view_domain[2]-1e-15 and physical_grid_local_corners[:,1].max()<=physical_view_domain[3]+1e-15))
if not physical_view_covers_grid: raise RuntimeError(f"Frozen SampleView source bounds {tuple(physical_view_domain)} do not cover the concrete world-grid corners after pose inversion; got local extrema {np.array2string(physical_grid_local_corners.min(axis=0),precision=16)} to {np.array2string(physical_grid_local_corners.max(axis=0),precision=16)}.")
physical_xx,physical_yy=np.meshgrid(physical_sample_x,physical_sample_y,indexing="xy"); physical_centres_xy=np.stack((physical_xx.ravel(),physical_yy.ravel()),axis=-1); physical_waists=np.full(physical_beamlet_count,physical_width_over[0]*physical_grid_step_m)
physical_grid_bounds_xyxy_m=(float(physical_sample_x[0]-.5*physical_grid_step_m),float(physical_sample_x[-1]+.5*physical_grid_step_m),float(physical_sample_y[0]-.5*physical_grid_step_m),float(physical_sample_y[-1]+.5*physical_grid_step_m)); physical_centres_digest=hashlib.sha256(np.asarray(physical_centres_xy,dtype=np.float64).tobytes()).hexdigest(); physical_waist_digest=hashlib.sha256(np.asarray(physical_waists,dtype=np.float64).tobytes()).hexdigest()
physical_topology_id=frozen_topology_fingerprint(specimen_identity=config["sample"]["identity"],atlas_version=atlas.certificate.version,region_name=physical_nominal_view.selection.selected_region_name or "global_atlas",level_index=physical_nominal_view.selection.level_index,crop_bounds_xyxy_m=selection_bounds,sample_translation_xy_m=physical_nominal_view.selection.sample_translation_xy_m,projection_world_from_local=projection,route="sample_plane_reexpansion",selected_spacing_m=physical_nominal_view.selection.selected_spacing_m,routed_feature_scale_m=physical_retained_scale_m,centres_per_period=physical_retained_scale_m/physical_grid_step_m,overlap_factor=physical_width_over[0],interpolation_halo_knots=physical_nominal_view.selection.interpolation_halo_knots,topology_version="chosen-mode-concrete-v9-gpu",centres_xy_m=physical_centres_xy,waist_m=physical_waists)
physical_material_requests=tuple(replace(request,feature_family=physical_nominal_view.selection.feature_family,selected_region_name=physical_nominal_view.selection.selected_region_name,sample_level_index=physical_nominal_view.selection.level_index,sample_translation_xy_m=physical_nominal_view.selection.sample_translation_xy_m,sample_crop_bounds_xyxy_m=physical_nominal_view.selection.local_bounds_xyxy_m,selected_spacing_m=physical_nominal_view.selection.selected_spacing_m,interaction_route="sample_plane_reexpansion",frozen_topology_id=physical_topology_id) for request in physical_material_requests)  # crop/LOD remain atlas-local; the concrete world-grid lives in topology metadata
physical_sample_request,physical_wobble_minus,physical_wobble_plus=physical_material_requests
physical_demo_specs=(("certified sample",physical_sample_request),("through-hole probe",physical_vacuum_request),("calibrated +x",physical_position_request),("calibrated +tilt",physical_angle_request),("IL2 − wobble",physical_wobble_minus),("IL2 + wobble",physical_wobble_plus)); physical_demo_requests=tuple(request for _,request in physical_demo_specs)
physical_null_requests=tuple(replace(request,frame_id=physical_virtual_frame_base+10+index,optical_state_label=f"{request.optical_state_label}_matched_blank",measurement="matched_blank",feature_family="vacuum_blank",selected_region_name=None,sample_level_index=None,sample_translation_xy_m=(0.,0.),sample_crop_bounds_xyxy_m=None,selected_spacing_m=None,interaction_route="vacuum",frozen_topology_id=None) for index,request in enumerate(physical_material_requests)); physical_null_by_material_id={material.frame_id:null for material,null in zip(physical_material_requests,physical_null_requests,strict=True)}; physical_checkpoint_requests=physical_demo_requests+physical_null_requests
physical_topology_metadata={"topology_id":physical_topology_id,"topology_version":"chosen-mode-concrete-v9-gpu","grid_bounds_xyxy_m":list(physical_grid_bounds_xyxy_m),"selection_local_bounds_xyxy_m":selection_bounds.tolist(),"selection_world_bounds_xyxy_m":list(physical_selection_world_bounds_xyxy_m),"support_world_bounds_xyxy_m":list(physical_support_world_bounds_xyxy_m),"footprint_world_bounds_xyxy_m":list(physical_footprint_world_bounds_xyxy_m),"gaussian_support_radius_m":physical_gaussian_support_radius_m,"grid_origin_xy_m":[float(physical_sample_x[0]),float(physical_sample_y[0])],"grid_terminal_xy_m":[float(physical_sample_x[-1]),float(physical_sample_y[-1])],"grid_spacing_xy_m":[float(physical_grid_step_m),float(physical_grid_step_m)],"grid_shape_yx":list(physical_grid_shape_yx),"gaussian_waist_m":float(physical_waists[0]),"beamlet_count":physical_beamlet_count,"centres_sha256":physical_centres_digest,"waists_sha256":physical_waist_digest}
def unit_current_sample_gaussian(waist_m):
    amplitude=np.sqrt(2/np.pi)/waist_m
    return make_gaussian(x=np.full(1,sample_origin_xy[0]),y=np.full(1,sample_origin_xy[1]),dx=np.zeros(1),dy=np.zeros(1),z=np.full(1,column.sample_z_m),voltage=np.full(1,beam_cfg["voltage_v"]),amp=np.full(1,amplitude),waist_x=np.full(1,waist_m),waist_y=np.full(1,waist_m))
upstream_matrix=np.asarray(column.matrix(physical_parameters,physical_sample_request.raw_dac_values,segment="source_to_sample"))
def backpropagate_nominal_sample_beam(sample_beam): return apply_transfer_matrix_to_gaussians(sample_beam,np.linalg.inv(upstream_matrix),z_out_m=physical_parameters[0],axial_path_m=physical_parameters[0]-column.sample_z_m)
physical_source_beam=backpropagate_nominal_sample_beam(unit_current_sample_gaussian(sample_illumination_radius_m)); physical_vacuum_probe_waist_m=min(3e-9,.15*atlas.certificate.perforation_diameter_m); physical_vacuum_source_beam=backpropagate_nominal_sample_beam(unit_current_sample_gaussian(physical_vacuum_probe_waist_m))
physical_max_vacuum_displacement_m=max(np.hypot(request.sample_state.x,request.sample_state.y) for request in (physical_vacuum_request,physical_position_request,physical_angle_request)); physical_vacuum_clearance_m=.5*atlas.certificate.perforation_diameter_m-atlas.certificate.interface_width_m
if physical_max_vacuum_displacement_m+4*physical_vacuum_probe_waist_m>=physical_vacuum_clearance_m: raise RuntimeError("The calibrated through-hole probe does not have four-waist clearance from the certified perforation edge.")
physical_slice_envelope_dacs=np.vstack((slice_envelope_dacs,[request.raw_dac_values for request in physical_demo_requests],[request.dac_approach_start_values for request in physical_demo_requests])); physical_slice_safety_report=assess_gaussian_slice_safety(column,physical_parameters,physical_slice_envelope_dacs); physical_slice_base=2**int(np.ceil(np.log2(physical_slice_safety_report.recommended_slices_per_lens))); physical_slice_candidates=tuple(physical_slice_base*2**power for power in range(4)); physical_slice_probe_detector=replace(physical_detector,shape_yx=(9,9),mtf_sigma_px=0.); physical_slice_test_dacs=np.unique(np.asarray([request.raw_dac_values for request in physical_demo_requests]),axis=0)
physical_slice_fields=[]
for slice_count in physical_slice_candidates:
    source_fields=[]
    for test_dacs in physical_slice_test_dacs:
        for source_beam in (physical_source_beam,physical_vacuum_source_beam):
            propagated_probe=propagate_gaussians_through_column(source_beam,column,physical_parameters,test_dacs,segment="full",slices_per_lens=slice_count); source_fields.append(render_gaussian_detector_plane(propagated_probe,physical_slice_probe_detector,tile_rows=9,beam_batch_size=16,method=FORWARD_EVALUATION_METHOD).field_yx)
    physical_slice_fields.append(np.stack(source_fields))
def phase_registered_field_error(old_field,new_field):
    overlap=np.vdot(new_field.ravel(),old_field.ravel()); registration=1+0j if abs(overlap)<=np.finfo(float).tiny else overlap/abs(overlap); field_error=float(np.linalg.norm(registration*new_field-old_field)/max(np.linalg.norm(old_field),np.finfo(float).tiny)); intensity_error=float(np.linalg.norm(np.abs(new_field)**2-np.abs(old_field)**2)/max(np.linalg.norm(np.abs(old_field)**2),np.finfo(float).tiny)); return max(field_error,intensity_error),float(np.angle(registration))
physical_slice_transition_errors=[]; physical_slice_transition_phase_offsets=[]
for old_fields,new_fields in zip(physical_slice_fields[:-1],physical_slice_fields[1:],strict=True):
    comparisons=[phase_registered_field_error(old_field,new_field) for old_field,new_field in zip(old_fields,new_fields,strict=True)]; physical_slice_transition_errors.append(max(value[0] for value in comparisons)); physical_slice_transition_phase_offsets.append(tuple(value[1] for value in comparisons))
physical_slice_transition_errors=tuple(physical_slice_transition_errors); physical_slice_transition_phase_offsets=tuple(physical_slice_transition_phase_offsets); physical_slice_tolerance=1e-8; physical_slices_per_lens=None
for transition_index in range(len(physical_slice_transition_errors)-1):
    if max(physical_slice_transition_errors[transition_index:transition_index+2])<=physical_slice_tolerance: physical_slices_per_lens=physical_slice_candidates[transition_index]; break
if physical_slices_per_lens is None: raise RuntimeError(f"Metaplectic complex-field convergence failed for slice counts {physical_slice_candidates}: {physical_slice_transition_errors}.")
physical_selected_slice_report=assess_gaussian_slice_safety(column,physical_parameters,physical_slice_envelope_dacs,slices_per_lens=physical_slices_per_lens); assert physical_selected_slice_report.requested_is_safe
physical_detector_interactions=physical_beamlet_count*int(np.prod(physical_detector.shape_yx)); physical_bundle_mib=physical_beamlet_count*160/2**20; physical_estimated_eval_s=physical_detector_interactions/measured_interactions_per_second
print(f"Physical footprint-derived topology: {physical_grid_shape_yx[1]}×{physical_grid_shape_yx[0]}={physical_beamlet_count:,} beamlets, Δ={physical_grid_step_m*1e9:.4f} nm, specimen waist={sample_illumination_radius_m*1e9:.2f} nm, Gaussian support={physical_gaussian_support_radius_m*1e9:.2f} nm, ||B||={physical_b_norm:.3e} m, certified={physical_topology_certified}\n  topology={physical_topology_id}")
print(f"Metaplectic slice convergence {physical_slice_candidates}: per-case globally phase-registered complex-field/intensity transitions={physical_slice_transition_errors}; accepted={physical_slices_per_lens}")
print(f"GPU forward contract: {FORWARD_EVALUATION_METHOD} on {GPU_DEVICES}; {physical_detector_interactions:,} beamlet-pixel interactions/frame, bundle arrays≈{physical_bundle_mib:.1f} MiB, measured-kernel estimate≈{physical_estimated_eval_s:.2f} s/frame (excluding propagation/re-expansion and compilation).")
def render_physical_demo(request):
    sample_view=physical_nominal_view if request.interaction_route!="vacuum" else None; source_beam=physical_source_beam if sample_view is not None or request.measurement=="matched_blank" else physical_vacuum_source_beam
    return predict_frame(column,physical_parameters,request,physical_detector,source_beam,sample_view=sample_view,sample_x_coords_m=physical_sample_x if sample_view is not None else None,sample_y_coords_m=physical_sample_y if sample_view is not None else None,slices_per_lens=physical_slices_per_lens,width_over_step_yx=physical_width_over,tile_rows=16,beam_batch_size=128,evaluation_method=FORWARD_EVALUATION_METHOD)
physical_demo_checkpoint=render_frame_sequence_checkpointed(physical_checkpoint_requests,physical_detector,render_physical_demo,GENERATED/f"physical_checkpoint_{SIMULATION_MAGNIFICATION//1000}k_{physical_detector.shape_yx[1]}px_{physical_grid_shape_yx[1]}x{physical_grid_shape_yx[0]}_v11_gpu",render_signature=f"hard-edge-cubic-reexpand-v11-gpu-matched-blank-{physical_topology_id}-slices{physical_slices_per_lens}",render_metadata={"config_sha256":hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),"parameters":physical_parameters,"certificate":atlas.certificate.version,"sample_illumination_radius_m":sample_illumination_radius_m,"vacuum_probe_waist_m":physical_vacuum_probe_waist_m,"sample_grid_shape_yx":physical_grid_shape_yx,"sample_x_bounds_m":[physical_sample_x[0],physical_sample_x[-1]],"sample_y_bounds_m":[physical_sample_y[0],physical_sample_y[-1]],"sample_grid_step_m":physical_grid_step_m,"support_margin_m":physical_support_margin_m,"slices_per_lens":physical_slices_per_lens,"slice_candidates":physical_slice_candidates,"slice_complex_field_relative_errors":physical_slice_transition_errors,"slice_global_phase_offsets_rad":physical_slice_transition_phase_offsets,"slice_phase_safety":physical_selected_slice_report.human_summary(),"beamlet_pixel_interactions_per_frame":physical_detector_interactions,"width_over_step_yx":physical_width_over,"frame_topology_ids":{str(request.frame_id):request.frozen_topology_id for request in physical_checkpoint_requests},"matched_blank_frame_by_material_frame":{str(material.frame_id):null.frame_id for material,null in zip(physical_material_requests,physical_null_requests,strict=True)},"mode_b_norm_m":physical_b_norm,"nominal_b_norm_atol_m":physical_b_norm_atol,"intended_count_model":{"dose_electrons":surrogate_dose,"read_noise_e":surrogate_read_noise,"material_seed_rule":"51000 + frame_id","blank_seed_rule":"61000 + frame_id","incident_sum_rule":"unit-current analytic integral divided by optical pixel area"},"gpu_kernel_required":FORWARD_EVALUATION_METHOD=="gpu_kernel_required",**physical_topology_metadata,**FORWARD_RUNTIME_METADATA},store_sample_fields=True,storage_dtype="float32",max_new_frames=None if RUN_PHYSICAL_DEMO_SUBSET else 0)
print(f"Physical checkpoint: newly rendered={physical_demo_checkpoint.rendered_frame_ids}, resumed={physical_demo_checkpoint.resumed_frame_ids}, pending={physical_demo_checkpoint.pending_frame_ids}")
if physical_demo_checkpoint.complete:
    physical_images=np.stack([physical_demo_checkpoint.record(request.frame_id).load_intensity() for request in physical_demo_requests]); physical_null_images={material.frame_id:physical_demo_checkpoint.record(null.frame_id).load_intensity() for material,null in zip(physical_material_requests,physical_null_requests,strict=True)}; physical_optical_pixel_area_m2=float(np.prod(physical_detector.pixel_size_yx_m)/abs(np.linalg.det(physical_detector.affine_xy))); physical_incident_sum=1/physical_optical_pixel_area_m2
    physical_count_images=np.stack([simulate_detector_counts(image,dose_electrons=surrogate_dose,incident_intensity_sum=physical_incident_sum,read_noise_e=surrogate_read_noise,seed=51_000+request.frame_id) for image,request in zip(physical_images,physical_demo_requests,strict=True)]); physical_null_count_images={material.frame_id:simulate_detector_counts(physical_null_images[material.frame_id],dose_electrons=surrogate_dose,incident_intensity_sum=physical_incident_sum,read_noise_e=surrogate_read_noise,seed=61_000+null.frame_id) for material,null in zip(physical_material_requests,physical_null_requests,strict=True)}; physical_observables=[extract_image_observables(np.clip(image,0,None),n_fourier_vectors=12,dc_exclusion_radius_px=3.) for image in physical_count_images]; physical_observables_by_id={request.frame_id:obs for request,obs in zip(physical_demo_requests,physical_observables,strict=True)}
    physical_vmax=float(np.percentile(physical_count_images,99.8)); physical_local_contrast=[]
    for image in physical_images:
        smooth=ndimage.gaussian_filter(image,4.0); floor=.02*float(np.max(smooth)); physical_local_contrast.append((image-smooth)/np.maximum(smooth,floor))
    physical_local_contrast=np.stack(physical_local_contrast); material_indices=[index for index,request in enumerate(physical_demo_requests) if request.interaction_route!="vacuum"]; contrast_limit=max(float(np.percentile(np.abs(physical_local_contrast[material_indices]),99.5)),1e-6); physical_pixel_xy=np.asarray(physical_detector.pixel_size_yx_m)[::-1]; physical_fov_xy=physical_pixel_xy*np.asarray(physical_detector.shape_yx)[::-1]; physical_fft_centre=np.asarray(physical_detector.shape_yx)//2
    def physical_control_label(request):
        delta=np.asarray(request.raw_dac_values)-physical_reference_dacs; changed=np.flatnonzero(np.abs(delta)>1e-12)
        return "nominal optics (ΔDAC=0)" if not len(changed) else ", ".join(f"{plan.channel_names[index]} ΔDAC={delta[index]:+.4f}" for index in changed)
    def physical_scale_bar(request):
        request_J=physical_request_J(request); specimen_per_horizontal_pixel=np.linalg.norm(np.linalg.inv(request_J)@np.asarray((physical_detector.pixel_size_yx_m[1],0.))); candidates=np.asarray((.2,.5,1.,2.,5.,10.,20.,50.,100.,200.,500.)); valid=candidates[candidates*1e-9/specimen_per_horizontal_pixel<=.24*physical_detector.shape_yx[1]]; length_nm=float(valid[-1] if len(valid) else candidates[0]); return length_nm,length_nm*1e-9/specimen_per_horizontal_pixel
    physical_dc_exclusion_radius_px=3.0
    def physical_certificate_targets(request):
        family=physical_nominal_view.selection.feature_family
        if family=="silicon_lattice_vectors":
            spacing=float(atlas.silicon_lattice_region.lattice_spacing_m); labels=("Si g1","Si g2"); local_reciprocal=np.asarray(((1/spacing,0.),(0.,1/spacing)))
        elif family=="individual_layer_widths":
            spacing=float(atlas.certificate.multilayer_period_m); labels=("23 nm fundamental","10/13 nm duty-cycle harmonic"); local_reciprocal=np.asarray(((1/spacing,0.),(2/spacing,0.)))
        else:
            spacing=float(atlas.certificate.multilayer_period_m); labels=("multilayer period",); local_reciprocal=np.asarray(((1/spacing,0.),))
        projection=np.asarray(physical_nominal_view.projection_world_from_local); world_reciprocal=(np.linalg.inv(projection).T@local_reciprocal.T).T; request_J=physical_request_J(request); detector_reciprocal=np.linalg.solve(request_J.T,world_reciprocal.T).T; return labels,(detector_reciprocal*physical_fov_xy)[:,::-1]
    def physical_certificate_evidence(request,counts,matched_blank=None):
        counts=np.clip(np.asarray(counts,float),0,None); window=np.outer(np.hanning(counts.shape[0]),np.hanning(counts.shape[1])); yy,xx=np.indices(counts.shape,dtype=float)
        if request.interaction_route=="vacuum":
            total=max(float(counts.sum()),1.); variance=float(np.sum(window**2*(counts+surrogate_read_noise**2))/total**2); spectrum=np.abs(np.fft.fftshift(np.fft.fft2(window*(counts/total-counts.mean()/total))))/max(np.sqrt(variance),np.finfo(float).tiny); return spectrum,np.empty((0,2)),np.empty(0),(),np.empty(0,dtype=bool)
        if matched_blank is None: raise RuntimeError("Material certificate evidence requires a same-optics broad-beam blank.")
        blank=np.clip(np.asarray(matched_blank,float),0,None); sample_total=max(float(counts.sum()),1.); blank_total=max(float(blank.sum()),1.); difference=counts/sample_total-blank/blank_total; variance=float(np.sum(window**2*((counts+surrogate_read_noise**2)/sample_total**2+(blank+surrogate_read_noise**2)/blank_total**2))); noise=max(np.sqrt(variance),np.finfo(float).tiny); spectrum=np.abs(np.fft.fftshift(np.fft.fft2(window*difference)))/noise
        centre=np.asarray(counts.shape)//2; fy,fx=np.indices(counts.shape); spectrum[np.hypot(fy-centre[0],fx-centre[1])<=physical_dc_exclusion_radius_px]=np.nan; labels,targets_yx=physical_certificate_targets(request); scores=[]; eligible=[]
        for vector_yx in targets_yx:
            inside=bool(np.linalg.norm(vector_yx)>physical_dc_exclusion_radius_px+.75 and np.all(np.abs(vector_yx)<.5*np.asarray(counts.shape)-1)); eligible.append(inside)
            if not inside: scores.append(np.nan); continue
            phase=np.exp(-2j*np.pi*(vector_yx[0]*yy/counts.shape[0]+vector_yx[1]*xx/counts.shape[1])); scores.append(float(abs(np.sum(window*difference*phase))/noise))
        return spectrum,targets_yx,np.asarray(scores),labels,np.asarray(eligible,dtype=bool)
    def overlay_registered_boundaries(ax,request):
        region_name=physical_nominal_view.selection.selected_region_name
        if region_name is None: return
        region=atlas.region(region_name)
        if region.kind!="multilayer": return
        request_J=physical_request_J(request); detector_centre=(np.asarray(physical_detector.shape_yx)[::-1]-1)/2; x_start=region.bounds_xyxy_m[0]
        for boundary in atlas.certificate.layer_boundary_coordinates_m:
            local=np.asarray(((x_start+boundary,region.bounds_xyxy_m[2]),(x_start+boundary,region.bounds_xyxy_m[3]))); world=local@np.asarray(physical_nominal_view.projection_world_from_local).T+np.asarray(physical_nominal_view.centre_xy_m); detector_xy=(world-sample_origin_xy)@request_J.T; pixels_xy=detector_centre+detector_xy/physical_pixel_xy; ax.plot(pixels_xy[:,0],pixels_xy[:,1],color="#56B4E9",lw=.45,alpha=.65)
    physical_spectral_evidence=[physical_certificate_evidence(request,counts,physical_null_count_images.get(request.frame_id)) for request,counts in zip(physical_demo_requests,physical_count_images,strict=True)]; physical_certificate_peak_snr={str(request.frame_id):scores.tolist() for request,(_,_,scores,_,_) in zip(physical_demo_requests,physical_spectral_evidence,strict=True) if len(scores)}; physical_certificate_target_bins_by_frame={str(request.frame_id):targets.tolist() for request,(_,targets,_,_,_) in zip(physical_demo_requests,physical_spectral_evidence,strict=True) if len(targets)}; physical_certificate_labels={str(request.frame_id):list(labels) for request,(_,_,_,labels,_) in zip(physical_demo_requests,physical_spectral_evidence,strict=True) if labels}; physical_certificate_eligible={str(request.frame_id):eligible.tolist() for request,(_,_,_,_,eligible) in zip(physical_demo_requests,physical_spectral_evidence,strict=True) if len(eligible)}; nominal_evidence=physical_spectral_evidence[physical_demo_requests.index(physical_sample_request)]; nominal_scores=nominal_evidence[2][nominal_evidence[4]]; physical_certificate_pass=bool(len(nominal_scores) and np.all(np.isfinite(nominal_scores)) and np.min(nominal_scores)>=3.)
    fig_subset,subset_axes=plt.subplots(3,len(physical_demo_specs),figsize=(18.2,8.8),constrained_layout=True)
    for column_index,((caption,request),counts,contrast,obs,(spectral_snr,targets_yx,scores,target_labels,target_eligible)) in enumerate(zip(physical_demo_specs,physical_count_images,physical_local_contrast,physical_observables,physical_spectral_evidence,strict=True)):
        material=request.interaction_route!="vacuum"; region=(physical_nominal_view.selection.selected_region_name or "global atlas") if material else "vacuum"; family=physical_nominal_view.selection.feature_family if material else request.measurement; level=physical_nominal_view.selection.level_index if material else "—"; detected_fraction=obs.total_intensity/surrogate_dose; control_label=physical_control_label(request)
        subset_axes[0,column_index].imshow(counts,origin="lower",cmap="gray",vmin=0,vmax=physical_vmax); subset_axes[0,column_index].set_title(f"{caption} · frame {request.frame_id}\n{region}; {family}; LOD {level}\nI/Iref={detected_fraction:.3f}; {control_label}",fontsize=7.5)
        if material: overlay_registered_boundaries(subset_axes[0,column_index],request)
        bar_nm,bar_px=physical_scale_bar(request); subset_axes[0,column_index].plot((5,5+bar_px),(5,5),color="white",lw=2,solid_capstyle="butt"); subset_axes[0,column_index].text(5+.5*bar_px,8,f"{bar_nm:g} nm",color="white",ha="center",fontsize=7)
        subset_axes[1,column_index].imshow(contrast,origin="lower",cmap="coolwarm",vmin=-contrast_limit,vmax=contrast_limit); subset_axes[1,column_index].set_title(f"noiseless local contrast (display only)\nroute={physical_demo_checkpoint.record(request.frame_id).actual_interaction_route}",fontsize=7.5)
        fft_art=subset_axes[2,column_index].imshow(spectral_snr,origin="lower",cmap="magma",vmin=0,vmax=8)
        for vector_yx in targets_yx:
            for sign in (-1,1): subset_axes[2,column_index].scatter(physical_fft_centre[1]+sign*vector_yx[1],physical_fft_centre[0]+sign*vector_yx[0],s=38,facecolors="none",edgecolors="#56B4E9",linewidths=1.1)
        admitted_scores=scores[target_eligible]; peak_label="—" if not len(admitted_scores) else f"{np.nanmin(admitted_scores):.1f} min SNR"; target_label=", ".join(label for label,eligible in zip(target_labels,target_eligible,strict=True) if eligible); subset_axes[2,column_index].set_title(f"matched-blank certificate evidence: {peak_label}\n{target_label}" if material else f"vacuum/input spectrum; centroid=({obs.centroid_xy_px[0]:.2f},{obs.centroid_xy_px[1]:.2f}) px",fontsize=7.2)
    for ax in subset_axes.ravel(): ax.set(xticks=[],yticks=[])
    fig_subset.colorbar(fft_art,ax=list(subset_axes[2]),shrink=.72,pad=.01,label="matched Fourier coefficient SNR")
    fig_subset.suptitle(f"Physical Gaussian experimental set at {SIMULATION_MAGNIFICATION:,}× — shared raw-count scale; {physical_beamlet_count:,} frozen beamlets",fontsize=12)
    physical_request_conjugacy={str(request.frame_id):{"b_norm_m":float(np.linalg.norm(np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector"))[np.ix_([0,2],[1,3])])),"effective_specimen_defocus_m":float(np.linalg.norm(np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector"))[np.ix_([0,2],[1,3])])/max(np.linalg.norm(np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector"))[np.ix_([0,2],[0,2])]),np.finfo(float).tiny))} for request in physical_material_requests}
    physical_subset_exports=export_ray_figure(fig_subset,GENERATED/f"neoarm_physical_{SIMULATION_MAGNIFICATION//1000}k_experimental_set_v11_gpu",metadata={"title":"NEOARM physical Gaussian experimental image set","magnification":SIMULATION_MAGNIFICATION,"frame_ids":[request.frame_id for request in physical_demo_requests],"matched_blank_frame_ids":[request.frame_id for request in physical_null_requests],"raw_dac_order":plan.channel_names,"raw_dac_values":[request.raw_dac_values for request in physical_demo_requests],"detector_shape_yx":physical_detector.shape_yx,"sample_grid_shape_yx":physical_grid_shape_yx,"sample_beamlet_count":physical_beamlet_count,"sample_x_bounds_m":[physical_sample_x[0],physical_sample_x[-1]],"sample_y_bounds_m":[physical_sample_y[0],physical_sample_y[-1]],"sample_grid_step_m":physical_grid_step_m,"sample_illumination_radius_m":sample_illumination_radius_m,"vacuum_probe_waist_m":physical_vacuum_probe_waist_m,"slices_per_lens":physical_slices_per_lens,"slice_candidates":physical_slice_candidates,"slice_complex_field_relative_errors":physical_slice_transition_errors,"topology_id":physical_topology_id,"dose_electrons":surrogate_dose,"read_noise_e":surrogate_read_noise,"count_seeds":{str(request.frame_id):51000+request.frame_id for request in physical_demo_requests},"blank_count_seeds":{str(request.frame_id):61000+request.frame_id for request in physical_null_requests},"incident_intensity_sum":physical_incident_sum,"optical_pixel_area_m2":physical_optical_pixel_area_m2,"shared_count_range":[0,physical_vmax],"local_contrast_range":[-contrast_limit,contrast_limit],"certificate_targets_yx_bins":physical_certificate_target_bins_by_frame,"certificate_target_labels":physical_certificate_labels,"certificate_target_eligible":physical_certificate_eligible,"certificate_matched_blank_snr":physical_certificate_peak_snr,"certificate_admission_rule":"all non-DC eligible nominal targets >= 3 matched-blank coefficient SNR; wobble frames diagnostic only","certificate_evidence_pass":physical_certificate_pass,"request_conjugacy":physical_request_conjugacy,"checkpoint_sha256":physical_demo_checkpoint.manifest_sha256,"gpu_kernel_required":FORWARD_EVALUATION_METHOD=="gpu_kernel_required","scope":"sliced hard-edge source-to-detector Gaussian propagation with exact cubic multiplication and re-expansion",**physical_topology_metadata,**FORWARD_RUNTIME_METADATA}); print("Physical image-set exports:",physical_subset_exports); plt.show()
    physical_image_evidence_admitted=bool(physical_certificate_pass); print(f"Nominal certificate-targeted physical-image evidence >=3 matched-blank coefficient SNR: {physical_image_evidence_admitted}; scores={physical_certificate_peak_snr}")
    physical_basis_requests=(physical_position_request,physical_angle_request); physical_vacuum_observable=physical_observables_by_id[physical_vacuum_request.frame_id]; physical_pixel_metric_jax=jnp.asarray(np.diag(1/physical_pixel_xy)); physical_affine_jax=jnp.asarray(detector_affine); physical_position_indices=jnp.asarray((0,2)); physical_basis_blocks=[]
    for request in physical_basis_requests:
        obs=physical_observables_by_id[request.frame_id]; observed_shift=np.asarray(obs.centroid_xy_px)-np.asarray(physical_vacuum_observable.centroid_xy_px); matrix=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector")); response=np.diag(1/physical_pixel_xy)@detector_affine@matrix[np.ix_([0,2],np.arange(4))]; statistical=np.asarray(obs.covariance_xy_px2)/max(obs.total_intensity,1.)+np.asarray(physical_vacuum_observable.covariance_xy_px2)/max(physical_vacuum_observable.total_intensity,1.); calibration=response@(np.asarray(request.calibration_covariance)+np.asarray(physical_vacuum_request.calibration_covariance))@response.T; covariance=statistical+calibration+np.eye(2)*.01**2; physical_basis_blocks.append(FrameObservationBlock(request.frame_id,request.optical_state_index,observed_shift,covariance,("delta_centroid_x_px","delta_centroid_y_px")))
    physical_basis_dataset=HeterogeneousInversionDataset(plan,tuple(physical_basis_blocks),metadata={"source":"checkpointed physical Gaussian count images","observable":"extracted calibrated-basis centroids minus the through-hole zero-input centroid","matched_observables":["delta_centroid_x_px","delta_centroid_y_px"],"selection_rule":"largest finite local whitened-Jacobian column among downstream q biases","scope":"local observable-matched downstream one-parameter recovery; not full-column inversion"})
    def physical_basis_predictions(parameters):
        predictions=[]
        for request in physical_basis_requests:
            matrix=column.matrix(parameters,jnp.asarray(request.raw_dac_values),segment="sample_to_detector"); response=physical_pixel_metric_jax@physical_affine_jax@matrix[jnp.ix_(physical_position_indices,jnp.arange(4))]; predictions.append(response@jnp.asarray(request.sample_state.as_array()))
        return tuple(predictions)
    physical_downstream_names=("IL1.q_bias","IL2.q_bias","IL3.q_bias","PL1.q_bias"); physical_downstream_indices=np.asarray([column.parameter_names.index(name) for name in physical_downstream_names]); physical_downstream_scale=np.full(len(physical_downstream_indices),.02); physical_base_parameters=jnp.asarray(physical_parameters)
    def physical_diagnostic_model(u): return physical_basis_predictions(physical_base_parameters.at[jnp.asarray(physical_downstream_indices)].add(jnp.asarray(physical_downstream_scale)*u))
    physical_diagnostic_J=np.asarray(jax.jacfwd(lambda u: physical_basis_dataset.jax_whitened_residuals(physical_diagnostic_model(u)))(jnp.zeros(len(physical_downstream_indices)))); physical_column_norms=np.linalg.norm(physical_diagnostic_J,axis=0)
    if not np.all(np.isfinite(physical_column_norms)) or float(np.max(physical_column_norms))<=1e-8: raise RuntimeError("No downstream q-bias has finite local sensitivity to the extracted calibrated-basis centroids.")
    physical_resolved_local=int(np.argmax(physical_column_norms)); physical_fit_name=physical_downstream_names[physical_resolved_local]; physical_fit_index=int(physical_downstream_indices[physical_resolved_local]); physical_fit_scale=float(physical_downstream_scale[physical_resolved_local])
    def physical_one_parameter_model(u): return physical_basis_predictions(physical_base_parameters.at[physical_fit_index].add(physical_fit_scale*u[0]))
    physical_fit_initial=np.asarray((.5,)); physical_initial_wrms=float(np.sqrt(np.mean(np.asarray(physical_basis_dataset.jax_whitened_residuals(physical_one_parameter_model(jnp.asarray(physical_fit_initial))))**2))); physical_image_fit=fit_heterogeneous_dataset(physical_one_parameter_model,physical_basis_dataset,physical_fit_initial,bounds=(np.asarray((-2.,)),np.asarray((2.,))),parameter_names=(f"scaled:{physical_fit_name}",),loss="linear",f_scale=1.,multistart=4,seed=606,max_nfev=200)
    physical_fit_absolute_J=np.asarray(jax.jacfwd(lambda u:physical_basis_dataset.jax_whitened_residuals(physical_one_parameter_model(u)))(jnp.asarray(physical_image_fit.parameters))); physical_fit_fisher=float(np.sum(physical_fit_absolute_J**2)); physical_truth_value=float(physical_parameters[physical_fit_index]); physical_fitted_value=physical_truth_value+physical_fit_scale*float(physical_image_fit.parameters[0]); physical_fit_se=physical_fit_scale/np.sqrt(physical_fit_fisher) if physical_fit_fisher>0 else np.inf
    if not np.isfinite(physical_fit_se) or physical_fit_se<=0: raise RuntimeError("The local physical-image fit has no finite positive uncertainty.")
    physical_fit_z=(physical_fitted_value-physical_truth_value)/physical_fit_se
    print(f"Physical-image local recovery from calibrated basis centroids: {physical_fit_name}, WRMS {physical_initial_wrms:.3f}→{physical_image_fit.weighted_rmse:.3f}, truth={physical_truth_value:+.6f}, fit={physical_fitted_value:+.6f}±{physical_fit_se:.2e}, z={physical_fit_z:+.2f}")
    fig_physical_fit,physical_fit_axes=plt.subplots(1,2,figsize=(9.4,3.5),constrained_layout=True); observed_shifts=np.concatenate([block.values for block in physical_basis_blocks]); initial_shifts=np.concatenate([np.asarray(value) for value in physical_one_parameter_model(jnp.asarray(physical_fit_initial))]); fitted_shifts=np.concatenate([np.asarray(value) for value in physical_one_parameter_model(jnp.asarray(physical_image_fit.parameters))]); coordinate_index=np.arange(len(observed_shifts)); physical_fit_axes[0].plot(coordinate_index,observed_shifts,"o",color="black",label="physical image"); physical_fit_axes[0].plot(coordinate_index,initial_shifts,"s--",color="#D55E00",label="initial"); physical_fit_axes[0].plot(coordinate_index,fitted_shifts,"o-",color="#0072B2",label="fitted"); physical_fit_axes[0].set(xlabel="position/angle centroid coordinate",ylabel="vacuum-referenced shift [px]",title="Measured physical-image response"); physical_fit_axes[0].legend(frameon=False)
    physical_fit_axes[1].axhline(0,color="0.7",lw=1); physical_fit_axes[1].errorbar((0,),(physical_fitted_value-physical_truth_value,),yerr=(physical_fit_se,),fmt="o",capsize=4,color="#0072B2"); physical_fit_axes[1].set(xlim=(-.6,.6),xticks=(0,),xticklabels=(physical_fit_name.replace(".q_bias"," bias"),),ylabel="q estimate − truth",title="Local matched downstream recovery")
    physical_fit_exports=export_ray_figure(fig_physical_fit,GENERATED/f"neoarm_physical_{SIMULATION_MAGNIFICATION//1000}k_local_fit_v9_gpu",metadata={"title":"NEOARM local physical-image parameter recovery","frame_ids":[request.frame_id for request in physical_basis_requests]+[physical_vacuum_request.frame_id],"parameter":physical_fit_name,"truth":physical_truth_value,"estimate":physical_fitted_value,"standard_error":physical_fit_se,"weighted_rmse":physical_image_fit.weighted_rmse,"loss":"linear","absolute_fisher_information":physical_fit_fisher,"selected_whitened_jacobian_norm":float(physical_column_norms[physical_resolved_local]),"fit_bounds_scaled":[-2.,2.],"matched_observables":["delta_centroid_x_px","delta_centroid_y_px"],"selection_rule":"largest finite local whitened-Jacobian column among downstream q biases","dose_electrons":surrogate_dose,"read_noise_e":surrogate_read_noise,"checkpoint_sha256":physical_demo_checkpoint.manifest_sha256,"gpu_kernel_required":FORWARD_EVALUATION_METHOD=="gpu_kernel_required","scope":"one downstream q-bias from extracted through-hole-referenced calibrated-basis centroids; deliberately not a full-column recovery",**physical_topology_metadata,**FORWARD_RUNTIME_METADATA}); print("Physical local-fit exports:",physical_fit_exports); plt.show()
    physical_fit_interior=bool(-1.95<float(physical_image_fit.parameters[0])<1.95); physical_local_recovery_admitted=bool(physical_image_fit.success and physical_fit_interior and physical_image_fit.weighted_rmse<physical_initial_wrms and physical_image_fit.weighted_rmse<2. and np.isfinite(physical_fit_z) and abs(physical_fit_z)<=3.)
    if not physical_local_recovery_admitted: raise RuntimeError("The local physical-image recovery failed its success/interior/WRMS/truth-consistency gate.")
else: print("Physical image-set checkpoint is intentionally partial; rerun with RUN_PHYSICAL_DEMO_SUBSET=True to resume it.")

def known_sample_source(dacs,waist_m):
    sample_beam=unit_current_sample_gaussian(waist_m); upstream=np.asarray(column.matrix(physical_parameters,np.asarray(dacs),segment="source_to_sample")); return apply_transfer_matrix_to_gaussians(sample_beam,np.linalg.inv(upstream),z_out_m=physical_parameters[0],axial_path_m=physical_parameters[0]-column.sample_z_m)
def gaussian_source_digest(beam):
    digest=hashlib.sha256()
    for name in ("x","y","dx","dy","z","voltage","amplitude","pathlength","Q_inv"):
        values=np.ascontiguousarray(np.asarray(getattr(beam,name))); digest.update(name.encode()); digest.update(str(values.shape).encode()); digest.update(values.tobytes())
    return digest.hexdigest()
def compile_full_state_topology(label,seed_view,central_J,state_dacs,topology_version):
    envelope=declared_transfer_envelope(np.asarray(central_J)); footprints=np.concatenate([physical_detector_corners@np.linalg.inv(J).T+sample_origin_xy for J in envelope]); span=footprints.max(axis=0)-footprints.min(axis=0); retained=float(seed_view.selection.routed_feature_scale_m or 8*seed_view.selection.selected_spacing_m); preliminary_step=min(seed_view.selection.specimen_pixel_min_m/2,retained/10,atlas.certificate.interface_width_m/3 if seed_view.selection.feature_family=="individual_layer_widths" else np.inf); illumination=max(2*retained,.22*float(np.min(span)),12*preliminary_step); support=2.5*illumination; seed_world_to_local=np.linalg.inv(np.asarray(seed_view.projection_world_from_local)); footprint_local=(footprints-sample_origin_xy)@seed_world_to_local.T; support_world=np.asarray(((-support,-support),(support,-support),(support,support),(-support,support))); support_local=support_world@seed_world_to_local.T; support_margin=max(.5e-9,float(np.max(np.max(np.abs(support_local),axis=0)-np.max(np.abs(footprint_local),axis=0)))+2*preliminary_step)
    view=select_sample_view(atlas,transfer_jacobian_xy=np.asarray(central_J),transfer_jacobian_envelope_xy=envelope,detector_shape_yx=physical_detector.shape_yx,detector_pixel_size_yx_m=physical_detector.pixel_size_yx_m,sample_centre_xy_m=sample_origin_xy,sample_rotation_rad=sample_rotation,sample_tilt_xy_rad=sample_tilt_xy,region_name=seed_view.selection.selected_region_name,gaussian_radius_min_m=3.4e-9,lens_wobble_margin_m=.15e-9,calibrated_input_margin_m=1.10e-9,gaussian_support_margin_m=support_margin,pose_uncertainty_margin_m=.10e-9,interpolation_halo_margin_m=.05e-9,boundary=None,z=beam_cfg["sample_z_m"])
    selection=view.selection; retained=float(selection.routed_feature_scale_m or 8*selection.selected_spacing_m); spacing=min(selection.specimen_pixel_min_m/2,retained/10,atlas.certificate.interface_width_m/3 if selection.feature_family=="individual_layer_widths" else np.inf); projection=np.asarray(view.projection_world_from_local); centre=np.asarray(view.centre_xy_m); x0,x1,y0,y1=selection.local_bounds_xyxy_m; local_corners=np.asarray(((x0,y0),(x1,y0),(x1,y1),(x0,y1))); world_corners=local_corners@projection.T+centre; grid_min=np.minimum(footprints.min(axis=0),sample_origin_xy-support); grid_max=np.maximum(footprints.max(axis=0),sample_origin_xy+support); index_min=np.floor((grid_min-sample_origin_xy)/spacing).astype(int); index_max=np.ceil((grid_max-sample_origin_xy)/spacing).astype(int); x=sample_origin_xy[0]+np.arange(index_min[0],index_max[0]+1)*spacing; y=sample_origin_xy[1]+np.arange(index_min[1],index_max[1]+1)*spacing
    grid_corners=np.asarray(((x[0],y[0]),(x[-1],y[0]),(x[-1],y[-1]),(x[0],y[-1]))); local_grid=np.asarray(view.world_to_local(jnp.asarray(grid_corners))); domain=np.asarray(view.domain_bounds_xyxy_m); covered=bool(view.boundary=="periodic" or (local_grid[:,0].min()>=domain[0]-1e-15 and local_grid[:,0].max()<=domain[1]+1e-15 and local_grid[:,1].min()>=domain[2]-1e-15 and local_grid[:,1].max()<=domain[3]+1e-15))
    if not covered: raise RuntimeError(f"Full-state topology {label} exceeds its certified SampleView source bounds.")
    xx,yy=np.meshgrid(x,y,indexing="xy"); centres=np.stack((xx.ravel(),yy.ravel()),axis=-1); waists=np.full(len(centres),physical_width_over[0]*spacing); topology_id=frozen_topology_fingerprint(specimen_identity=config["sample"]["identity"],atlas_version=atlas.certificate.version,region_name=selection.selected_region_name or "global_atlas",level_index=selection.level_index,crop_bounds_xyxy_m=selection.local_bounds_xyxy_m,sample_translation_xy_m=selection.sample_translation_xy_m,projection_world_from_local=projection,route="sample_plane_reexpansion",selected_spacing_m=selection.selected_spacing_m,routed_feature_scale_m=retained,centres_per_period=retained/spacing,overlap_factor=physical_width_over[0],interpolation_halo_knots=selection.interpolation_halo_knots,topology_version=topology_version,centres_xy_m=centres,waist_m=waists); source=known_sample_source(state_dacs,illumination); result={"label":label,"view":view,"x":x,"y":y,"spacing_m":spacing,"illumination_waist_m":illumination,"beam_count":len(centres),"topology_id":topology_id,"x_sha256":hashlib.sha256(np.asarray(x,dtype=np.float64).tobytes()).hexdigest(),"y_sha256":hashlib.sha256(np.asarray(y,dtype=np.float64).tobytes()).hexdigest(),"source":source,"source_sha256":gaussian_source_digest(source)}; del xx,yy,centres,waists; return result
def bind_full_material_request(request,item):
    selection=item["view"].selection; return replace(request,feature_family=selection.feature_family,selected_region_name=selection.selected_region_name,sample_level_index=selection.level_index,sample_translation_xy_m=selection.sample_translation_xy_m,sample_crop_bounds_xyxy_m=selection.local_bounds_xyxy_m,selected_spacing_m=selection.selected_spacing_m,interaction_route="sample_plane_reexpansion",frozen_topology_id=item["topology_id"])
full_state_items={}
for state in plan.states:
    state_J=detector_affine@np.asarray(column.sample_position_jacobian(physical_parameters,np.asarray(state.dac_values))); full_state_items[state.label]=compile_full_state_topology(state.label,view_by_state[state.label],state_J,state.dac_values,"full-imaging-state-v5-gpu")
full_vacuum_source_by_state={state.label:known_sample_source(state.dac_values,physical_vacuum_probe_waist_m) for state in plan.states}; full_vacuum_source_digests={label:gaussian_source_digest(source) for label,source in full_vacuum_source_by_state.items()}; full_vacuum_requests=tuple(request for request in manifest if request.interaction_route=="vacuum")
if max(np.hypot(request.sample_state.x,request.sample_state.y) for request in full_vacuum_requests)+4*physical_vacuum_probe_waist_m>=physical_vacuum_clearance_m: raise RuntimeError("At least one full-acquisition vacuum/basis request violates four-waist perforation clearance.")
full_imaging_requests=tuple(bind_full_material_request(request,full_state_items[request.optical_state_label]) if request.measurement in {"sample","sample_repeat"} else request for request in manifest if request.measurement!="indexed_diffraction")
diffraction_detector=detector; diffraction_reference_label=next(state.label for state in plan.states if state.role=="nominal" and state.target_magnification==600_000); diffraction_reference_state=next(state for state in plan.states if state.label==diffraction_reference_label); diffraction_reference_J=detector_affine@np.asarray(column.sample_position_jacobian(physical_parameters,np.asarray(diffraction_reference_state.dac_values))); full_diffraction_item=compile_full_state_topology("indexed-diffraction",views[600_000],diffraction_reference_J,diffraction_reference_state.dac_values,"native-hard-edge-diffraction-v3-gpu")
if full_diffraction_item["view"].selection.feature_family!="silicon_lattice_vectors" or full_diffraction_item["view"].boundary!="periodic": raise RuntimeError("Native diffraction requires the registered periodic silicon-lattice SampleView.")
unbound_full_diffraction_requests=tuple(bind_full_material_request(request,full_diffraction_item) for request in manifest if request.measurement=="indexed_diffraction")
native_diffraction_camera_lengths_m=np.linspace(.30,.40,6); native_diffraction_adjusted_channels=("IL1","PL1"); native_diffraction_dac_bounds=tuple(float(value) for value in experiment_cfg["nominal_dac_search_bounds"]); native_diffraction_max_bandlimit_fraction=.85
if len(unbound_full_diffraction_requests)!=len(native_diffraction_camera_lengths_m): raise RuntimeError(f"Expected six indexed-diffraction requests, got {len(unbound_full_diffraction_requests)}.")
native_diffraction_bindings=bind_native_diffraction_requests(unbound_full_diffraction_requests,column,physical_parameters,native_diffraction_camera_lengths_m,adjusted_channel_names=native_diffraction_adjusted_channels,dac_bounds=native_diffraction_dac_bounds,multistart_count=16)
full_diffraction_requests=tuple(binding.request for binding in native_diffraction_bindings); native_diffraction_binding_by_frame_id={binding.request.frame_id:binding for binding in native_diffraction_bindings}
native_adjusted_indices=np.asarray([column.channel_names.index(name) for name in native_diffraction_adjusted_channels]); native_unadjusted_indices=np.asarray([index for index in range(len(column.channel_names)) if index not in set(native_adjusted_indices)]); native_source_to_sample_invariance=[]
for original,binding in zip(unbound_full_diffraction_requests,native_diffraction_bindings,strict=True):
    old_endpoint=np.asarray(original.raw_dac_values); new_endpoint=np.asarray(binding.request.raw_dac_values); old_history=np.asarray(original.dac_approach_start_values)-old_endpoint; new_history=np.asarray(binding.request.dac_approach_start_values)-new_endpoint
    if not np.array_equal(new_endpoint[native_unadjusted_indices],old_endpoint[native_unadjusted_indices]): raise RuntimeError(f"Native diffraction binding changed an undeclared DAC for frame {original.frame_id}.")
    if not np.allclose(new_history,old_history,rtol=0,atol=2e-14) or binding.request.dac_approach!=original.dac_approach or binding.request.frame_id!=original.frame_id or binding.request.frozen_topology_id!=original.frozen_topology_id: raise RuntimeError(f"Native diffraction binding did not preserve request history for frame {original.frame_id}.")
    old_upstream=np.asarray(column.matrix(physical_parameters,old_endpoint,segment="source_to_sample")); new_upstream=np.asarray(column.matrix(physical_parameters,new_endpoint,segment="source_to_sample")); native_source_to_sample_invariance.append(float(np.max(np.abs(old_upstream-new_upstream))))
if max(native_source_to_sample_invariance)>2e-12: raise RuntimeError("The selected native diffraction controls are not downstream-only in this column geometry.")
full_diffraction_source_by_frame_id={request.frame_id:known_sample_source(request.raw_dac_values,full_diffraction_item["illumination_waist_m"]) for request in unbound_full_diffraction_requests}
diffraction_spacing_m=float(atlas.silicon_lattice_region.lattice_spacing_m); diffraction_projection=np.asarray(full_diffraction_item["view"].projection_world_from_local); diffraction_g_local=np.eye(2)/diffraction_spacing_m; diffraction_g_world=(np.linalg.inv(diffraction_projection).T@diffraction_g_local.T).T; diffraction_theta_world=wavelength_m*diffraction_g_world; diffraction_pixel_xy=np.asarray(diffraction_detector.pixel_size_yx_m)[::-1]; diffraction_optical_corners=np.asarray(diffraction_detector.optical_corners_xy_m)-np.asarray(diffraction_detector.centre_xy_m); diffraction_angular_nyquist=wavelength_m/(2*full_diffraction_item["spacing_m"])
native_diffraction_diagnostics=[]
for binding in native_diffraction_bindings:
    mode=binding.mode; spot_column_xy=(mode.b_block@diffraction_theta_world.T).T; spot_detector_xy=(np.asarray(diffraction_detector.affine_xy)@spot_column_xy.T).T; spot_radii_px=np.linalg.norm(spot_detector_xy/diffraction_pixel_xy,axis=1); detector_angles=(np.linalg.solve(mode.b_block,diffraction_optical_corners.T)).T; bandlimit_fraction=float(np.max(np.abs(detector_angles))/diffraction_angular_nyquist); native_diffraction_diagnostics.append({"frame_id":binding.request.frame_id,"state":binding.request.optical_state_label,"target_camera_length_m":mode.target_camera_length_m,"achieved_camera_length_m":mode.achieved_camera_length_m,"first_order_spot_radii_px":spot_radii_px,"detector_bandlimit_fraction":bandlimit_fraction,"a_block_norm":mode.a_block_norm,"camera_length_relative_error":mode.camera_length_relative_error,"b_rotation_relative_residual":mode.b_rotation_relative_residual,"round_structure_defect":mode.round_structure_defect,"symplectic_defect_norm":mode.symplectic_defect_norm})
native_first_order_radii_px=np.concatenate([row["first_order_spot_radii_px"] for row in native_diffraction_diagnostics]); native_max_bandlimit_fraction=max(row["detector_bandlimit_fraction"] for row in native_diffraction_diagnostics); native_diffraction_sampling_certified=bool(full_diffraction_item["spacing_m"]<=diffraction_spacing_m/8*(1+1e-12) and np.min(native_first_order_radii_px)>=3 and np.max(native_first_order_radii_px)<=96 and native_max_bandlimit_fraction<=native_diffraction_max_bandlimit_fraction and all(binding.mode.feasible for binding in native_diffraction_bindings))
print(f"{'frame':>5s} {'state':>14s} {'L target/actual [m]':>25s} {'Si spots [px]':>18s} {'Nyquist use':>12s} {'||A||':>11s} {'rel |B|-L':>11s}")
for row in native_diffraction_diagnostics: print(f"{row['frame_id']:5d} {row['state']:>14s} {row['target_camera_length_m']:.3f}/{row['achieved_camera_length_m']:.9f} {np.array2string(row['first_order_spot_radii_px'],precision=2):>18s} {row['detector_bandlimit_fraction']:11.3f} {row['a_block_norm']:11.2e} {row['camera_length_relative_error']:11.2e}")
print(f"Native-mode maxima: B/L rotation residual={max(row['b_rotation_relative_residual'] for row in native_diffraction_diagnostics):.2e}; round-structure defect={max(row['round_structure_defect'] for row in native_diffraction_diagnostics):.2e}; symplectic defect={max(row['symplectic_defect_norm'] for row in native_diffraction_diagnostics):.2e}; full-detector angular-Nyquist use={native_max_bandlimit_fraction:.3f}.")
if not native_diffraction_sampling_certified: raise RuntimeError(f"Native diffraction sampling failed closed: first-order range={native_first_order_radii_px.min():.3f}--{native_first_order_radii_px.max():.3f} px, detector/Nyquist={native_max_bandlimit_fraction:.3f}, sample step/d={full_diffraction_item['spacing_m']/diffraction_spacing_m:.3f}.")
material_prediction_cache={}
def material_prediction_key(request):
    state=request.sample_state
    return (request.optical_state_label,tuple(float(value) for value in request.raw_dac_values),(float(state.x),float(state.theta_x),float(state.y),float(state.theta_y)),request.frozen_topology_id,tuple(float(value) for value in request.sample_translation_xy_m))
material_request_count=sum(request.measurement in {"sample","sample_repeat"} for request in full_imaging_requests); unique_material_prediction_count=len({material_prediction_key(request) for request in full_imaging_requests if request.measurement in {"sample","sample_repeat"}})
def render_full_imaging(request):
    material=request.measurement in {"sample","sample_repeat"}
    if material:
        key=material_prediction_key(request)
        cached=material_prediction_cache.get(key)
        if cached is not None: return replace(cached,request=request)
    item=full_state_items[request.optical_state_label] if material else None; source=item["source"] if material else full_vacuum_source_by_state[request.optical_state_label]
    prediction=predict_frame(column,physical_parameters,request,detector,source,sample_view=None if item is None else item["view"],sample_x_coords_m=None if item is None else item["x"],sample_y_coords_m=None if item is None else item["y"],slices_per_lens=physical_slices_per_lens,width_over_step_yx=physical_width_over,tile_rows=16,beam_batch_size=128,evaluation_method=FORWARD_EVALUATION_METHOD)
    if material:
        prediction=replace(prediction,incident_sample_field_yx=None,transmitted_sample_field_yx=None); material_prediction_cache[key]=prediction
    return prediction
def render_full_diffraction(request):
    binding=native_diffraction_binding_by_frame_id[request.frame_id]; source=full_diffraction_source_by_frame_id[request.frame_id]; at_sample=propagate_gaussians_through_column(source,column,physical_parameters,np.asarray(request.raw_dac_values),segment="source_to_sample",slices_per_lens=physical_slices_per_lens); state=request.sample_state; at_sample=at_sample.derive(x=at_sample.x+state.x,y=at_sample.y+state.y,dx=at_sample.dx+state.theta_x,dy=at_sample.dy+state.theta_y); interaction=interact_gaussian_with_sample(full_diffraction_item["view"],at_sample,mode="reexpand",x_coords_m=full_diffraction_item["x"],y_coords_m=full_diffraction_item["y"],width_over_step_yx=physical_width_over,evaluation_method=FORWARD_EVALUATION_METHOD,evaluation_batch_size=128); render=render_native_diffraction_detector(interaction.beam,column,physical_parameters,binding.mode,diffraction_detector,slices_per_lens=physical_slices_per_lens,tile_rows=16,beam_batch_size=128,method=FORWARD_EVALUATION_METHOD,normalise=False); return FramePrediction(request=request,render=render,observables=extract_image_observables(render.intensity_yx),interaction_route=interaction.route,incident_sample_field_yx=interaction.incident_field_yx,transmitted_sample_field_yx=interaction.transmitted_field_yx)
full_topology_metadata=[{"label":label,"topology_id":item["topology_id"],"beam_count":item["beam_count"],"spacing_m":item["spacing_m"],"illumination_waist_m":item["illumination_waist_m"],"x_sha256":item["x_sha256"],"y_sha256":item["y_sha256"],"source_sha256":item["source_sha256"]} for label,item in full_state_items.items()]; configured_chunk=os.environ.get("NEOARM_PHYSICAL_CHUNK","").strip(); full_max_new=(None if not configured_chunk else int(configured_chunk)) if RUN_FULL_RENDERS else 0; full_metadata={"config_sha256":hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),"parameters":physical_parameters,"certificate":atlas.certificate.version,"state_topologies":full_topology_metadata,"vacuum_source_sha256_by_state":full_vacuum_source_digests,"vacuum_four_waist_clearance_verified":True,"slices_per_lens":physical_slices_per_lens,"slice_candidates":physical_slice_candidates,"slice_complex_field_relative_errors":physical_slice_transition_errors,"slice_phase_safety":physical_selected_slice_report.human_summary(),"width_over_step_yx":physical_width_over,"gpu_kernel_required":FORWARD_EVALUATION_METHOD=="gpu_kernel_required",**FORWARD_RUNTIME_METADATA}
full_imaging_checkpoint=render_frame_sequence_checkpointed(full_imaging_requests,detector,render_full_imaging,GENERATED/"physical_checkpoint_native_imaging_v5_gpu",render_signature=f"hard-edge-cubic-native-imaging-v5-gpu-slices{physical_slices_per_lens}",render_metadata=full_metadata,storage_dtype="float32",max_new_frames=full_max_new)
signed_defocus_base_request=next(request for request in manifest if request.optical_state_label==diffraction_reference_label and request.measurement=="sample" and request.dac_approach=="forward" and request.repeat_index==0); signed_defocus_history=np.asarray(signed_defocus_base_request.dac_approach_start_values)-np.asarray(signed_defocus_base_request.raw_dac_values)
signed_defocus_items={}; signed_defocus_unbound=[]; signed_defocus_states=[]
for signed_index,(signed_dacs,signed_target,signed_achieved) in enumerate(zip(signed_defocus_design.dac_values,signed_defocus_design.target_object_defocus_m,signed_defocus_design.achieved_magnifications,strict=True)):
    signed_label=f"{diffraction_reference_label}_object_defocus_{'minus' if signed_target<0 else 'plus'}"; signed_state_index=plan.n_states+signed_index; signed_request=replace(signed_defocus_base_request,frame_id=6_000_000+signed_index,optical_state_index=signed_state_index,optical_state_label=signed_label,measurement="sample",raw_dac_values=tuple(signed_dacs),predicted_effective_magnification=float(signed_achieved),dac_approach_start_values=tuple(np.asarray(signed_dacs)+signed_defocus_history),repeat_index=0); signed_J=detector_affine@np.asarray(column.sample_position_jacobian(physical_parameters,np.asarray(signed_dacs))); signed_item=compile_full_state_topology(signed_label,views[600_000],signed_J,signed_dacs,"signed-defocus-600k-v1-gpu"); signed_defocus_items[signed_label]=signed_item; signed_defocus_unbound.append(signed_request); signed_defocus_states.append(AcquisitionState(label=signed_label,dac_values=tuple(signed_dacs),role="nominal",frame_count=1,target_magnification=float(signed_defocus_design.nominal_magnification)))
signed_defocus_requests=tuple(bind_full_material_request(request,signed_defocus_items[request.optical_state_label]) for request in signed_defocus_unbound); physical_acquisition_plan=replace(plan,states=plan.states+tuple(signed_defocus_states))
for request in signed_defocus_requests:
    state=physical_acquisition_plan.states[request.optical_state_index]; actual_history=np.asarray(request.dac_approach_start_values)-np.asarray(request.raw_dac_values)
    if state.label!=request.optical_state_label or not np.allclose(state.dac_values,request.raw_dac_values,rtol=0,atol=1e-14) or not np.allclose(actual_history,signed_defocus_history,rtol=0,atol=1e-14): raise RuntimeError(f"Signed-defocus request {request.frame_id} is not registered to its augmented optical state/history.")
signed_defocus_endpoint_dacs=np.asarray([request.raw_dac_values for request in signed_defocus_requests],dtype=float); signed_defocus_approach_start_dacs=np.asarray([request.dac_approach_start_values for request in signed_defocus_requests],dtype=float)
signed_defocus_ramp_dacs=np.concatenate([np.linspace(np.asarray(request.dac_approach_start_values,dtype=float),np.asarray(request.raw_dac_values,dtype=float),int(request.dac_ramp_steps)+1) for request in signed_defocus_requests],axis=0); signed_defocus_slice_envelope_dacs=np.vstack((signed_defocus_endpoint_dacs,signed_defocus_approach_start_dacs,signed_defocus_ramp_dacs))
signed_defocus_slice_safety_report=assess_gaussian_slice_safety(column,physical_parameters,signed_defocus_slice_envelope_dacs,slices_per_lens=physical_slices_per_lens); signed_defocus_slice_envelope_sha256=hashlib.sha256(np.ascontiguousarray(signed_defocus_slice_envelope_dacs,dtype=np.float64).tobytes()).hexdigest()
if signed_defocus_slice_safety_report.requested_is_safe is not True: raise RuntimeError(f"The frozen {physical_slices_per_lens}-slice topology is unsafe across the signed-defocus endpoints/approach histories: {signed_defocus_slice_safety_report.human_summary()}")
print("Signed-defocus Gaussian slice audit:",signed_defocus_slice_safety_report.human_summary())
def render_signed_defocus(request):
    item=signed_defocus_items[request.optical_state_label]; prediction=predict_frame(column,physical_parameters,request,detector,item["source"],sample_view=item["view"],sample_x_coords_m=item["x"],sample_y_coords_m=item["y"],slices_per_lens=physical_slices_per_lens,width_over_step_yx=physical_width_over,tile_rows=16,beam_batch_size=128,evaluation_method=FORWARD_EVALUATION_METHOD); return replace(prediction,incident_sample_field_yx=None,transmitted_sample_field_yx=None)
signed_defocus_topology_metadata=[{"label":label,"topology_id":item["topology_id"],"beam_count":item["beam_count"],"spacing_m":item["spacing_m"],"illumination_waist_m":item["illumination_waist_m"],"x_sha256":item["x_sha256"],"y_sha256":item["y_sha256"],"source_sha256":item["source_sha256"]} for label,item in signed_defocus_items.items()]
signed_defocus_metadata={**full_metadata,"experiment":"600k signed object-defocus phase contrast","target_object_defocus_m":signed_defocus_design.target_object_defocus_m,"achieved_object_defocus_m":signed_defocus_design.achieved_object_defocus_m,"achieved_magnifications":signed_defocus_design.achieved_magnifications,"rotation_changes_rad":signed_defocus_design.cumulative_rotation_changes_rad,"selected_channel_names":signed_defocus_design.selected_channel_names,"dac_perturbations":signed_defocus_design.dac_perturbations,"state_topologies":signed_defocus_topology_metadata,"signed_defocus_slice_phase_safety":signed_defocus_slice_safety_report.human_summary(),"signed_defocus_slice_envelope_policy":"each endpoint, approach start, and every declared linear ramp point","signed_defocus_slice_history_state_count":int(len(signed_defocus_slice_envelope_dacs)),"signed_defocus_slice_envelope_sha256":signed_defocus_slice_envelope_sha256,"signed_defocus_endpoint_dacs":signed_defocus_endpoint_dacs,"signed_defocus_approach_start_dacs":signed_defocus_approach_start_dacs,"signed_defocus_ramp_steps_by_frame":{str(request.frame_id):int(request.dac_ramp_steps) for request in signed_defocus_requests},"signed_defocus_recommended_slices_per_lens":signed_defocus_slice_safety_report.recommended_slices_per_lens}
signed_defocus_checkpoint=render_frame_sequence_checkpointed(signed_defocus_requests,detector,render_signed_defocus,GENERATED/"physical_checkpoint_600k_signed_defocus_v1_gpu",render_signature=f"hard-edge-cubic-600k-signed-defocus-v1-gpu-slices{physical_slices_per_lens}",render_metadata=signed_defocus_metadata,storage_dtype="float32",max_new_frames=full_max_new)
native_diffraction_mode_digest=hashlib.sha256("|".join(binding.mode.model_signature for binding in native_diffraction_bindings).encode()).hexdigest()[:16]; native_diffraction_mode_metadata=[{**dict(binding.render_metadata),**row} for binding,row in zip(native_diffraction_bindings,native_diffraction_diagnostics,strict=True)]; native_diffraction_source_digests={str(frame_id):gaussian_source_digest(source) for frame_id,source in full_diffraction_source_by_frame_id.items()}
native_diffraction_render_signature=f"hard-edge-cubic-native-diffraction-v3-gpu-slices{physical_slices_per_lens}-modes-{native_diffraction_mode_digest}"; legacy_ideal_diffraction_checkpoint=GENERATED/"physical_checkpoint_native_diffraction_ideal_camera_v2_gpu"
full_diffraction_checkpoint=render_frame_sequence_checkpointed(full_diffraction_requests,diffraction_detector,render_full_diffraction,GENERATED/"physical_checkpoint_native_diffraction_v3_gpu",render_signature=native_diffraction_render_signature,render_metadata={**full_metadata,"model_scope":"neoarm_physical_column","renderer_family":"hard_edge_native_gaussian","uses_ideal_relay":False,"downstream_diffraction_model":"exact HardEdgeColumnParameterization sample_to_detector segment through IL1, IL2, IL3, and PL1","diffraction_topology_id":full_diffraction_item["topology_id"],"diffraction_beam_count":full_diffraction_item["beam_count"],"diffraction_sample_spacing_m":full_diffraction_item["spacing_m"],"silicon_lattice_spacing_m":diffraction_spacing_m,"sample_angular_nyquist_rad":diffraction_angular_nyquist,"maximum_detector_bandlimit_fraction":native_max_bandlimit_fraction,"maximum_allowed_detector_bandlimit_fraction":native_diffraction_max_bandlimit_fraction,"first_order_spot_radius_range_px":[float(native_first_order_radii_px.min()),float(native_first_order_radii_px.max())],"adjusted_channel_names":native_diffraction_adjusted_channels,"dac_bounds":native_diffraction_dac_bounds,"source_to_sample_invariance_max_abs":max(native_source_to_sample_invariance),"source_sha256_by_frame_id":native_diffraction_source_digests,"native_modes":native_diffraction_mode_metadata,"native_mode_digest":native_diffraction_mode_digest,"legacy_ideal_relay_checkpoint":str(legacy_ideal_diffraction_checkpoint.relative_to(REPO)),"legacy_ideal_relay_status":"visibility-only; excluded from readiness and acceptance"},storage_dtype="float32",max_new_frames=full_max_new)
native_diffraction_contract_ready=bool(native_diffraction_sampling_certified and len(native_diffraction_bindings)==6 and len({binding.mode.model_signature for binding in native_diffraction_bindings})==6 and all(binding.mode.feasible and binding.render_metadata["model_scope"]=="neoarm_physical_column" and binding.render_metadata["uses_ideal_relay"] is False for binding in native_diffraction_bindings))
if not native_diffraction_contract_ready: raise RuntimeError("The physical native-diffraction contract failed closed.")
print(f"Native physical checkpoints: imaging {len(full_imaging_checkpoint.records)}/{len(full_imaging_requests)} complete, 600k signed defocus {len(signed_defocus_checkpoint.records)}/{len(signed_defocus_requests)} complete, physical-column diffraction {len(full_diffraction_checkpoint.records)}/{len(full_diffraction_requests)} complete. Material requests require {unique_material_prediction_count} distinct deterministic mean-field renders for {material_request_count} frames; repeated exposures receive independent detector noise downstream. The legacy ideal-relay checkpoint is visibility-only and excluded. State-specific crops/topologies, source digests, and six native mode signatures are frozen. Set NEOARM_RUN_FULL_RENDERS=1; optional NEOARM_PHYSICAL_CHUNK limits new frames per run.")

### Production Gaussian refinement at the chosen conjugate mode

The small Collins experiment above is a local propagation unit test; it cannot certify the physical hard-edge column. This gate instead reuses the checkpointed chosen-mode specimen frame and renders two strictly denser versions of the **same world crop** through the actual conjugate hard-edge state, detector affine, MTF, and GPU kernel. A registered specimen Fourier vector is extracted from each material image and inverted to the same locally selected downstream $q$ parameter when that parameter has finite Fourier sensitivity. Complex sample fields are reconstructed on one fixed coarse-plus-feature validation stencil, while detector and observable changes are measured in the declared Poisson-plus-read-noise units.

Passing this cell establishes production Gaussian **self-convergence** over two successive refinements. It also evaluates an independent grid-wave reference in the exact specimen-conjugate ($B\to0$) limit by inverse-mapping detector pixels through $A$, multiplying the analytic incident intensity by the cubic specimen intensity transmission, and applying the measured detector MTF. This path never constructs or evaluates the Gaussian expansion. The omitted angular displacement is bounded from the sample Nyquist angle and must be below $10^{-3}$ detector pixel. A raw Collins integral is ill-conditioned in this limit; deliberately defocused validation states continue to use Collins/CZT. `physical_production_images_admitted` becomes true only if both refinements and this signature-bound independent reference pass all $0.2\sigma$ detector/observable gates.

In [ ]:
physical_production_refinement_ready=bool(physical_demo_checkpoint.complete)
physical_production_topology_converged=False; physical_production_images_admitted=False
if not physical_production_refinement_ready:
    print("Production refinement is pending the complete chosen-mode v11 physical checkpoint.")
else:
    if FORWARD_EVALUATION_METHOD!="gpu_kernel_required" or not GPU_DEVICES: raise RuntimeError("Production refinement requires the proven GPU kernel.")
    if physical_b_norm>physical_b_norm_atol*(1+1e-12): raise RuntimeError("Production refinement requires a specimen-conjugate nominal state.")
    production_step_fractions=(1.0,.8,.64); production_fixed_world_bounds=np.asarray(physical_grid_bounds_xyxy_m,float); production_topologies=[]
    production_topologies.append({"level":0,"step_fraction":1.,"x":np.asarray(physical_sample_x),"y":np.asarray(physical_sample_y),"spacing_yx_m":(physical_grid_step_m,physical_grid_step_m),"waist_yx_m":(physical_width_over[0]*physical_grid_step_m,physical_width_over[1]*physical_grid_step_m),"topology_id":physical_topology_id,"request":physical_sample_request,"beam_count":physical_beamlet_count})
    production_refinement_frame_base=2_000_000+100*physical_mode_index
    for level_index,step_fraction in enumerate(production_step_fractions[1:],start=1):
        nx=int(np.ceil(len(physical_sample_x)/step_fraction-1e-12)); ny=int(np.ceil(len(physical_sample_y)/step_fraction-1e-12)); dx=(production_fixed_world_bounds[1]-production_fixed_world_bounds[0])/nx; dy=(production_fixed_world_bounds[3]-production_fixed_world_bounds[2])/ny
        x=production_fixed_world_bounds[0]+(np.arange(nx)+.5)*dx; y=production_fixed_world_bounds[2]+(np.arange(ny)+.5)*dy; refined_world_corners=np.asarray(((x[0],y[0]),(x[-1],y[0]),(x[-1],y[-1]),(x[0],y[-1]))); refined_local_corners=np.asarray(physical_nominal_view.world_to_local(jnp.asarray(refined_world_corners))); refined_domain=np.asarray(physical_nominal_view.domain_bounds_xyxy_m); refined_view_covers_grid=bool(physical_nominal_view.boundary=="periodic" or (refined_local_corners[:,0].min()>=refined_domain[0]-1e-15 and refined_local_corners[:,0].max()<=refined_domain[1]+1e-15 and refined_local_corners[:,1].min()>=refined_domain[2]-1e-15 and refined_local_corners[:,1].max()<=refined_domain[3]+1e-15))
        if not refined_view_covers_grid: raise RuntimeError(f"Production refinement {level_index} exceeds the frozen SampleView source bounds.")
        xx,yy=np.meshgrid(x,y,indexing="xy"); centres=np.stack((xx.ravel(),yy.ravel()),axis=-1); equivalent_waist=np.sqrt((physical_width_over[0]*dy)*(physical_width_over[1]*dx)); waists=np.full(nx*ny,equivalent_waist)
        topology_id=frozen_topology_fingerprint(specimen_identity=config["sample"]["identity"],atlas_version=atlas.certificate.version,region_name=physical_nominal_view.selection.selected_region_name or "global_atlas",level_index=physical_nominal_view.selection.level_index,crop_bounds_xyxy_m=selection_bounds,sample_translation_xy_m=physical_nominal_view.selection.sample_translation_xy_m,projection_world_from_local=projection,route="sample_plane_reexpansion",selected_spacing_m=physical_nominal_view.selection.selected_spacing_m,routed_feature_scale_m=physical_retained_scale_m,centres_per_period=physical_retained_scale_m/max(dx,dy),overlap_factor=float(np.sqrt(np.prod(physical_width_over))),interpolation_halo_knots=physical_nominal_view.selection.interpolation_halo_knots,topology_version="production-refinement-v2-gpu",centres_xy_m=centres,waist_m=waists)
        request=replace(physical_sample_request,frame_id=production_refinement_frame_base+level_index,optical_state_label=f"{physical_nominal_label}_production_refinement_{level_index}",sample_crop_bounds_xyxy_m=physical_nominal_view.selection.local_bounds_xyxy_m,selected_spacing_m=physical_nominal_view.selection.selected_spacing_m,frozen_topology_id=topology_id)
        production_topologies.append({"level":level_index,"step_fraction":step_fraction,"x":x,"y":y,"spacing_yx_m":(dy,dx),"waist_yx_m":(physical_width_over[0]*dy,physical_width_over[1]*dx),"topology_id":topology_id,"request":request,"beam_count":nx*ny}); del xx,yy,centres,waists
    production_topology_by_frame={item["request"].frame_id:item for item in production_topologies[1:]}; production_refinement_requests=tuple(item["request"] for item in production_topologies[1:]); production_matched_blank_request=physical_null_by_material_id[physical_sample_request.frame_id]
    def render_production_refinement(request):
        item=production_topology_by_frame[request.frame_id]
        return predict_frame(column,physical_parameters,request,physical_detector,physical_source_beam,sample_view=physical_nominal_view,sample_x_coords_m=item["x"],sample_y_coords_m=item["y"],slices_per_lens=physical_slices_per_lens,width_over_step_yx=physical_width_over,tile_rows=16,beam_batch_size=128,evaluation_method=FORWARD_EVALUATION_METHOD)
    production_refinement_chunk=os.environ.get("NEOARM_PRODUCTION_REFINEMENT_CHUNK","").strip(); production_refinement_max_new=(None if not production_refinement_chunk else int(production_refinement_chunk)) if RUN_PHYSICAL_DEMO_SUBSET else 0
    if production_refinement_max_new is not None and production_refinement_max_new<0: raise ValueError("NEOARM_PRODUCTION_REFINEMENT_CHUNK must be non-negative.")
    production_topology_metadata=[{"level":item["level"],"step_fraction":item["step_fraction"],"shape_yx":[len(item["y"]),len(item["x"])],"spacing_yx_m":list(item["spacing_yx_m"]),"waist_yx_m":list(item["waist_yx_m"]),"beam_count":item["beam_count"],"topology_id":item["topology_id"],"x_sha256":hashlib.sha256(np.asarray(item["x"],dtype=np.float64).tobytes()).hexdigest(),"y_sha256":hashlib.sha256(np.asarray(item["y"],dtype=np.float64).tobytes()).hexdigest()} for item in production_topologies]
    production_signature_digest=hashlib.sha256("|".join(item["topology_id"] for item in production_topologies).encode()).hexdigest()[:16]
    production_refinement_checkpoint=render_frame_sequence_checkpointed(production_refinement_requests,physical_detector,render_production_refinement,GENERATED/f"physical_checkpoint_production_refinement_{SIMULATION_MAGNIFICATION//1000}k_v2_gpu",render_signature=f"production-refinement-v2-gpu-{production_signature_digest}-slices{physical_slices_per_lens}",render_metadata={"base_checkpoint_sha256":physical_demo_checkpoint.manifest_sha256,"matched_blank_frame_id":production_matched_blank_request.frame_id,"matched_blank_contract":"same nominal raw DAC and broad unit-current source; topology-independent vacuum propagation","fixed_world_bounds_xyxy_m":production_fixed_world_bounds,"topologies":production_topology_metadata,"selected_q_parameter":physical_fit_name if "physical_fit_name" in globals() else None,"slices_per_lens":physical_slices_per_lens,"mode_b_norm_m":physical_b_norm,"gpu_kernel_required":True,**FORWARD_RUNTIME_METADATA},store_sample_fields=True,storage_dtype="float32",max_new_frames=production_refinement_max_new)
    print(f"Production refinement checkpoint: newly rendered={production_refinement_checkpoint.rendered_frame_ids}, resumed={production_refinement_checkpoint.resumed_frame_ids}, pending={production_refinement_checkpoint.pending_frame_ids}")
    if production_refinement_checkpoint.complete:
        production_records=(physical_demo_checkpoint.record(physical_sample_request.frame_id),)+tuple(production_refinement_checkpoint.record(request.frame_id) for request in production_refinement_requests)
        if any(record.actual_interaction_route!="sample_plane_reexpansion" or record.transmitted_sample_field_path is None for record in production_records): raise RuntimeError("Every production refinement requires a checkpointed re-expanded sample field.")
        production_expected_counts=tuple(np.asarray(record.load_intensity(),float)*surrogate_dose/physical_incident_sum for record in production_records); production_blank_record=physical_demo_checkpoint.record(production_matched_blank_request.frame_id)
        if production_blank_record.actual_interaction_route!="vacuum": raise RuntimeError("The production matched-blank record is not a vacuum propagation.")
        production_blank_expected_counts=np.asarray(production_blank_record.load_intensity(),float)*surrogate_dose/physical_incident_sum
        production_downstream_matrix=np.asarray(column.matrix(physical_parameters,np.asarray(physical_sample_request.raw_dac_values),segment="sample_to_detector")); production_finest_spacing_m=min(min(item["spacing_yx_m"]) for item in production_topologies); physical_production_grid_reference=conjugate_gaussian_grid_reference(physical_nominal_view,production_downstream_matrix,physical_detector,incident_centre_xy_m=sample_origin_xy,incident_waist_xy_m=(sample_illumination_radius_m,sample_illumination_radius_m),wavelength_m=wavelength_m,sample_bandlimit_spacing_m=production_finest_spacing_m,maximum_b_displacement_px=1e-3); production_reference_expected_counts=np.asarray(physical_production_grid_reference.intensity_yx,float)*surrogate_dose/physical_incident_sum
        if physical_nominal_view.selection.feature_family=="silicon_lattice_vectors": production_g_local=np.asarray((1/float(atlas.silicon_lattice_region.lattice_spacing_m),0.))
        else: production_g_local=np.asarray((1/float(atlas.certificate.multilayer_period_m),0.))
        production_g_world=np.linalg.inv(np.asarray(physical_nominal_view.projection_world_from_local)).T@production_g_local; production_position_indices=jnp.asarray((0,2)); production_fov_xy_jax=jnp.asarray(np.asarray(physical_detector.shape_yx)[::-1]*np.asarray(physical_detector.pixel_size_yx_m)[::-1]); production_affine_jax=jnp.asarray(detector_affine); production_g_world_jax=jnp.asarray(production_g_world)
        def production_fourier_prediction(parameters):
            matrix=column.matrix(parameters,jnp.asarray(physical_sample_request.raw_dac_values),segment="sample_to_detector"); real_map=production_affine_jax@matrix[jnp.ix_(production_position_indices,production_position_indices)]; detector_frequency_xy=jnp.linalg.solve(real_map.T,production_g_world_jax); return (detector_frequency_xy*production_fov_xy_jax)[::-1]
        production_target_yx=np.asarray(production_fourier_prediction(jnp.asarray(physical_parameters))); production_peak_match_radius_px=2.5
        def production_matched_fourier(image_yx,blank_yx):
            material=np.clip(np.asarray(image_yx,float),0,None); blank=np.clip(np.asarray(blank_yx,float),0,None); difference=material/max(float(material.sum()),1.)-blank/max(float(blank.sum()),1.); nonnegative_difference=difference-float(difference.min())+np.finfo(float).tiny; vectors=extract_image_observables(nonnegative_difference,n_fourier_vectors=12,dc_exclusion_radius_px=physical_dc_exclusion_radius_px).fourier_vectors_yx_px
            if not len(vectors): raise RuntimeError("Production material frame contains no non-DC Fourier peak.")
            signed=np.concatenate((vectors,-vectors),axis=0); matched=signed[np.argmin(np.linalg.norm(signed-production_target_yx,axis=1))]
            if np.linalg.norm(matched-production_target_yx)>production_peak_match_radius_px: raise RuntimeError(f"Certificate peak missed its local matching window: target={production_target_yx}, observed={matched}.")
            return matched
        def production_observable_summary(image_yx,blank_yx):
            image=np.clip(np.asarray(image_yx,float),0,None); obs=extract_image_observables(image,n_fourier_vectors=12,dc_exclusion_radius_px=physical_dc_exclusion_radius_px); vector=production_matched_fourier(image,blank_yx); return {"scale":float(np.linalg.norm(vector)),"rotation":float(np.rad2deg(np.arctan2(vector[0],vector[1]))),"centroid":np.asarray(obs.centroid_xy_px),"fourier_vector":np.asarray(vector),"intensity":float(obs.total_intensity)}
        production_summaries=tuple(production_observable_summary(image,production_blank_expected_counts) for image in production_expected_counts)
        production_reference_summary=production_observable_summary(production_reference_expected_counts,production_blank_expected_counts)
        production_certificate_evidence=tuple(physical_certificate_evidence(physical_sample_request,image,production_blank_expected_counts) for image in production_expected_counts); production_peak_z=np.asarray([float(np.min(scores[eligible])) if np.any(eligible) and np.all(np.isfinite(scores[eligible])) else np.nan for _,_,scores,_,eligible in production_certificate_evidence])
        if not np.all(np.isfinite(production_peak_z)) or np.min(production_peak_z)<3.: raise RuntimeError(f"Matched-blank certificate Fourier evidence failed during production refinement: {production_peak_z}.")
        bootstrap_rng=np.random.default_rng(91_000+physical_mode_index); bootstrap_summaries=[]
        for _ in range(32):
            mean=production_expected_counts[-1]; draw=bootstrap_rng.poisson(mean).astype(float)+bootstrap_rng.normal(0,surrogate_read_noise,size=mean.shape); blank_draw=bootstrap_rng.poisson(production_blank_expected_counts).astype(float)+bootstrap_rng.normal(0,surrogate_read_noise,size=production_blank_expected_counts.shape); bootstrap_summaries.append(production_observable_summary(np.clip(draw,0,None),np.clip(blank_draw,0,None)))
        reference_summary=production_summaries[-1]; bootstrap_scale=np.asarray([item["scale"] for item in bootstrap_summaries]); bootstrap_rotation=np.asarray([((item["rotation"]-reference_summary["rotation"]+180)%360)-180 for item in bootstrap_summaries]); bootstrap_centroid=np.asarray([item["centroid"] for item in bootstrap_summaries]); bootstrap_fourier=np.asarray([item["fourier_vector"] for item in bootstrap_summaries]); bootstrap_intensity=np.asarray([item["intensity"] for item in bootstrap_summaries])
        production_observable_errors={"scale":max(float(np.std(bootstrap_scale,ddof=1)),.01),"rotation":max(float(np.std(bootstrap_rotation,ddof=1)),.05),"centroid":np.maximum(np.std(bootstrap_centroid,axis=0,ddof=1),.02),"fourier_vector":np.maximum(np.std(bootstrap_fourier,axis=0,ddof=1),.035),"intensity":max(float(np.std(bootstrap_intensity,ddof=1)),np.sqrt(reference_summary["intensity"]+np.prod(physical_detector.shape_yx)*surrogate_read_noise**2))}
        production_reference_rotation_delta=((production_summaries[0]["rotation"]-production_reference_summary["rotation"]+180)%360)-180; physical_production_reference_observable_residuals={"scale":abs(production_summaries[0]["scale"]-production_reference_summary["scale"])/production_observable_errors["scale"],"rotation":abs(production_reference_rotation_delta)/production_observable_errors["rotation"],"centroid":float(np.max(np.abs(production_summaries[0]["centroid"]-production_reference_summary["centroid"])/production_observable_errors["centroid"])),"fourier_vector":float(np.max(np.abs(production_summaries[0]["fourier_vector"]-production_reference_summary["fourier_vector"])/production_observable_errors["fourier_vector"])),"intensity":abs(production_summaries[0]["intensity"]-production_reference_summary["intensity"])/production_observable_errors["intensity"]}; physical_production_reference_noise=np.sqrt(np.maximum(production_reference_expected_counts,1.)+surrogate_read_noise**2); physical_production_reference_validation=validate_forward_models(production_expected_counts[0],production_reference_expected_counts,physical_production_reference_noise,observable_residuals_in_sigma=physical_production_reference_observable_residuals,require_complete=False); physical_production_reference_signature=production_signature_digest
        production_fourier_covariance=np.cov(bootstrap_fourier,rowvar=False,ddof=1)+np.eye(2)*.035**2
        candidate_indices=np.asarray(physical_downstream_indices); candidate_scales=np.full(len(candidate_indices),.02); production_base_parameters=jnp.asarray(physical_parameters)
        def production_candidate_vectors(u): return production_fourier_prediction(production_base_parameters.at[jnp.asarray(candidate_indices)].add(jnp.asarray(candidate_scales)*u))
        production_candidate_J=np.asarray(jax.jacfwd(production_candidate_vectors)(jnp.zeros(len(candidate_indices)))); production_candidate_norms=np.linalg.norm(production_candidate_J,axis=0); preferred_local=int(np.flatnonzero(candidate_indices==physical_fit_index)[0]) if "physical_fit_index" in globals() and np.any(candidate_indices==physical_fit_index) else int(np.argmax(production_candidate_norms)); strongest_local=int(np.argmax(production_candidate_norms)); production_q_local=preferred_local if np.isfinite(production_candidate_norms[preferred_local]) and production_candidate_norms[preferred_local]>max(production_candidate_norms[strongest_local],np.finfo(float).tiny)*1e-6 else strongest_local
        production_q_index=int(candidate_indices[production_q_local]); production_q_name=column.parameter_names[production_q_index]; production_q_scale=float(candidate_scales[production_q_local]); production_q_truth=float(physical_parameters[production_q_index])
        production_q_fits=[]
        for item,summary in zip(production_topologies,production_summaries,strict=True):
            block=FrameObservationBlock(item["request"].frame_id,item["request"].optical_state_index,np.asarray(summary["fourier_vector"]),production_fourier_covariance,("certificate_fourier_y_bin","certificate_fourier_x_bin")); dataset=HeterogeneousInversionDataset(plan,(block,),metadata={"source":"production Gaussian specimen image","topology_id":item["topology_id"]})
            def q_model(u): return (production_fourier_prediction(production_base_parameters.at[production_q_index].add(production_q_scale*u[0])),)
            fit=fit_heterogeneous_dataset(q_model,dataset,np.asarray((.35,)),bounds=(np.asarray((-2.,)),np.asarray((2.,))),parameter_names=(f"scaled:{production_q_name}",),loss="linear",multistart=3,seed=92_000+item["level"],max_nfev=160); fit_J=np.asarray(jax.jacfwd(lambda u:dataset.jax_whitened_residuals(q_model(u)))(jnp.asarray(fit.parameters))); fisher=float(np.sum(fit_J**2)); standard_error=production_q_scale/np.sqrt(fisher) if fisher>0 else np.inf; estimate=production_q_truth+production_q_scale*float(fit.parameters[0])
            if not fit.success or fit.jacobian_rank!=1 or not np.isfinite(standard_error) or standard_error<=0: raise RuntimeError(f"The material-image proxy did not identify {production_q_name} at refinement {item['level']}.")
            production_q_fits.append((estimate,standard_error,fit.weighted_rmse))
        coarse_x=np.linspace(physical_selection_world_bounds_xyxy_m[0],physical_selection_world_bounds_xyxy_m[1],65); coarse_y=np.linspace(physical_selection_world_bounds_xyxy_m[2],physical_selection_world_bounds_xyxy_m[3],33); fine_half=min(max(2*physical_retained_scale_m,8*physical_grid_step_m),.45*min(physical_selection_world_bounds_xyxy_m[1]-physical_selection_world_bounds_xyxy_m[0],physical_selection_world_bounds_xyxy_m[3]-physical_selection_world_bounds_xyxy_m[2])); fine_x=np.linspace(sample_origin_xy[0]-fine_half,sample_origin_xy[0]+fine_half,65); fine_y=np.linspace(sample_origin_xy[1]-fine_half,sample_origin_xy[1]+fine_half,65)
        def reconstructed_validation_field(record,item):
            transmitted=np.load(record.transmitted_sample_field_path,allow_pickle=False); coefficients=gaussian_cardinal_coefficients(transmitted,width_over_step_yx=physical_width_over,kernel_tolerance=1e-13,ridge=1e-10); dy,dx=item["spacing_yx_m"]; wy,wx=item["waist_yx_m"]
            def patch(x_query,y_query): return np.exp(-((y_query[:,None]-item["y"][None,:])/wy)**2)@coefficients@np.exp(-((x_query[:,None]-item["x"][None,:])/wx)**2).T
            return np.concatenate((patch(coarse_x,coarse_y),patch(fine_x,fine_y)),axis=0)
        production_validation_fields=tuple(reconstructed_validation_field(record,item) for record,item in zip(production_records,production_topologies,strict=True)); production_field_reference_rms=float(np.sqrt(np.mean(np.abs(production_validation_fields[-1])**2))); production_sample_field_noise=max(.02*production_field_reference_rms,np.finfo(float).tiny); production_detector_noise=np.sqrt(np.maximum(production_expected_counts[-1],1.)+surrogate_read_noise**2)
        production_snapshots=tuple(GaussianRefinementSnapshot(topology_id=item["topology_id"],beam_count=item["beam_count"],detector_intensity_yx=image,sample_field_yx=field,observables=summary,fit_parameters=np.asarray((fit_result[0],)),fit_standard_errors=np.asarray((fit_result[1],)),identifiable_parameter_mask=np.asarray((True,))) for item,image,field,summary,fit_result in zip(production_topologies,production_expected_counts,production_validation_fields,production_summaries,production_q_fits,strict=True))
        physical_production_refinement_report=validate_gaussian_refinement_sequence(production_snapshots,detector_noise_sigma_yx=production_detector_noise,sample_field_noise_sigma=production_sample_field_noise,observable_standard_errors=production_observable_errors,required_successive_refinements=2); physical_production_topology_converged=bool(physical_production_refinement_report.passed)
        for step in physical_production_refinement_report.steps: print(f"production {step.previous_beam_count:,}→{step.current_beam_count:,}: detector={step.forward_validation.detector_rms_in_noise_sigma:.3e}σ, field={step.forward_validation.sample_field_rms_in_noise_sigma:.3e}σ, q={step.forward_validation.parameter_change_in_sigma:.3e}σ, pass={step.passed}")
        if not physical_production_topology_converged: raise RuntimeError(f"Production Gaussian refinement failed closed: {physical_production_refinement_report.failure_reasons}.")
        physical_production_reference_report=globals().get("physical_production_reference_validation"); physical_production_reference_signature=globals().get("physical_production_reference_signature"); physical_production_reference_passed=bool(physical_production_reference_report is not None and getattr(physical_production_reference_report,"passed",False) and physical_production_reference_signature==production_signature_digest); physical_production_images_admitted=bool(physical_production_topology_converged and physical_production_reference_passed)
        print(f"Production topology converged for {production_q_name}; q estimates={[value[0] for value in production_q_fits]}; independent conjugate-grid reference={physical_production_reference_passed} (detector={physical_production_reference_validation.detector_rms_in_noise_sigma:.3e}σ, max B displacement={physical_production_grid_reference.maximum_b_displacement_px:.3e}px); chosen-mode physical image validated for local observable tests={physical_production_images_admitted}.")
        production_reference_residual_sigma=(production_expected_counts[0]-production_reference_expected_counts)/physical_production_reference_noise; production_reference_residual_limit=max(float(np.max(np.abs(production_reference_residual_sigma))),1e-6); fig_production_reference,production_reference_axes=plt.subplots(1,3,figsize=(11.6,3.6),constrained_layout=True); production_reference_vmax=float(np.percentile(np.stack((production_expected_counts[0],production_reference_expected_counts)),99.8)); production_reference_axes[0].imshow(production_expected_counts[0],origin="lower",cmap="gray",vmin=0,vmax=production_reference_vmax); production_reference_axes[0].set_title(f"Gaussian · {physical_beamlet_count:,} beamlets"); production_reference_axes[1].imshow(production_reference_expected_counts,origin="lower",cmap="gray",vmin=0,vmax=production_reference_vmax); production_reference_axes[1].set_title("Independent conjugate grid"); residual_art=production_reference_axes[2].imshow(production_reference_residual_sigma,origin="lower",cmap="RdBu_r",vmin=-production_reference_residual_limit,vmax=production_reference_residual_limit); production_reference_axes[2].set_title(f"difference / noise · RMS {physical_production_reference_validation.detector_rms_in_noise_sigma:.2e}σ"); fig_production_reference.colorbar(residual_art,ax=production_reference_axes[2],shrink=.82,label="experimental σ"); [ax.set(xticks=[],yticks=[]) for ax in production_reference_axes]; fig_production_reference.suptitle(f"Independent specimen-conjugate grid-wave validation at {SIMULATION_MAGNIFICATION:,}×",fontsize=12); physical_production_reference_exports=export_ray_figure(fig_production_reference,GENERATED/f"neoarm_physical_{SIMULATION_MAGNIFICATION//1000}k_conjugate_grid_reference_v1",metadata={"title":"Independent specimen-conjugate grid-wave validation","method":"B-to-zero inverse-A grid sampling; analytic incident Gaussian; cubic intensity transmission; detector MTF; no Gaussian re-expansion/evaluation","topology_signature":production_signature_digest,"detector_rms_in_noise_sigma":physical_production_reference_validation.detector_rms_in_noise_sigma,"observable_residuals_in_sigma":physical_production_reference_observable_residuals,"maximum_b_displacement_px":physical_production_grid_reference.maximum_b_displacement_px,"angular_bandlimit_rad":physical_production_grid_reference.angular_bandlimit_rad,"b_displacement_limit_px":1e-3,"passed":physical_production_reference_passed,"gaussian_forward_gpu_required":True,**FORWARD_RUNTIME_METADATA}); print("Independent conjugate-grid exports:",physical_production_reference_exports); plt.show()
        production_fine_fields=tuple(field[len(coarse_y):] for field in production_validation_fields)
        production_detector_residuals_sigma=tuple((image-production_expected_counts[-1])/production_detector_noise for image in production_expected_counts); production_detector_residual_rms=np.asarray([float(np.sqrt(np.mean(residual**2))) for residual in production_detector_residuals_sigma])
        count_vmax=float(np.percentile(np.stack(production_expected_counts),99.8)); field_vmax=max(float(np.max(np.abs(field))) for field in production_fine_fields); residual_limit=max(.2,float(np.percentile(np.abs(np.stack(production_detector_residuals_sigma)),99.8))); field_extent_nm=((fine_x[0]-sample_origin_xy[0])*1e9,(fine_x[-1]-sample_origin_xy[0])*1e9,(fine_y[0]-sample_origin_xy[1])*1e9,(fine_y[-1]-sample_origin_xy[1])*1e9)
        fig_production_refinement,production_axes=plt.subplots(3,3,figsize=(11.8,9.2),constrained_layout=True)
        for index,(item,image,field,fit_result,peak_z,residual,residual_rms) in enumerate(zip(production_topologies,production_expected_counts,production_fine_fields,production_q_fits,production_peak_z,production_detector_residuals_sigma,production_detector_residual_rms,strict=True)):
            count_art=production_axes[0,index].imshow(image,origin="lower",cmap="gray",vmin=0,vmax=count_vmax); production_axes[0,index].set_title(f"Topology {item['level']} · {item['beam_count']:,} beamlets\ncertificate peak {peak_z:.1f}σ",fontsize=8.5)
            field_art=production_axes[1,index].imshow(np.abs(field),origin="lower",extent=field_extent_nm,cmap="magma",vmin=0,vmax=field_vmax); production_axes[1,index].set_title("fine physical sample patch · |ψₛ|",fontsize=8.5); production_axes[1,index].set_xlabel("sample x about probe [nm]",fontsize=7)
            residual_art=production_axes[2,index].imshow(residual,origin="lower",cmap="RdBu_r",vmin=-residual_limit,vmax=residual_limit); reference_label="finest reference" if index==len(production_topologies)-1 else "relative to finest"; production_axes[2,index].set_title(f"{reference_label} · RMS {residual_rms:.2e}σ\n{production_q_name}={fit_result[0]:+.6f}±{fit_result[1]:.1e}",fontsize=8.2)
        for ax in production_axes[0]: ax.set(xticks=[],yticks=[])
        for index,ax in enumerate(production_axes[1]):
            ax.tick_params(labelsize=6.5,labelleft=index==0)
            if index==0: ax.set_ylabel("sample y about probe [nm]",fontsize=7)
        for ax in production_axes[2]: ax.set(xticks=[],yticks=[])
        production_axes[0,0].set_ylabel("expected detector counts",fontsize=7.5); production_axes[2,0].set_ylabel("detector residual / σ",fontsize=7.5)
        fig_production_refinement.colorbar(count_art,ax=production_axes[0,:].tolist(),shrink=.76,pad=.012,label="expected electrons / pixel"); fig_production_refinement.colorbar(field_art,ax=production_axes[1,:].tolist(),shrink=.76,pad=.012,label="transmitted field magnitude [arb.]"); fig_production_refinement.colorbar(residual_art,ax=production_axes[2,:].tolist(),shrink=.76,pad=.012,label="difference / experimental σ")
        fig_production_refinement.suptitle(f"Chosen-mode production Gaussian convergence at {SIMULATION_MAGNIFICATION:,}×\nDetector residuals use the finest topology as reference; admission target <0.2σ RMS",fontsize=12); physical_production_refinement_exports=export_ray_figure(fig_production_refinement,GENERATED/f"neoarm_physical_{SIMULATION_MAGNIFICATION//1000}k_production_refinement_v2_gpu",metadata={"title":"NEOARM production Gaussian refinement","topology_signature":production_signature_digest,"topologies":production_topology_metadata,"matched_blank_frame_id":production_matched_blank_request.frame_id,"q_parameter":production_q_name,"q_truth":production_q_truth,"q_estimates":[value[0] for value in production_q_fits],"q_standard_errors":[value[1] for value in production_q_fits],"certificate_matched_blank_snr":production_peak_z,"detector_residual_reference":"finest topology","detector_residual_rms_in_noise_sigma":production_detector_residual_rms,"detector_residual_display_limit_sigma":residual_limit,"field_display":"fine registered physical patch; coarse validation patch omitted from the figure","topology_converged":physical_production_topology_converged,"independent_reference_method":"B-to-zero inverse-A grid sampling","independent_reference_detector_rms_in_noise_sigma":physical_production_reference_validation.detector_rms_in_noise_sigma,"independent_reference_observable_residuals_in_sigma":physical_production_reference_observable_residuals,"independent_reference_maximum_b_displacement_px":physical_production_grid_reference.maximum_b_displacement_px,"independent_reference_signature":physical_production_reference_signature,"independent_reference_passed":physical_production_reference_passed,"chosen_mode_local_observable_images_admitted":physical_production_images_admitted,"base_checkpoint_sha256":physical_demo_checkpoint.manifest_sha256,"refinement_checkpoint_sha256":production_refinement_checkpoint.manifest_sha256,"slices_per_lens":physical_slices_per_lens,"gpu_kernel_required":True,**FORWARD_RUNTIME_METADATA}); print("Production refinement exports:",physical_production_refinement_exports); plt.show()
        if not physical_production_images_admitted: raise RuntimeError("The chosen-mode physical image failed its Gaussian-refinement and independent conjugate-grid gate.")
    else:
        print("Production refinement checkpoint is partial; rerun with RUN_PHYSICAL_DEMO_SUBSET=True or increase NEOARM_PRODUCTION_REFINEMENT_CHUNK.")

### Physical checkpoint data, registered observables, and identifiable fit

The images above are not decorative. Their detector noise is sampled independently exposure by exposure, robust observables carry registered within-frame covariance, and only quantities supported by the first-order column enter the inverse problem. Synthetic checkpoint covariance can be estimated by bootstrap, but bootstrap regeneration is not an optical-identifiability requirement: `NEOARM_BOOTSTRAP_REPLICATES` controls it, the smoke route uses the minimum eight replicates, and real data may bind an externally estimated covariance. Material intensity and arbitrary image moments are deliberately excluded from the optical fit. Repeated specimen images are precision-pooled only after their certificate and rendering identities agree.

The imaging likelihood uses calibrated basis centroids relative to the same-state perforation frame plus certificate-defined reciprocal vectors. Indexed diffraction uses physical Bragg-spot displacement, $D B(\lambda g)$, measured from Friedel pairs; it does **not** confuse spot position with the inverse-transpose law for Fourier covectors. The condenser aperture/critical-line data are generated again from the same physical truth as these GPU images so that the joint fit has one consistent instrument.


In [ ]:
physical_observation_dataset_ready=False; physical_global_dataset=None; physical_global_specs=(); physical_diffraction_blocks=(); physical_diffraction_specs=()

physical_global_noise=DetectorNoiseModel(dose_electrons=surrogate_dose,incident_intensity_sum=1/float(np.prod(detector.pixel_size_yx_m)/abs(np.linalg.det(detector.affine_xy))),read_noise_e=surrogate_read_noise)

physical_fourier_features_by_frame={}; physical_fourier_covariance_by_frame={}; physical_fourier_routing_failures={}

for request in full_imaging_requests:

    if request.measurement not in {"sample","sample_repeat"}: continue

    item=full_state_items[request.optical_state_label]; feature=known_fourier_features_from_sample_view(atlas,item["view"])

    if feature is None: continue  # certified individual interfaces are added by the boundary-metrology block below

    matrix=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector")); position_block=matrix[np.ix_([0,2],[0,2])]; angle_block=matrix[np.ix_([0,2],[1,3])]
    if request.feature_family=="silicon_lattice_vectors":
        effective_defocus_m=float(.5*np.trace(np.linalg.solve(position_block,angle_block))); lattice_spacing_m=float(atlas.silicon_lattice_region.lattice_spacing_m); phase_ctf=abs(np.sin(np.pi*wavelength_m*effective_defocus_m/lattice_spacing_m**2)); optimum_defocus_m=lattice_spacing_m**2/(2*wavelength_m)
        if phase_ctf<.2: physical_fourier_routing_failures[request.frame_id]={"reason":"phase-only lattice has insufficient in-focus CTF","effective_defocus_m":effective_defocus_m,"phase_ctf":phase_ctf,"first_maximum_defocus_m":optimum_defocus_m,"action":"exclude this real-space Fourier block; acquire signed defocus diversity or use native diffraction"}; continue
    H=np.asarray(detector.affine_xy)@position_block; fov_xy=np.asarray(detector.shape_yx)[::-1]*np.asarray(detector.pixel_size_yx_m)[::-1]; registration=(np.linalg.solve(H.T,np.asarray(feature.world_reciprocal_vectors_xy_per_m).T).T*fov_xy)[:,::-1]

    registered_feature=replace(feature,registration_vectors_yx_px=registration); physical_fourier_features_by_frame[request.frame_id]=registered_feature
    if registered_feature.covariance_world_per_m2 is not None:
        imaging_part=imaging_reciprocal_vectors_from_feature_set(request,registered_feature); physical_fourier_covariance_by_frame[request.frame_id]=fourier_observable_covariance_augmentation(column,physical_parameters,detector,imaging_part)

if physical_fourier_routing_failures: print("Fail-closed real-space Fourier routes:",physical_fourier_routing_failures)
physical_signed_defocus_blocks=(); physical_signed_defocus_specs=(); physical_signed_defocus_product=None; signed_defocus_features_by_frame={}; signed_defocus_covariance_by_frame={}
if signed_defocus_checkpoint.complete:
    for request in signed_defocus_requests:
        item=signed_defocus_items[request.optical_state_label]; feature=known_fourier_features_from_sample_view(atlas,item["view"]); matrix=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector")); position_block=matrix[np.ix_([0,2],[0,2])]; H=np.asarray(detector.affine_xy)@position_block; fov_xy=np.asarray(detector.shape_yx)[::-1]*np.asarray(detector.pixel_size_yx_m)[::-1]; registration=(np.linalg.solve(H.T,np.asarray(feature.world_reciprocal_vectors_xy_per_m).T).T*fov_xy)[:,::-1]; registered=replace(feature,registration_vectors_yx_px=registration); signed_defocus_features_by_frame[request.frame_id]=registered; signed_part=imaging_reciprocal_vectors_from_feature_set(request,registered); signed_defocus_covariance_by_frame[request.frame_id]=fourier_observable_covariance_augmentation(column,physical_parameters,detector,signed_part)
    physical_signed_defocus_product=build_heterogeneous_dataset_from_checkpoint(physical_acquisition_plan,signed_defocus_requests,detector,signed_defocus_checkpoint.checkpoint_directory,render_signature=f"hard-edge-cubic-600k-signed-defocus-v1-gpu-slices{physical_slices_per_lens}",default_noise_model=physical_global_noise,render_metadata=signed_defocus_metadata,storage_dtype="float32",seed=20260906,bootstrap_replicates=PHYSICAL_BOOTSTRAP_REPLICATES,fourier_features_by_frame=signed_defocus_features_by_frame,fourier_covariance_augmentation_by_frame=signed_defocus_covariance_by_frame,moment_observables_by_measurement={"sample":()},covariance_shrinkage=.08)
    physical_signed_defocus_blocks=physical_signed_defocus_product.dataset.blocks; physical_signed_defocus_specs=tuple(PhysicalObservableBlockSpecification(request.frame_id,request.optical_state_index,(imaging_reciprocal_vectors_from_feature_set(request,signed_defocus_features_by_frame[request.frame_id]),),provenance="600k_signed_defocus_checkpoint") for request in signed_defocus_requests)
    for block,spec in zip(physical_signed_defocus_blocks,physical_signed_defocus_specs,strict=True): assert block.observable_names==spec.observable_names
    signed_counts=tuple(np.asarray(physical_signed_defocus_product.frame(request.frame_id).counts_yx,float) for request in signed_defocus_requests); signed_sum=np.maximum(signed_counts[0]+signed_counts[1],1.); signed_difference=2*(signed_counts[1]-signed_counts[0])/signed_sum; signed_difference-=np.mean(signed_difference); signed_spectrum=np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(signed_difference)))**2)
    fig_signed,signed_axes=plt.subplots(1,3,figsize=(11.2,3.7),constrained_layout=True); signed_axes[0].imshow(signed_counts[0],origin="lower",cmap="gray",vmin=0,vmax=np.percentile(signed_counts[0],99.8)); signed_axes[1].imshow(signed_counts[1],origin="lower",cmap="gray",vmin=0,vmax=np.percentile(signed_counts[1],99.8)); signed_axes[2].imshow(signed_spectrum,origin="lower",cmap="magma",vmin=np.percentile(signed_spectrum,5),vmax=np.percentile(signed_spectrum,99.9)); signed_axes[0].set_title(f"−{abs(signed_defocus_design.target_object_defocus_m[0])*1e9:.2f} nm object defocus\nraw detector counts"); signed_axes[1].set_title(f"+{signed_defocus_design.target_object_defocus_m[1]*1e9:.2f} nm object defocus\nraw detector counts"); signed_axes[2].set_title("signed normalized difference\nFourier power"); [axis.set(xticks=[],yticks=[]) for axis in signed_axes]; fig_signed.suptitle("600k× Si {111} phase contrast from one GPU-rendered signed-defocus pair",fontsize=12.5); signed_exports=export_ray_figure(fig_signed,GENERATED/"neoarm_physical_600k_signed_defocus_v1_gpu",metadata={"title":"600k signed object-defocus phase-contrast images","frame_ids":[request.frame_id for request in signed_defocus_requests],"target_object_defocus_m":signed_defocus_design.target_object_defocus_m,"achieved_object_defocus_m":signed_defocus_design.achieved_object_defocus_m,"magnifications":signed_defocus_design.achieved_magnifications,"rotation_changes_rad":signed_defocus_design.cumulative_rotation_changes_rad,"checkpoint_sha256":signed_defocus_checkpoint.manifest_sha256,"gpu_forward":True,**FORWARD_RUNTIME_METADATA}); plt.show(); plt.close(fig_signed); print("600k signed-defocus exports:",signed_exports)
else: print("600k signed-defocus checkpoint is incomplete; the nominal in-focus lattice remains excluded.")
if full_imaging_checkpoint.complete:

    physical_layer_metrology_by_frame={}; physical_layer_registration_diagnostics={}; physical_layer_registration_failures={}

    individual_layer_labels=tuple(dict.fromkeys(request.optical_state_label for request in full_imaging_requests if request.measurement in {"sample","sample_repeat"} and request.feature_family=="individual_layer_widths"))

    for label in individual_layer_labels:

        state_requests=tuple(request for request in full_imaging_requests if request.optical_state_label==label and request.measurement in {"sample","sample_repeat"} and request.feature_family=="individual_layer_widths")

        reference_request=next((request for request in state_requests if request.measurement=="sample" and request.dac_approach=="forward"),state_requests[0]); item=full_state_items[label]; matrix=np.asarray(column.matrix(physical_parameters,np.asarray(reference_request.raw_dac_values),segment="sample_to_detector")); H=np.asarray(detector.affine_xy)@matrix[np.ix_([0,2],[0,2])]; reference_intensity=np.asarray(full_imaging_checkpoint.record(reference_request.frame_id).load_intensity(),float)

        try:

            metrology=registered_layer_metrology_from_sample_view(atlas,item["view"],reference_intensity,detector_position_map_xy_m_per_m=H,detector_pixel_size_yx_m=detector.pixel_size_yx_m,observable_mode="interval_widths",minimum_visible_intervals=2)

            augmentation=layer_observable_covariance_augmentation(metrology); residual=registered_layer_reference_residuals_px(metrology); covariance=np.asarray(augmentation.certificate_covariance_px2); normalized_rms=float(np.sqrt(residual@np.linalg.pinv(covariance)@residual/max(len(residual),1))) if np.any(covariance) else np.inf

            for request in state_requests: physical_layer_metrology_by_frame[request.frame_id]=metrology

            physical_layer_registration_diagnostics[label]={"reference_frame_id":reference_request.frame_id,"n_boundaries":metrology.geometry.n_boundaries,"n_intervals":metrology.geometry.n_intervals,"observable_names":metrology.observable_names,"reference_residual_rms_px":float(np.sqrt(np.mean(residual**2))),"certificate_normalized_rms":normalized_rms}

        except (ValueError,RuntimeError,np.linalg.LinAlgError) as error:

            physical_layer_registration_failures[label]=f"{type(error).__name__}: {error}"

    print(f"Registered certified layer metrology for {len(physical_layer_registration_diagnostics)}/{len(individual_layer_labels)} eligible optical states and {len(physical_layer_metrology_by_frame)} detector exposures; unresolved={physical_layer_registration_failures}.")

    physical_imaging_product=build_heterogeneous_dataset_from_checkpoint(plan,full_imaging_requests,detector,full_imaging_checkpoint.checkpoint_directory,render_signature=f"hard-edge-cubic-native-imaging-v5-gpu-slices{physical_slices_per_lens}",default_noise_model=physical_global_noise,render_metadata=full_metadata,storage_dtype="float32",seed=20260903,bootstrap_replicates=PHYSICAL_BOOTSTRAP_REPLICATES,fourier_features_by_frame=physical_fourier_features_by_frame,fourier_covariance_augmentation_by_frame=physical_fourier_covariance_by_frame,layer_metrology_by_frame=physical_layer_metrology_by_frame,covariance_shrinkage=.08)

    physical_imaging_dataset=aggregate_checkpoint_observables_for_fit(physical_imaging_product,material_repeat_policy="pool")

    # The deliberately defocused held_out_0 frame has a frozen, explicit same-transition period schema. Its noisy pixels are never used to select the registration.
    held_out_0_period_request=next(request for request in full_imaging_requests if request.optical_state_label=="held_out_0" and request.measurement=="sample" and request.dac_approach=="forward"); assert held_out_0_period_request.frame_id==209
    held_out_0_period_item=full_state_items[held_out_0_period_request.optical_state_label]; held_out_0_period_matrix=np.asarray(column.matrix(physical_parameters,np.asarray(held_out_0_period_request.raw_dac_values),segment="sample_to_detector")); held_out_0_period_H=np.asarray(detector.affine_xy)@held_out_0_period_matrix[np.ix_([0,2],[0,2])]; held_out_0_period_reference_intensity=np.asarray(full_imaging_checkpoint.record(held_out_0_period_request.frame_id).load_intensity(),float)
    held_out_0_period_metrology=registered_layer_period_metrology_from_sample_view(atlas,held_out_0_period_item["view"],held_out_0_period_reference_intensity,detector_position_map_xy_m_per_m=held_out_0_period_H,detector_pixel_size_yx_m=detector.pixel_size_yx_m,minimum_peak_snr=1.5)
    held_out_0_period_noisy_frame=physical_imaging_product.frame(held_out_0_period_request.frame_id); held_out_0_period_values,held_out_0_period_names=held_out_0_period_metrology(held_out_0_period_noisy_frame.counts_yx,held_out_0_period_request,physical_global_noise); held_out_0_period_values=np.asarray(held_out_0_period_values,float); held_out_0_period_names=tuple(held_out_0_period_names)
    held_out_0_period_bootstrap=[]; held_out_0_period_bootstrap_seeds=[]
    for replicate in range(PHYSICAL_BOOTSTRAP_REPLICATES):
        seed=int(np.random.SeedSequence((20260903,held_out_0_period_request.frame_id,replicate+1)).generate_state(1)[0]); held_out_0_period_bootstrap_seeds.append(seed); draw=simulate_detector_counts(held_out_0_period_reference_intensity,dose_electrons=physical_global_noise.dose_electrons,incident_intensity_sum=physical_global_noise.incident_intensity_sum,read_noise_e=physical_global_noise.read_noise_e,dark_offset_e=physical_global_noise.dark_offset_e,seed=seed); values,names=held_out_0_period_metrology(draw,held_out_0_period_request,physical_global_noise)
        if tuple(names)!=held_out_0_period_names: raise RuntimeError("Held-out period extractor changed schema during the detector-noise bootstrap.")
        held_out_0_period_bootstrap.append(np.asarray(values,float))
    held_out_0_period_bootstrap=np.asarray(held_out_0_period_bootstrap); held_out_0_period_bootstrap_covariance=np.atleast_2d(np.cov(held_out_0_period_bootstrap,rowvar=False,ddof=1)); held_out_0_period_diagonal=np.maximum(np.diag(held_out_0_period_bootstrap_covariance),0.); held_out_0_period_bootstrap_covariance=.92*held_out_0_period_bootstrap_covariance+.08*np.diag(held_out_0_period_diagonal); held_out_0_period_floor=np.maximum.reduce((held_out_0_period_diagonal*1e-9,(np.abs(held_out_0_period_values)*1e-12)**2,np.full(held_out_0_period_diagonal.shape,1e-18))); held_out_0_period_bootstrap_covariance=.5*(held_out_0_period_bootstrap_covariance+held_out_0_period_bootstrap_covariance.T)+np.diag(held_out_0_period_floor)
    held_out_0_period_geometry=held_out_0_period_metrology.geometry; held_out_0_period_pixel_xy=np.asarray(detector.pixel_size_yx_m)[::-1]; held_out_0_period_world_axis=np.asarray(held_out_0_period_geometry.sample_projection_world_from_local)[:,0]; held_out_0_period_axis_scale_px_per_m=float(np.asarray(held_out_0_period_metrology.registration.detector_normal_xy)@((held_out_0_period_H@held_out_0_period_world_axis)/held_out_0_period_pixel_xy)); held_out_0_period_certificate_covariance=held_out_0_period_axis_scale_px_per_m**2*np.asarray(held_out_0_period_metrology.span_certificate_covariance_local_m2); held_out_0_period_covariance=held_out_0_period_bootstrap_covariance+held_out_0_period_certificate_covariance
    held_out_0_blocks=[(index,block) for index,block in enumerate(physical_imaging_dataset.blocks) if block.optical_state_index==held_out_0_period_request.optical_state_index]; assert len(held_out_0_blocks)==1; held_out_0_block_index,held_out_0_block=held_out_0_blocks[0]; held_out_0_period_prefixed_names=tuple(f"material_frame_{held_out_0_period_request.frame_id}.{name}" for name in held_out_0_period_names); held_out_0_values=np.concatenate((held_out_0_block.values,held_out_0_period_values)); held_out_0_covariance=np.zeros((len(held_out_0_values),len(held_out_0_values))); held_out_0_covariance[:len(held_out_0_block.values),:len(held_out_0_block.values)]=held_out_0_block.covariance; held_out_0_covariance[len(held_out_0_block.values):,len(held_out_0_block.values):]=held_out_0_period_covariance; held_out_0_augmented_block=FrameObservationBlock(held_out_0_block.frame_id,held_out_0_block.optical_state_index,held_out_0_values,held_out_0_covariance,held_out_0_block.observable_names+held_out_0_period_prefixed_names)
    physical_imaging_blocks=list(physical_imaging_dataset.blocks); physical_imaging_blocks[held_out_0_block_index]=held_out_0_augmented_block; physical_imaging_metadata=dict(physical_imaging_dataset.metadata); physical_imaging_aggregation=dict(physical_imaging_metadata["aggregation"]); physical_imaging_state_blocks={key:dict(value) for key,value in physical_imaging_aggregation["state_blocks"].items()}; held_out_0_state_metadata=dict(physical_imaging_state_blocks[str(held_out_0_block.frame_id)]); held_out_0_material_metadata=list(held_out_0_state_metadata.get("material",())); assert not any(held_out_0_period_request.frame_id in tuple(entry.get("member_frame_ids",())) for entry in held_out_0_material_metadata)
    held_out_0_period_covariance_metadata={"observable_names":list(held_out_0_period_names),"bootstrap_covariance_px2":held_out_0_period_bootstrap_covariance.tolist(),"certificate_covariance_px2":held_out_0_period_certificate_covariance.tolist(),"total_covariance_px2":held_out_0_period_covariance.tolist(),"certificate_pushforward_axis_scale_px_per_m":held_out_0_period_axis_scale_px_per_m,"certificate_covariance_added_once":True}; held_out_0_material_metadata.append({"mode":"retained","member_frame_ids":[held_out_0_period_request.frame_id],"known_fourier_features":None,"known_layer_metrology":None,"known_layer_period_metrology":held_out_0_period_metrology.to_metadata(),"layer_period_covariance_augmentation":held_out_0_period_covariance_metadata}); held_out_0_state_metadata["material"]=held_out_0_material_metadata; physical_imaging_state_blocks[str(held_out_0_block.frame_id)]=held_out_0_state_metadata; physical_imaging_aggregation["state_blocks"]=physical_imaging_state_blocks; physical_imaging_metadata["aggregation"]=physical_imaging_aggregation
    physical_imaging_metadata["held_out_0_layer_period"]={"frame_id":held_out_0_period_request.frame_id,"checkpoint_manifest_sha256":full_imaging_checkpoint.manifest_sha256,"actual_detector_noise_seed":held_out_0_period_noisy_frame.noise_seed,"bootstrap_base_seed":20260903,"bootstrap_seed_sequence":"numpy.SeedSequence((base_seed, frame_id, replicate_index_plus_one))","bootstrap_replicates":PHYSICAL_BOOTSTRAP_REPLICATES,"covariance_shrinkage":.08,"registration_image_role":"deterministic pre-fit synthetic checkpoint intensity; held-out noisy pixels never select boundary identity","instrument_transfer_requirement":"freeze period identities from an independent certificate/calibration registration before instrument held-out acquisition","observable_role":"same-transition i-to-i+2 accumulated layer periods; not relabelled individual widths","metrology":held_out_0_period_metrology.to_metadata(),"covariance":held_out_0_period_covariance_metadata}; physical_imaging_dataset=HeterogeneousInversionDataset(physical_imaging_dataset.plan,tuple(physical_imaging_blocks),metadata=physical_imaging_metadata)
    assert held_out_0_period_noisy_frame.request.frame_id==held_out_0_period_request.frame_id and held_out_0_period_noisy_frame.noise_seed==int(np.random.SeedSequence((20260903,held_out_0_period_request.frame_id,0)).generate_state(1)[0]) and np.all(np.isfinite(held_out_0_period_values)) and np.linalg.eigvalsh(held_out_0_period_covariance).min()>0

    physical_imaging_specs=specifications_from_aggregated_dataset(physical_imaging_dataset,full_imaging_requests)

    # A detector-native contact sheet shows the synthetic noisy detector images supplied to metrology, not a schematic sample preview or measured instrument data.

    nominal_material_requests=tuple(next(request for request in full_imaging_requests if request.optical_state_label==state.label and request.measurement=="sample" and request.dac_approach=="forward") for state in plan.states if state.role=="nominal")

    nominal_count_images=tuple(np.asarray(physical_imaging_product.frame(request.frame_id).counts_yx) for request in nominal_material_requests); fig_physical_nominal,nominal_axes=plt.subplots(2,len(nominal_material_requests),figsize=(15.8,6.2),constrained_layout=True)

    for column_index,(request,counts) in enumerate(zip(nominal_material_requests,nominal_count_images,strict=True)):

        display_max=float(np.percentile(counts,99.8)); nominal_axes[0,column_index].imshow(counts,origin="lower",cmap="gray",vmin=0,vmax=display_max); nominal_axes[0,column_index].set_title(f"{int(request.target_magnification)//1000}k× · frame {request.frame_id}\n{request.selected_region_name or 'global atlas'} · {request.feature_family}\nraw detector counts",fontsize=8)

        centred=np.asarray(counts,float)-ndimage.gaussian_filter(np.asarray(counts,float),5.); spectrum=np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(centred)))**2); nominal_axes[1,column_index].imshow(spectrum,origin="lower",cmap="magma",vmin=np.percentile(spectrum,5),vmax=np.percentile(spectrum,99.8)); fourier_status=("excluded: in-focus phase lattice; use ±defocus/diffraction" if request.frame_id in physical_fourier_routing_failures else "measured Fourier evidence admitted"); nominal_axes[1,column_index].set_title(fourier_status,fontsize=8)

    for axis in nominal_axes.ravel(): axis.set(xticks=[],yticks=[])

    fig_physical_nominal.suptitle("One registered MAG*I*CAL-like specimen · physical GPU images from 40k× to 600k×",fontsize=12.5)

    physical_nominal_exports=export_ray_figure(fig_physical_nominal,GENERATED/"neoarm_physical_nominal_images_40k_600k_v1_gpu",metadata={"title":"Physical noisy nominal images across magnification","frame_ids":[request.frame_id for request in nominal_material_requests],"magnifications":[request.target_magnification for request in nominal_material_requests],"feature_families":[request.feature_family for request in nominal_material_requests],"fourier_routing_failures":physical_fourier_routing_failures,"checkpoint_sha256":full_imaging_checkpoint.manifest_sha256,"dose_electrons":physical_global_noise.dose_electrons,"read_noise_e":physical_global_noise.read_noise_e,"detector_shape_yx":detector.shape_yx,"gpu_forward":True,**FORWARD_RUNTIME_METADATA}); plt.show(); plt.close(fig_physical_nominal); print("Physical nominal-image exports:",physical_nominal_exports)

else: print("Physical imaging checkpoint is incomplete; covariance-aware image data remain fail-closed.")

if full_diffraction_checkpoint.complete:

    diffraction_feature=known_fourier_features_from_sample_view(atlas,full_diffraction_item["view"])

    if diffraction_feature is None or len(diffraction_feature.labels)!=2: raise RuntimeError("Native diffraction requires two certified Si reciprocal vectors.")

    diffraction_blocks=[]; diffraction_specs=[]; diffraction_noisy_counts={}; diffraction_registration={}; diffraction_covariances={}; diffraction_covariance_augmentations={}
    synthetic_high_voltage_sigma_eV=20.; wavelength_sigma_m=abs(float(jax.grad(energy2wavelength)(jnp.asarray(float(beam_cfg["voltage_v"])))))*synthetic_high_voltage_sigma_eV; wavelength_variance_m2=wavelength_sigma_m**2

    for binding in native_diffraction_bindings:

        request=binding.request; part=indexed_diffraction_bragg_spots_from_feature_set(request,diffraction_feature,wavelength_m,"200 kV accelerating-voltage calibration via energy2wavelength; synthetic 1σ=20 eV, replace with instrument readback",wavelength_variance_m2=wavelength_variance_m2,name_prefix=f"diffraction_frame_{request.frame_id}.",wavelength_calibration_group_name="neoarm-electron-wavelength"); spec=PhysicalObservableBlockSpecification(request.frame_id,request.optical_state_index,(part,)); registration=np.asarray(predict_physical_observable_blocks(column,jnp.asarray(physical_parameters),detector,(spec,))[0]).reshape(-1,2); diffraction_covariance_augmentations[request.frame_id]=fourier_observable_covariance_augmentation(column,physical_parameters,detector,part)

        expected=np.asarray(full_diffraction_checkpoint.record(request.frame_id).load_intensity(),float); actual_seed=int(np.random.SeedSequence((20260903,request.frame_id,0)).generate_state(1)[0]); counts=simulate_detector_counts(expected,dose_electrons=physical_global_noise.dose_electrons,incident_intensity_sum=physical_global_noise.incident_intensity_sum,read_noise_e=physical_global_noise.read_noise_e,seed=actual_seed); measured=extract_registered_bragg_spot_offsets(counts,registration,diffraction_feature.labels,search_radius_px=6.).values

        bootstrap=[]

        for replicate in range(PHYSICAL_BOOTSTRAP_REPLICATES):

            seed=int(np.random.SeedSequence((20260903,request.frame_id,replicate+1)).generate_state(1)[0]); draw=simulate_detector_counts(expected,dose_electrons=physical_global_noise.dose_electrons,incident_intensity_sum=physical_global_noise.incident_intensity_sum,read_noise_e=physical_global_noise.read_noise_e,seed=seed); bootstrap.append(extract_registered_bragg_spot_offsets(draw,registration,diffraction_feature.labels,search_radius_px=6.).values)

        bootstrap=np.asarray(bootstrap); covariance=np.cov(bootstrap,rowvar=False,ddof=1); diagonal=np.maximum(np.diag(covariance),1e-8); covariance=.92*covariance+.08*np.diag(diagonal)+np.diag(np.maximum(diagonal*1e-9,1e-10))+diffraction_covariance_augmentations[request.frame_id].total_covariance_observable2; names=spec.observable_names; diffraction_blocks.append(FrameObservationBlock(request.frame_id,request.optical_state_index,measured,covariance,names)); diffraction_specs.append(spec); diffraction_noisy_counts[request.frame_id]=counts; diffraction_registration[request.frame_id]=registration; diffraction_covariances[request.frame_id]=covariance

    physical_diffraction_blocks=tuple(diffraction_blocks); physical_diffraction_specs=tuple(diffraction_specs)

    if full_imaging_checkpoint.complete:

        physical_global_specs=physical_imaging_specs+physical_signed_defocus_specs+physical_diffraction_specs; held_out_state_indices=set(np.flatnonzero(plan.held_out_mask)); held_out_states_with_declared_material={spec.optical_state_index for spec in physical_global_specs if spec.optical_state_index in held_out_state_indices and any(part.request.measurement in {"sample","sample_repeat","indexed_diffraction"} for part in spec.parts)}; assert held_out_states_with_declared_material==held_out_state_indices, f"Held-out material schema is incomplete: expected {sorted(held_out_state_indices)}, found {sorted(held_out_states_with_declared_material)}."; assert any(part.request.frame_id==held_out_0_period_request.frame_id and any("layer_period_" in name for name in part.observable_names) for spec in physical_global_specs for part in spec.parts)
        physical_uncertainty_status=("provisional_fixed_covariance_smoke" if PHYSICAL_BOOTSTRAP_REPLICATES<64 else "synthetic_parametric_bootstrap"); physical_global_metadata={**dict(physical_imaging_dataset.metadata),"source":"GPU physical imaging, 600k signed defocus, and native hard-edge diffraction checkpoints","data_kind":"synthetic_gpu_checkpoint","instrument_observation_vector_bound":False,"instrument_covariance_bound":False,"imaging":physical_imaging_dataset.metadata,"signed_defocus_checkpoint_manifest_sha256":(None if not signed_defocus_checkpoint.complete else signed_defocus_checkpoint.manifest_sha256),"diffraction_checkpoint_manifest_sha256":full_diffraction_checkpoint.manifest_sha256,"diffraction_observable":"Friedel-pair Bragg displacement D B lambda g","diffraction_wavelength_m":wavelength_m,"diffraction_wavelength_sigma_m":wavelength_sigma_m,"diffraction_wavelength_provenance":"200 kV accelerating-voltage calibration; synthetic 1σ=20 eV, replace with instrument readback","layer_registration_diagnostics":physical_layer_registration_diagnostics,"layer_registration_failures":physical_layer_registration_failures,"fourier_routing_failures":physical_fourier_routing_failures,"layer_certificate_covariance":"propagated per block; preserved across same-state repeat pooling; cross-state shared correlation requires nuisance-factor likelihood","held_out_material_schema_state_indices":sorted(held_out_states_with_declared_material),"unsupported_observables_excluded":["unresolved in-focus silicon-lattice Fourier vectors","arbitrary material moments"],"uncertainty_estimation":{"status":physical_uncertainty_status,"bootstrap_replicates":PHYSICAL_BOOTSTRAP_REPLICATES,"covariance_shrinkage":.08,"claim":"synthetic likelihood weighting only; instrument-grade uncertainty requires registered experimental covariance"}}
        global_fourier_prior_entries=list(physical_global_metadata.get("fourier_calibration_covariance",{}).get("shared_input_priors",()))
        if physical_signed_defocus_product is not None: global_fourier_prior_entries.extend(physical_signed_defocus_product.dataset.metadata.get("fourier_calibration_covariance",{}).get("shared_input_priors",()))
        for augmentation in diffraction_covariance_augmentations.values(): global_fourier_prior_entries.extend(augmentation.to_metadata().get("shared_input_priors",()))
        global_fourier_priors_by_group={}
        for entry in global_fourier_prior_entries:
            group=str(entry.get("group_name","")); previous=global_fourier_priors_by_group.get(group)
            if not group or (previous is not None and previous!=entry): raise RuntimeError(f"Inconsistent global Fourier calibration prior {group!r}.")
            global_fourier_priors_by_group[group]=entry
        global_fourier_metadata=dict(physical_global_metadata.get("fourier_calibration_covariance",{})); global_fourier_metadata["shared_input_priors"]=list(global_fourier_priors_by_group.values()); global_fourier_metadata["per_frame_output_covariance_included"]=bool(global_fourier_priors_by_group); physical_global_metadata["fourier_calibration_covariance"]=global_fourier_metadata
        physical_global_frame_metadata=dict(physical_global_metadata.get("frame_metadata",{}))
        for request in signed_defocus_requests:
            if physical_signed_defocus_product is not None: physical_global_frame_metadata[str(request.frame_id)]=dict(physical_signed_defocus_product.dataset.metadata["frame_metadata"][str(request.frame_id)])
        for request in (binding.request for binding in native_diffraction_bindings): physical_global_frame_metadata[str(request.frame_id)]={"measurement":request.measurement,"fourier_covariance_augmentation":diffraction_covariance_augmentations[request.frame_id].to_metadata()}
        physical_global_metadata["frame_metadata"]=physical_global_frame_metadata; physical_global_dataset=HeterogeneousInversionDataset(physical_acquisition_plan,physical_imaging_dataset.blocks+physical_signed_defocus_blocks+physical_diffraction_blocks,metadata=physical_global_metadata); physical_statistical_dataset=strip_shared_multilayer_certificate_covariance(physical_global_dataset,column,physical_parameters,detector,physical_global_specs); physical_calibration_priors,physical_calibration_predictor=make_calibration_aware_physical_predictor(column,physical_parameters,detector,physical_global_specs,dataset=physical_statistical_dataset); physical_observation_layout=GlobalObservationLayout.from_dataset(physical_statistical_dataset); physical_shared_likelihood=SharedLatentNuisanceLikelihood(physical_observation_layout,column.n_parameters,physical_calibration_predictor,calibration_priors=physical_calibration_priors); physical_observation_dataset_ready=True
        physical_training_block_indices=tuple(index for index,block in enumerate(physical_statistical_dataset.blocks) if physical_statistical_dataset.plan.states[block.optical_state_index].role!="held_out"); physical_held_out_block_indices=tuple(index for index,block in enumerate(physical_statistical_dataset.blocks) if physical_statistical_dataset.plan.states[block.optical_state_index].role=="held_out"); assert physical_observation_layout.selected_block_indices==physical_training_block_indices and physical_held_out_block_indices and set(physical_training_block_indices).isdisjoint(physical_held_out_block_indices)

        assert sum(block.values.size for block in physical_global_dataset.blocks)==physical_global_dataset.n_observations

        print(f"Physical inverse dataset: {len(physical_global_dataset.blocks)} image/defocus/diffraction blocks, {physical_global_dataset.n_observations} correlated observables; training blocks={len(physical_training_block_indices)}, excluded held-out blocks={len(physical_held_out_block_indices)}; shared Fourier/wavelength/unified-layer-certificate latents={physical_shared_likelihood.n_calibration_latents}; real/reciprocal certificate covariance stripped exactly once={physical_statistical_dataset.metadata['multilayer_certificate_covariance']['stripped_for_shared_latent_likelihood']}; uncertainty={physical_uncertainty_status} ({PHYSICAL_BOOTSTRAP_REPLICATES} parametric draws, 0.08 shrinkage; not an instrument-grade covariance claim); imaging checkpoint={full_imaging_checkpoint.manifest_sha256[:12]}, signed defocus={(signed_defocus_checkpoint.manifest_sha256[:12] if signed_defocus_checkpoint.complete else 'incomplete')}, diffraction={full_diffraction_checkpoint.manifest_sha256[:12]}.")

else: print("Native diffraction checkpoint is incomplete; its physical Bragg observables remain fail-closed.")

### Exact physical-render signatures and independent grid-wave/analytic admission

A topology hash is not a forward-model validation. The same specimen grid can be illuminated by a different calibrated input, propagated by a different downstream map, sampled by a different detector affine, or evaluated by a different renderer. The ledger below therefore predeclares every mean field used by the likelihood—including material, perforation, calibrated position/angle basis, signed-defocus, and native-diffraction frames—by a complete `PhysicalRenderSignature`: acquisition kind, frozen topology, interaction route, calibrated source and sample input, exact DAC/approach-bound downstream matrix, detector, checkpoint renderer, and the explicit `neoarm_physical_column` scope. The visibility-only ideal relay is structurally incapable of satisfying this contract.

Two genuinely separate references are used. Material and diffraction validation starts with an analytic one-Gaussian illumination, multiplies it by the cubic sample on the frozen grid, and propagates the resulting field with `round_solenoid_grid_reference`. Its fail-closed `auto` dispatcher uses the certified inverse-map limit for conjugate states, a padded scaled-Fresnel FFT for near-conjugate states, and the Simpson/CZT Collins integral for general or diffraction states. Imaging retains two intermediate samples per detector pixel. Every native-diffraction reference is computed independently at 7× and 8× CZT refinement, must change by at most 0.01 experimental standard deviations in both detector RMS and every retained observable, and uses the 8× result for the unchanged 0.2σ candidate gate. Vacuum/perforation and calibrated-basis frames instead use a NumPy closed-form single-Gaussian ABCD intensity evaluator, sharing neither the production lifted-Gaussian propagation nor its GPU field kernel; a conservative elliptical-Gaussian tail bound must certify less than $10^{-10}$ power clipped by the finite, uncertainty-reduced perforation. Absolute counts are never normalized: the total-count uncertainty is evaluated analytically as $[\sum_p(\mu_p+\sigma_{\rm read}^2)]^{1/2}$, while nonlinear geometric observables retain the registered bootstrap. This separates reference convergence from candidate agreement and records both grids, shapes, totals, and diagnostics. A candidate is admitted only when detector RMS is at most 0.2 experimental noise standard deviations, every acquisition-relevant observable changes by at most 0.2 of its uncertainty, and—where sample multiplication is present—the off-grid Gaussian re-expansion error passes the same 0.2-noise-sigma field gate.

Highly magnifying round maps can evaluate $AD-BC$ by subtracting two numbers of order $10^8$. The notebook therefore retains the raw determinant residual but certifies it against an explicit IEEE-binary64 $\gamma_4(|AD|+|BC|+1)$ backward-error bound; it never projects or alters the map coefficients. The deliberately defocused held-out multilayer frame also keeps its failed individual-width association as provenance. Its distinct validation observable is four certificate-defined same-transition $i\rightarrow i+2$ accumulated spans (`layer_period`), with the original SNR and step-improvement gates intact; it is never silently relabelled as an individual layer width.

The ledger is a validation-only gate; its scalar residual map is not inserted into the inverse likelihood. For later nuisance-bias analysis, each evidence record additionally preserves the signed component-ordered candidate-minus-reference vector, its bootstrap covariance, and the effective standard errors used by the gate. Failed records from an older numerical policy remain as historical audit evidence, but a passing record whose stored grid options do not match the current per-signature policy is never restored as admission evidence.

Parameter stability is deliberately **not** assigned per frame: one image cannot possess a fitted full-column parameter change. Consequently the per-signature requirement keeps `require_parameter_stability=False`. Once every exact likelihood mean field is admitted, the next inversion cell maps the signed candidate-minus-independent-reference changes into the exact pooled observation order and whitens them with the fit covariance. The acceptance quantity then projects both the signed bias and the scaled optical Jacobian off all numerically resolved nuisance response before restricting them to the admitted optical quotient. Its basis-invariant optical Fisher displacement must be strictly below $0.1\sigma$ at both the declared and final linearizations. A separate 61-dimensional optical-plus-nuisance Fisher displacement is retained as a deliberately non-gating diagnostic; allowing calibration latents to move must not be confused with instability of recovered microscope optics. Per-likelihood-block signed Fisher-mode and fixed-information leave-block diagnostics identify useful targeted rerenders without deleting those blocks or their information from the experiment.

The two real, strictly denser 150k Gaussian-topology transitions above must also remain below $0.1\sigma$, and an independent-reference comparison is never mislabeled as a second refinement. Those transitions currently refit one conditional `IL1.q_bias` one-parameter proxy while every other optical and nuisance coordinate is fixed; they establish representative chosen-mode Gaussian convergence, **not** literal two-refinement convergence of an identifiable full-column combination or all 23 admitted optical directions. Until equally dense global rerenders/refits are run, the all-identifiable-parameter successive-refinement acceptance remains explicitly provisional even when the independent-reference optical influence gate passes. Missing signed evidence, schema mismatch, optical-rank loss, an incomplete additive block decomposition, or missing representative refinement history all fail closed.

With `NEOARM_RUN_FULL_RENDERS=1`, the default reference selection evaluates six deliberately distinct representatives: conjugate imaging, scaled-Fresnel/FFT imaging, general-CZT imaging, native physical-column diffraction, and both signed 600k defocus states. This demonstrates the numerical routes and the signed phase-contrast pair without implying complete coverage. Set `NEOARM_GRID_REFERENCE_SELECTION=all` (and `NEOARM_RUN_PHYSICAL_GRID_REFERENCES=1`) for the expensive signature-by-signature admission run. Missing or failed entries remain in the JSON ledger and keep the physical-image global fit disabled.


In [ ]:
PHYSICAL_OPTICAL_SCOPE="neoarm_physical_column"; PHYSICAL_CANDIDATE_RENDERER="hard_edge_sliced_cubic_gaussian_reexpansion_gpu"; PHYSICAL_REFERENCE_RENDERER="independent_scaled_grid_wave_scipy"; PHYSICAL_REFERENCE_NOTEBOOK_VERSION="neoarm-notebook-grid-validation-v2"
PHYSICAL_ANALYTIC_REFERENCE_RENDERER="independent_analytic_single_gaussian_numpy"; PHYSICAL_ANALYTIC_REFERENCE_NOTEBOOK_VERSION="neoarm-notebook-vacuum-validation-v1"; PHYSICAL_VACUUM_TOPOLOGY_ID="vacuum-perforation:single-gaussian:no-reexpansion:v1"
physical_reference_implementation_sha256=hashlib.sha256((REPO/"src/temgym_core/grid_wave_reference.py").read_bytes()).hexdigest(); physical_analytic_reference_implementation_sha256=hashlib.sha256((REPO/"src/temgym_core/topology_validation.py").read_bytes()).hexdigest()
physical_validation_thresholds=PhysicalValidationThresholds(sample_field_rms_in_noise_sigma=.2,detector_rms_in_noise_sigma=.2,observable_residual_in_sigma=.2,parameter_change_in_sigma=.1)
physical_grid_reference_options=GridWaveOptions(conjugate_maximum_b_displacement_px=1e-3,czt_maximum_kernel_phase_step_rad=.8*np.pi,maximum_input_edge_power_fraction=1e-3,scaled_fresnel_fft_enabled=True,fft_padding_factor=3.,fft_maximum_nyquist_shift_fraction_of_padding=.8,fft_maximum_paraxial_angle_rad=.1,fft_minimum_samples_per_detector_pixel=1.,output_oversampling=2.,interpolation_halo_pixels=3,maximum_intermediate_points=int(os.environ.get("NEOARM_GRID_REFERENCE_MAX_POINTS","96000000")),require_power_conservation=False)
PHYSICAL_DIFFRACTION_PREVIOUS_OUTPUT_OVERSAMPLING=7.
PHYSICAL_DIFFRACTION_OUTPUT_OVERSAMPLING=8.
def physical_grid_reference_options_for_signature(signature):
    return replace(physical_grid_reference_options,output_oversampling=max(physical_grid_reference_options.output_oversampling,PHYSICAL_DIFFRACTION_OUTPUT_OVERSAMPLING)) if signature.acquisition_kind=="diffraction" else physical_grid_reference_options

def physical_beam_state_arrays(beam):
    vector=beam.to_vector(); return {name:np.asarray(getattr(vector,name)) for name in ("x","y","dx","dy","z","voltage","amplitude","pathlength","Q_inv")}

def physical_source_provenance(request,source):
    arrays={f"source_{name}":value for name,value in physical_beam_state_arrays(source).items()}; arrays["sample_state_xytheta"]=np.asarray(request.sample_state.as_array()); arrays["sample_state_covariance"]=np.asarray(request.calibration_covariance); arrays["source_to_sample_matrix"]=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="source_to_sample")); return stable_validation_signature("neoarm-calibrated-source-v1",arrays=arrays,metadata={"voltage_v":beam_cfg["voltage_v"],"sample_z_m":column.sample_z_m,"wavelength_m":wavelength_m,"input_calibration":"metres_and_radians","slices_per_lens":physical_slices_per_lens})

def physical_downstream_provenance(request,*,native_mode_signature=None):
    arrays={"physical_parameters":physical_parameters,"raw_dac_values":np.asarray(request.raw_dac_values),"dac_approach_start_values":np.asarray(request.dac_approach_start_values),"sample_to_detector_matrix":np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector"))}; return stable_validation_signature("neoarm-downstream-map-v1",arrays=arrays,metadata={"raw_dac_order":list(plan.channel_names),"dac_approach":request.dac_approach,"dac_ramp_steps":request.dac_ramp_steps,"settling_time_s":request.settling_time_s,"slices_per_lens":physical_slices_per_lens,"native_mode_signature":native_mode_signature})

physical_detector_signature=stable_validation_signature("medipix-detector-v1",arrays={"x_coords_m":detector.x_coords_m,"y_coords_m":detector.y_coords_m,"affine_xy":detector.affine_xy},metadata={"shape_yx":list(detector.shape_yx),"pixel_size_yx_m":list(detector.pixel_size_yx_m),"centre_xy_m":list(detector.centre_xy_m),"z_m":detector.z_m,"mtf_sigma_px":detector.mtf_sigma_px,"coordinate_convention":"pixel_centres_yx_storage"})
full_imaging_render_identity=stable_validation_signature("neoarm-candidate-render-v1",metadata={"checkpoint_manifest_sha256":full_imaging_checkpoint.manifest_sha256,"checkpoint_render_signature":f"hard-edge-cubic-native-imaging-v5-gpu-slices{physical_slices_per_lens}","config_sha256":hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),"certificate":atlas.certificate.version,"renderer_family":PHYSICAL_CANDIDATE_RENDERER,"model_scope":PHYSICAL_OPTICAL_SCOPE})
signed_defocus_render_identity=stable_validation_signature("neoarm-candidate-render-v1",metadata={"checkpoint_manifest_sha256":signed_defocus_checkpoint.manifest_sha256,"checkpoint_render_signature":f"hard-edge-cubic-600k-signed-defocus-v1-gpu-slices{physical_slices_per_lens}","config_sha256":hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),"certificate":atlas.certificate.version,"renderer_family":PHYSICAL_CANDIDATE_RENDERER,"model_scope":PHYSICAL_OPTICAL_SCOPE,"phase_contrast_route":"signed_object_defocus"})
full_diffraction_render_identity=stable_validation_signature("neoarm-candidate-render-v1",metadata={"checkpoint_manifest_sha256":full_diffraction_checkpoint.manifest_sha256,"checkpoint_render_signature":native_diffraction_render_signature,"config_sha256":hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),"certificate":atlas.certificate.version,"renderer_family":PHYSICAL_CANDIDATE_RENDERER,"model_scope":PHYSICAL_OPTICAL_SCOPE,"uses_ideal_relay":False})

physical_signature_by_frame_id={}; physical_request_by_signature={}; physical_item_by_signature={}; physical_source_by_signature={}; physical_checkpoint_by_signature={}; physical_expected_route_by_signature={}; physical_signature_requirements={}
def physical_likelihood_mean_field_frame_ids(specifications):
    frame_ids=set()
    for specification in specifications:
        for part in specification.parts:
            frame_ids.add(part.request.frame_id)
            baseline=getattr(part,"perforation_request",None)
            if baseline is not None: frame_ids.add(baseline.frame_id)
    return frozenset(frame_ids)
physical_likelihood_frame_ids=physical_likelihood_mean_field_frame_ids(physical_global_specs); physical_vacuum_likelihood_measurements={"perforation","position_basis","angle_basis","basis_orthogonal","basis_negative"}
def physical_layer_validation_observable(request):
    return "layer_period" if (request.optical_state_label,request.measurement,request.dac_approach)==("held_out_0","sample","forward") else "layer_interval"
for request in full_imaging_requests:
    if request.measurement not in {"sample","sample_repeat"}: continue
    item=full_state_items[request.optical_state_label]; signature=PhysicalRenderSignature(acquisition_kind="imaging",frozen_topology_id=request.frozen_topology_id,interaction_route=request.interaction_route,source_signature=physical_source_provenance(request,item["source"]),downstream_signature=physical_downstream_provenance(request),detector_signature=physical_detector_signature,render_signature=full_imaging_render_identity,renderer_family=PHYSICAL_CANDIDATE_RENDERER,optical_scope=PHYSICAL_OPTICAL_SCOPE); physical_signature_by_frame_id[request.frame_id]=signature; physical_request_by_signature.setdefault(signature,request); physical_item_by_signature.setdefault(signature,item); physical_source_by_signature.setdefault(signature,item["source"]); physical_checkpoint_by_signature.setdefault(signature,full_imaging_checkpoint)
    feature=physical_fourier_features_by_frame.get(request.frame_id); required=("scale","rotation","centroid","fourier_vector","intensity") if feature is not None else (("scale","centroid",physical_layer_validation_observable(request),"intensity") if request.feature_family=="individual_layer_widths" else ("centroid","intensity")); physical_signature_requirements.setdefault(signature,PhysicalValidationRequirements(required_observables=required,require_sample_field=True,require_parameter_stability=False,require_independent_reference=True,allowed_optical_scopes=(PHYSICAL_OPTICAL_SCOPE,),thresholds=physical_validation_thresholds))
for request in full_imaging_requests:
    if request.frame_id not in physical_likelihood_frame_ids or request.measurement not in physical_vacuum_likelihood_measurements: continue
    source=full_vacuum_source_by_state[request.optical_state_label]; signature=PhysicalRenderSignature(acquisition_kind="imaging",frozen_topology_id=PHYSICAL_VACUUM_TOPOLOGY_ID,interaction_route="vacuum",source_signature=physical_source_provenance(request,source),downstream_signature=physical_downstream_provenance(request),detector_signature=physical_detector_signature,render_signature=full_imaging_render_identity,renderer_family=PHYSICAL_CANDIDATE_RENDERER,optical_scope=PHYSICAL_OPTICAL_SCOPE); physical_signature_by_frame_id[request.frame_id]=signature; physical_request_by_signature.setdefault(signature,request); physical_item_by_signature.setdefault(signature,None); physical_source_by_signature.setdefault(signature,source); physical_checkpoint_by_signature.setdefault(signature,full_imaging_checkpoint); physical_signature_requirements.setdefault(signature,PhysicalValidationRequirements(required_observables=("centroid","intensity"),require_sample_field=False,require_parameter_stability=False,require_independent_reference=True,allowed_optical_scopes=(PHYSICAL_OPTICAL_SCOPE,),thresholds=physical_validation_thresholds))
for request in signed_defocus_requests:
    item=signed_defocus_items[request.optical_state_label]; signature=PhysicalRenderSignature(acquisition_kind="imaging",frozen_topology_id=request.frozen_topology_id,interaction_route=request.interaction_route,source_signature=physical_source_provenance(request,item["source"]),downstream_signature=physical_downstream_provenance(request),detector_signature=physical_detector_signature,render_signature=signed_defocus_render_identity,renderer_family=PHYSICAL_CANDIDATE_RENDERER,optical_scope=PHYSICAL_OPTICAL_SCOPE); physical_signature_by_frame_id[request.frame_id]=signature; physical_request_by_signature.setdefault(signature,request); physical_item_by_signature.setdefault(signature,item); physical_source_by_signature.setdefault(signature,item["source"]); physical_checkpoint_by_signature.setdefault(signature,signed_defocus_checkpoint); physical_signature_requirements.setdefault(signature,PhysicalValidationRequirements(required_observables=("scale","rotation","centroid","fourier_vector","intensity"),require_sample_field=True,require_parameter_stability=False,require_independent_reference=True,allowed_optical_scopes=(PHYSICAL_OPTICAL_SCOPE,),thresholds=physical_validation_thresholds))
for binding in native_diffraction_bindings:
    request=binding.request; source=full_diffraction_source_by_frame_id[request.frame_id]; signature=PhysicalRenderSignature(acquisition_kind="diffraction",frozen_topology_id=request.frozen_topology_id,interaction_route=request.interaction_route,source_signature=physical_source_provenance(request,source),downstream_signature=physical_downstream_provenance(request,native_mode_signature=binding.mode.model_signature),detector_signature=physical_detector_signature,render_signature=full_diffraction_render_identity,renderer_family=PHYSICAL_CANDIDATE_RENDERER,optical_scope=PHYSICAL_OPTICAL_SCOPE); physical_signature_by_frame_id[request.frame_id]=signature; physical_request_by_signature.setdefault(signature,request); physical_item_by_signature.setdefault(signature,full_diffraction_item); physical_source_by_signature.setdefault(signature,source); physical_checkpoint_by_signature.setdefault(signature,full_diffraction_checkpoint); physical_expected_route_by_signature[signature]="czt"; physical_signature_requirements.setdefault(signature,PhysicalValidationRequirements(required_observables=("scale","rotation","centroid","bragg_vector","intensity"),require_sample_field=True,require_parameter_stability=False,require_independent_reference=True,allowed_optical_scopes=(PHYSICAL_OPTICAL_SCOPE,),thresholds=physical_validation_thresholds))
physical_unbound_likelihood_frame_ids=physical_likelihood_frame_ids-set(physical_signature_by_frame_id)
if physical_unbound_likelihood_frame_ids: raise RuntimeError(f"Likelihood mean-field frames lack exact render signatures: {sorted(physical_unbound_likelihood_frame_ids)}.")
physical_likelihood_signatures=frozenset(physical_signature_by_frame_id[frame_id] for frame_id in physical_likelihood_frame_ids); physical_vacuum_signatures=tuple(signature for signature in physical_likelihood_signatures if signature.interaction_route=="vacuum")
physical_validation_ledger=PhysicalValidationLedger(physical_signature_requirements)
if any(signature.optical_scope!=PHYSICAL_OPTICAL_SCOPE or "ideal" in signature.renderer_family.lower() for signature in physical_validation_ledger.required_signatures): raise RuntimeError("Only physical-column candidate signatures may enter the validation ledger.")
physical_validation_ledger_path=GENERATED/"neoarm_physical_validation_ledger_v2.json"

def restore_physical_validation_ledger(path):
    if not path.is_file(): return 0
    try:
        document=json.loads(path.read_text(encoding="utf-8")); stored_sha=document.pop("sha256"); calculated_sha=hashlib.sha256(json.dumps(document,sort_keys=True,separators=(",",":"),ensure_ascii=False,allow_nan=False).encode("utf-8")).hexdigest()
        if stored_sha!=calculated_sha or document.get("schema")!="temgym_core.physical_topology_validation_ledger" or document.get("version")!=1: raise ValueError("ledger checksum/schema mismatch")
        current_by_digest={signature.digest:signature for signature in physical_validation_ledger.required_signatures}; restored=0
        for entry in document.get("entries",()):
            signature=current_by_digest.get(entry.get("signature_digest"))
            if signature is None or entry.get("signature")!=signature.to_payload(): continue
            for payload in entry.get("evidence",()):
                provenance=payload.get("provenance",{}); reference_family=payload.get("reference_renderer_family")
                if payload.get("signature_digest")!=signature.digest: continue
                if reference_family==PHYSICAL_REFERENCE_RENDERER:
                    implementation_is_current=provenance.get("reference_implementation_sha256")==physical_reference_implementation_sha256 and provenance.get("reference_notebook_version")==PHYSICAL_REFERENCE_NOTEBOOK_VERSION; stored_grid_options=provenance.get("grid_wave_options"); expected_grid_options=vars(physical_grid_reference_options_for_signature(signature)); legacy_imaging_options=stored_grid_options is None and signature.acquisition_kind=="imaging"; numerical_policy_is_current=legacy_imaging_options or stored_grid_options==expected_grid_options
                elif reference_family==PHYSICAL_ANALYTIC_REFERENCE_RENDERER:
                    implementation_is_current=provenance.get("reference_implementation_sha256")==physical_analytic_reference_implementation_sha256 and provenance.get("reference_notebook_version")==PHYSICAL_ANALYTIC_REFERENCE_NOTEBOOK_VERSION; numerical_policy_is_current=provenance.get("perforation_loss_bound_policy")=="elliptical-Gaussian chi-square-2 upper bound with uncertain edge subtracted; maximum 1e-10"
                else: continue
                if not implementation_is_current or not numerical_policy_is_current: continue
                evidence=PhysicalValidationEvidence(signature=signature,reference_method=payload["reference_method"],reference_signature=payload["reference_signature"],reference_renderer_family=payload["reference_renderer_family"],reference_optical_scope=payload["reference_optical_scope"],detector_rms_in_noise_sigma=payload["detector_rms_in_noise_sigma"],observable_residuals_in_sigma=payload["observable_residuals_in_sigma"],sample_field_rms_in_noise_sigma=payload.get("sample_field_rms_in_noise_sigma"),parameter_change_in_sigma=payload.get("parameter_change_in_sigma"),provenance=payload.get("provenance"))
                physical_validation_ledger.record(evidence); restored+=1
        return restored
    except (KeyError,TypeError,ValueError,json.JSONDecodeError) as error:
        print(f"Existing physical-validation ledger was not trusted and no evidence was restored: {error}"); return 0

physical_restored_evidence_count=restore_physical_validation_ledger(physical_validation_ledger_path)
if physical_restored_evidence_count: print(f"Restored {physical_restored_evidence_count} checksum-verified evidence records whose complete signatures still match.")

def physical_analytic_vacuum_observable_residuals(candidate_counts,reference_counts):
    candidate_counts=np.asarray(candidate_counts,float); reference_counts=np.asarray(reference_counts,float); candidate_observables=extract_image_observables(np.clip(candidate_counts,0,None)); reference_observables=extract_image_observables(np.clip(reference_counts,0,None)); centroid_difference=np.asarray(candidate_observables.centroid_xy_px)-np.asarray(reference_observables.centroid_xy_px); yy,xx=np.indices(reference_counts.shape,dtype=float); centroid_xy=np.asarray(reference_observables.centroid_xy_px); noise_variance=np.maximum(reference_counts,0.)+physical_global_noise.read_noise_e**2; total=max(float(reference_counts.sum()),1.); dx=xx-centroid_xy[0]; dy=yy-centroid_xy[1]; centroid_covariance=np.asarray(((np.sum(noise_variance*dx*dx),np.sum(noise_variance*dx*dy)),(np.sum(noise_variance*dx*dy),np.sum(noise_variance*dy*dy))))/total**2; centroid_sigma=np.maximum(np.sqrt(np.maximum(np.diag(centroid_covariance),0.)),.02); intensity_difference=float(candidate_observables.total_intensity-reference_observables.total_intensity); intensity_sigma=max(poisson_read_total_standard_error(reference_counts,read_noise_e=physical_global_noise.read_noise_e),1.); residuals={"centroid":float(np.max(np.abs(centroid_difference)/centroid_sigma)),"intensity":abs(intensity_difference)/intensity_sigma}; signed={"component_names":["centroid_x_px","centroid_y_px","intensity"],"candidate_minus_reference":[*centroid_difference.tolist(),intensity_difference],"reference_values":[*np.asarray(reference_observables.centroid_xy_px).tolist(),float(reference_observables.total_intensity)],"candidate_values":[*np.asarray(candidate_observables.centroid_xy_px).tolist(),float(candidate_observables.total_intensity)],"effective_standard_errors":[*centroid_sigma.tolist(),intensity_sigma],"standardized_signed_residuals":[*(centroid_difference/centroid_sigma).tolist(),intensity_difference/intensity_sigma],"analytic_covariance":np.block([[centroid_covariance,np.zeros((2,1))],[np.zeros((1,2)),np.asarray(((intensity_sigma**2,),))]]),"standard_error_methods":["Poisson/read-noise centroid delta method","Poisson/read-noise centroid delta method","analytic independent Poisson plus read-noise detector sum"],"bootstrap_replicates_retained":0,"likelihood_use":"validation-only; not propagated into the inverse likelihood"}; return residuals,signed

def physical_round_map(matrix):
    values=np.asarray(matrix,float); position=np.asarray((0,2)); angle=np.asarray((1,3)); blocks=(values[np.ix_(position,position)],values[np.ix_(position,angle)],values[np.ix_(angle,position)],values[np.ix_(angle,angle)]); pivot=blocks[int(np.argmax([np.linalg.norm(block) for block in blocks]))]; U,_,Vh=np.linalg.svd(pivot); rotation=U@Vh
    if np.linalg.det(rotation)<0: U[:,-1]*=-1; rotation=U@Vh
    radial=tuple(float(.5*np.trace(rotation.T@block)) for block in blocks); defect=max(float(np.linalg.norm(block-scalar*rotation))/max(float(np.linalg.norm(block)),1.) for scalar,block in zip(radial,blocks,strict=True)); symplectic_certificate=certify_radial_symplectic_map(radial,absolute_tolerance=1e-8)
    if defect>2e-8 or not symplectic_certificate.passed: raise RuntimeError(f"The requested downstream map is not a certified round symplectic map (round-block defect={defect:.3e}, determinant error={symplectic_certificate.determinant_error:.3e}, evaluation bound={symplectic_certificate.determinant_evaluation_roundoff_bound:.3e}).")
    return radial,float(np.arctan2(rotation[1,0],rotation[0,0])),defect,symplectic_certificate

def physical_analytic_single_gaussian(beam,xy):
    vector=beam.to_vector(); centres=np.asarray(vector.r_xy); slopes=np.asarray(vector.d_xy); amplitudes=np.asarray(vector.amplitude); Q=np.asarray(vector.Q_inv); wave_numbers=np.asarray(vector.k)
    if centres.shape[0]!=1: raise ValueError("The independent atlas reference expects exactly one analytic illumination Gaussian before sample multiplication.")
    delta=np.asarray(xy,float)-centres[0]; action=np.einsum("...i,i->...",delta,slopes[0])+.5*np.einsum("...i,ij,...j->...",delta,Q[0],delta); return amplitudes[0]*np.exp(1j*wave_numbers[0]*action)

def physical_at_sample_beam(request,source):
    upstream=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="source_to_sample")); source_z=float(np.ravel(np.asarray(source.z))[0]); beam=apply_transfer_matrix_to_gaussians(source,upstream,z_out_m=column.sample_z_m,axial_path_m=column.sample_z_m-source_z); state=request.sample_state; return beam.derive(x=beam.x+state.x,y=beam.y+state.y,dx=beam.dx+state.theta_x,dy=beam.dy+state.theta_y)

def physical_exact_sample_product(request,item,source):
    x=np.asarray(item["x"]); y=np.asarray(item["y"]); xx,yy=np.meshgrid(x,y,indexing="xy"); xy=np.stack((xx,yy),axis=-1); beam=physical_at_sample_beam(request,source); incident=physical_analytic_single_gaussian(beam,xy); actions=np.asarray(item["view"].evaluate_actions(jnp.asarray(xy.reshape(-1,2)))).reshape((*xy.shape[:2],2)); transmitted=incident*np.exp(actions[...,1]+2j*np.pi*actions[...,0]/wavelength_m); return beam,transmitted

def physical_off_grid_reexpansion_error(request,item,beam,transmitted):
    x=np.asarray(item["x"]); y=np.asarray(item["y"]); dx=float(np.diff(x).mean()); dy=float(np.diff(y).mean()); centre=np.asarray((float(np.ravel(np.asarray(beam.x))[0]),float(np.ravel(np.asarray(beam.y))[0]))); waist=float(item["illumination_waist_m"]); half=min(1.6*waist,.45*(x[-1]-x[0]),.45*(y[-1]-y[0])); qx=np.linspace(max(x[2],centre[0]-half),min(x[-3],centre[0]+half),23)+.271*dx; qy=np.linspace(max(y[2],centre[1]-half),min(y[-3],centre[1]+half),21)-.193*dy; qx=np.clip(qx,x[2],x[-3]); qy=np.clip(qy,y[2],y[-3]); qxx,qyy=np.meshgrid(qx,qy,indexing="xy"); qxy=np.stack((qxx,qyy),axis=-1); actions=np.asarray(item["view"].evaluate_actions(jnp.asarray(qxy.reshape(-1,2)))).reshape((*qxy.shape[:2],2)); exact=physical_analytic_single_gaussian(beam,qxy)*np.exp(actions[...,1]+2j*np.pi*actions[...,0]/wavelength_m); coefficients=np.asarray(gaussian_cardinal_coefficients(transmitted,width_over_step_yx=physical_width_over,kernel_tolerance=1e-13,ridge=1e-10,method="scipy")); gx=np.exp(-((qx[:,None]-x[None,:])/(physical_width_over[1]*dx))**2); gy=np.exp(-((qy[:,None]-y[None,:])/(physical_width_over[0]*dy))**2); reconstructed=gy@coefficients@gx.T; overlap=np.vdot(exact,reconstructed)
    if abs(overlap)>0: reconstructed*=np.exp(-1j*np.angle(overlap))
    exact_rms=float(np.sqrt(np.mean(np.abs(exact)**2))); field_noise=max(.02*exact_rms,np.finfo(float).tiny); return float(np.sqrt(np.mean(np.abs(reconstructed-exact)**2))/field_noise),{"validation_shape_yx":list(exact.shape),"field_noise_fraction":.02,"exact_field_rms":exact_rms}

def physical_registered_fourier_vectors(image,target_vectors_yx):
    values=np.asarray(image,float); spectrum=np.abs(np.fft.fftshift(np.fft.fft2(values-ndimage.gaussian_filter(values,5.))))**2; centre=np.asarray(values.shape)//2; found=[]
    for target in np.asarray(target_vectors_yx,float):
        location=centre+np.rint(target).astype(int); y0,y1=max(0,location[0]-4),min(values.shape[0],location[0]+5); x0,x1=max(0,location[1]-4),min(values.shape[1],location[1]+5); patch=spectrum[y0:y1,x0:x1]; background=float(np.median(patch)); weights=np.maximum(patch-background,0.)
        if not np.any(weights>0): raise RuntimeError("A registered Fourier window contains no positive peak weight.")
        py,px=np.mgrid[y0:y1,x0:x1]; found.append((np.sum(weights*py)/np.sum(weights)-centre[0],np.sum(weights*px)/np.sum(weights)-centre[1]))
    return np.asarray(found)

def physical_reference_summary_factory(request,item,reference_counts):
    if request.measurement=="indexed_diffraction":
        feature=known_fourier_features_from_sample_view(atlas,item["view"]); binding=native_diffraction_binding_by_frame_id[request.frame_id]; part=IndexedDiffractionBraggSpots(request,feature.labels,feature.world_reciprocal_vectors_xy_per_m,wavelength_m,"200 kV accelerating-voltage calibration via energy2wavelength"); spec=PhysicalObservableBlockSpecification(request.frame_id,request.optical_state_index,(part,)); registration=np.asarray(predict_physical_observable_blocks(column,jnp.asarray(physical_parameters),detector,(spec,))[0]).reshape(-1,2)
        def summary(image):
            vectors=extract_registered_bragg_spot_offsets(image,registration,feature.labels,search_radius_px=6.).offsets_yx_px; obs=extract_image_observables(np.clip(image,0,None)); return {"scale":np.linalg.norm(vectors,axis=1),"rotation":np.rad2deg(np.arctan2(vectors[:,0],vectors[:,1])),"centroid":np.asarray(obs.centroid_xy_px),"bragg_vector":vectors.reshape(-1),"intensity":np.asarray((obs.total_intensity,))}
        return summary
    feature=physical_fourier_features_by_frame.get(request.frame_id,signed_defocus_features_by_frame.get(request.frame_id))
    if feature is not None:
        registration=np.asarray(feature.registration_vectors_yx_px)
        def summary(image):
            vectors=physical_registered_fourier_vectors(image,registration); obs=extract_image_observables(np.clip(image,0,None)); return {"scale":np.linalg.norm(vectors,axis=1),"rotation":np.rad2deg(np.arctan2(vectors[:,0],vectors[:,1])),"centroid":np.asarray(obs.centroid_xy_px),"fourier_vector":vectors.reshape(-1),"intensity":np.asarray((obs.total_intensity,))}
        return summary
    if request.feature_family=="individual_layer_widths":
        downstream=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector")); H=np.asarray(detector.affine_xy)@downstream[np.ix_([0,2],[0,2])]; layer_observable=physical_layer_validation_observable(request)
        if layer_observable=="layer_period": metrology=registered_layer_period_metrology_from_sample_view(atlas,item["view"],reference_counts,detector_position_map_xy_m_per_m=H,detector_pixel_size_yx_m=detector.pixel_size_yx_m,minimum_peak_snr=1.5)
        else: metrology=registered_layer_metrology_from_sample_view(atlas,item["view"],reference_counts,detector_position_map_xy_m_per_m=H,detector_pixel_size_yx_m=detector.pixel_size_yx_m,observable_mode="interval_widths",minimum_peak_snr=1.5)
        def summary(image):
            values=np.asarray(metrology.values(metrology.extract(image))); obs=extract_image_observables(np.clip(image,0,None)); return {"scale":values,"centroid":np.asarray(obs.centroid_xy_px),layer_observable:values,"intensity":np.asarray((obs.total_intensity,))}
        return summary
    def summary(image):
        obs=extract_image_observables(np.clip(image,0,None)); return {"centroid":np.asarray(obs.centroid_xy_px),"intensity":np.asarray((obs.total_intensity,))}
    return summary

def physical_observable_residuals(candidate_counts,reference_counts,summary,request):
    reference=summary(reference_counts); candidate=summary(candidate_counts); bootstrap=[]; rng=np.random.default_rng(1_406_000+request.frame_id)
    for _ in range(32):
        draw=rng.poisson(np.maximum(reference_counts,0.)).astype(float)
        if physical_global_noise.read_noise_e>0: draw+=rng.normal(0,physical_global_noise.read_noise_e,size=draw.shape)
        try: bootstrap.append(summary(np.clip(draw,0,None)))
        except (RuntimeError,ValueError): pass
    if len(bootstrap)<16: raise RuntimeError(f"Only {len(bootstrap)} of 32 observable bootstraps retained their frozen registration.")
    floors={"scale":.01,"rotation":.05,"centroid":.02,"fourier_vector":.035,"layer_interval":.03,"layer_period":.03,"bragg_vector":.035,"intensity":1.}; residuals={}; component_names=[]; signed_differences=[]; effective_errors=[]; standardized_signed=[]; bootstrap_columns=[]; reference_components=[]; candidate_components=[]; standard_error_methods=[]
    for name,reference_value in reference.items():
        ref=np.ravel(np.asarray(reference_value,float)); cand=np.ravel(np.asarray(candidate[name],float)); draws=np.stack([np.ravel(np.asarray(row[name],float)) for row in bootstrap]); differences=cand-ref
        if name=="rotation": differences=(differences+180.)%360.-180.; draws=ref+(draws-ref+180.)%360.-180.
        if name=="intensity": sigma=np.asarray((max(poisson_read_total_standard_error(reference_counts,read_noise_e=physical_global_noise.read_noise_e),floors[name]),)); method="analytic independent Poisson plus read-noise detector sum"
        else: sigma=np.maximum(np.std(draws,axis=0,ddof=1),floors[name]); method="registered parametric image bootstrap with declared floor"
        residuals[name]=float(np.max(np.abs(differences)/sigma)); component_names.extend(f"{name}[{index}]" for index in range(len(ref))); signed_differences.extend(differences.tolist()); effective_errors.extend(sigma.tolist()); standardized_signed.extend((differences/sigma).tolist()); bootstrap_columns.append(draws); reference_components.extend(ref.tolist()); candidate_components.extend(cand.tolist()); standard_error_methods.extend([method]*len(ref))
    bootstrap_matrix=np.concatenate(bootstrap_columns,axis=1); covariance=np.atleast_2d(np.cov(bootstrap_matrix,rowvar=False,ddof=1)); signed_metadata={"component_names":component_names,"candidate_minus_reference":signed_differences,"reference_values":reference_components,"candidate_values":candidate_components,"effective_standard_errors":effective_errors,"standardized_signed_residuals":standardized_signed,"standard_error_methods":standard_error_methods,"bootstrap_covariance":covariance,"bootstrap_replicates_retained":len(bootstrap),"intensity_standard_error_formula":"sqrt(sum_p(mu_p + read_noise_e**2)); raw absolute counts, no normalization","likelihood_use":"validation-only; not propagated into the inverse likelihood"}; return residuals,signed_metadata

def validate_physical_signature(signature,expected_route=None):
    request=physical_request_by_signature[signature]; item=physical_item_by_signature[signature]; source=physical_source_by_signature[signature]; checkpoint=physical_checkpoint_by_signature[signature]; record=checkpoint.record(request.frame_id)
    if record.frozen_topology_id!=signature.frozen_topology_id or record.actual_interaction_route!=signature.interaction_route: raise RuntimeError("Checkpoint record does not match its exact topology/interaction signature.")
    reference_options=physical_grid_reference_options_for_signature(signature); beam,transmitted=physical_exact_sample_product(request,item,source); sample_error,field_provenance=physical_off_grid_reexpansion_error(request,item,beam,transmitted); matrix=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector")); radial,rotation,round_defect,radial_symplectic_certificate=physical_round_map(matrix)
    count_scale=physical_global_noise.dose_electrons/physical_global_noise.incident_intensity_sum; previous_counts=None; previous_reference_provenance=None
    if signature.acquisition_kind=="diffraction":
        previous_options=replace(reference_options,output_oversampling=PHYSICAL_DIFFRACTION_PREVIOUS_OUTPUT_OVERSAMPLING); previous_reference=round_solenoid_grid_reference(transmitted,np.asarray(item["x"]),np.asarray(item["y"]),detector.x_coords_m,detector.y_coords_m,radial_abcd=radial,cumulative_rotation_rad=rotation,detector_affine_xy=detector.affine_xy,detector_centre_xy_m=detector.centre_xy_m,wavelength_m=wavelength_m,detector_mtf_sigma_px=detector.mtf_sigma_px,route="auto",options=previous_options); previous_counts=np.asarray(previous_reference.intensity_yx,float)*count_scale+physical_global_noise.dark_offset_e; previous_reference_provenance={"grid_wave_options":vars(previous_options),"grid_wave_diagnostics":vars(previous_reference.diagnostics),"intermediate_shape_yx":list(previous_reference.diagnostics.intermediate_shape_yx),"total_counts":float(np.sum(previous_counts))}; del previous_reference; gc.collect()
    reference=round_solenoid_grid_reference(transmitted,np.asarray(item["x"]),np.asarray(item["y"]),detector.x_coords_m,detector.y_coords_m,radial_abcd=radial,cumulative_rotation_rad=rotation,detector_affine_xy=detector.affine_xy,detector_centre_xy_m=detector.centre_xy_m,wavelength_m=wavelength_m,detector_mtf_sigma_px=detector.mtf_sigma_px,route="auto",options=reference_options)
    if expected_route is not None and reference.diagnostics.route!=expected_route: raise RuntimeError(f"Representative {request.optical_state_label!r} selected {reference.diagnostics.route!r}, expected {expected_route!r}.")
    candidate=np.asarray(record.load_intensity(),float); candidate_counts=candidate*count_scale+physical_global_noise.dark_offset_e; reference_counts=np.asarray(reference.intensity_yx,float)*count_scale+physical_global_noise.dark_offset_e; detector_noise=np.sqrt(np.maximum(reference_counts,1.)+physical_global_noise.read_noise_e**2); detector_rms=float(np.sqrt(np.mean(((candidate_counts-reference_counts)/detector_noise)**2))); summary=physical_reference_summary_factory(request,item,reference_counts)
    successive_reference_convergence=None
    if previous_counts is not None:
        refinement_observable_residuals,refinement_signed_discrepancy=physical_observable_residuals(previous_counts,reference_counts,summary,request); convergence=assess_grid_reference_convergence(previous_counts,reference_counts,detector_noise,observable_residuals_in_sigma=refinement_observable_residuals,maximum_detector_rms_in_noise_sigma=.01,maximum_observable_residual_in_sigma=.01); successive_reference_convergence={**vars(convergence),"previous_reference":previous_reference_provenance,"refined_grid_wave_options":vars(reference_options),"refined_grid_wave_diagnostics":vars(reference.diagnostics),"refined_intermediate_shape_yx":list(reference.diagnostics.intermediate_shape_yx),"refined_total_counts":float(np.sum(reference_counts)),"signed_observable_discrepancy":refinement_signed_discrepancy}
        if not convergence.passed: raise RuntimeError(f"Successive 7x-to-8x diffraction reference failed closed: {convergence.failure_reasons}.")
    layer_validation_route=physical_layer_validation_observable(request) if request.feature_family=="individual_layer_widths" else None; observable_residuals,signed_observable_discrepancy=physical_observable_residuals(candidate_counts,reference_counts,summary,request); reference_signature=stable_validation_signature("neoarm-independent-grid-wave-reference-v2",arrays={"transmitted_sample_field_yx":transmitted,"radial_abcd":np.asarray(radial),"candidate_intensity_yx":candidate},metadata={"candidate_signature":signature.digest,"route":reference.diagnostics.route,"grid_wave_options":vars(reference_options),"round_structure_defect":round_defect,"radial_symplectic_certificate":vars(radial_symplectic_certificate),"layer_metrology_validation_route":layer_validation_route,"successive_reference_convergence":successive_reference_convergence,"reference_implementation_sha256":physical_reference_implementation_sha256,"reference_notebook_version":PHYSICAL_REFERENCE_NOTEBOOK_VERSION})
    evidence=PhysicalValidationEvidence(signature=signature,reference_method=f"round_solenoid_grid_reference:auto->{reference.diagnostics.route}",reference_signature=reference_signature,reference_renderer_family=PHYSICAL_REFERENCE_RENDERER,reference_optical_scope=PHYSICAL_OPTICAL_SCOPE,detector_rms_in_noise_sigma=detector_rms,observable_residuals_in_sigma=observable_residuals,sample_field_rms_in_noise_sigma=sample_error,parameter_change_in_sigma=None,provenance={"frame_id":request.frame_id,"optical_state_label":request.optical_state_label,"checkpoint_manifest_sha256":checkpoint.manifest_sha256,"reference_implementation_sha256":physical_reference_implementation_sha256,"reference_notebook_version":PHYSICAL_REFERENCE_NOTEBOOK_VERSION,"grid_wave_options":vars(reference_options),"round_structure_defect":round_defect,"radial_symplectic_certificate":vars(radial_symplectic_certificate),"layer_metrology_validation_route":layer_validation_route,"successive_reference_convergence":successive_reference_convergence,"shared_upstream":"calibrated analytic source propagated by the exact source-to-sample first-order map; independent reference begins before cubic multiplication","sample_field_candidate":"Gaussian cardinal re-expansion on the exact frozen grid, assessed at deterministic off-grid points after global-phase registration","sample_field":field_provenance,"grid_wave_diagnostics":vars(reference.diagnostics),"signed_observable_discrepancy":signed_observable_discrepancy,"noise_model":{"dose_electrons":physical_global_noise.dose_electrons,"incident_intensity_sum":physical_global_noise.incident_intensity_sum,"read_noise_e":physical_global_noise.read_noise_e,"dark_offset_e":physical_global_noise.dark_offset_e}}); compact_reference=replace(reference,intermediate_field_yx=np.empty((0,0),dtype=np.complex128),intermediate_x_coords_m=np.empty(0),intermediate_y_coords_m=np.empty(0)); return evidence,compact_reference

def validate_analytic_vacuum_signature(signature):
    request=physical_request_by_signature[signature]; source=physical_source_by_signature[signature]; checkpoint=physical_checkpoint_by_signature[signature]; record=checkpoint.record(request.frame_id)
    if signature.interaction_route!="vacuum" or signature.frozen_topology_id!=PHYSICAL_VACUUM_TOPOLOGY_ID or record.frozen_topology_id is not None or record.actual_interaction_route!="vacuum": raise RuntimeError("Vacuum checkpoint record does not match its explicit no-reexpansion render signature.")
    at_sample=physical_at_sample_beam(request,source); perforation_world_centre=np.asarray(request.sample_translation_xy_m)+perforation_projection@np.asarray(atlas.perforation.centre_xy_m); perforation_certificate=certify_single_gaussian_perforation(at_sample,perforation_centre_xy_m=perforation_world_centre,perforation_radius_m=.5*atlas.certificate.perforation_diameter_m,uncertain_edge_width_m=atlas.certificate.interface_width_m,maximum_lost_power_fraction=1e-10)
    if not perforation_certificate.passed: raise RuntimeError(f"Frame {request.frame_id} is not certified as untruncated by the finite perforation: loss bound={perforation_certificate.lost_power_fraction_upper_bound:.3e}.")
    matrix=np.asarray(column.matrix(physical_parameters,np.asarray(request.raw_dac_values),segment="sample_to_detector")); reference=analytic_single_gaussian_detector_reference(at_sample,matrix,detector); candidate=np.asarray(record.load_intensity(),float); count_scale=physical_global_noise.dose_electrons/physical_global_noise.incident_intensity_sum; candidate_counts=candidate*count_scale+physical_global_noise.dark_offset_e; reference_counts=np.asarray(reference.intensity_yx,float)*count_scale+physical_global_noise.dark_offset_e; detector_noise=np.sqrt(np.maximum(reference_counts,1.)+physical_global_noise.read_noise_e**2); detector_rms=float(np.sqrt(np.mean(((candidate_counts-reference_counts)/detector_noise)**2))); observable_residuals,signed_observable_discrepancy=physical_analytic_vacuum_observable_residuals(candidate_counts,reference_counts)
    reference_signature=stable_validation_signature("neoarm-independent-analytic-vacuum-reference-v1",arrays={**{f"sample_{name}":value for name,value in physical_beam_state_arrays(at_sample).items()},"sample_to_detector_matrix":matrix,"reference_intensity_yx":reference.intensity_yx},metadata={"candidate_signature":signature.digest,"reference_implementation_sha256":physical_analytic_reference_implementation_sha256,"reference_notebook_version":PHYSICAL_ANALYTIC_REFERENCE_NOTEBOOK_VERSION,"perforation_loss_bound_policy":"elliptical-Gaussian chi-square-2 upper bound with uncertain edge subtracted; maximum 1e-10","detector_mtf_model":"declared discrete Gaussian PSF, scipy ndimage, nearest boundary","finite_detector_grid":True})
    evidence=PhysicalValidationEvidence(signature=signature,reference_method="closed-form single-Gaussian ABCD intensity on finite detector plus certified negligible perforation clipping",reference_signature=reference_signature,reference_renderer_family=PHYSICAL_ANALYTIC_REFERENCE_RENDERER,reference_optical_scope=PHYSICAL_OPTICAL_SCOPE,detector_rms_in_noise_sigma=detector_rms,observable_residuals_in_sigma=observable_residuals,sample_field_rms_in_noise_sigma=None,parameter_change_in_sigma=None,provenance={"frame_id":request.frame_id,"measurement":request.measurement,"optical_state_label":request.optical_state_label,"checkpoint_manifest_sha256":checkpoint.manifest_sha256,"reference_implementation_sha256":physical_analytic_reference_implementation_sha256,"reference_notebook_version":PHYSICAL_ANALYTIC_REFERENCE_NOTEBOOK_VERSION,"perforation_loss_bound_policy":"elliptical-Gaussian chi-square-2 upper bound with uncertain edge subtracted; maximum 1e-10","perforation_certificate":vars(perforation_certificate),"analytic_detector":{"sampled_power_fraction":reference.sampled_power_fraction,"mtf_sampled_power_fraction":reference.mtf_sampled_power_fraction,"detector_centre_column_xy_m":reference.detector_centre_column_xy_m,"detector_intensity_covariance_column_m2":reference.detector_intensity_covariance_column_m2,"detector_intensity_covariance_pixel2":reference.detector_intensity_covariance_pixel2,"detector_mtf_sigma_px":reference.detector_mtf_sigma_px,"finite_shape_yx":list(reference.intensity_yx.shape)},"independence":"NumPy closed-form intensity quadratic; no lifted Gaussian propagation and no production GPU field evaluator","signed_observable_discrepancy":signed_observable_discrepancy,"noise_model":{"dose_electrons":physical_global_noise.dose_electrons,"incident_intensity_sum":physical_global_noise.incident_intensity_sum,"read_noise_e":physical_global_noise.read_noise_e,"dark_offset_e":physical_global_noise.dark_offset_e}}); return evidence,reference

physical_analytic_reference_results={}; physical_analytic_reference_validation_errors={}; analytic_already_accepted=set(physical_validation_ledger.coverage.accepted_signature_digests)
for index,signature in enumerate(sorted(physical_vacuum_signatures,key=lambda value:value.digest),1):
    if signature.digest in analytic_already_accepted: continue
    request=physical_request_by_signature[signature]
    try:
        evidence,reference=validate_analytic_vacuum_signature(signature); decision=physical_validation_ledger.record(evidence); physical_analytic_reference_results[signature.digest]=(evidence,reference,decision)
        if not decision.passed: raise RuntimeError(f"Analytic vacuum evidence failed admission: {decision.failure_reasons}.")
    except (RuntimeError,ValueError,KeyError,OSError,np.linalg.LinAlgError) as error:
        physical_analytic_reference_validation_errors[signature.digest]=f"{type(error).__name__}: {error}"; print(f"analytic vacuum reference {index}/{len(physical_vacuum_signatures)} · {request.optical_state_label} · frame {request.frame_id}: FAILED CLOSED · {error}")
physical_validation_ledger.write_json(physical_validation_ledger_path)
if physical_vacuum_signatures: print(f"Analytic finite-perforation/vacuum coverage: {sum(signature.digest in physical_validation_ledger.coverage.accepted_signature_digests for signature in physical_vacuum_signatures)}/{len(physical_vacuum_signatures)} exact likelihood signatures accepted; newly evaluated={len(physical_analytic_reference_results)}, errors={len(physical_analytic_reference_validation_errors)}.")

representative_specs=(("nominal_2","conjugate"),("single_IL3_minus","scaled_fresnel_fft"),("single_IL2_minus","czt")); representative_signatures=[]
for label,route in representative_specs:
    request=next(request for request in full_imaging_requests if request.optical_state_label==label and request.measurement=="sample" and request.dac_approach=="forward"); signature=physical_signature_by_frame_id[request.frame_id]; representative_signatures.append(signature); physical_expected_route_by_signature[signature]=route
representative_signatures.append(physical_signature_by_frame_id[native_diffraction_bindings[0].request.frame_id])
representative_signatures.extend(physical_signature_by_frame_id[request.frame_id] for request in signed_defocus_requests)
selection=os.environ.get("NEOARM_GRID_REFERENCE_SELECTION","representative").strip().lower()
if selection not in {"representative","all"}: raise ValueError("NEOARM_GRID_REFERENCE_SELECTION must be 'representative' or 'all'.")
requested_signatures=tuple(signature for signature in physical_validation_ledger.required_signatures if signature.interaction_route!="vacuum") if selection=="all" else tuple(dict.fromkeys(representative_signatures)); already_accepted=set(physical_validation_ledger.coverage.accepted_signature_digests); selected_signatures=tuple(signature for signature in requested_signatures if signature.digest not in already_accepted); physical_reference_results={}; physical_reference_validation_errors={}
if RUN_PHYSICAL_GRID_REFERENCES:
    for index,signature in enumerate(selected_signatures,1):
        request=physical_request_by_signature[signature]
        try:
            evidence,reference=validate_physical_signature(signature,physical_expected_route_by_signature.get(signature)); decision=physical_validation_ledger.record(evidence); physical_validation_ledger.write_json(physical_validation_ledger_path); physical_reference_results[signature.digest]=(evidence,reference,decision); print(f"grid reference {index}/{len(selected_signatures)} · {request.optical_state_label} · frame {request.frame_id} · {reference.diagnostics.route}: detector={evidence.detector_rms_in_noise_sigma:.3e}σ, observables={evidence.observable_residual_map}, field={evidence.sample_field_rms_in_noise_sigma:.3e}σ, pass={decision.passed}")
        except (GridWaveSamplingError,RuntimeError,ValueError,KeyError,OSError,np.linalg.LinAlgError) as error:
            physical_reference_validation_errors[signature.digest]=f"{type(error).__name__}: {error}"; print(f"grid reference {index}/{len(selected_signatures)} · {request.optical_state_label} · frame {request.frame_id}: FAILED CLOSED · {error}")
        finally: gc.collect()
else: print("Independent material/diffraction grid references are disabled; any unrestored material signature remains missing. Analytic vacuum/basis references were still evaluated above.")
physical_validation_ledger.write_json(physical_validation_ledger_path); physical_validation_coverage=physical_validation_ledger.coverage; physical_render_set_complete=bool(full_imaging_checkpoint.complete and signed_defocus_checkpoint.complete and full_diffraction_checkpoint.complete and native_diffraction_contract_ready); physical_observation_dataset_ready=isinstance(globals().get("physical_global_dataset"),HeterogeneousInversionDataset); physical_image_global_fit_ready=bool(physical_render_set_complete and physical_observation_dataset_ready and physical_validation_coverage.complete)
print(f"Exact physical-render coverage: {physical_validation_coverage.accepted_count}/{physical_validation_coverage.required_count} accepted, {physical_validation_coverage.failed_count} failed, {physical_validation_coverage.missing_count} missing; likelihood mean-field signatures={len(physical_likelihood_signatures)} including analytic vacuum/basis={len(physical_vacuum_signatures)}; renders complete={physical_render_set_complete}; dataset assembled={physical_observation_dataset_ready}; global-fit inputs ready={physical_image_global_fit_ready}.")
print("Validation ledger:",physical_validation_ledger_path.relative_to(REPO))
if selection!="all": print("Representative validation never implies complete coverage; use NEOARM_GRID_REFERENCE_SELECTION=all for admission.")

# Build one signed renderer-bias vector in the exact downstream likelihood schema. Parameter change is evaluated jointly in the next cell, never copied into per-frame ledger records.
physical_fit_bias_ready=False; physical_fit_bias_whitened=None; physical_fit_bias_provenance={}; physical_fit_bias_by_frame={}
def physical_latest_accepted_evidence(signature):
    accepted=[evidence for evidence,decision in zip(physical_validation_ledger.evidence_for(signature),physical_validation_ledger.decisions_for(signature),strict=True) if decision.passed]
    if not accepted: raise RuntimeError(f"No accepted signed evidence exists for {signature.digest}.")
    return accepted[-1]
def physical_signed_component(evidence,family):
    signed=evidence.provenance.get("signed_observable_discrepancy",{}); names=tuple(str(value) for value in signed.get("component_names",())); values=np.asarray(signed.get("candidate_minus_reference",()),float)
    if values.shape!=(len(names),) or not np.all(np.isfinite(values)): raise RuntimeError(f"Malformed signed discrepancy for {evidence.signature.digest}.")
    if family=="centroid" and {"centroid_x_px","centroid_y_px"}.issubset(names): return np.asarray((values[names.index("centroid_x_px")],values[names.index("centroid_y_px")]))
    prefix=f"{family}["; indices=[index for index,name in enumerate(names) if name.startswith(prefix) and name.endswith("]")]; component_indices=[int(names[index][len(prefix):-1]) for index in indices]
    if not indices or component_indices!=list(range(len(indices))): raise RuntimeError(f"Signed {family!r} components are missing or unordered for {evidence.signature.digest}.")
    return values[np.asarray(indices,dtype=int)]
def physical_material_source_block(frame_id):
    matches=[block for block in physical_imaging_product.dataset.blocks if block.frame_id==frame_id]
    if len(matches)!=1: raise RuntimeError(f"Expected one raw material block for frame {frame_id}, found {len(matches)}.")
    return matches[0]
def physical_frame_fit_bias(frame_id):
    request=next((candidate for candidate in (*full_imaging_requests,*signed_defocus_requests,*(binding.request for binding in native_diffraction_bindings)) if candidate.frame_id==frame_id),None)
    if request is None: raise RuntimeError(f"No request exists for fit-bias frame {frame_id}.")
    signature=physical_signature_by_frame_id[frame_id]; evidence=physical_latest_accepted_evidence(signature)
    if signature.interaction_route=="vacuum": names=("centroid_x_px","centroid_y_px"); values=physical_signed_component(evidence,"centroid")
    elif request.measurement=="indexed_diffraction": names=next(spec.observable_names for spec in physical_diffraction_specs if spec.frame_id==frame_id); values=physical_signed_component(evidence,"bragg_vector")
    elif frame_id in {candidate.frame_id for candidate in signed_defocus_requests}: names=next(block.observable_names for block in physical_signed_defocus_blocks if block.frame_id==frame_id); values=physical_signed_component(evidence,"fourier_vector")
    elif physical_layer_validation_observable(request)=="layer_period": names=held_out_0_period_names; values=physical_signed_component(evidence,"layer_period")
    else:
        source=physical_material_source_block(frame_id); names=tuple(name for name in source.observable_names if name.startswith(("fourier_","layer_")))
        if not names: return {"values_by_name":{},"signature_digest":signature.digest,"frame_id":frame_id}
        family="fourier_vector" if names[0].startswith("fourier_") else "layer_interval"; values=physical_signed_component(evidence,family)
    if len(names)!=len(values): raise RuntimeError(f"Signed evidence/schema length mismatch for frame {frame_id}: {len(values)} versus {len(names)}.")
    return {"values_by_name":dict(zip(names,np.asarray(values,float),strict=True)),"signature_digest":signature.digest,"frame_id":frame_id}
def physical_independent_pool_covariance(frame_id,names):
    source=physical_material_source_block(frame_id); indices=np.asarray([source.observable_names.index(name) for name in names],dtype=int); covariance=np.asarray(source.covariance)[np.ix_(indices,indices)].copy(); metadata=physical_imaging_product.dataset.metadata["frame_metadata"][str(frame_id)]; augmentation=metadata.get("layer_covariance_augmentation") or metadata.get("fourier_covariance_augmentation")
    if isinstance(augmentation,dict): covariance-=np.asarray(augmentation["total_covariance_px2"] if "total_covariance_px2" in augmentation else augmentation["total_covariance_observable2"],float)
    covariance=.5*(covariance+covariance.T); eigenvalues=np.linalg.eigvalsh(covariance); scale=max(float(np.max(np.abs(covariance),initial=0.)),1e-18); floor=max(1e-18,1e-12*scale)
    if eigenvalues.min()<-1e-8*scale: raise RuntimeError(f"Frame {frame_id} covariance is smaller than its shared calibration term.")
    if eigenvalues.min()<floor: covariance+=np.eye(len(names))*(floor-eigenvalues.min())
    return covariance
if physical_image_global_fit_ready:
    physical_fit_bias_by_frame={frame_id:physical_frame_fit_bias(frame_id) for frame_id in physical_signature_by_frame_id}
    bias_blocks=[]; assignment_blocks=[]; used_signatures=set(); used_signatures_by_block=[]; aggregation=physical_statistical_dataset.metadata["aggregation"]["state_blocks"]
    for block in physical_statistical_dataset.blocks:
        bias=np.zeros(block.values.size); assigned=np.zeros(block.values.size,dtype=bool); block_used_signatures=set(); name_to_index={name:index for index,name in enumerate(block.observable_names)}; state_metadata=aggregation.get(str(block.frame_id))
        if isinstance(state_metadata,dict):
            baseline_id=int(state_metadata["baseline_frame_id"]); baseline_record=physical_fit_bias_by_frame[baseline_id]; baseline=np.asarray((baseline_record["values_by_name"]["centroid_x_px"],baseline_record["values_by_name"]["centroid_y_px"])); used_signatures.add(baseline_record["signature_digest"]); block_used_signatures.add(baseline_record["signature_digest"])
            for frame_id in state_metadata.get("response_frame_ids",()):
                frame_id=int(frame_id); record=physical_fit_bias_by_frame[frame_id]; request=next(candidate for candidate in full_imaging_requests if candidate.frame_id==frame_id); delta=np.asarray((record["values_by_name"]["centroid_x_px"],record["values_by_name"]["centroid_y_px"]))-baseline; targets=(f"{request.measurement}_frame_{frame_id}.delta_centroid_x_px",f"{request.measurement}_frame_{frame_id}.delta_centroid_y_px")
                for name,value in zip(targets,delta,strict=True): bias[name_to_index[name]]=value; assigned[name_to_index[name]]=True
                used_signatures.add(record["signature_digest"]); block_used_signatures.add(record["signature_digest"])
            for entry in state_metadata.get("material",()):
                member_ids=tuple(int(value) for value in entry["member_frame_ids"]); member_records=tuple(physical_fit_bias_by_frame[frame_id] for frame_id in member_ids); schemas=tuple(tuple(record["values_by_name"]) for record in member_records)
                if not schemas or len(set(schemas))!=1 or not schemas[0]: raise RuntimeError(f"Material bias schemas do not match for frames {member_ids}.")
                raw_names=schemas[0]; member_values=tuple(np.asarray([record["values_by_name"][name] for name in raw_names]) for record in member_records)
                if entry["mode"]=="precision_pool":
                    covariances=tuple(physical_independent_pool_covariance(frame_id,raw_names) for frame_id in member_ids); precision=sum(np.linalg.inv(value) for value in covariances); pooled_covariance=np.linalg.inv(precision); values=pooled_covariance@sum(np.linalg.solve(covariance,value) for covariance,value in zip(covariances,member_values,strict=True)); prefix="material_pool."
                elif entry["mode"]=="retained" and len(member_ids)==1: values=member_values[0]; prefix=f"material_frame_{member_ids[0]}."
                else: raise RuntimeError(f"Unsupported material-bias aggregation {entry['mode']!r} for frames {member_ids}.")
                for name,value in zip((prefix+name for name in raw_names),values,strict=True): bias[name_to_index[name]]=value; assigned[name_to_index[name]]=True
                used_signatures.update(record["signature_digest"] for record in member_records); block_used_signatures.update(record["signature_digest"] for record in member_records)
        else:
            record=physical_fit_bias_by_frame[block.frame_id]
            for name,value in record["values_by_name"].items(): bias[name_to_index[name]]=value; assigned[name_to_index[name]]=True
            used_signatures.add(record["signature_digest"]); block_used_signatures.add(record["signature_digest"])
        bias_blocks.append(bias); assignment_blocks.append(assigned); used_signatures_by_block.append(frozenset(block_used_signatures))
    physical_fit_bias_all_unwhitened=np.concatenate(bias_blocks); physical_fit_bias_all_assigned=np.concatenate(assignment_blocks)
    if not np.all(physical_fit_bias_all_assigned): missing=np.asarray(physical_observation_layout.all_observation_keys,dtype=object)[~physical_fit_bias_all_assigned]; raise RuntimeError(f"Signed renderer bias is missing {len(missing)} likelihood components: {missing[:8].tolist()}.")
    training_used_signatures=set().union(*(used_signatures_by_block[index] for index in physical_observation_layout.selected_block_indices)); assert training_used_signatures.issubset(used_signatures) and len(used_signatures_by_block)==len(physical_statistical_dataset.blocks)
    physical_fit_bias_training_unwhitened=physical_fit_bias_all_unwhitened[np.asarray(physical_observation_layout.included_observation_mask)]; physical_fit_bias_whitened=np.linalg.solve(np.asarray(physical_observation_layout.statistical_cholesky),physical_fit_bias_training_unwhitened); physical_fit_bias_ready=bool(np.all(np.isfinite(physical_fit_bias_whitened)))
    physical_fit_bias_provenance={"meaning":"Gaussian-candidate minus independent-reference mean-field observables","aggregation":"exact response differencing and the same independent-covariance precision pooling as the inverse dataset","whitening":"physical_statistical_dataset training covariance after shared-certificate terms are replaced by explicit global latents","likelihood_observation_count":int(physical_fit_bias_all_unwhitened.size),"training_observation_count":int(physical_fit_bias_whitened.size),"all_likelihood_used_signature_count":len(used_signatures),"training_used_signature_count":len(training_used_signatures),"total_accepted_signature_count":physical_validation_coverage.accepted_count,"excluded_rows":"held-out observation rows are validated per signature but do not perturb a training-only fit; material intensity is validation-only and is absent from this physical likelihood","per_signature_parameter_values_fabricated":False}
    print(f"Signed candidate−reference bias mapped to all {physical_fit_bias_all_unwhitened.size} downstream likelihood components using {len(used_signatures)} all-likelihood signatures; {physical_fit_bias_whitened.size} training components use the exact {len(training_used_signatures)}-signature subset. Joint Fisher projection follows after the global Jacobian is formed.")


In [ ]:
if full_diffraction_checkpoint.complete and "diffraction_noisy_counts" in globals():
    diffraction_pairs=[]
    for binding in native_diffraction_bindings:
        request=binding.request; mean_intensity=np.asarray(full_diffraction_checkpoint.record(request.frame_id).load_intensity(),float); mean_counts=mean_intensity*(physical_global_noise.dose_electrons/physical_global_noise.incident_intensity_sum)+physical_global_noise.dark_offset_e; noisy=np.asarray(diffraction_noisy_counts[request.frame_id],float); diffraction_pairs.append((binding,mean_counts,noisy,np.asarray(diffraction_registration[request.frame_id])))
    log_limits=[np.percentile(np.log1p(np.maximum(np.concatenate((mean,noisy),axis=1),0.)),99.9) for _,mean,noisy,_ in diffraction_pairs]; fig_diffraction,diffraction_axes=plt.subplots(2,3,figsize=(12.2,7.7),constrained_layout=True)
    for axis,(binding,mean,noisy,registration),vmax in zip(diffraction_axes.ravel(),diffraction_pairs,log_limits,strict=True):
        paired=np.concatenate((mean,np.maximum(noisy,0.)),axis=1); ny,nx=mean.shape; axis.imshow(np.log1p(paired),origin="lower",cmap="magma",vmin=0,vmax=vmax); axis.axvline(nx-.5,color="white",lw=.8,alpha=.8); centre=.5*(np.asarray(mean.shape,float)-1.)
        for vector_index,vector_yx in enumerate(registration):
            color=("#56B4E9","#D55E00")[vector_index%2]
            for sign in (-1,1):
                yx=centre+sign*vector_yx; axis.scatter((yx[1],yx[1]+nx),(yx[0],yx[0]),s=35,facecolors="none",edgecolors=color,linewidths=1.1)
        axis.text(.02,.97,"noiseless mean",transform=axis.transAxes,ha="left",va="top",color="white",fontsize=7.5); axis.text(.52,.97,"one detector draw",transform=axis.transAxes,ha="left",va="top",color="white",fontsize=7.5); axis.set_title(f"{binding.request.optical_state_label} · frame {binding.request.frame_id}\nL={binding.mode.achieved_camera_length_m:.3f} m · native IL1–PL1",fontsize=8.5); axis.set(xticks=[],yticks=[])
    fig_diffraction.suptitle("Native physical-column Si diffraction · mean and synthetic detector-count draw\ncyan/vermillion rings: predicted registered Friedel-pair locations",fontsize=12.3); physical_diffraction_contact_exports=export_ray_figure(fig_diffraction,GENERATED/"neoarm_native_diffraction_synthetic_images_v2_gpu",metadata={"title":"Native NEOARM diffraction synthetic detector images","frame_ids":[binding.request.frame_id for binding in native_diffraction_bindings],"optical_state_labels":[binding.request.optical_state_label for binding in native_diffraction_bindings],"camera_lengths_m":[binding.mode.achieved_camera_length_m for binding in native_diffraction_bindings],"native_mode_signatures":[binding.mode.model_signature for binding in native_diffraction_bindings],"predicted_registration_offsets_yx_px":{str(binding.request.frame_id):diffraction_registration[binding.request.frame_id] for binding in native_diffraction_bindings},"checkpoint_manifest_sha256":full_diffraction_checkpoint.manifest_sha256,"checkpoint_render_signature":native_diffraction_render_signature,"model_scope":PHYSICAL_OPTICAL_SCOPE,"uses_ideal_relay":False,"synthetic_not_measured":True,"dose_electrons":physical_global_noise.dose_electrons,"read_noise_e":physical_global_noise.read_noise_e,"display":"log1p counts; each panel pairs noiseless mean (left) and one synthetic Poisson-plus-read-noise draw (right)",**FORWARD_RUNTIME_METADATA}); plt.show(); print("Native diffraction-image exports:",physical_diffraction_contact_exports)
else: print("Native diffraction contact sheet remains fail-closed until all six physical-column frames and their registered noisy data products exist.")


In [ ]:
physical_global_fit=None; physical_subspace_design=None; physical_postfit_subspace_design=None; physical_identifiable_subspace=None; physical_profile_results=(); physical_profile_postfit_gate=False; selected_condenser_route_postfit_gate=False; physical_multistart_diagnostics=None; physical_matched_recovery=None; physical_named_optical_recovery=(); physical_matched_recovery_gate=False; physical_local_null_report=None; physical_optimization_gate=False; physical_global_fit_ready=False; final_instrument_acceptance=False; physical_preliminary_blockwise_rank=None; physical_image_nuisance_diagnostics=None; physical_held_out_evaluation=None; physical_held_out_statistical_gate=False; physical_held_out_engineering_reports=(); physical_held_out_engineering_gate=False; physical_condenser_held_out_evaluation=None; physical_condenser_held_out_gate=False; physical_downstream_instrument_dataset_bound=False; physical_parameter_stability_rough_report=None; physical_parameter_stability_initial_report=None; physical_parameter_stability_postfit_report=None; physical_joint_parameter_stability_rough_report=None; physical_joint_parameter_stability_initial_report=None; physical_joint_parameter_stability_postfit_report=None; physical_optical_stability_rough_diagnostics=None; physical_optical_stability_initial_diagnostics=None; physical_optical_stability_postfit_diagnostics=None; physical_forward_model_parameter_stability_ready=False; physical_pilot_inputs_ready=False; physical_global_fit_inputs_ready=False; physical_literal_global_two_refinement_acceptance=False; physical_terminal_result_kind=None
entrance_covariance_nuisance_count=10; hardware_deflector_gain_nuisance_count=0 if CONDENSER_ROUTE_BRANCH=="ASTIGMATIC" else 16
if CONDENSER_ROUTE_BRANCH=="A":
    physical_condenser_covariance_reference=np.asarray(hardware_zero_covariance_coordinates); physical_condenser_gain_reference=np.asarray(hardware_deflector_gain_truth); physical_condenser_extra_reference=np.empty(0); physical_condenser_outer_names=tuple(f"actual_hardware.entrance.{name}" for name in hardware_conditional_chart.coordinate_names)+tuple(f"actual_hardware.{name}" for name in hardware_deflector_map.parameter_names[4:]); physical_condenser_outer_scales=np.concatenate((hardware_branch_A_covariance_scales,hardware_branch_A_gain_scales)); physical_condenser_outer_lower=np.concatenate((np.full(10,-.35),hardware_deflector_gain_truth-.25)); physical_condenser_outer_upper=np.concatenate((np.full(10,.35),hardware_deflector_gain_truth+.25))
elif CONDENSER_ROUTE_BRANCH=="B":
    physical_condenser_covariance_reference=np.asarray(reusable_branch_B_truth_parameters[12:22]); physical_condenser_gain_reference=np.asarray(reusable_branch_B_truth_parameters[26:42]); physical_condenser_extra_reference=np.asarray(reusable_branch_B_extra_nuisance_reference); physical_condenser_outer_names=tuple(reusable_branch_B_synthetic_likelihood.parameter_names[12:22])+tuple(reusable_branch_B_synthetic_likelihood.parameter_names[26:42])+tuple(reusable_branch_B_extra_nuisance_names); physical_branch_B_extra_scales=np.concatenate(([reusable_branch_B_experiment.aperture_log_radius_prior_sigma,reusable_branch_B_experiment.aperture_stage_log_scale_prior_sigma],np.tile((1.,1.,.1),4))); physical_condenser_outer_scales=np.concatenate((np.full(10,.1),np.full(16,.05),physical_branch_B_extra_scales)); physical_branch_B_extra_lower=np.concatenate(((-5*reusable_branch_B_experiment.aperture_log_radius_prior_sigma,-5*reusable_branch_B_experiment.aperture_stage_log_scale_prior_sigma),np.tile((-5.,-5.,-.5),4))); physical_branch_B_extra_upper=-physical_branch_B_extra_lower; physical_condenser_outer_lower=np.concatenate((np.full(10,-.35),physical_condenser_gain_reference-.25,physical_branch_B_extra_lower)); physical_condenser_outer_upper=np.concatenate((np.full(10,.35),physical_condenser_gain_reference+.25,physical_branch_B_extra_upper))
else:
    if selected_astigmatic_likelihood is None or selected_astigmatic_adapter is None or selected_astigmatic_held_out_split is None: raise RuntimeError(f"{CONDENSER_ROUTE} is fail-closed: a checksum-audited likelihood, matching full-column adapter, and held-out split must all be bound.")
    physical_astigmatic_outer_contract=selected_astigmatic_adapter.outer_contract; physical_condenser_covariance_reference=np.asarray(physical_astigmatic_outer_contract.reference[:10]); physical_condenser_gain_reference=np.empty(0); physical_condenser_extra_reference=np.asarray(physical_astigmatic_outer_contract.reference[10:]); physical_condenser_outer_names=physical_astigmatic_outer_contract.names; physical_condenser_outer_scales=np.asarray(physical_astigmatic_outer_contract.scales); physical_condenser_outer_lower=np.asarray(physical_astigmatic_outer_contract.lower); physical_condenser_outer_upper=np.asarray(physical_astigmatic_outer_contract.upper)
physical_condenser_extra_nuisance_count=len(physical_condenser_extra_reference); physical_outer_nuisance_count=entrance_covariance_nuisance_count+hardware_deflector_gain_nuisance_count+physical_condenser_extra_nuisance_count; physical_condenser_outer_reference=np.concatenate((physical_condenser_covariance_reference,physical_condenser_gain_reference,physical_condenser_extra_reference)); assert physical_outer_nuisance_count==len(physical_condenser_outer_reference)==len(physical_condenser_outer_names)==len(physical_condenser_outer_scales)
physical_optical_identifiability_scales=np.asarray(design_parameter_scales,float).copy(); assert np.all(np.isfinite(physical_optical_identifiability_scales)) and np.all(physical_optical_identifiability_scales>0)  # local mechanical/design scales; invariant to the arbitrary z-origin
physical_downstream_data_kind=(str(physical_global_dataset.metadata.get("data_kind","synthetic_gpu_checkpoint")) if physical_observation_dataset_ready else "unbound"); physical_downstream_instrument_dataset_bound=bool(physical_observation_dataset_ready and physical_downstream_data_kind=="instrument" and physical_global_dataset.metadata.get("instrument_observation_vector_bound",False) and physical_global_dataset.metadata.get("instrument_covariance_bound",False)); physical_absolute_intensity_fit_observable=("88 unnormalised C-aperture Poisson count rows" if CONDENSER_ROUTE_BRANCH=="ASTIGMATIC" else "route-specific condenser response observables"); physical_sample_frame_total_count_role="validation_only_until_a_downstream_clipping_and_detection_emulator_is_validated"
print(f"Absolute-intensity contract: {physical_absolute_intensity_fit_observable} enter the selected condenser likelihood. Sample-frame total counts are omitted from the inverse: their apparent invariance here is a consequence of known_sample_source truth-backpropagation with unit-current normalization and the absence of a validated downstream clipping/detection emulator, not a general invariance to upstream optics. They remain {physical_sample_frame_total_count_role} and are not claimed as lens information.")
def selected_condenser_global_residual(optical,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters):
    if CONDENSER_ROUTE=="synthetic_branch_a_conditional": return hardware_branch_A_synthetic_global_residual(optical,covariance_coordinates,deflector_gain_parameters)
    if CONDENSER_ROUTE=="synthetic_branch_b_unconditional": return hardware_branch_B_synthetic_global_residual(optical,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters)
    if CONDENSER_ROUTE_BRANCH=="ASTIGMATIC":
        if selected_astigmatic_likelihood is None or selected_astigmatic_adapter is None or selected_astigmatic_held_out_split is None: raise RuntimeError(f"{CONDENSER_ROUTE} has no complete astigmatic likelihood/adapter/held-out binding.")
        native=selected_astigmatic_adapter.likelihood_parameters_from_parts(optical,jnp.concatenate((covariance_coordinates,extra_nuisance_parameters))); return astigmatic_training_residual(selected_astigmatic_likelihood,native,selected_astigmatic_held_out_split)
    if CONDENSER_ROUTE=="instrument_branch_a_conditional":
        if not hardware_branch_A_instrument_likelihood_bound_to_global: raise RuntimeError("Instrument Branch A selected, but no registered observation vector/full covariance/Obj-prefield anchor is bound.")
        return hardware_branch_A_dataset_residual(optical,covariance_coordinates,deflector_gain_parameters,hardware_branch_A_instrument_observed_vector,hardware_branch_A_instrument_covariance_cholesky)
    if CONDENSER_ROUTE=="instrument_branch_b_unconditional":
        if not hardware_branch_B_instrument_likelihood_bound_to_global or hardware_branch_B_instrument_likelihood is None: raise RuntimeError("Instrument Branch B selected, but no payload-registered instrument observations/covariances are bound.")
        return hardware_branch_B_instrument_likelihood.residual(hardware_branch_B_parameters_from_shared_physical(optical,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters))
    raise RuntimeError(f"Unhandled condenser route {CONDENSER_ROUTE!r}.")
if physical_observation_dataset_ready:
    if CONDENSER_ROUTE_IS_INSTRUMENT and not selected_condenser_route_global_gate_ready: raise RuntimeError(f"{CONDENSER_ROUTE} is fail-closed: registered instrument observations, covariance, calibration, and pre-fit hardware contracts are not bound.")
    if CONDENSER_ROUTE_IS_INSTRUMENT and not physical_downstream_instrument_dataset_bound: raise RuntimeError(f"{CONDENSER_ROUTE} is fail-closed: the downstream likelihood is still {physical_downstream_data_kind!r}, not a separately registered instrument dataset with observation and covariance provenance.")
    preliminary_base_reference=np.asarray(physical_shared_likelihood.join_parameters(physical_parameters)); preliminary_joint_reference=np.concatenate((preliminary_base_reference,physical_condenser_outer_reference)); preliminary_joint_scales=np.concatenate((physical_optical_identifiability_scales,np.ones(physical_shared_likelihood.n_nuisance_latents),physical_condenser_outer_scales))
    def split_physical_outer_joint(joint):
        base_joint=joint[:-physical_outer_nuisance_count]; outer=joint[-physical_outer_nuisance_count:]; gain_stop=entrance_covariance_nuisance_count+hardware_deflector_gain_nuisance_count; covariance_coordinates=outer[:entrance_covariance_nuisance_count]; deflector_gain_parameters=outer[entrance_covariance_nuisance_count:gain_stop]; extra_nuisance_parameters=outer[gain_stop:]; optical,calibration,discrepancy=physical_shared_likelihood.split_joint_parameters(base_joint); return base_joint,optical,calibration,discrepancy,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters
    def preliminary_condenser_residual(joint):
        _,optical,_,_,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters=split_physical_outer_joint(joint); return selected_condenser_global_residual(optical,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters)
    def preliminary_image_residual(joint): return physical_shared_likelihood.data_residual(joint[:-physical_outer_nuisance_count])
    assert len(preliminary_joint_reference)==len(preliminary_joint_scales)==column.n_parameters+physical_shared_likelihood.n_nuisance_latents+physical_outer_nuisance_count
    preliminary_condenser_J=np.asarray(jax.jacfwd(preliminary_condenser_residual)(jnp.asarray(preliminary_joint_reference))); preliminary_image_J=np.asarray(jax.jacfwd(preliminary_image_residual)(jnp.asarray(preliminary_joint_reference))); assert np.max(np.abs(preliminary_image_J[:,-physical_outer_nuisance_count:]))==0, "Calibrated sample-plane image likelihood unexpectedly depends on condenser-route nuisances."; physical_preliminary_blockwise_rank=diagnose_blockwise_identifiability(((f"{CONDENSER_ROUTE}_condenser",preliminary_condenser_J),("physical_image_and_diffraction",preliminary_image_J)),optical_parameter_count=column.n_parameters,parameter_scales=preliminary_joint_scales,rank_rtol=1e-7); physical_image_nuisance_diagnostics=diagnose_nuisance_identifiability(physical_shared_likelihood,physical_parameters,optical_parameter_scales=design_parameter_scales,rank_rtol=1e-7)
    print("Pre-admission design diagnostic (not a fit):",{report.block_name:{"conditional_optical":report.column_equilibrated_conditional_optical.rank,"nuisance_profiled_optical":report.column_equilibrated_profiled_optical.rank} for report in physical_preliminary_blockwise_rank.block_reports},"combined profiled optical",physical_preliminary_blockwise_rank.combined.column_equilibrated_profiled_optical.rank,"/",column.n_parameters,"selected condenser route",CONDENSER_ROUTE,"shared calibration latents",physical_shared_likelihood.n_calibration_latents,"entrance covariance latents",entrance_covariance_nuisance_count,"double-deflector gains",hardware_deflector_gain_nuisance_count,"route-specific extra latents",physical_condenser_extra_nuisance_count)
else: print("Pre-admission rank diagnostic unavailable until the physical observable dataset exists.")

if physical_image_global_fit_ready:

    if CONDENSER_ROUTE_REQUIRES_OBJ_PREFIELD_ANCHOR and not selected_obj_prefield_anchor_bound: raise RuntimeError(f"Cannot build the full global likelihood: {CONDENSER_ROUTE} is conditional on an independent Obj-prefield anchor, and its explicit anchor-binding flag is false.")

    if CONDENSER_ROUTE_IS_INSTRUMENT and not selected_condenser_route_global_gate_ready: raise RuntimeError(f"Cannot build the full global likelihood: {CONDENSER_ROUTE} has no admitted instrument observations.")
    if CONDENSER_ROUTE_IS_INSTRUMENT and not physical_downstream_instrument_dataset_bound: raise RuntimeError(f"Cannot build the instrument likelihood from downstream data_kind={physical_downstream_data_kind!r}; a registered instrument observation/covariance dataset is required.")
    assert (CONDENSER_ROUTE!="synthetic_branch_a_conditional" or (hardware_branch_A_synthetic_global_observed_tomography.size==120 and hardware_branch_A_synthetic_global_observed_deflector.size==48)) and (CONDENSER_ROUTE!="synthetic_branch_b_unconditional" or (len(reusable_branch_B_synthetic_likelihood.parameter_names)==56 and len(reusable_branch_B_synthetic_likelihood.residual_names)==458))
    # Synthetic A uses 168 actual-geometry response observations; synthetic B uses the checksum-bound 296-frame likelihood; the preferred synthetic astigmatic route uses 88 raw aperture frames plus 48 seven-observable tag acquisitions and no commanded tilt. Instrument selections never substitute synthetic observations.

    unrestricted_optical_bounds=column.default_bounds(position_half_width_m=3e-3,length_ratio=(.4,2.5),q_half_width=.8,slope_half_width=.8); physical_fit_plane_constraints=tuple(HardEdgeInternalPlaneConstraint(float(z_m),"CL3","C-mini",APERTURE_FIT_MINIMUM_CLEARANCE_M) for z_m in (actual_c_aperture_z_m,*actual_deflector_planes_m)); physical_fit_plane_certificates=restrict_hard_edge_internal_planes_bounds(column,unrestricted_optical_bounds,physical_fit_plane_constraints,reference_parameters=physical_parameters); optical_lower,optical_upper=physical_fit_plane_certificates[0].bounds; physical_fit_worst_plane_clearance_m=min(certificate.worst_case_clearance_m for certificate in physical_fit_plane_certificates); assert tuple(certificate.plane_z_m for certificate in physical_fit_plane_certificates)==(actual_c_aperture_z_m,*tuple(actual_deflector_planes_m))
    physical_initial_scaled_offset=.08*np.sin((np.arange(column.n_parameters)+1)*np.sqrt(2.)); physical_fit_initial_parameters=np.clip(np.asarray(physical_parameters)+design_parameter_scales*physical_initial_scaled_offset,optical_lower+1e-10*np.maximum(1.,np.abs(optical_lower)),optical_upper-1e-10*np.maximum(1.,np.abs(optical_upper))); assert np.all(np.isfinite(np.asarray(column.matrix(physical_fit_initial_parameters,reference_dacs,segment="full")))) and physical_fit_worst_plane_clearance_m>=APERTURE_FIT_MINIMUM_CLEARANCE_M
    print("Fixed-plane fit-box certificates:",[(certificate.plane_z_m,certificate.upstream_lens_name,certificate.downstream_lens_name,1e3*certificate.worst_case_clearance_m) for certificate in physical_fit_plane_certificates],"(z, upstream, downstream, worst clearance mm)")
    # The affine practical-null gauge is centred at a deterministic truth-perturbed matched-synthetic initializer. This tests local recovery but is not advertised as discovery from an independently measured rough column.
    physical_base_reference=np.asarray(physical_shared_likelihood.join_parameters(physical_fit_initial_parameters)); physical_joint_reference=np.concatenate((physical_base_reference,physical_condenser_outer_reference)); physical_joint_scales=np.concatenate((physical_optical_identifiability_scales,np.ones(physical_shared_likelihood.n_nuisance_latents),physical_condenser_outer_scales)); physical_joint_names=tuple(column.parameter_names)+physical_shared_likelihood.calibration_latent_names+physical_shared_likelihood.discrepancy_latent_names+physical_condenser_outer_names; assert len(physical_joint_reference)==len(physical_joint_scales)==len(physical_joint_names)

    def physical_global_residual(joint_parameters):
        _,optical,calibration,discrepancy,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters=split_physical_outer_joint(joint_parameters); condenser=selected_condenser_global_residual(optical,covariance_coordinates,deflector_gain_parameters,extra_nuisance_parameters); return jnp.concatenate((condenser,physical_shared_likelihood.joint_residual_from_parts(optical,calibration,discrepancy)))
    physical_global_residual_compiled=jax.jit(physical_global_residual); physical_global_residual_compiled(jnp.asarray(physical_joint_reference)).block_until_ready()

    physical_minimum_optical_snr=75.; physical_nuisance_rank_rtol=128*np.finfo(float).eps; physical_lower=np.concatenate((optical_lower,np.full(physical_shared_likelihood.n_nuisance_latents,-5.),physical_condenser_outer_lower)); physical_upper=np.concatenate((optical_upper,np.full(physical_shared_likelihood.n_nuisance_latents,5.),physical_condenser_outer_upper))
    def physical_design_at_reference(reference):
        jacobian=np.asarray(jax.jacfwd(physical_global_residual_compiled)(jnp.asarray(reference))); design=nuisance_profiled_identifiable_subspace(jacobian,reference,physical_joint_scales,optical_parameter_count=column.n_parameters,optical_minimum_singular_value=physical_minimum_optical_snr,nuisance_rank_rtol=physical_nuisance_rank_rtol,parameter_names=physical_joint_names); return jacobian,design
    def physical_condenser_target_spectrum(jacobian):
        scaled=jacobian*physical_joint_scales[None,:]; target_indices=np.asarray(hardware_condenser_indices,dtype=int); competing_indices=np.setdiff1d(np.arange(len(physical_joint_reference)),target_indices); target=scaled[:,target_indices]; competing=scaled[:,competing_indices]; left,singular,_=np.linalg.svd(competing,full_matrices=False); threshold=np.finfo(float).eps*max(competing.shape)*singular[0]; rank=int(np.sum(singular>threshold)); profiled=target-left[:,:rank]@(left[:,:rank].T@target); values=np.linalg.svd(profiled,compute_uv=False); return values,int(np.sum(values>physical_minimum_optical_snr))
    physical_rough_global_J,physical_rough_subspace_design=physical_design_at_reference(physical_joint_reference); physical_rough_identifiable_subspace=physical_rough_subspace_design.subspace; physical_global_J=physical_rough_global_J; physical_subspace_design=physical_rough_subspace_design; physical_identifiable_subspace=physical_rough_identifiable_subspace; physical_initial_profiled_condenser_target_singular,physical_initial_profiled_condenser_target_rank=physical_condenser_target_spectrum(physical_rough_global_J); physical_profiled_condenser_target_singular=physical_initial_profiled_condenser_target_singular; physical_profiled_condenser_target_rank=physical_initial_profiled_condenser_target_rank
    physical_structural_condenser_rank_source_by_branch={"ASTIGMATIC":"selected_astigmatic_likelihood.local_rank_diagnostics().profiled_target_rank","A":"hardware_conditional_audit.combined.profiled_optical.rank","B":"reusable_branch_B_profiled_diagnostics.target_rank"}; physical_structural_condenser_target_rank=int({"ASTIGMATIC":selected_astigmatic_rank_summary,"A":hardware_conditional_audit.combined.profiled_optical.rank,"B":reusable_branch_B_profiled_diagnostics.target_rank}[CONDENSER_ROUTE_BRANCH]); physical_structural_condenser_target_gate=bool(physical_structural_condenser_target_rank==len(hardware_condenser_indices)); physical_declared_profiled_condenser_target_rank=None; physical_practical_condenser_rank_stable=False; physical_practical_condenser_rank_gate=False; physical_full_condenser_coordinate_gate=False; physical_rank_gate_for_route=False
    if not physical_fit_bias_ready or physical_fit_bias_whitened is None: raise RuntimeError("The full signed candidate−reference likelihood bias is unavailable despite complete render coverage.")
    physical_condenser_residual_count=physical_global_J.shape[0]-physical_observation_layout.n_observations-physical_shared_likelihood.n_nuisance_latents
    if physical_condenser_residual_count<1 or len(physical_fit_bias_whitened)!=physical_observation_layout.n_observations: raise RuntimeError("The signed renderer bias cannot be aligned with the global residual rows.")
    physical_global_fit_bias_whitened=np.zeros(physical_global_J.shape[0]); physical_global_fit_bias_whitened[physical_condenser_residual_count:physical_condenser_residual_count+physical_observation_layout.n_observations]=physical_fit_bias_whitened
    physical_global_fit_bias_blocks=[]; physical_bias_block_provenance={}; physical_bias_cursor=0; physical_aggregation=physical_statistical_dataset.metadata["aggregation"]["state_blocks"]
    for selected_block_index in physical_observation_layout.selected_block_indices:
        block=physical_statistical_dataset.blocks[selected_block_index]; block_size=block.values.size; block_start=physical_condenser_residual_count+physical_bias_cursor; block_stop=block_start+block_size; state=physical_statistical_dataset.plan.states[block.optical_state_index]; label=f"{state.label}:block_frame_{block.frame_id}"; block_bias=np.zeros_like(physical_global_fit_bias_whitened); block_bias[block_start:block_stop]=physical_fit_bias_whitened[physical_bias_cursor:physical_bias_cursor+block_size]; state_metadata=physical_aggregation.get(str(block.frame_id))
        if isinstance(state_metadata,dict):
            source_frame_ids=[int(state_metadata["baseline_frame_id"]),*(int(value) for value in state_metadata.get("response_frame_ids",()))]
            for entry in state_metadata.get("material",()): source_frame_ids.extend(int(value) for value in entry["member_frame_ids"])
        else: source_frame_ids=[int(block.frame_id)]
        source_frame_ids=tuple(dict.fromkeys(source_frame_ids)); signature_digests=tuple(physical_fit_bias_by_frame[frame_id]["signature_digest"] for frame_id in source_frame_ids); physical_global_fit_bias_blocks.append((label,block_bias)); physical_bias_block_provenance[label]={"dataset_block_index":int(selected_block_index),"block_frame_id":int(block.frame_id),"optical_state_index":int(block.optical_state_index),"optical_state_label":state.label,"observable_names":list(block.observable_names),"source_frame_ids":list(source_frame_ids),"source_signature_digests":list(signature_digests),"global_residual_row_start":int(block_start),"global_residual_row_stop":int(block_stop)}; physical_bias_cursor+=block_size
    if physical_bias_cursor!=physical_observation_layout.n_observations or not np.allclose(np.sum([value for _,value in physical_global_fit_bias_blocks],axis=0),physical_global_fit_bias_whitened,rtol=1e-12,atol=1e-14): raise RuntimeError("Likelihood-block renderer biases do not exactly reconstruct the global signed bias.")
    physical_production_parameter_history=tuple(float(step.forward_validation.parameter_change_in_sigma) for step in getattr(globals().get("physical_production_refinement_report"),"steps",()))
    physical_joint_parameter_stability_rough_report=assess_fisher_parameter_stability(physical_rough_global_J,physical_global_fit_bias_whitened,physical_joint_scales,physical_rough_identifiable_subspace.active_basis_scaled,successive_refinement_changes_in_sigma=physical_production_parameter_history,required_successive_refinements=2,parameter_change_threshold_sigma=physical_validation_thresholds.parameter_change_in_sigma)
    physical_optical_stability_rough_diagnostics=assess_nuisance_profiled_optical_stability(physical_rough_global_J,physical_global_fit_bias_whitened,physical_joint_scales,optical_parameter_count=column.n_parameters,optical_active_basis_scaled=physical_rough_subspace_design.optical_subspace.active_basis_scaled,nuisance_rank=physical_rough_subspace_design.nuisance_rank,optical_parameter_names=column.parameter_names,bias_blocks=physical_global_fit_bias_blocks,successive_refinement_changes_in_sigma=physical_production_parameter_history,required_successive_refinements=2,parameter_change_threshold_sigma=physical_validation_thresholds.parameter_change_in_sigma); physical_parameter_stability_rough_report=physical_optical_stability_rough_diagnostics.stability
    physical_forward_model_parameter_stability_ready=False; physical_pilot_inputs_ready=bool(physical_image_global_fit_ready and physical_fit_bias_ready); physical_global_fit_inputs_ready=False; physical_refresh_history=[]; physical_terminal_attempt_provenance=None; physical_terminal_stationarity_provenance=None; physical_parameter_stability_path=GENERATED/"neoarm_physical_parameter_stability_v3.json"
    def write_physical_parameter_stability_report(terminal_optical=None,terminal_joint=None,terminal_result_kind=None):
        declared_rank=(physical_parameter_stability_rough_report.active_rank if physical_parameter_stability_initial_report is None else physical_parameter_stability_initial_report.active_rank); terminal_key=(None if terminal_result_kind is None else ("converged_terminal" if terminal_result_kind=="converged_terminal_fit" else "failed_terminal_attempt_diagnostic")); payload={"schema":"temgym_core.neoarm_fisher_parameter_stability","version":3,"primary_gate_scope":f"nuisance-profiled {declared_rank}-direction quotient declared only after a successful stable pilot; strict Fisher norm below 0.1 sigma plus two representative 150k refinement transitions","optical_rough_reference_diagnostic":physical_optical_stability_rough_diagnostics.to_payload(),"optical_declared_pre_global":None if physical_optical_stability_initial_diagnostics is None else physical_optical_stability_initial_diagnostics.to_payload(),"optical_terminal_result_kind":terminal_key,"optical_terminal":None if terminal_optical is None else terminal_optical.to_payload(),"joint_optical_plus_nuisance_rough_reference_non_gating":physical_joint_parameter_stability_rough_report.to_payload(),"joint_optical_plus_nuisance_declared_pre_global_non_gating":None if physical_joint_parameter_stability_initial_report is None else physical_joint_parameter_stability_initial_report.to_payload(),"joint_optical_plus_nuisance_terminal_non_gating":None if terminal_joint is None else terminal_joint.to_payload(),"pilot_refresh_history":physical_refresh_history,"pilot_refresh_converged":bool(physical_refresh_history and physical_refresh_history[-1].get("stable",False)),"terminal_attempt_provenance":physical_terminal_attempt_provenance,"terminal_stationarity_provenance":physical_terminal_stationarity_provenance,"bias_provenance":physical_fit_bias_provenance,"bias_block_provenance":physical_bias_block_provenance,"ledger_sha256":physical_validation_ledger.to_payload()["sha256"],"representative_successive_refinement_scope":f"three 150k production topologies (480249, 751689, 1172889 beamlets) refit one conditional IL1.q_bias one-parameter proxy while every other optical and nuisance coordinate is fixed; this is representative chosen-mode evidence, not {declared_rank}-direction global successive-refinement evidence","literal_all_identifiable_parameter_two_successive_refinement_acceptance":physical_literal_global_two_refinement_acceptance,"limitation":"the optical candidate-versus-independent-reference influence is a first-order Fisher diagnostic recomputed at the successfully converged pre-global reference and, only when optimizer/rank admission succeeds, at the converged terminal fit; a failed terminal candidate is labelled diagnostic only. Global two-successive-topology refits of every admitted optical direction have not been run, so literal all-identifiable-parameter refinement acceptance remains provisional"}; canonical=json.dumps(payload,sort_keys=True,separators=(",",":"),ensure_ascii=False,allow_nan=False); payload["sha256"]=hashlib.sha256(canonical.encode("utf-8")).hexdigest(); physical_parameter_stability_path.write_text(json.dumps(payload,indent=2,sort_keys=True)+"\n",encoding="utf-8")
    write_physical_parameter_stability_report()
    physical_blinded_joint=physical_joint_reference.copy(); physical_active_x0=np.zeros(physical_identifiable_subspace.rank); physical_blind_factor=1.; physical_strict_bound_margin=1024*np.finfo(float).eps*np.maximum(1.,np.maximum(np.abs(physical_lower),np.abs(physical_upper))); physical_projected_initial_joint=np.asarray(physical_identifiable_subspace.parameters_from_active(jnp.asarray(physical_active_x0))); physical_projected_initial_interior=bool(np.all(physical_projected_initial_joint>physical_lower+physical_strict_bound_margin) and np.all(physical_projected_initial_joint<physical_upper-physical_strict_bound_margin)); assert physical_projected_initial_interior
    physical_truth_joint=(np.concatenate((np.asarray(physical_shared_likelihood.join_parameters(physical_parameters)),physical_condenser_outer_reference)) if CONDENSER_ROUTE_IS_SYNTHETIC else None)
    if physical_truth_joint is None: physical_initial_truth_active_distance=np.nan; physical_initial_truth_null_gauge_distance=np.nan; physical_initial_to_truth_residual_norm=np.nan
    else:
        physical_initial_truth_scaled=(physical_truth_joint-physical_joint_reference)/physical_joint_scales; physical_initial_truth_active_distance=float(np.linalg.norm(physical_identifiable_subspace.active_basis_scaled.T@physical_initial_truth_scaled)); physical_initial_truth_null_gauge_distance=float(np.linalg.norm(physical_identifiable_subspace.null_basis_scaled.T@physical_initial_truth_scaled)); physical_initial_to_truth_residual_norm=float(np.linalg.norm(np.asarray(physical_global_residual_compiled(jnp.asarray(physical_joint_reference)))-np.asarray(physical_global_residual_compiled(jnp.asarray(physical_truth_joint)))))
    physical_discarded_initial_scaled_norm=physical_initial_truth_null_gauge_distance; physical_discarded_initial_residual_norm=physical_initial_to_truth_residual_norm  # legacy metadata keys are removed before export below

    physical_optical_influence_preview=sorted(physical_optical_stability_rough_diagnostics.block_influences,key=lambda value:abs(value.signed_fisher_squared_contribution),reverse=True)[:5]
    print(f"Physical shared-latent checkpoint likelihood ({CONDENSER_ROUTE}) at the rough reference: residual Jacobian {physical_rough_global_J.shape}, nuisance-profiled optical rank={physical_rough_subspace_design.optical_rank}/{column.n_parameters}, retained numerical nuisance rank={physical_rough_subspace_design.nuisance_rank}/{physical_rough_subspace_design.nuisance_parameter_count}, joint active/null={physical_rough_identifiable_subspace.rank}/{physical_rough_identifiable_subspace.nullity}. Route-native structural condenser rank={physical_structural_condenser_target_rank}/12; rough-reference full-column practical condenser rank={physical_initial_profiled_condenser_target_rank}/12 at SNR>{physical_minimum_optical_snr:.0f}, weakest SNR={physical_initial_profiled_condenser_target_singular[-1]:.3f}. Rough-reference nuisance-profiled optical Fisher displacement={physical_parameter_stability_rough_report.fisher_mahalanobis_change_in_sigma:.3e} sigma across {physical_parameter_stability_rough_report.active_rank} provisional directions; this is diagnostic and cannot admit the global fit. Representative conditional IL1.q_bias topology changes={physical_parameter_stability_rough_report.successive_refinement_changes_in_sigma[-2:].tolist()}; a successfully converged stable pilot must declare and freeze the quotient before 64 starts. Report={physical_parameter_stability_path.relative_to(REPO)}. Top signed rough-reference optical-bias blocks={[ (value.name,round(value.signed_fisher_squared_fraction,4),round(value.leave_block_fisher_change_in_sigma,6)) for value in physical_optical_influence_preview ]}. Scales are local mechanical/design scales and invariant to z-origin. The local matched-synthetic gauge is centred at a deterministic truth-perturbed initializer; synthetic truth lies {physical_initial_truth_active_distance:.3f} active-scale and {physical_initial_truth_null_gauge_distance:.3f} practical-null-scale units away (N/A for instrument data).")

    plt.close("all")  # release exported preview figures before fitting and downstream plots
    if RUN_PHYSICAL_GLOBAL_FIT:
        if not physical_pilot_inputs_ready: raise RuntimeError("Global-fit pilot is blocked: the registered image likelihood or signed renderer-bias evidence is incomplete.")

        physical_subspace_projector_tolerance=1e-3; physical_subspace_update_tolerance_sigma=.1; physical_refresh_history=[]; physical_refresh_reference=physical_joint_reference.copy(); physical_refresh_maximum=3
        for refresh_index in range(physical_refresh_maximum):
            refresh_J,refresh_design=physical_design_at_reference(physical_refresh_reference)
            refresh_fit=fit_nuisance_profiled_polytope(physical_global_residual_compiled,refresh_design,bounds=(physical_lower,physical_upper),active_x0=np.zeros(refresh_design.subspace.rank),multistart=8,seed=20260905+1000*refresh_index,hit_and_run_chains=4,max_nfev=600,optimizer_cost_transform="log_soft_norm")
            refresh_start_costs=np.asarray(refresh_fit.active_fit.start_costs,dtype=float); refresh_finite_costs=refresh_start_costs[np.isfinite(refresh_start_costs)]; refresh_cost_quantiles=({"minimum":float(np.quantile(refresh_finite_costs,0.)),"q25":float(np.quantile(refresh_finite_costs,.25)),"median":float(np.quantile(refresh_finite_costs,.5)),"q75":float(np.quantile(refresh_finite_costs,.75)),"maximum":float(np.quantile(refresh_finite_costs,1.))} if refresh_finite_costs.size else {"minimum":None,"q25":None,"median":None,"q75":None,"maximum":None}); refresh_record={"iteration":refresh_index,"cost":float(refresh_fit.cost),"success":bool(refresh_fit.success),"message":str(refresh_fit.active_fit.message),"nfev":int(refresh_fit.active_fit.nfev),"njev":None if refresh_fit.active_fit.njev is None else int(refresh_fit.active_fit.njev),"successful_starts":int(np.sum(np.asarray(refresh_fit.active_fit.start_success,dtype=bool))),"start_cost_quantiles":refresh_cost_quantiles,"optimizer_cost_transform":"log_soft_norm","fit_rank":int(refresh_fit.fit_rank),"joint_rank":int(refresh_design.subspace.rank),"optical_rank":int(refresh_design.optical_rank),"nuisance_rank":int(refresh_design.nuisance_rank)}
            if not refresh_fit.success or refresh_fit.fit_rank!=refresh_design.subspace.rank:
                refresh_record.update({"candidate_joint_rank":None,"candidate_optical_rank":None,"candidate_nuisance_rank":None,"joint_projector_distance":None,"optical_projector_distance":None,"joint_maximum_principal_angle_rad":None,"optical_maximum_principal_angle_rad":None,"active_update_max_sigma":None,"promoted":False,"stable":False}); physical_refresh_history.append(refresh_record); print(f"Pilot refresh {refresh_index} failed success/rank admission and was not promoted: success={refresh_fit.success}, fit rank={refresh_fit.fit_rank}/{refresh_design.subspace.rank}, nfev={refresh_fit.active_fit.nfev}, costs={refresh_cost_quantiles}, message={refresh_fit.active_fit.message}"); continue
            refresh_post_J,refresh_post_design=physical_design_at_reference(refresh_fit.parameters); refresh_joint_stability=compare_scaled_active_subspaces(refresh_design.subspace,refresh_post_design.subspace); refresh_optical_stability=compare_scaled_active_subspaces(refresh_design.optical_subspace,refresh_post_design.optical_subspace); refresh_sigma=np.sqrt(np.maximum(np.diag(refresh_fit.active_fisher_covariance),0.)); refresh_update=np.asarray(refresh_fit.active_coordinates); refresh_update_sigma=float(np.max(np.divide(np.abs(refresh_update),refresh_sigma,out=np.full_like(refresh_update,np.inf),where=refresh_sigma>0))) if refresh_update.size else 0.; refresh_stable=bool(refresh_joint_stability.ranks_match and refresh_optical_stability.ranks_match and max(refresh_joint_stability.spectral_projector_distance,refresh_optical_stability.spectral_projector_distance)<=physical_subspace_projector_tolerance and np.all(np.isfinite(refresh_sigma)) and refresh_update_sigma<physical_subspace_update_tolerance_sigma); refresh_record.update({"candidate_joint_rank":int(refresh_post_design.subspace.rank),"candidate_optical_rank":int(refresh_post_design.optical_rank),"candidate_nuisance_rank":int(refresh_post_design.nuisance_rank),"joint_projector_distance":float(refresh_joint_stability.spectral_projector_distance),"optical_projector_distance":float(refresh_optical_stability.spectral_projector_distance),"joint_maximum_principal_angle_rad":float(refresh_joint_stability.maximum_principal_angle_rad),"optical_maximum_principal_angle_rad":float(refresh_optical_stability.maximum_principal_angle_rad),"active_update_max_sigma":None if not np.isfinite(refresh_update_sigma) else float(refresh_update_sigma),"promoted":True,"stable":refresh_stable}); physical_refresh_history.append(refresh_record); physical_refresh_reference=np.asarray(refresh_fit.parameters,dtype=float)
            if refresh_stable: break
        physical_refresh_converged=bool(physical_refresh_history and physical_refresh_history[-1]["stable"])
        if not physical_refresh_converged:
            write_physical_parameter_stability_report(); raise RuntimeError(f"Global fit aborted before 64 starts: no successful stable pilot refresh was obtained, and failed candidates were never promoted. Diagnostic artifact={physical_parameter_stability_path.relative_to(REPO)}; refreshes={physical_refresh_history}")
        physical_declared_fit_reference=physical_refresh_reference.copy(); physical_global_J,physical_subspace_design=physical_design_at_reference(physical_declared_fit_reference); physical_identifiable_subspace=physical_subspace_design.subspace; physical_rough_to_declared_joint_stability=compare_scaled_active_subspaces(physical_rough_subspace_design.subspace,physical_subspace_design.subspace); physical_rough_to_declared_optical_stability=compare_scaled_active_subspaces(physical_rough_subspace_design.optical_subspace,physical_subspace_design.optical_subspace)
        physical_fit_reference_profiled_condenser_target_singular,physical_fit_reference_profiled_condenser_target_rank=physical_condenser_target_spectrum(physical_global_J); physical_declared_profiled_condenser_target_rank=int(physical_fit_reference_profiled_condenser_target_rank); physical_practical_condenser_rank_gate=bool(physical_declared_profiled_condenser_target_rank>0); physical_rank_gate_for_route=bool(physical_structural_condenser_target_gate and physical_subspace_design.optical_rank>0 and physical_practical_condenser_rank_gate)
        physical_joint_parameter_stability_initial_report=assess_fisher_parameter_stability(physical_global_J,physical_global_fit_bias_whitened,physical_joint_scales,physical_identifiable_subspace.active_basis_scaled,successive_refinement_changes_in_sigma=physical_production_parameter_history,required_successive_refinements=2,parameter_change_threshold_sigma=physical_validation_thresholds.parameter_change_in_sigma); physical_optical_stability_initial_diagnostics=assess_nuisance_profiled_optical_stability(physical_global_J,physical_global_fit_bias_whitened,physical_joint_scales,optical_parameter_count=column.n_parameters,optical_active_basis_scaled=physical_subspace_design.optical_subspace.active_basis_scaled,nuisance_rank=physical_subspace_design.nuisance_rank,optical_parameter_names=column.parameter_names,bias_blocks=physical_global_fit_bias_blocks,successive_refinement_changes_in_sigma=physical_production_parameter_history,required_successive_refinements=2,parameter_change_threshold_sigma=physical_validation_thresholds.parameter_change_in_sigma); physical_parameter_stability_initial_report=physical_optical_stability_initial_diagnostics.stability; physical_forward_model_parameter_stability_ready=bool(physical_optical_stability_initial_diagnostics.passed); physical_global_fit_inputs_ready=bool(physical_pilot_inputs_ready and physical_forward_model_parameter_stability_ready); write_physical_parameter_stability_report()
        print(f"Stable pilot declared and froze the pre-global quotient: rough/declared joint ranks={physical_rough_identifiable_subspace.rank}/{physical_identifiable_subspace.rank}, rough/declared optical ranks={physical_rough_subspace_design.optical_rank}/{physical_subspace_design.optical_rank}, rough→declared joint/optical projector distances={physical_rough_to_declared_joint_stability.spectral_projector_distance:.3e}/{physical_rough_to_declared_optical_stability.spectral_projector_distance:.3e}. A rough-reference rank change is retained as diagnostic provenance, never silently relabeled as pre-global stability. Declared forward-error Fisher displacement={physical_parameter_stability_initial_report.fisher_mahalanobis_change_in_sigma:.3e} sigma; gate={physical_forward_model_parameter_stability_ready}.")
        if not physical_global_fit_inputs_ready: raise RuntimeError(f"Global fit aborted before 64 starts: the successfully converged pre-global reference fails the nuisance-profiled optical forward-error gate. Diagnostic artifact={physical_parameter_stability_path.relative_to(REPO)}; declared={physical_optical_stability_initial_diagnostics.to_payload()}")

        physical_global_fit=fit_nuisance_profiled_polytope(physical_global_residual_compiled,physical_subspace_design,bounds=(physical_lower,physical_upper),active_x0=np.zeros(physical_identifiable_subspace.rank),multistart=64,seed=20263905,hit_and_run_chains=8,max_nfev=900,optimizer_cost_transform="log_soft_norm")
        physical_terminal_costs=np.asarray(physical_global_fit.active_fit.start_costs,dtype=float); physical_terminal_finite_costs=physical_terminal_costs[np.isfinite(physical_terminal_costs)]; physical_terminal_cost_quantiles=({"minimum":float(np.quantile(physical_terminal_finite_costs,0.)),"q25":float(np.quantile(physical_terminal_finite_costs,.25)),"median":float(np.quantile(physical_terminal_finite_costs,.5)),"q75":float(np.quantile(physical_terminal_finite_costs,.75)),"maximum":float(np.quantile(physical_terminal_finite_costs,1.))} if physical_terminal_finite_costs.size else {"minimum":None,"q25":None,"median":None,"q75":None,"maximum":None}); physical_terminal_optimizer_admitted=bool(physical_global_fit.success and physical_global_fit.fit_rank==physical_identifiable_subspace.rank and np.asarray(physical_global_fit.fit_null_basis_active).shape[1]==0); physical_terminal_result_kind=("converged_terminal_fit" if physical_terminal_optimizer_admitted else "failed_terminal_attempt"); physical_terminal_attempt_provenance={"result_kind":physical_terminal_result_kind,"optimizer_success":bool(physical_global_fit.success),"optimizer_rank_admitted":physical_terminal_optimizer_admitted,"message":str(physical_global_fit.active_fit.message),"nfev":int(physical_global_fit.active_fit.nfev),"njev":None if physical_global_fit.active_fit.njev is None else int(physical_global_fit.active_fit.njev),"successful_starts":int(np.sum(np.asarray(physical_global_fit.active_fit.start_success,dtype=bool))),"start_cost_quantiles":physical_terminal_cost_quantiles,"optimizer_cost_transform":"log_soft_norm","fit_rank":int(physical_global_fit.fit_rank),"declared_joint_rank":int(physical_identifiable_subspace.rank),"declared_optical_rank":int(physical_subspace_design.optical_rank),"declared_nuisance_rank":int(physical_subspace_design.nuisance_rank)}
        physical_terminal_full_J,physical_terminal_subspace_design=physical_design_at_reference(physical_global_fit.parameters); physical_joint_subspace_stability=compare_scaled_active_subspaces(physical_subspace_design.subspace,physical_terminal_subspace_design.subspace); physical_optical_subspace_stability=compare_scaled_active_subspaces(physical_subspace_design.optical_subspace,physical_terminal_subspace_design.optical_subspace); physical_final_active_sigma=np.sqrt(np.maximum(np.diag(physical_global_fit.active_fisher_covariance),0.)); physical_final_update_sigma=float(np.max(np.divide(np.abs(physical_global_fit.active_coordinates),physical_final_active_sigma,out=np.full_like(physical_global_fit.active_coordinates,np.inf),where=physical_final_active_sigma>0))) if physical_global_fit.active_coordinates.size else 0.; physical_final_active_rank_gate=physical_terminal_optimizer_admitted; physical_final_subspace_stable=bool(physical_terminal_optimizer_admitted and physical_joint_subspace_stability.ranks_match and physical_optical_subspace_stability.ranks_match and max(physical_joint_subspace_stability.spectral_projector_distance,physical_optical_subspace_stability.spectral_projector_distance)<=physical_subspace_projector_tolerance and np.all(np.isfinite(physical_final_active_sigma)) and physical_final_update_sigma<physical_subspace_update_tolerance_sigma)
        physical_joint_parameter_stability_terminal_report=assess_fisher_parameter_stability(physical_terminal_full_J,physical_global_fit_bias_whitened,physical_joint_scales,physical_terminal_subspace_design.subspace.active_basis_scaled,successive_refinement_changes_in_sigma=physical_production_parameter_history,required_successive_refinements=2,parameter_change_threshold_sigma=physical_validation_thresholds.parameter_change_in_sigma); physical_optical_stability_terminal_diagnostics=assess_nuisance_profiled_optical_stability(physical_terminal_full_J,physical_global_fit_bias_whitened,physical_joint_scales,optical_parameter_count=column.n_parameters,optical_active_basis_scaled=physical_terminal_subspace_design.optical_subspace.active_basis_scaled,nuisance_rank=physical_terminal_subspace_design.nuisance_rank,optical_parameter_names=column.parameter_names,bias_blocks=physical_global_fit_bias_blocks,successive_refinement_changes_in_sigma=physical_production_parameter_history,required_successive_refinements=2,parameter_change_threshold_sigma=physical_validation_thresholds.parameter_change_in_sigma); physical_parameter_stability_terminal_report=physical_optical_stability_terminal_diagnostics.stability; physical_forward_model_parameter_stability_ready=bool(physical_terminal_optimizer_admitted and physical_optical_stability_initial_diagnostics.passed and physical_optical_stability_terminal_diagnostics.passed); write_physical_parameter_stability_report(physical_optical_stability_terminal_diagnostics,physical_joint_parameter_stability_terminal_report,physical_terminal_result_kind)
        if not physical_terminal_optimizer_admitted:
            print(f"Failed terminal 64-start attempt (diagnostic only; not a post-fit/final result): {physical_terminal_attempt_provenance}. Artifact={physical_parameter_stability_path.relative_to(REPO)}"); raise RuntimeError("The terminal 64-start attempt failed optimizer-success or full-active-rank admission. Its candidate parameters and relinearized design were not promoted, and no fit/profile/held-out artifact will be produced.")
        if not physical_final_subspace_stable: raise RuntimeError(f"The converged terminal optimizer changed the frozen declared quotient or rank: declared/terminal joint={physical_identifiable_subspace.rank}/{physical_terminal_subspace_design.subspace.rank}, optical={physical_subspace_design.optical_rank}/{physical_terminal_subspace_design.optical_rank}, projector distances={physical_joint_subspace_stability.spectral_projector_distance:.3e}/{physical_optical_subspace_stability.spectral_projector_distance:.3e}, active update={physical_final_update_sigma:.3e} sigma. No relinearized quotient is promoted.")
        physical_fitted_full_J=physical_terminal_full_J; physical_postfit_subspace_design=physical_terminal_subspace_design; physical_joint_parameter_stability_postfit_report=physical_joint_parameter_stability_terminal_report; physical_optical_stability_postfit_diagnostics=physical_optical_stability_terminal_diagnostics; physical_parameter_stability_postfit_report=physical_parameter_stability_terminal_report
        physical_profiled_condenser_target_singular,physical_profiled_condenser_target_rank=physical_condenser_target_spectrum(physical_terminal_full_J); physical_rank_hysteresis_values=np.concatenate((np.asarray(physical_subspace_design.profiled_optical_singular_values),np.asarray(physical_terminal_subspace_design.profiled_optical_singular_values),np.asarray(physical_fit_reference_profiled_condenser_target_singular),np.asarray(physical_profiled_condenser_target_singular))); physical_rank_hysteresis_clear=bool(not np.any((physical_rank_hysteresis_values>=.9*physical_minimum_optical_snr)&(physical_rank_hysteresis_values<=1.1*physical_minimum_optical_snr))); physical_practical_condenser_rank_stable=bool(physical_declared_profiled_condenser_target_rank==physical_profiled_condenser_target_rank); physical_practical_condenser_rank_gate=bool(physical_declared_profiled_condenser_target_rank>0 and physical_practical_condenser_rank_stable and physical_rank_hysteresis_clear); physical_full_condenser_coordinate_gate=bool(physical_structural_condenser_target_gate and physical_practical_condenser_rank_gate and physical_declared_profiled_condenser_target_rank==len(hardware_condenser_indices)); physical_rank_gate_for_route=bool(physical_structural_condenser_target_gate and physical_subspace_design.optical_rank>0 and physical_terminal_subspace_design.optical_rank==physical_subspace_design.optical_rank and physical_practical_condenser_rank_gate)
        print(f"Condenser/rank audit: route-native structural={physical_structural_condenser_target_rank}/12; full-column practical at SNR>{physical_minimum_optical_snr:.0f}: rough diagnostic={physical_initial_profiled_condenser_target_rank}/12, declared pre-global={physical_declared_profiled_condenser_target_rank}/12, converged terminal={physical_profiled_condenser_target_rank}/12; rough/declared/terminal optical ranks={physical_rough_subspace_design.optical_rank}/{physical_subspace_design.optical_rank}/{physical_terminal_subspace_design.optical_rank}; declared→terminal joint/optical projector distances={physical_joint_subspace_stability.spectral_projector_distance:.3e}/{physical_optical_subspace_stability.spectral_projector_distance:.3e}; threshold hysteresis clear={physical_rank_hysteresis_clear}; stable declared-quotient gate={physical_practical_condenser_rank_gate}; final active fit rank={physical_global_fit.fit_rank}/{physical_identifiable_subspace.rank}, gate={physical_final_active_rank_gate}. Primary nuisance-profiled optical Fisher displacement declared/terminal={physical_parameter_stability_initial_report.fisher_mahalanobis_change_in_sigma:.3e}/{physical_parameter_stability_postfit_report.fisher_mahalanobis_change_in_sigma:.3e} sigma; rough reference remains diagnostic at {physical_parameter_stability_rough_report.fisher_mahalanobis_change_in_sigma:.3e} sigma; influence gate={physical_forward_model_parameter_stability_ready}.")
        if CONDENSER_ROUTE_BRANCH=="ASTIGMATIC":
            _,astigmatic_fitted_optical,_,_,astigmatic_fitted_covariance,_,astigmatic_fitted_extra=split_physical_outer_joint(jnp.asarray(physical_global_fit.parameters)); astigmatic_fitted_native=selected_astigmatic_adapter.likelihood_parameters_from_parts(astigmatic_fitted_optical,jnp.concatenate((astigmatic_fitted_covariance,astigmatic_fitted_extra))); physical_condenser_held_out_evaluation=evaluate_astigmatic_held_out(selected_astigmatic_likelihood,astigmatic_fitted_native,selected_astigmatic_held_out_split); physical_condenser_held_out_gate=bool(physical_condenser_held_out_evaluation.rms_standardized_residual<1.2 and physical_condenser_held_out_evaluation.maximum_absolute_standardized_residual<4.); print(f"Excluded astigmatic condenser states {physical_condenser_held_out_evaluation.state_indices} / acquisitions {physical_condenser_held_out_evaluation.acquisition_indices}: {physical_condenser_held_out_evaluation.degrees_of_freedom} observables, chi2={physical_condenser_held_out_evaluation.chi_square:.3f}, WRMS={physical_condenser_held_out_evaluation.rms_standardized_residual:.3f} sigma, max|z|={physical_condenser_held_out_evaluation.maximum_absolute_standardized_residual:.3f}, gate={physical_condenser_held_out_gate}.")
        else:
            print(f"Selected comparison route {CONDENSER_ROUTE} has no registered condenser held-out split; route acceptance remains fail-closed.")
        physical_fitted_base_joint=np.asarray(physical_global_fit.parameters[:-physical_outer_nuisance_count]); physical_held_out_evaluation=evaluate_held_out_shared_likelihood(physical_shared_likelihood,physical_statistical_dataset,physical_fitted_base_joint); physical_held_out_per_block_gate=bool(all(block.weighted_rms_sigma<2. and block.maximum_absolute_whitened_residual<4. for block in physical_held_out_evaluation.blocks)); physical_held_out_statistical_gate=bool(physical_held_out_evaluation.weighted_rms_sigma<1.2 and physical_held_out_evaluation.maximum_absolute_whitened_residual<4. and physical_held_out_per_block_gate)
        print(f"Excluded downstream physical held-out prediction ({physical_downstream_data_kind}): {physical_held_out_evaluation.n_blocks} blocks, {physical_held_out_evaluation.n_observations} observables, chi2={physical_held_out_evaluation.chi_square:.3f}, WRMS={physical_held_out_evaluation.weighted_rms_sigma:.3f} sigma, max|z|={physical_held_out_evaluation.maximum_absolute_whitened_residual:.3f}.")
        for block in physical_held_out_evaluation.blocks: print(f"  frame {block.frame_id} / {block.optical_state_label}: {len(block.observable_names)} obs, chi2={block.chi_square:.3f}, WRMS={block.weighted_rms_sigma:.3f} sigma, max|z|={block.maximum_absolute_whitened_residual:.3f}")
        if CONDENSER_ROUTE_IS_SYNTHETIC:
            physical_held_out_known_state=np.array([.30e-6,.20e-3,-.15e-6,-.12e-3]); physical_held_out_w0=.18e-6; physical_held_out_theta0=.14e-3; physical_position_indices=np.array([0,2]); physical_angle_indices=np.array([1,3]); physical_pixel_xy=np.asarray(detector.pixel_size_yx_m)[::-1]; physical_held_out_diffraction_specs={spec.optical_state_index:spec for spec in physical_diffraction_specs if plan.states[spec.optical_state_index].role=="held_out"}; assert set(physical_held_out_diffraction_specs)==set(np.flatnonzero(plan.held_out_mask))
            def physical_held_out_engineering_observables(parameters,imaging_dacs,diffraction_spec):
                matrix=np.asarray(column.matrix(parameters,imaging_dacs,segment="sample_to_detector")); A=detector.affine_xy@matrix[np.ix_(physical_position_indices,physical_position_indices)]; B=detector.affine_xy@matrix[np.ix_(physical_position_indices,physical_angle_indices)]; affine=fit_fourier_affine(np.eye(2),A.T); centroid=(detector.affine_xy@(matrix@physical_held_out_known_state)[physical_position_indices])/physical_pixel_xy; covariance=A@(physical_held_out_w0**2*np.eye(2))@A.T+B@(physical_held_out_theta0**2*np.eye(2))@B.T; diameter=2*np.sqrt(np.trace(covariance)/2)/np.mean(physical_pixel_xy); diffraction=np.asarray(predict_physical_observable_blocks(column,jnp.asarray(parameters),detector,(diffraction_spec,))[0]).reshape(-1,2); return affine.isotropic_scale,np.rad2deg(affine.rotation_rad),centroid,diameter,diffraction
            physical_held_out_engineering_reports=[]; physical_fitted_optical=np.asarray(physical_global_fit.parameters[:column.n_parameters])
            for state_index in np.flatnonzero(plan.held_out_mask):
                state=np.asarray(plan.control_array())[state_index]; diffraction_spec=physical_held_out_diffraction_specs[int(state_index)]; predicted=physical_held_out_engineering_observables(physical_fitted_optical,state,diffraction_spec); reference=physical_held_out_engineering_observables(physical_parameters,state,diffraction_spec); physical_held_out_engineering_reports.append(evaluate_held_out_metrics(predicted_magnification=predicted[0],reference_magnification=reference[0],predicted_rotation_deg=predicted[1],reference_rotation_deg=reference[1],predicted_centroid_xy_px=predicted[2],reference_centroid_xy_px=reference[2],predicted_beam_diameter_px=predicted[3],reference_beam_diameter_px=reference[3],predicted_diffraction_vectors_yx_px=predicted[4],reference_diffraction_vectors_yx_px=reference[4],require_diffraction_vectors=True))
            physical_held_out_engineering_reports=tuple(physical_held_out_engineering_reports); physical_held_out_engineering_gate=bool(physical_held_out_engineering_reports and all(report.passed for report in physical_held_out_engineering_reports))
            for index,report in enumerate(physical_held_out_engineering_reports,1): print(f"  synthetic engineering held-out {index}: mag={100*report.magnification_relative_error:.4f}% rot={report.rotation_error_deg:.4f} deg centroid={report.centroid_error_px:.4f}px diameter={100*report.beam_diameter_relative_error:.4f}% diffraction={report.diffraction_vector_error_px:.4f}px pass={report.passed}")
        else:
            physical_held_out_engineering_reports=(); physical_held_out_engineering_gate=False; print("Instrument route: engineering held-out metrics against physical_parameters are forbidden. No measured magnification/rotation/centroid/diameter/diffraction-vector evaluator is bound, so the engineering and final-instrument gates remain false.")

        physical_active_polytope_marginal_half_width=.5*(np.asarray(physical_global_fit.active_coordinate_bounds[1])-np.asarray(physical_global_fit.active_coordinate_bounds[0])); assert np.all(np.isfinite(physical_active_polytope_marginal_half_width)) and np.all(physical_active_polytope_marginal_half_width>0); physical_basin_relative_polytope_threshold=.05; physical_multistart_diagnostics=diagnose_multistart_basins(physical_global_fit.active_fit,delta_cost=1.,scaled_distance_threshold=physical_basin_relative_polytope_threshold,parameter_scales=physical_active_polytope_marginal_half_width); successful=np.asarray(physical_global_fit.active_fit.start_success,dtype=bool); near_best_count=len(physical_multistart_diagnostics.near_best_start_indices); best_basin=physical_multistart_diagnostics.basins[0] if physical_multistart_diagnostics.basins else None
        if CONDENSER_ROUTE_IS_SYNTHETIC:
            physical_truth_scaled_delta=(physical_truth_joint-physical_identifiable_subspace.reference_parameters)/physical_joint_scales; physical_truth_active_coordinates=physical_identifiable_subspace.active_basis_scaled.T@physical_truth_scaled_delta; physical_truth_null_coordinates=physical_identifiable_subspace.null_basis_scaled.T@physical_truth_scaled_delta; active_sigma=np.sqrt(np.maximum(np.diag(physical_global_fit.active_fisher_covariance),0)); active_z=np.divide(physical_global_fit.active_coordinates-physical_truth_active_coordinates,active_sigma,out=np.full_like(active_sigma,np.inf),where=active_sigma>0); physical_matched_recovery=assess_matched_model_recovery(physical_subspace_design.optical_subspace,physical_global_fit.parameters[:column.n_parameters],physical_parameters,named_metric_policy="physical_optical",optical_parameter_names=tuple(column.parameter_names)); physical_named_optical_recovery=tuple({"name":item.name,"identifiable_individually":item.identifiable_individually,"null_space_weight":item.null_space_weight,"relative_or_log_error":item.named_metric_error,"metric":item.named_metric_kind,"passed":item.one_percent_gate_passed} for item in physical_matched_recovery.parameters); physical_individual_named_gates=tuple(item.one_percent_gate_passed for item in physical_matched_recovery.parameters if item.identifiable_individually); physical_named_recovery_gate=bool(all(value is True for value in physical_individual_named_gates)); physical_matched_recovery_gate=bool(physical_matched_recovery.active_one_percent_gate_passed and physical_named_recovery_gate); physical_recovery_gate_for_route=physical_matched_recovery_gate
        else:
            physical_truth_active_coordinates=None; physical_truth_null_coordinates=None; active_sigma=None; active_z=np.empty(0); physical_matched_recovery=None; physical_named_optical_recovery=(); physical_matched_recovery_gate=False; physical_recovery_gate_for_route=True  # Instrument data have no known physical truth; recovery-vs-synthetic nominal is inapplicable and never gates them.
        physical_postfit_rank_gate=bool(physical_final_subspace_stable and physical_postfit_subspace_design.optical_rank==physical_subspace_design.optical_rank and physical_postfit_subspace_design.nuisance_rank==physical_subspace_design.nuisance_rank and physical_rank_hysteresis_clear); physical_fit_equivalent_family=EquivalentSolutionFamily(parameters=np.asarray(physical_global_fit.parameters),parameter_names=physical_joint_names,basis=np.asarray(physical_global_fit.subspace.null_basis_physical),singular_values=np.asarray(physical_subspace_design.joint_singular_values),rank=physical_global_fit.subspace.rank,threshold=physical_minimum_optical_snr); physical_local_null_report=verify_local_null_families(physical_fit_equivalent_family,physical_global_residual_compiled,bounds=(physical_lower,physical_upper),parameter_scales=physical_joint_scales,delta_cost_tolerance=1.,dominant_loading_count=4)
        physical_null_report_path=GENERATED/"neoarm_physical_null_families_v3.json"; physical_null_report_payload={"schema":"neoarm-local-null-family-v3","base_cost":physical_local_null_report.base_cost,"delta_cost_tolerance":physical_local_null_report.delta_cost_tolerance,"straight_sample_count":physical_local_null_report.straight_sample_count,"rank":physical_fit_equivalent_family.rank,"nullity":physical_fit_equivalent_family.nullity,"families":[{"family_index":family.family_index,"status":family.status,"maximum_absolute_endpoint_delta_cost":family.maximum_absolute_endpoint_delta_cost,"maximum_absolute_sampled_delta_cost":family.maximum_absolute_sampled_delta_cost,"sampled_scaled_coordinates":family.sampled_scaled_coordinates,"sampled_costs":family.sampled_costs,"sampled_delta_costs":family.sampled_delta_costs,"requested_negative_scaled_coordinate":family.requested_negative_scaled_coordinate,"requested_positive_scaled_coordinate":family.requested_positive_scaled_coordinate,"negative_physical_bound_reached":family.negative_physical_bound_reached,"positive_physical_bound_reached":family.positive_physical_bound_reached,"dominant_loadings":[{"parameter_name":loading.parameter_name,"scaled_loading":loading.scaled_loading} for loading in family.dominant_loadings],"negative_endpoint":{"coordinate":family.negative_endpoint.signed_scaled_coordinate,"delta_cost":family.negative_endpoint.delta_cost,"active_bounds":family.negative_endpoint.active_bounds},"positive_endpoint":{"coordinate":family.positive_endpoint.signed_scaled_coordinate,"delta_cost":family.positive_endpoint.delta_cost,"active_bounds":family.positive_endpoint.active_bounds}} for family in physical_local_null_report.families]}; physical_null_report_path.write_text(json.dumps(physical_null_report_payload,indent=2,sort_keys=True,default=lambda value:np.asarray(value).tolist()),encoding="utf-8")
        verified_null_count=sum(family.verified_equivalent for family in physical_local_null_report.families); optical_identifiable_count=(sum(item.identifiable_individually for item in physical_matched_recovery.parameters) if physical_matched_recovery is not None else 0)
        if physical_matched_recovery is not None: print(f"Synthetic matched-model optical recovery: max retained optical-combination error={100*physical_matched_recovery.maximum_active_combination_error_scaled:.4f}% of its natural physical scale; exact physical-optical gate={physical_matched_recovery_gate}; independently identifiable named optical coordinates={optical_identifiable_count}/{column.n_parameters}. Non-identifiable names remain family-valued.")
        else: print("Instrument route: matched-model truth recovery is N/A and is not used as a gate.")
        print(f"Local-null straight-segment audit: {len(physical_local_null_report.families)} families sampled at {physical_local_null_report.straight_sample_count} bounded coordinates each; {verified_null_count} residual-equivalent at Δcost≤1, {len(physical_local_null_report.families)-verified_null_count} retained as local-null candidates; postfit rank stable={physical_postfit_rank_gate}; report={physical_null_report_path.relative_to(REPO)}.")
        _,physical_fit_optical,physical_fit_calibration,physical_fit_discrepancy,physical_fit_covariance,physical_fit_deflector,physical_fit_extra=split_physical_outer_joint(jnp.asarray(physical_global_fit.parameters)); physical_condenser_total_residual=np.asarray(selected_condenser_global_residual(physical_fit_optical,physical_fit_covariance,physical_fit_deflector,physical_fit_extra))
        if CONDENSER_ROUTE_BRANCH=="ASTIGMATIC": physical_condenser_data_residual=np.concatenate((physical_condenser_total_residual[:88],physical_condenser_total_residual[92:])); physical_condenser_prior_residual=physical_condenser_total_residual[88:92]
        else: physical_condenser_data_residual=physical_condenser_total_residual; physical_condenser_prior_residual=np.empty(0)
        physical_downstream_training_data_residual=np.asarray(physical_shared_likelihood.data_residual_from_parts(physical_fit_optical,physical_fit_calibration,physical_fit_discrepancy)); physical_training_data_residual=np.concatenate((physical_condenser_data_residual,physical_downstream_training_data_residual)); physical_training_data_wrms=float(np.sqrt(np.mean(physical_training_data_residual**2))); physical_prior_residual=np.concatenate((physical_condenser_prior_residual,np.asarray(physical_fit_calibration),np.asarray(physical_fit_discrepancy))); physical_prior_penalty=float(.5*physical_prior_residual@physical_prior_residual); physical_prior_max_abs=float(np.max(np.abs(physical_prior_residual))) if physical_prior_residual.size else 0.; physical_prior_gate=bool(physical_prior_max_abs<4.)
        near_best_spread=np.inf if best_basin is None else best_basin.scaled_radius; sampled_single_basin=physical_multistart_diagnostics.single_sampled_near_best_basin; representative_joint=(None if best_basin is None else np.asarray(physical_identifiable_subspace.parameters_from_active(jnp.asarray(best_basin.representative_parameters)))); physical_terminal_joint=np.asarray(physical_global_fit.parameters,dtype=float); physical_bound_hit_tolerance=1e-7*np.maximum(physical_upper-physical_lower,np.finfo(float).tiny); representative_bound_hits=(() if representative_joint is None else tuple(f"{name}:{side}" for name,value,lo,hi,tol in zip(physical_joint_names,representative_joint,physical_lower,physical_upper,physical_bound_hit_tolerance,strict=True) for side in (("lower",) if abs(value-lo)<=tol else ("upper",) if abs(value-hi)<=tol else ()))); physical_terminal_bound_hits=tuple(f"{name}:{side}" for name,value,lo,hi,tol in zip(physical_joint_names,physical_terminal_joint,physical_lower,physical_upper,physical_bound_hit_tolerance,strict=True) for side in (("lower",) if abs(value-lo)<=tol else ("upper",) if abs(value-hi)<=tol else ())); representative_gradient_norm=np.inf if best_basin is None else best_basin.representative_gradient_norm
        physical_terminal_active_residual=np.asarray(physical_global_fit.active_fit.residuals,dtype=float); physical_terminal_active_J=np.asarray(physical_global_fit.active_fit.jacobian,dtype=float); physical_terminal_raw_active_gradient=physical_terminal_active_J.T@physical_terminal_active_residual; physical_terminal_raw_active_gradient_inf_norm=float(np.max(np.abs(physical_terminal_raw_active_gradient))) if physical_terminal_raw_active_gradient.size else 0.; physical_terminal_active_left,physical_terminal_active_singular,physical_terminal_active_right_t=np.linalg.svd(physical_terminal_active_J,full_matrices=False); physical_terminal_active_svd_rank=int(np.sum(physical_terminal_active_singular>physical_global_fit.fit_threshold)); physical_terminal_fisher_rank_gate=bool(physical_terminal_active_svd_rank==physical_identifiable_subspace.rank==physical_global_fit.fit_rank)
        if physical_terminal_fisher_rank_gate:
            physical_terminal_active_vectors=physical_terminal_active_right_t[:physical_terminal_active_svd_rank].T; physical_terminal_newton_correction=-physical_terminal_active_vectors@((physical_terminal_active_left[:,:physical_terminal_active_svd_rank].T@physical_terminal_active_residual)/physical_terminal_active_singular[:physical_terminal_active_svd_rank]); physical_terminal_fisher_covariance_explicit=physical_terminal_active_vectors@np.diag(1./physical_terminal_active_singular[:physical_terminal_active_svd_rank]**2)@physical_terminal_active_vectors.T; physical_terminal_fisher_sigma=np.sqrt(np.maximum(np.diag(physical_terminal_fisher_covariance_explicit),0.)); physical_terminal_coordinate_standardized_newton_components=np.divide(np.abs(physical_terminal_newton_correction),physical_terminal_fisher_sigma,out=np.full_like(physical_terminal_newton_correction,np.inf),where=physical_terminal_fisher_sigma>0); physical_terminal_fisher_standardized_components=np.abs((physical_terminal_active_right_t[:physical_terminal_active_svd_rank]@physical_terminal_raw_active_gradient)/physical_terminal_active_singular[:physical_terminal_active_svd_rank]); physical_terminal_fisher_standardized_components_from_residual=np.abs(physical_terminal_active_left[:,:physical_terminal_active_svd_rank].T@physical_terminal_active_residual); physical_terminal_fisher_mode_identity_max_abs_error=float(np.max(np.abs(physical_terminal_fisher_standardized_components-physical_terminal_fisher_standardized_components_from_residual))) if physical_terminal_fisher_standardized_components.size else 0.; physical_terminal_fisher_standardized_interior_score=float(np.max(physical_terminal_fisher_standardized_components)) if physical_terminal_fisher_standardized_components.size else 0.
        else: physical_terminal_newton_correction=np.full(physical_identifiable_subspace.rank,np.nan); physical_terminal_fisher_sigma=np.full(physical_identifiable_subspace.rank,np.nan); physical_terminal_coordinate_standardized_newton_components=np.full(physical_identifiable_subspace.rank,np.inf); physical_terminal_fisher_standardized_components=np.full(physical_identifiable_subspace.rank,np.inf); physical_terminal_fisher_standardized_components_from_residual=np.full(physical_identifiable_subspace.rank,np.inf); physical_terminal_fisher_mode_identity_max_abs_error=np.inf; physical_terminal_fisher_standardized_interior_score=np.inf
        physical_terminal_fisher_standardized_interior_tolerance=1e-3; physical_terminal_interior_stationarity_gate=bool(physical_terminal_fisher_rank_gate and not physical_terminal_bound_hits and np.isfinite(physical_terminal_fisher_standardized_interior_score) and physical_terminal_fisher_standardized_interior_score<physical_terminal_fisher_standardized_interior_tolerance); physical_terminal_stationarity_provenance={"raw_active_gradient_inf_norm_diagnostic":physical_terminal_raw_active_gradient_inf_norm,"active_svd_rank":physical_terminal_active_svd_rank,"declared_active_rank":physical_identifiable_subspace.rank,"fisher_rank_gate":physical_terminal_fisher_rank_gate,"retained_mode_formula":"eta_i = abs((V.T @ g)_i / s_i) = abs((U.T @ r)_i)","retained_fisher_mode_eta_components":physical_terminal_fisher_standardized_components.tolist(),"retained_fisher_mode_eta_from_residual_components":physical_terminal_fisher_standardized_components_from_residual.tolist(),"retained_fisher_mode_identity_max_abs_error":physical_terminal_fisher_mode_identity_max_abs_error,"retained_fisher_mode_eta_inf_norm":physical_terminal_fisher_standardized_interior_score,"coordinatewise_newton_over_marginal_sigma_components_secondary_non_gating":physical_terminal_coordinate_standardized_newton_components.tolist(),"fisher_standardized_tolerance":physical_terminal_fisher_standardized_interior_tolerance,"physical_bound_hits":physical_terminal_bound_hits,"interior_stationarity_gate":physical_terminal_interior_stationarity_gate}; write_physical_parameter_stability_report(physical_optical_stability_terminal_diagnostics,physical_joint_parameter_stability_terminal_report,physical_terminal_result_kind); physical_active_truth_gate=(bool(np.all(np.isfinite(active_z)) and np.max(np.abs(active_z))<3.5) if CONDENSER_ROUTE_IS_SYNTHETIC else True); physical_optimization_gate=bool(physical_global_fit.success and physical_forward_model_parameter_stability_ready and physical_final_active_rank_gate and physical_refresh_converged and physical_final_subspace_stable and physical_rank_gate_for_route and physical_recovery_gate_for_route and physical_active_truth_gate and physical_postfit_rank_gate and successful.sum()>=48 and near_best_count>=48 and sampled_single_basin and near_best_spread<physical_basin_relative_polytope_threshold and physical_terminal_interior_stationarity_gate and physical_training_data_wrms<1.2 and physical_prior_gate and physical_condenser_held_out_gate and physical_held_out_statistical_gate and physical_held_out_engineering_gate and selected_condenser_route_global_gate_ready and (not CONDENSER_ROUTE_IS_INSTRUMENT or physical_downstream_instrument_dataset_bound))

        physical_active_recovery_text=(f"{100*physical_matched_recovery.maximum_active_combination_error_scaled:.4f}% natural scale ({np.max(np.abs(active_z)):.2f} sigma)" if physical_matched_recovery is not None else "N/A for instrument data"); print(f"64-start selected-route global fit ({CONDENSER_ROUTE}) over the complete active physical-bound polytope: success={physical_global_fit.success}, converged={successful.sum()}/64, near-best={near_best_count}/64 in {physical_multistart_diagnostics.n_near_best_basins} sampled basin(s), high-cost={len(physical_multistart_diagnostics.high_cost_start_indices)}, data-only training WRMS={physical_training_data_wrms:.3f} sigma, posterior objective WRMS={physical_global_fit.weighted_rmse:.3f}, prior penalty={physical_prior_penalty:.3f} (max |z|={physical_prior_max_abs:.3f}), retained rank={physical_global_fit.fit_rank}/{physical_identifiable_subspace.rank}, active recovery={physical_active_recovery_text}, pilot refresh converged={physical_refresh_converged}, declared→terminal joint/optical projector distances={physical_joint_subspace_stability.spectral_projector_distance:.3e}/{physical_optical_subspace_stability.spectral_projector_distance:.3e}, terminal active update={physical_final_update_sigma:.3e} sigma, near-best basin radius={near_best_spread:.3f} relative marginal-polytope units (threshold {physical_basin_relative_polytope_threshold:.3f}), raw selected active-gradient infinity norm={physical_terminal_raw_active_gradient_inf_norm:.3e} (diagnostic only), retained Fisher-mode stationarity eta infinity norm={physical_terminal_fisher_standardized_interior_score:.3e} (threshold {physical_terminal_fisher_standardized_interior_tolerance:.1e}), selected physical-bound hits={physical_terminal_bound_hits}, representative-basin gradient/bound hits={representative_gradient_norm:.3e}/{representative_bound_hits}, synthetic matched-recovery gate={physical_matched_recovery_gate if CONDENSER_ROUTE_IS_SYNTHETIC else 'N/A'}, condenser/downstream-statistical/engineering held-out gates={physical_condenser_held_out_gate}/{physical_held_out_statistical_gate}/{physical_held_out_engineering_gate}, pre-fit selected-route gate={selected_condenser_route_global_gate_ready}, optimization/held-out gate before profiles={physical_optimization_gate}. Multistart basin evidence is sampled, never a mathematical proof of uniqueness; final route acceptance is evaluated only after every requested profile and the local-null audit.")

        profile_directions=[]

        if physical_subspace_design.optical_rank:
            weakest_optical_seed=np.concatenate((physical_subspace_design.optical_subspace.active_basis_scaled[:,physical_subspace_design.optical_rank-1],np.zeros(physical_subspace_design.nuisance_parameter_count))); weakest_resolved_direction=physical_global_fit.subspace.active_basis_scaled@(physical_global_fit.subspace.active_basis_scaled.T@weakest_optical_seed); profile_directions.append(("weakest admitted optical combination",weakest_resolved_direction,None))

        for null_index,family in enumerate(physical_local_null_report.families): profile_directions.append((f"null {null_index+1}: {family.status.replace('_',' ')}",family.direction_scaled,(family.negative_endpoint.signed_scaled_coordinate,family.positive_endpoint.signed_scaled_coordinate)))

        def bounded_profile_grid(direction,endpoint_coordinates=None,requested_half_width=.08,interior_fraction=.95):
            direction=np.asarray(direction,dtype=float); direction=direction/np.linalg.norm(direction); physical_direction=physical_joint_scales*direction
            if endpoint_coordinates is None:
                negative_candidates=np.full_like(physical_direction,-np.inf); positive_candidates=np.full_like(physical_direction,np.inf); positive_mask=physical_direction>0; negative_mask=physical_direction<0; negative_candidates[positive_mask]=(physical_lower[positive_mask]-physical_global_fit.parameters[positive_mask])/physical_direction[positive_mask]; negative_candidates[negative_mask]=(physical_upper[negative_mask]-physical_global_fit.parameters[negative_mask])/physical_direction[negative_mask]; positive_candidates[positive_mask]=(physical_upper[positive_mask]-physical_global_fit.parameters[positive_mask])/physical_direction[positive_mask]; positive_candidates[negative_mask]=(physical_lower[negative_mask]-physical_global_fit.parameters[negative_mask])/physical_direction[negative_mask]; negative=max(float(np.max(negative_candidates)),-requested_half_width); positive=min(float(np.min(positive_candidates)),requested_half_width)
            else: negative,positive=(float(value) for value in endpoint_coordinates)
            negative=min(0.,negative)*interior_fraction; positive=max(0.,positive)*interior_fraction; grid=np.unique(np.concatenate((np.linspace(negative,0.,4),np.linspace(0.,positive,4))));
            if grid.size<2: raise RuntimeError("No non-zero in-bounds displacement is available for the requested profile direction.")
            return grid
        profiles=[]; physical_profile_failures=[]; physical_profile_prerequisite_gate=bool(physical_optimization_gate)

        if physical_profile_prerequisite_gate:
            for label,direction,endpoint_coordinates in profile_directions:
                try: profiles.append(profile_nuisance_profiled_polytope_direction(physical_global_residual_compiled,physical_global_fit,direction,bounded_profile_grid(direction,endpoint_coordinates),bounds=(physical_lower,physical_upper),direction_name=label,multistart=4,seed=20260917+len(profiles),hit_and_run_chains=4,max_nfev=350,optimizer_cost_transform="log_soft_norm"))
                except (ValueError,RuntimeError,FloatingPointError) as error: physical_profile_failures.append({"direction_name":label,"error":f"{type(error).__name__}: {error}"})
        else: physical_profile_failures.append({"direction_name":"all","error":"profiles skipped because the finite/rank/recovery/held-out optimization prerequisite gate failed"})

        physical_profile_results=tuple(profiles); physical_profile_execution_gate=bool(len(physical_profile_results)==len(profile_directions) and not physical_profile_failures and all(np.all(np.asarray(profile.success,dtype=bool)) and np.all(np.isfinite(profile.costs)) for profile in physical_profile_results)); physical_profile_base_cost_tolerance=1e-5*max(1.,abs(physical_global_fit.cost)); physical_profile_base_consistency_gate=bool(physical_profile_results and all(np.any(np.isclose(np.asarray(profile.grid),0.,rtol=0.,atol=1e-14)) and abs(float(np.asarray(profile.costs)[np.argmin(np.abs(np.asarray(profile.grid)))])-physical_global_fit.cost)<=physical_profile_base_cost_tolerance for profile in physical_profile_results)); physical_profile_lower_alternative_delta_cost_threshold=physical_profile_base_cost_tolerance; physical_profile_no_lower_alternative_gate=bool(physical_profile_results and all(float(np.min(np.asarray(profile.costs)-physical_global_fit.cost))>=-physical_profile_lower_alternative_delta_cost_threshold for profile in physical_profile_results)); physical_resolved_profile_delta_cost_threshold=1.; resolved_profile=next((profile for profile in physical_profile_results if profile.direction_name=="weakest admitted optical combination"),None)
        if resolved_profile is None: physical_resolved_profile_curvature_gate=False
        else:
            resolved_grid=np.asarray(resolved_profile.grid); resolved_delta=np.asarray(resolved_profile.costs)-physical_global_fit.cost; resolved_negative=resolved_grid<0; resolved_positive=resolved_grid>0; physical_resolved_profile_curvature_gate=bool(np.any(resolved_negative) and np.any(resolved_positive) and np.max(resolved_delta[resolved_negative])>=physical_resolved_profile_delta_cost_threshold and np.max(resolved_delta[resolved_positive])>=physical_resolved_profile_delta_cost_threshold)
        physical_null_profile_classifications=[]
        for family_index,family in enumerate(physical_local_null_report.families):
            prefix=f"null {family_index+1}:"; profile=next((candidate for candidate in physical_profile_results if candidate.direction_name.startswith(prefix)),None); signed_delta=(np.asarray((np.inf,)) if profile is None else np.asarray(profile.costs)-physical_global_fit.cost); maximum_delta=float(np.max(np.abs(signed_delta))); minimum_delta=float(np.min(signed_delta)); classification=("profile_failed" if profile is None else "lower_cost_alternative_beyond_numerical_tolerance" if minimum_delta<-physical_profile_lower_alternative_delta_cost_threshold else "conditionally_flat_abs_delta_cost_le_1" if maximum_delta<=physical_local_null_report.delta_cost_tolerance else "conditionally_curved_without_lower_alternative"); physical_null_profile_classifications.append({"family_index":family_index,"straight_segment_status":family.status,"conditional_profile_status":classification,"minimum_signed_profiled_delta_cost":minimum_delta,"maximum_absolute_profiled_delta_cost":maximum_delta})
        physical_null_profile_classifications=tuple(physical_null_profile_classifications); physical_null_segment_domain_gate=bool(all(family.status not in {"finite_domain_truncated","degenerate_bounded_segment","invalid_straight_segment"} and family.negative_physical_bound_reached and family.positive_physical_bound_reached and np.all(np.isfinite(np.asarray(family.sampled_costs))) and np.isfinite(family.maximum_absolute_sampled_delta_cost) for family in physical_local_null_report.families)); physical_profile_postfit_gate=bool(physical_profile_execution_gate and physical_profile_base_consistency_gate and physical_profile_no_lower_alternative_gate and physical_resolved_profile_curvature_gate and physical_null_segment_domain_gate and len(physical_null_profile_classifications)==len(physical_local_null_report.families))
        if CONDENSER_ROUTE_BRANCH=="ASTIGMATIC": selected_condenser_route_postfit_gate=bool(physical_condenser_held_out_evaluation is not None and physical_condenser_held_out_gate and physical_profile_postfit_gate)
        else: selected_condenser_route_postfit_gate=False  # Comparison branches have no registered condenser held-out evaluator in this notebook and remain fail-closed.
        physical_global_fit_ready=bool(physical_forward_model_parameter_stability_ready and physical_optimization_gate and physical_profile_postfit_gate and (CONDENSER_ROUTE_BRANCH!="ASTIGMATIC" or selected_condenser_route_postfit_gate))
        final_instrument_acceptance=bool(CONDENSER_ROUTE_IS_INSTRUMENT and physical_downstream_instrument_dataset_bound and selected_condenser_route_global_gate_ready and physical_global_fit_ready and selected_condenser_route_postfit_gate and physical_literal_global_two_refinement_acceptance)
        print(f"Post-fit route audit: all profiles executed={physical_profile_execution_gate}; profile zero agrees with global cost within {physical_profile_base_cost_tolerance:.3g}={physical_profile_base_consistency_gate}; no profiled alternative improves cost by more than {physical_profile_lower_alternative_delta_cost_threshold:g}={physical_profile_no_lower_alternative_gate}; weakest resolved two-sided Δcost≥{physical_resolved_profile_delta_cost_threshold:g} curvature={physical_resolved_profile_curvature_gate}; null segments reached both declared physical bounds without a model-domain truncation={physical_null_segment_domain_gate}; null classifications={physical_null_profile_classifications}; failures={physical_profile_failures}; combined profile gate={physical_profile_postfit_gate}; route-specific held-out/profile evidence={selected_condenser_route_postfit_gate}; final instrument acceptance={final_instrument_acceptance}. Hit-and-run multistarts sample the complete active physical-bound polytope, but remain sampled evidence and never a mathematical proof of global uniqueness.")

        fig_identifiable,identified_axes=plt.subplots(1,2,figsize=(11.4,4.0),constrained_layout=True); singular=physical_subspace_design.profiled_optical_singular_values; identified_axes[0].semilogy(np.arange(1,len(singular)+1),singular,"o-",color="#0072B2"); identified_axes[0].axhline(physical_minimum_optical_snr,color="#D55E00",ls="--",label=f"optical admission SNR>{physical_minimum_optical_snr:.0f}"); identified_axes[0].set(xlabel="nuisance-profiled optical direction",ylabel="whitened singular value",title=f"Optical rank {physical_subspace_design.optical_rank}/{column.n_parameters}; nuisance {physical_subspace_design.nuisance_rank}/{physical_subspace_design.nuisance_parameter_count}"); identified_axes[0].legend(frameon=False)

        profile_colors=("#0072B2","#D55E00","#009E73","#CC79A7","#E69F00");
        for profile_index,profile in enumerate(physical_profile_results): identified_axes[1].plot(profile.grid,np.asarray(profile.costs)-physical_global_fit.cost,"o-",color=profile_colors[profile_index%len(profile_colors)],label=profile.direction_name)

        identified_axes[1].axhline(0.,color="#6B7280",lw=.7); identified_axes[1].set(xlabel="scaled direction coordinate",ylabel="conditional cost minus global-fit cost",title="Resolved curvature versus local-null audit"); identified_axes[1].legend(frameon=False); [axis.grid(alpha=.2) for axis in identified_axes]; fig_identifiable.suptitle("What the compact NEOARM experiment can—and cannot—identify",fontsize=12.5)
        physical_identifiable_metadata={"title":"Physical NEOARM nuisance-profiled identifiable subspace","selected_condenser_route":CONDENSER_ROUTE,"selected_route_status":selected_condenser_route_status,"condenser_data_kind":selected_condenser_route_status["data_kind"],"downstream_data_kind":physical_downstream_data_kind,"downstream_instrument_dataset_bound":physical_downstream_instrument_dataset_bound,"absolute_intensity_fit_observable":physical_absolute_intensity_fit_observable,"sample_frame_total_counts_enter_fit":False,"sample_frame_total_count_role":physical_sample_frame_total_count_role,"joint_active_rank":physical_identifiable_subspace.rank,"joint_practical_nullity":physical_identifiable_subspace.nullity,"nuisance_profiled_optical_rank":physical_subspace_design.optical_rank,"numerical_nuisance_rank":physical_subspace_design.nuisance_rank,"nuisance_parameter_count":physical_subspace_design.nuisance_parameter_count,"optical_minimum_snr":physical_minimum_optical_snr,"nuisance_rank_rtol":physical_nuisance_rank_rtol,"profiled_optical_singular_values":singular,"profiled_condenser_target_singular_values":physical_profiled_condenser_target_singular,"profiled_condenser_target_rank":physical_profiled_condenser_target_rank,"fit_success":physical_global_fit.success,"optimization_gate_before_profiles":physical_optimization_gate,"selected_route_fit_accepted":physical_global_fit_ready,"final_instrument_acceptance":final_instrument_acceptance,"training_weighted_rms_sigma":physical_global_fit.weighted_rmse,"matched_recovery_applicable":CONDENSER_ROUTE_IS_SYNTHETIC,"matched_recovery_gate":(physical_matched_recovery_gate if CONDENSER_ROUTE_IS_SYNTHETIC else None),"maximum_active_recovery_fraction_of_scale":(None if physical_matched_recovery is None else physical_matched_recovery.maximum_active_combination_error_scaled),"named_optical_recovery":physical_named_optical_recovery,"postfit_rank_stable":physical_postfit_rank_gate,"condenser_held_out_gate":physical_condenser_held_out_gate,"condenser_held_out_state_indices":(None if physical_condenser_held_out_evaluation is None else physical_condenser_held_out_evaluation.state_indices),"condenser_held_out_acquisition_indices":(None if physical_condenser_held_out_evaluation is None else physical_condenser_held_out_evaluation.acquisition_indices),"condenser_held_out_weighted_rms_sigma":(None if physical_condenser_held_out_evaluation is None else physical_condenser_held_out_evaluation.rms_standardized_residual),"condenser_held_out_maximum_absolute_residual_sigma":(None if physical_condenser_held_out_evaluation is None else physical_condenser_held_out_evaluation.maximum_absolute_standardized_residual),"held_out_statistical_gate":physical_held_out_statistical_gate,"held_out_engineering_gate":physical_held_out_engineering_gate,"held_out_weighted_rms_sigma":physical_held_out_evaluation.weighted_rms_sigma,"held_out_maximum_absolute_whitened_residual":physical_held_out_evaluation.maximum_absolute_whitened_residual,"held_out_blocks":[{"frame_id":block.frame_id,"optical_state_label":block.optical_state_label,"observable_names":block.observable_names,"chi_square":block.chi_square,"weighted_rms_sigma":block.weighted_rms_sigma,"maximum_absolute_whitened_residual":block.maximum_absolute_whitened_residual} for block in physical_held_out_evaluation.blocks],"held_out_engineering":[{"magnification_relative_error":report.magnification_relative_error,"rotation_error_deg":report.rotation_error_deg,"centroid_error_px":report.centroid_error_px,"beam_diameter_relative_error":report.beam_diameter_relative_error,"diffraction_vector_error_px":report.diffraction_vector_error_px,"passed":report.passed} for report in physical_held_out_engineering_reports],"converged_multistarts":int(successful.sum()),"near_best_multistarts":near_best_count,"high_cost_multistarts":len(physical_multistart_diagnostics.high_cost_start_indices),"sampled_near_best_basin_count":physical_multistart_diagnostics.n_near_best_basins,"multistarts":64,"maximum_active_truth_error_sigma":(None if not CONDENSER_ROUTE_IS_SYNTHETIC else float(np.max(np.abs(active_z)))),"near_best_basin_radius_relative_active_polytope_marginals":near_best_spread,"basin_relative_polytope_threshold":physical_basin_relative_polytope_threshold,"active_polytope_marginal_half_widths":physical_active_polytope_marginal_half_width,"representative_gradient_norm":representative_gradient_norm,"representative_physical_bound_hits":representative_bound_hits,"proposed_initial_scaled_offset":physical_initial_scaled_offset,"accepted_blind_factor":physical_blind_factor,"discarded_initial_scaled_norm":physical_discarded_initial_scaled_norm,"discarded_initial_residual_norm":physical_discarded_initial_residual_norm,"entrance_covariance_released":True,"condenser_outer_parameter_names":physical_condenser_outer_names,"condenser_extra_nuisance_count":physical_condenser_extra_nuisance_count,"astigmatic_design_sha256":fullcolumn_astigmatic_design.sha256,"astigmatic_observation_kind":selected_astigmatic_observation_kind,"astigmatic_commanded_tilt_used":False,"astigmatic_instrument_ready":(fullcolumn_astigmatic_instrument_ready if CONDENSER_ROUTE_IS_INSTRUMENT else None),"fixed_internal_plane_z_m":[certificate.plane_z_m for certificate in physical_fit_plane_certificates],"fixed_internal_plane_worst_clearance_m":physical_fit_worst_plane_clearance_m,"profile_directions":[profile.direction_name for profile in physical_profile_results],"profile_grids":[profile.grid for profile in physical_profile_results],"local_null_report":str(physical_null_report_path.relative_to(REPO)),"verified_endpoint_equivalent_null_families":verified_null_count,"local_null_candidate_count":len(physical_local_null_report.families)-verified_null_count,"interpretation":"Nuisance directions are retained before optical admission. Optical directions use an explicit SNR>75 screen; exact one-percent matched-synthetic recovery is gated separately. One reference start plus 63 starts from eight independently seeded hit-and-run chains sample the complete bound-feasible active polytope and are optimized with its exact linear constraints; this is strong sampled-basin evidence, never a mathematical proof of uniqueness. The initializer is a deterministic perturbation of the matched synthetic truth, so this run tests local recovery rather than rough-model discovery. Every requested profile must execute, the weakest admitted optical direction must show two-sided Δcost≥1 curvature, and null-family model-domain truncation fails closed. The absolute-intensity rows are unnormalised C-aperture Poisson counts. Sample-frame totals are validation-only because these renders truth-backpropagate a unit-current known_sample_source and no downstream clipping/detection emulator is validated. Instrument routes require separately registered condenser and downstream observations/covariances and never use synthetic truth recovery."}
        if CONDENSER_ROUTE_BRANCH!="ASTIGMATIC":
            for route_specific_key in ("astigmatic_design_sha256","astigmatic_observation_kind","astigmatic_commanded_tilt_used","astigmatic_instrument_ready"): physical_identifiable_metadata.pop(route_specific_key,None)
            physical_identifiable_metadata["astigmatic_route_metadata"]="not_applicable_to_selected_condenser_route"
        for stale_key in ("training_weighted_rms_sigma","accepted_blind_factor","discarded_initial_scaled_norm","discarded_initial_residual_norm"): physical_identifiable_metadata.pop(stale_key,None)
        physical_identifiable_metadata.update({"data_only_training_weighted_rms_sigma":physical_training_data_wrms,"posterior_objective_weighted_rms":physical_global_fit.weighted_rmse,"prior_penalty":physical_prior_penalty,"prior_maximum_absolute_z":physical_prior_max_abs,"prior_gate":physical_prior_gate,"initial_truth_active_distance_scaled":physical_initial_truth_active_distance,"initial_truth_null_gauge_distance_scaled":physical_initial_truth_null_gauge_distance,"initial_to_truth_residual_norm":physical_initial_to_truth_residual_norm,"truth_null_coordinates":physical_truth_null_coordinates,"fit_reference_is_synthetic_truth":False,"axial_scale_policy":"origin-invariant local mechanical/design scales","subspace_refresh_history":physical_refresh_history,"subspace_refresh_converged":physical_refresh_converged,"subspace_projector_tolerance":physical_subspace_projector_tolerance,"subspace_active_update_tolerance_sigma":physical_subspace_update_tolerance_sigma,"final_joint_projector_distance":physical_joint_subspace_stability.spectral_projector_distance,"final_optical_projector_distance":physical_optical_subspace_stability.spectral_projector_distance,"final_joint_maximum_principal_angle_rad":physical_joint_subspace_stability.maximum_principal_angle_rad,"final_optical_maximum_principal_angle_rad":physical_optical_subspace_stability.maximum_principal_angle_rad,"final_active_update_max_sigma":physical_final_update_sigma,"final_subspace_stable":physical_final_subspace_stable,"final_active_fit_rank_gate":physical_final_active_rank_gate,"initial_profiled_condenser_target_singular_values":physical_initial_profiled_condenser_target_singular,"initial_profiled_condenser_target_rank":physical_initial_profiled_condenser_target_rank,"fit_reference_profiled_condenser_target_singular_values":physical_fit_reference_profiled_condenser_target_singular,"fit_reference_profiled_condenser_target_rank":physical_fit_reference_profiled_condenser_target_rank,"final_profiled_condenser_target_singular_values":physical_profiled_condenser_target_singular,"final_profiled_condenser_target_rank":physical_profiled_condenser_target_rank,"rank_hysteresis_clear":physical_rank_hysteresis_clear,"held_out_per_block_gate":physical_held_out_per_block_gate})
        physical_identifiable_metadata.update({"fit_result_kind":physical_terminal_result_kind,"terminal_attempt_provenance":physical_terminal_attempt_provenance,"optimizer_cost_transform":"log_soft_norm","rough_reference_joint_rank":physical_rough_identifiable_subspace.rank,"rough_reference_optical_rank":physical_rough_subspace_design.optical_rank,"rough_reference_nuisance_rank":physical_rough_subspace_design.nuisance_rank,"declared_pre_global_joint_rank":physical_identifiable_subspace.rank,"declared_pre_global_optical_rank":physical_subspace_design.optical_rank,"declared_pre_global_nuisance_rank":physical_subspace_design.nuisance_rank,"converged_terminal_joint_rank":physical_terminal_subspace_design.subspace.rank,"converged_terminal_optical_rank":physical_terminal_subspace_design.optical_rank,"converged_terminal_nuisance_rank":physical_terminal_subspace_design.nuisance_rank,"rough_to_declared_joint_ranks_match_non_gating":physical_rough_to_declared_joint_stability.ranks_match,"rough_to_declared_optical_ranks_match_non_gating":physical_rough_to_declared_optical_stability.ranks_match,"rough_to_declared_joint_projector_distance_non_gating":physical_rough_to_declared_joint_stability.spectral_projector_distance,"rough_to_declared_optical_projector_distance_non_gating":physical_rough_to_declared_optical_stability.spectral_projector_distance,"declared_to_terminal_joint_ranks_match":physical_joint_subspace_stability.ranks_match,"declared_to_terminal_optical_ranks_match":physical_optical_subspace_stability.ranks_match,"terminal_raw_active_gradient_inf_norm_diagnostic":physical_terminal_raw_active_gradient_inf_norm,"terminal_retained_fisher_mode_formula":"eta_i = abs((V.T @ g)_i / s_i) = abs((U.T @ r)_i)","terminal_retained_fisher_mode_eta_components":physical_terminal_fisher_standardized_components,"terminal_retained_fisher_mode_identity_max_abs_error":physical_terminal_fisher_mode_identity_max_abs_error,"terminal_coordinatewise_newton_over_marginal_sigma_components_secondary_non_gating":physical_terminal_coordinate_standardized_newton_components,"terminal_fisher_standardized_interior_score":physical_terminal_fisher_standardized_interior_score,"terminal_fisher_standardized_interior_tolerance":physical_terminal_fisher_standardized_interior_tolerance,"terminal_fisher_rank_gate":physical_terminal_fisher_rank_gate,"terminal_physical_bound_hits":physical_terminal_bound_hits,"terminal_interior_stationarity_gate":physical_terminal_interior_stationarity_gate})
        physical_identifiable_metadata.update({"profile_execution_gate":physical_profile_execution_gate,"profile_prerequisite_gate":physical_profile_prerequisite_gate,"resolved_profile_two_sided_curvature_gate":physical_resolved_profile_curvature_gate,"resolved_profile_delta_cost_threshold":physical_resolved_profile_delta_cost_threshold,"null_segment_domain_gate":physical_null_segment_domain_gate,"profile_postfit_gate":physical_profile_postfit_gate,"profile_failures":physical_profile_failures,"null_profile_classifications":physical_null_profile_classifications,"selected_route_postfit_gate":selected_condenser_route_postfit_gate,"declared_prefit_instrument_status_not_used_for_acceptance":selected_condenser_route_declared_prefit_instrument_status,"selected_obj_prefield_anchor_bound":selected_obj_prefield_anchor_bound,"rank_gate_for_route":physical_rank_gate_for_route,"selected_astigmatic_observation_kind":selected_astigmatic_observation_kind,"selected_astigmatic_route_ready":selected_astigmatic_route_ready,"max_nfev_per_start":900,"multistart_strategy":("truth-perturbed matched-synthetic pilot recentering followed by one reference start and 63 starts distributed across eight independently seeded hit-and-run chains, each optimized by SLSQP under the exact active physical-box constraints" if CONDENSER_ROUTE_IS_SYNTHETIC else "declared nominal/reference pilot recentering followed by one reference start and 63 starts distributed across eight independently seeded hit-and-run chains, each optimized by SLSQP under the exact active physical-box constraints"),"sampled_uniqueness_scope":"complete_bound_feasible_active_polytope; eight-chain 64-start evidence, never proof of global uniqueness","shared_layer_certificate_latent_groups":[prior.group_name for prior in physical_calibration_priors if prior.group_name.startswith("specimen_layers:")],"unified_real_reciprocal_certificate_covariance_stripped":physical_statistical_dataset.metadata["multilayer_certificate_covariance"]["stripped_for_shared_latent_likelihood"]})
        physical_identifiable_metadata.update({"structural_condenser_target_rank_source":physical_structural_condenser_rank_source_by_branch[CONDENSER_ROUTE_BRANCH],"structural_condenser_target_rank":physical_structural_condenser_target_rank,"structural_condenser_target_gate":physical_structural_condenser_target_gate,"condenser_target_rank_kind":"full_column_practical_snr_thresholded","declared_practical_condenser_target_rank":physical_declared_profiled_condenser_target_rank,"practical_condenser_rank_stable":physical_practical_condenser_rank_stable,"practical_condenser_rank_gate":physical_practical_condenser_rank_gate,"full_condenser_coordinate_gate":physical_full_condenser_coordinate_gate,"full_condenser_coordinate_gate_used_for_acceptance":False,"uncertainty_estimation":physical_global_dataset.metadata.get("uncertainty_estimation",{})})
        physical_identifiable_metadata.update({"forward_model_parameter_stability_gate":physical_forward_model_parameter_stability_ready,"forward_model_parameter_stability_scope":"nuisance-profiled identifiable optical quotient frozen at the successfully converged pre-global pilot reference","forward_model_parameter_stability_rough_reference_diagnostic":physical_optical_stability_rough_diagnostics.to_payload(),"forward_model_parameter_stability_declared_pre_global":physical_optical_stability_initial_diagnostics.to_payload(),"forward_model_parameter_stability_converged_terminal":physical_optical_stability_terminal_diagnostics.to_payload(),"joint_optical_plus_nuisance_stability_rough_reference_non_gating":physical_joint_parameter_stability_rough_report.to_payload(),"joint_optical_plus_nuisance_stability_declared_pre_global_non_gating":physical_joint_parameter_stability_initial_report.to_payload(),"joint_optical_plus_nuisance_stability_converged_terminal_non_gating":physical_joint_parameter_stability_terminal_report.to_payload(),"forward_model_parameter_stability_report":str(physical_parameter_stability_path.relative_to(REPO)),"forward_model_bias_provenance":physical_fit_bias_provenance,"forward_model_bias_block_provenance":physical_bias_block_provenance,"per_signature_parameter_values_fabricated":False,"representative_two_refinement_gate":physical_parameter_stability_terminal_report.successive_refinements_passed,"literal_all_identifiable_parameter_two_successive_refinement_acceptance":physical_literal_global_two_refinement_acceptance,"parameter_stability_interpretation":"strict basis-invariant nuisance-profiled optical Fisher displacement below 0.1 sigma at the successfully converged declared pre-global reference and converged terminal fit; the rough reference is retained as non-gating diagnostic provenance. The two actual representative 150k topology transitions refit only one conditional IL1.q_bias parameter while every other optical and nuisance coordinate is fixed; they are not full-quotient refinement evidence, so literal all-optical-direction two-successive-refinement acceptance remains provisional pending global rerenders/refits"})
        if not CONDENSER_ROUTE_IS_SYNTHETIC:
            physical_identifiable_metadata.update({"initial_truth_active_distance_scaled":None,"initial_truth_null_gauge_distance_scaled":None,"initial_to_truth_residual_norm":None,"truth_null_coordinates":None,"initializer_scope":"declared nominal/reference model; no synthetic truth is available or consulted","interpretation":"Nuisance directions are retained before optical admission. Optical directions use an explicit SNR>75 screen and are required to remain rank-stable at the declared fit reference and final solution. One reference start plus 63 starts from eight independently seeded hit-and-run chains sample the complete bound-feasible active polytope; this is sampled-basin evidence, never a mathematical proof of uniqueness. Every requested profile and held-out instrument gate must pass; matched-synthetic truth recovery is inapplicable and is never substituted."})
        else: physical_identifiable_metadata["initializer_scope"]="deterministic perturbation of matched synthetic truth; local recovery test, not independent rough-model discovery"
        physical_identifiable_metadata["condenser_rank_interpretation"]=(f"The selected route has {physical_structural_condenser_target_rank}/12 route-native structural rank, while the unified full-column likelihood admits {physical_declared_profiled_condenser_target_rank}/12 stable practical condenser combinations at SNR>{physical_minimum_optical_snr:.0f}. Acceptance recovers and profiles only that identifiable quotient; all twelve named centre/length/strength coordinates require stronger axial-diversity data or credible centre/length priors.")
        physical_identifiable_metadata.update({"conditional_profile_strategy":"one warm/base start plus three starts from three independently seeded hit-and-run chains, each optimized by SLSQP over the complete fixed-direction active-polytope slice","conditional_profile_multistarts_per_grid_point":4,"conditional_profile_requested_hit_and_run_chains_per_grid_point":4,"conditional_profile_instantiated_hit_and_run_chains_per_grid_point":3,"conditional_profile_max_nfev_per_start":350})
        physical_identifiable_metadata.update({"profile_zero_cost_consistency_gate":physical_profile_base_consistency_gate,"profile_zero_cost_tolerance":physical_profile_base_cost_tolerance,"profile_no_lower_alternative_gate":physical_profile_no_lower_alternative_gate,"profile_lower_alternative_delta_cost_threshold":physical_profile_lower_alternative_delta_cost_threshold})
        physical_identifiable_exports=export_ray_figure(fig_identifiable,GENERATED/"neoarm_physical_identifiable_subspace_v2",metadata=physical_identifiable_metadata); plt.show(); plt.close(fig_identifiable); print("Identifiability exports:",physical_identifiable_exports)

    else: print(f"Physical pilot and 64-start fit are configured but disabled by NEOARM_RUN_PHYSICAL_GLOBAL_FIT=0. The rough-reference {physical_parameter_stability_rough_report.active_rank}-direction influence result is diagnostic only; no quotient has been declared and literal all-direction two-successive-refinement acceptance remains provisional.")

else: print(f"Physical global inverse ({CONDENSER_ROUTE}) remains fail-closed until both checkpoints, the registered downstream dataset, and every exact physical-render signature pass the independent grid-wave ledger.")
final_instrument_acceptance=bool(physical_global_fit_ready and CONDENSER_ROUTE_IS_INSTRUMENT and physical_downstream_instrument_dataset_bound and selected_condenser_route_global_gate_ready and selected_condenser_route_postfit_gate and physical_literal_global_two_refinement_acceptance)

## 5. Differentiable hard-edge column and identifiability

Each lens has a center, positive physical length, integrated Larmor bias, and—when controllable—a raw-DAC slope. The sample plane fixes the axial gauge. The exact finite-body map contains Larmor rotation and a stable $\sin(q)/q$ term and is symplectic to numerical precision. The full-global adapter selects one explicitly named condenser route. The preferred astigmatic route maps the one shared 36-parameter physical column into its exact 43-parameter likelihood and appends only 27 acquisition nuisances: ten entrance-covariance coordinates, aperture/stage scale, twelve edge-local coordinates, two specimen/diffraction scales, and one common tag orientation. It contains no commanded tilt, deflector, or K16 coordinate. Branch A and B remain explicit comparison routes. All routes use the actual 0.250/0.290/0.295 m fixed-plane bounds. Synthetic observations can establish computational recovery but cannot produce instrument acceptance; instrument selections fail closed. Held-out manifest blocks are the exact complement of the training layout and are evaluated only after the full fit.

There are two deliberately separate evidence tracks. The primary `HeterogeneousInversionDataset` is built from the 230 GPU-rendered detector frames after every required material/diffraction topology passes a signature-bound independent FFT/CZT or analytic Gaussian reference. Its correlated physical observables and shared calibration latents enter the identifiable-subspace global likelihood together with the selected condenser block. The later transfer-response fit remains only a cheaper analytic diagnostic. Parameters are nondimensionalized with local mechanical/design scales that are invariant to the arbitrary axial origin; only directions above the declared practical threshold are released. The astigmatic acquisition has route-native local structural rank 12/12, but the unified full-column likelihood currently admits only six of those twelve condenser combinations at SNR greater than 75. The rough-reference rank is retained explicitly as a non-gating diagnostic. Only a successful, full-rank, stable pilot may declare and freeze the pre-global quotient; that declared rank and its scaled projectors must then remain unchanged at the converged terminal solution. Failed pilot candidates are never promoted, and a failed/rank-deficient terminal attempt stops before any fit, profile, held-out, or ray artifact is labelled final. The physical polytope and conditional-profile optimizers use a strictly monotone log-soft conditioning of the scalar cost while every reported residual, likelihood cost, gradient, and Fisher quantity remains the original whitened least-squares quantity. In the matched-synthetic demonstration, the practical-null gauge and first Jacobian are centred on a deterministic truth-perturbed initializer; this is an honest local-recovery test, not a claim of discovery from an independently measured rough column. The terminal interior-stationarity gate uses the basis-invariant retained singular/Fisher-mode score `eta_i = abs((V.T @ g)_i / s_i) = abs((U.T @ r)_i)` and requires `max(eta_i) < 1e-3` with no physical-bound hit; the raw gradient and coordinatewise Newton-correction diagnostic remain non-gating. These checks are reported separately, so neither a small singular value nor a failed global gate is hidden by fixing a named lens parameter.

A truth-free instrument initializer must not reuse the synthetic `physical_parameters`. Begin from measured mechanical lens centres/lengths and the declared nominal DAC laws. First fit IL1/IL2/IL3/PL1 strengths with the condenser fixed, using the calibrated sample-plane position/angle responses, image affine scale/rotation, and diffraction vectors. Next hold that downstream solution and fit CL1/CL3/C-mini strengths plus entrance-covariance, aperture, and dose nuisances from the translated C-aperture counts and deliberately astigmatic tag covariance/orientation data; no commanded tilt or deflector calibration is assumed. Then release all strength biases/slopes jointly, retain mechanical centre/length priors, and release centre/length combinations only when the combined D-optimal data admit them above the SNR threshold. Each stage may relinearize only after optimizer success and full active rank. Once the physical pilot meets the update and projector tolerances, freeze its declared quotient for the 64-start fit and never redefine that quotient from the terminal result.

In [ ]:
p_initial=column.nominal_parameters.copy(); names=column.parameter_names; parameter_scales=design_parameter_scales.copy(); state_controls=jnp.asarray(plan.control_array())
def predict_state_downstream(parameters): return jax.vmap(lambda dac:response_observable_vector(column.matrix(parameters,dac,segment="sample_to_detector")))(state_controls).reshape(-1)
def predict_responses(parameters): return jnp.concatenate((complete_condenser_response(parameters),predict_state_downstream(parameters)))
downstream_response_sigma=np.array([2e-5]*4+[2e-6]*4); downstream_truth=np.asarray(predict_state_downstream(jnp.asarray(p_truth))); downstream_sigma=np.tile(downstream_response_sigma,plan.n_states); downstream_observed=downstream_truth+RNG.normal(scale=.04*downstream_sigma)
truth_values=np.concatenate((np.asarray(complete_condenser_response(jnp.asarray(p_truth))),downstream_truth)); observed=np.concatenate((complete_condenser_observed,downstream_observed)); sigma=np.concatenate((complete_condenser_sigma,downstream_sigma))
state_training_mask=np.asarray([state.role!="held_out" for state in plan.states]); combined_training_mask=np.concatenate((np.ones(len(complete_condenser_observed),dtype=bool),np.repeat(state_training_mask,len(downstream_response_sigma)))); combined_training_indices=jnp.asarray(np.flatnonzero(combined_training_mask))
def combined_residual(parameters): return (predict_responses(parameters)[combined_training_indices]-jnp.asarray(observed)[combined_training_indices])/jnp.asarray(sigma)[combined_training_indices]
def whitened_from_scaled(u): return (predict_responses(jnp.asarray(p_truth)+jnp.asarray(parameter_scales)*u)[combined_training_indices]-jnp.asarray(truth_values)[combined_training_indices])/jnp.asarray(sigma)[combined_training_indices]
J=np.asarray(jax_jacobian(whitened_from_scaled,jnp.zeros(column.n_parameters)))
svd=jacobian_svd(J,rtol=1e-7); condenser_row_count=len(complete_condenser_observed); blockwise_rank=diagnose_blockwise_identifiability((("upstream_condenser",J[:condenser_row_count]),("downstream_sample_to_detector",J[condenser_row_count:])),optical_parameter_count=column.n_parameters,parameter_scales=np.ones(column.n_parameters),rank_rtol=1e-7)
J_physical=J/parameter_scales[None,:]
family=equivalent_solution_family(J_physical,p_truth,parameter_names=names,parameter_scales=parameter_scales,rtol=1e-7)
transfer_response_fit_ready=bool(np.all(np.isfinite(observed)) and np.all(np.isfinite(sigma)) and np.all(sigma>0) and np.all(np.isfinite(J)))
physical_validation_coverage=physical_validation_ledger.coverage; physical_render_set_complete=bool(full_imaging_checkpoint.complete and signed_defocus_checkpoint.complete and full_diffraction_checkpoint.complete and native_diffraction_contract_ready); physical_observation_dataset_ready=isinstance(globals().get("physical_global_dataset"),HeterogeneousInversionDataset); physical_image_global_fit_ready=bool(physical_render_set_complete and physical_observation_dataset_ready and physical_validation_coverage.complete)
print("Physical-image global fit readiness:",f"{physical_validation_coverage.accepted_count}/{physical_validation_coverage.required_count} exact render signatures accepted;",f"failed={physical_validation_coverage.failed_count}; missing={physical_validation_coverage.missing_count};",f"native diffraction contract={native_diffraction_contract_ready};",f"renders complete={physical_render_set_complete};",f"dataset assembled={physical_observation_dataset_ready};",f"ready={physical_image_global_fit_ready}")
active_observations=int(combined_training_mask.sum()); print(f"exact combined vector: condenser block once ({len(complete_condenser_observed)} observables from {2*len(all_recon_scan_values)} reconnaissance + {3*len(condenser_probe_dacs)} probe frames) + one joint A/B response per optical state assembled from the 230-frame manifest; training observations={active_observations}; scaled Jacobian {J.shape}; raw-global rank={svd.rank}; condition={svd.condition_number:.3e}")
print("Blockwise column-equilibrated structural ranks:",{report.block_name:report.column_equilibrated_joint.rank for report in blockwise_rank.block_reports},"combined",blockwise_rank.combined.column_equilibrated_joint.rank,"/",column.n_parameters,"(column equilibration diagnoses rank; it does not reweight the likelihood).")
for k in range(min(4,svd.right_singular_vectors.shape[0])):
    vector=svd.right_singular_vectors[-1-k]; strongest=np.argsort(np.abs(vector))[-4:][::-1]
    print(f"weak direction {k+1}: "+", ".join(f"{names[i]}={vector[i]:+.3f}" for i in strongest))

fig,axes=plt.subplots(1,2,figsize=(10.5,3.8),constrained_layout=True)
axes[0].semilogy(np.arange(1,len(svd.singular_values)+1),svd.singular_values,"o-",color="#0072B2"); axes[0].axhline(svd.threshold,color="#D55E00",ls="--"); axes[0].set(xlabel="singular direction",ylabel="singular value",title="Whitened, scaled Jacobian spectrum")
steps=np.linspace(-3,3,101); weakest=svd.singular_values[-min(3,len(svd.singular_values)):]
for value in weakest: axes[1].plot(steps,.5*(value*steps)**2,label=f"σ={value:.2g}")
axes[1].set(xlabel="local singular coordinate",ylabel="linearized Δ cost",title="Local profile families"); axes[1].legend(frameon=False)
plt.show()

In [ ]:
from temgym_core.microscope_inversion import fit_least_squares
smoke_scale=np.full(3,.02)
def smoke_model_scaled(u):
    full=jnp.asarray(p_initial).at[jnp.asarray(smoke_idx)].set(jnp.asarray(p_initial[smoke_idx])+jnp.asarray(smoke_scale)*u)
    return predict_responses(full)
smoke_lower=np.full(3,-4.); smoke_upper=np.full(3,4.)
def smoke_residual(u): return (smoke_model_scaled(u)[combined_training_indices]-jnp.asarray(observed)[combined_training_indices])/jnp.asarray(sigma)[combined_training_indices]
smoke_fit=fit_least_squares(smoke_residual,np.zeros(3),bounds=(smoke_lower,smoke_upper),parameter_names=tuple(f"scaled:{name}" for name in smoke_names),loss="linear",multistart=3,seed=11,jitter_scale=.15,max_nfev=200)
p_fitted=p_initial.copy(); p_fitted[smoke_idx]=p_initial[smoke_idx]+smoke_scale*smoke_fit.parameters
active_weighted_rms=float(np.sqrt(np.mean(smoke_fit.residuals**2))); smoke_relative_errors=np.abs(p_fitted[smoke_idx]-p_truth[smoke_idx])/np.maximum(np.abs(p_truth[smoke_idx]),np.finfo(float).tiny); smoke_matched_recovery_pass=bool(smoke_fit.success and smoke_fit.jacobian_rank==len(smoke_idx) and active_weighted_rms<1.0 and np.all(smoke_relative_errors<.01)); print(f"smoke fit success={smoke_fit.success}; active weighted RMS={active_weighted_rms:.4f}; rank={smoke_fit.jacobian_rank}/3; DoF={smoke_fit.degrees_of_freedom}; all parameters within 1%={np.all(smoke_relative_errors<.01)}")
for name,truth,estimate in zip(smoke_names,p_truth[smoke_idx],p_fitted[smoke_idx],strict=True): print(f"  {name:18s} truth={truth:+.6f} fit={estimate:+.6f} error={estimate-truth:+.2e}")
assert smoke_matched_recovery_pass
nearby=equivalent_solutions(smoke_fit,delta_cost=1,parameter_scales=np.ones(3)); print("distinct statistically equivalent smoke optima:",len(nearby.costs))
profile_grid=np.linspace(smoke_fit.parameters[0]-.4,smoke_fit.parameters[0]+.4,7)
profile=profile_likelihood(smoke_residual,smoke_fit.parameters,0,profile_grid,bounds=(smoke_lower,smoke_upper),parameter_names=tuple(f"scaled:{name}" for name in smoke_names),max_nfev=35)
fig,ax=plt.subplots(figsize=(5.2,3.3),constrained_layout=True); ax.plot(profile.grid,profile.delta_cost,"o-",color="#0072B2"); ax.set(xlabel=profile.parameter_name,ylabel="profile Δ cost",title="Conditional profile likelihood"); plt.show()

unrestricted_physical_bounds=column.default_bounds(position_half_width_m=3e-3,length_ratio=(.4,2.5),q_half_width=.8,slope_half_width=.8); transfer_aperture_fit_bounds=restrict_hard_edge_internal_plane_bounds(column,unrestricted_physical_bounds,plane_z_m=float(aperture_cfg["z_m"]),upstream_lens_name="CL1",downstream_lens_name="CL3",minimum_clearance_m=APERTURE_FIT_MINIMUM_CLEARANCE_M,reference_parameters=p_initial); physical_lower,physical_upper=transfer_aperture_fit_bounds.bounds
assert transfer_aperture_fit_bounds.worst_case_clearance_m>=APERTURE_FIT_MINIMUM_CLEARANCE_M; print(f"Transfer-response aperture fit box: restricted={transfer_aperture_fit_bounds.was_restricted}, worst clearance={transfer_aperture_fit_bounds.worst_case_clearance_m*1e3:.3f} mm.")
u_lower=(physical_lower-p_initial)/parameter_scales; u_upper=(physical_upper-p_initial)/parameter_scales
def transfer_response_model_scaled(u): return predict_responses(jnp.asarray(p_initial)+jnp.asarray(parameter_scales)*u)
def transfer_response_residual_scaled(u): return (transfer_response_model_scaled(u)[combined_training_indices]-jnp.asarray(observed)[combined_training_indices])/jnp.asarray(sigma)[combined_training_indices]
TRANSFER_RESPONSE_FIT_OPTIONS=dict(multistart=64,seed=20260902,jitter_scale=.3,loss="soft_l1",f_scale=1,max_nfev=1200)
transfer_response_fit=None
if RUN_TRANSFER_RESPONSE_GLOBAL_FIT:
    if not transfer_response_fit_ready: raise RuntimeError("The synthetic transfer-response dataset failed its finite-value readiness check.")
    transfer_response_fit=fit_least_squares(transfer_response_residual_scaled,np.zeros(column.n_parameters),bounds=(u_lower,u_upper),parameter_names=tuple(f"scaled:{n}" for n in names),**TRANSFER_RESPONSE_FIT_OPTIONS)
    print("64-start transfer-response feasibility fit weighted RMS:",float(np.sqrt(np.mean(transfer_response_fit.residuals**2))),"rank/DoF:",transfer_response_fit.jacobian_rank,transfer_response_fit.degrees_of_freedom,"equivalent optima:",len(equivalent_solutions(transfer_response_fit).costs))
else: print("64-start transfer-response feasibility fit configured but not run; this is separate from the fail-closed physical-image global fit.")

### Held-out acceptance

Held-out **measurements** are not used to tune topology, hierarchy, or parameters. Their crops and topology IDs are predeclared from the design model and uncertainty envelope before acquisition; each held-out setting includes indexed diffraction. The full shared-latent likelihood is built from the exact non-held-out complement. The astigmatic condenser route independently reserves states 2, 3, and 7 (acquisitions 4, 5, and 19); removing their 21 summaries retains route-local structural rank 12/12 and a 0.9980% worst local target uncertainty under that route's native scaling. This does not override the unified full-column SNR-thresholded practical rank. Dense train/test covariance is split before whitening, and the held-out score uses the conditional innovation and Schur covariance. After—and only after—the 64-start fit, fitted coordinates predict both this condenser holdout and the excluded GPU image/defocus/diffraction blocks. We report covariance-whitened residuals separately from the engineering limits: magnification 0.5%, rotation 0.5°, centroid 0.5 pixel, beam diameter 2%, and indexed diffraction vectors one pixel. If the full fit was not run, this section fails closed rather than substituting the earlier three-parameter smoke fit.

In [ ]:
if physical_global_fit is None or physical_held_out_evaluation is None:
    heldout_reports=(); print("Full physical held-out acceptance is fail-closed: run NEOARM_RUN_PHYSICAL_GLOBAL_FIT=1 after all render/reference signatures pass. The three-parameter smoke fit is not substituted.")
else:
    heldout_reports=physical_held_out_engineering_reports
    if physical_condenser_held_out_evaluation is not None: print(f"Condenser holdout: states={physical_condenser_held_out_evaluation.state_indices}, observations={physical_condenser_held_out_evaluation.degrees_of_freedom}, WRMS={physical_condenser_held_out_evaluation.rms_standardized_residual:.3f} sigma, max|z|={physical_condenser_held_out_evaluation.maximum_absolute_standardized_residual:.3f}, gate={physical_condenser_held_out_gate}.")
    print(f"Full-fit downstream held-out summary: blocks={physical_held_out_evaluation.n_blocks}, observations={physical_held_out_evaluation.n_observations}, WRMS={physical_held_out_evaluation.weighted_rms_sigma:.3f} sigma, max|z|={physical_held_out_evaluation.maximum_absolute_whitened_residual:.3f}, statistical gate={physical_held_out_statistical_gate}, engineering gate={physical_held_out_engineering_gate}.")
    for index,report in enumerate(heldout_reports,1): print(f"held-out {index}: mag={100*report.magnification_relative_error:.4f}% rot={report.rotation_error_deg:.4f} deg centroid={report.centroid_error_px:.4f}px diameter={100*report.beam_diameter_relative_error:.4f}% diffraction={report.diffraction_vector_error_px:.4f}px pass={report.passed}")

### Model-hierarchy gate

A thin rotating lens supplies a stable initializer. The finite hard-edge model is the physical default because it adds body propagation and Larmor rotation with few parameters. A normalized Glaser field adds one profile-width degree of freedom and is promoted only when its held-out RMS falls by more than 0.2 experimental standard deviations **and** by at least 20% relative to the hard-edge result. The executable synthetic check below deliberately uses a smooth field, so the promotion gate should open; this is a unit test of the decision rule, not evidence that the NEOARM needs a Glaser profile. A generic round symplectic section remains a predictive fallback and is never interpreted as pole-piece geometry.

In [ ]:
hierarchy_length_m=.020; truth_glaser_width_m=.25*hierarchy_length_m
q_train=jnp.asarray([-.70,-.40,.40,.70]); q_held=jnp.asarray([-.85,-.55,.55,.85])
coordinate_scale=jnp.diag(jnp.asarray([1/hierarchy_length_m,1.,1/hierarchy_length_m,1.])); coordinate_unscale=jnp.linalg.inv(coordinate_scale)
@jax.jit
def glaser_family(width,q_values):
    return jax.vmap(lambda q: coordinate_scale@glaser_solenoid_matrix(q,0.,width,-hierarchy_length_m/2,hierarchy_length_m/2,n_slices=32)@coordinate_unscale)(q_values)
@jax.jit
def hard_edge_family(q_values): return jax.vmap(lambda q: coordinate_scale@hard_edge_solenoid_matrix(q,hierarchy_length_m)@coordinate_unscale)(q_values)
@jax.jit
def thin_family(q_values): return jax.vmap(lambda q: coordinate_scale@thin_lens_matrix(q*q/hierarchy_length_m,-q)@coordinate_unscale)(q_values)
truth_train=glaser_family(truth_glaser_width_m,q_train)
width_candidates=jnp.linspace(.10*hierarchy_length_m,.50*hierarchy_length_m,21)
width_losses=jax.vmap(lambda width:jnp.mean((glaser_family(width,q_train)-truth_train)**2))(width_candidates)
fitted_glaser_width_m=float(width_candidates[jnp.argmin(width_losses)])
truth_held=glaser_family(truth_glaser_width_m,q_held); matrix_noise_sigma=.020
thin_rms=float(jnp.sqrt(jnp.mean((thin_family(q_held)-truth_held)**2))/matrix_noise_sigma)
hard_rms=float(jnp.sqrt(jnp.mean((hard_edge_family(q_held)-truth_held)**2))/matrix_noise_sigma)
glaser_rms=float(jnp.sqrt(jnp.mean((glaser_family(fitted_glaser_width_m,q_held)-truth_held)**2))/matrix_noise_sigma)
gain_sigma=hard_rms-glaser_rms; relative_gain=gain_sigma/hard_rms
promote_glaser=bool(gain_sigma>.2 and relative_gain>.20)
hierarchy_rows=(("thin rotating initializer",2,thin_rms,"initialize only"),("hard-edge solenoid",2,hard_rms,"default"),("Glaser profile",3,glaser_rms,"promote" if promote_glaser else "reject"),("generic symplectic section",4,np.nan,"predictive fallback"))
print(f"Synthetic Glaser width: truth={1e3*truth_glaser_width_m:.3f} mm, training fit={1e3*fitted_glaser_width_m:.3f} mm")
print(f"{'model':29s} {'parameters':>10s} {'held-out RMS [sigma]':>21s}  decision")
for model,n_parameter,residual,decision in hierarchy_rows:
    residual_text='—' if not np.isfinite(residual) else f'{residual:.3f}'
    print(f"{model:29s} {n_parameter:10d} {residual_text:>21s}  {decision}")
print(f"Glaser held-out gain={gain_sigma:.3f} sigma ({100*relative_gain:.1f}%); promotion={promote_glaser}")
assert promote_glaser==(gain_sigma>.2 and relative_gain>.20)


## 6. Publication ray diagrams

The two colored curves are basis responses, not the edges of a ray bundle. For a Gaussian launched at a waist,

$$x_h(z)=A(z)w_0,\qquad x_\alpha(z)=B(z)\frac{\lambda}{\pi w_0},\qquad w(z)=\sqrt{x_h(z)^2+x_\alpha(z)^2}.$$

The figures use a true linear $z$ overview plus condenser, objective/sample, and projector detail panels with independent, labelled transverse scales. Finite lens bodies, the disabled corrector, selected sample tile, $A=0$ crossovers, $B=0$ conjugates, waist minima, and cumulative Larmor rotation are explicit. Truth/initial/fitted and nominal/negative/positive wobble traces share panel extents. A second synthetic diagnostic diagram uses calibrated position and tilt bases to validate downstream conventions; it is not a required instrument acquisition.

In [ ]:
def decoded_lenses(parameters,dacs):
    parameters=np.asarray(parameters,float); dacs=np.asarray(dacs,float); cursor=2; result=[]
    for lens in column.lenses:
        zc=parameters[cursor]; length=np.exp(parameters[cursor+1]); q0=parameters[cursor+2]; cursor+=3
        slope=0. if lens.channel_index is None else parameters[cursor]
        if lens.channel_index is not None: cursor+=1
        displacement=0. if lens.channel_index is None else dacs[lens.channel_index]-lens.reference_dac
        result.append((lens,zc,length,q0+slope*displacement))
    return result

def hard_edge_basis_trace(parameters,dacs,position_scale,angle_scale,label,lens_samples=17,source_basis_states=None,coordinate="x_rot"):
    parameters=np.asarray(parameters,float); states=np.zeros((4,2)) if source_basis_states is None else np.asarray(source_basis_states,float).copy()
    if source_basis_states is None: states[0,0]=position_scale; states[1,1]=angle_scale
    if states.shape!=(4,2) or not np.all(np.isfinite(states)): raise ValueError("source_basis_states must be a finite (4,2) [x, alpha_x, y, alpha_y] basis.")
    if coordinate not in {"x","x_rot"}: raise ValueError("hard_edge_basis_trace coordinate must be 'x' or 'x_rot'.")
    z_cursor=float(parameters[0]); cumulative_rotation=0.; z_values=[]; position=[]; angle=[]; sample_recorded=False
    def record(z,frame_angle):
        # Project lab (x,y) into the co-rotating Larmor frame; the orthogonal component is numerical zero for a round column.
        axis_xy=np.array([1.,0.]) if coordinate=="x" else np.array([np.cos(frame_angle),np.sin(frame_angle)])
        z_values.append(float(z)); position.append(float(axis_xy@states[[0,2],0])); angle.append(float(axis_xy@states[[0,2],1]))
    record(z_cursor,cumulative_rotation)
    def drift_to(z):
        nonlocal states,z_cursor
        states=np.asarray(drift_matrix(float(z)-z_cursor))@states; z_cursor=float(z); record(z_cursor,cumulative_rotation)
    for lens,zc,length,q in decoded_lenses(parameters,dacs):
        entrance,exit_=zc-length/2,zc+length/2
        if not sample_recorded and z_cursor<=column.sample_z_m<=entrance:
            drift_to(column.sample_z_m); sample_recorded=True
        drift_to(entrance); entrance_states=states.copy(); rotation_at_entrance=cumulative_rotation
        fractions=np.linspace(1/lens_samples,1,lens_samples)
        if entrance<column.sample_z_m<exit_: fractions=np.unique(np.append(fractions,(column.sample_z_m-entrance)/length))
        for fraction in np.sort(fractions):
            states=np.asarray(hard_edge_solenoid_matrix(q*fraction,length*fraction))@entrance_states
            z_cursor=float(entrance+fraction*length); cumulative_rotation=rotation_at_entrance-float(q)*fraction
            record(z_cursor,cumulative_rotation)
            if np.isclose(z_cursor,column.sample_z_m,rtol=0,atol=1e-12): sample_recorded=True
    if not sample_recorded and z_cursor<=column.sample_z_m:
        drift_to(column.sample_z_m); sample_recorded=True
    drift_to(float(parameters[1]))
    position=np.asarray(position); angle=np.asarray(angle); z_values=np.asarray(z_values)
    return PositionAngleTrace(z=z_values,centroid=np.zeros_like(z_values),position=position,angle=angle,radius=gaussian_basis_envelope(position,angle),coordinate=coordinate,position_scale=position_scale,angle_scale=angle_scale,label=label,voltage=beam_cfg["voltage_v"])

def plotting_components(parameters,dacs):
    Rc=float(np.asarray(compute_Rc_from_voltage(beam_cfg["voltage_v"])))
    components=[]; labels=[]; roles=[]
    for lens,zc,length,q in decoded_lenses(parameters,dacs):
        excitation=-float(q)/Rc
        if abs(excitation)<1e-15: excitation=1e-15
        components.append(ElectromagneticLens(z=zc-length/2,turns=1.,current=excitation,Gc=1.,Tc=length/abs(excitation)))
        labels.append(lens.name)
        roles.append("objective_lens" if lens.name.startswith("Obj") else ("condenser_lens" if lens.name in {"CL1","CL3","C-mini"} else ("projector_lens" if lens.name.startswith("PL") else "intermediate_lens")))
    return tuple(components),tuple(labels),tuple(roles)

nominal_dac=nominal_dacs[3].copy(); minus_dac=nominal_dac.copy(); plus_dac=nominal_dac.copy(); minus_dac[1]-=perturbations[1]; plus_dac[1]+=perturbations[1]
selected=view300.selection; selected_label=(selected.selected_region_name or "global_atlas")+f"; feature={selected.feature_family}; P={selected.routed_feature_scale_m*1e9:.3g} nm; grid Δ={selected.selected_spacing_m*1e9:.3g} nm"
sample_short=(selected.selected_region_name or "global atlas").replace("multilayer_region_","ML").replace("silicon_lattice_region","Si lattice")
def passive_elements_for(parameters): return (ColumnElement(column.sample_z_m,f"Sample: {sample_short} · P={selected.routed_feature_scale_m*1e9:.2f} nm · grid Δ={selected.selected_spacing_m*1e9:.3f} nm","sample"),ColumnElement(column.sample_z_m-.018,"Corrector (off)","corrector_off",column.sample_z_m-.008),ColumnElement(float(np.asarray(parameters)[1]),"Medipix 256×256","detector"))
passive_elements=passive_elements_for(p_truth)
sections=(ColumnSection("Condenser",p_truth[0],column.sample_z_m-.004),ColumnSection("Objective and sample",column.sample_z_m-.032,column.sample_z_m+.036),ColumnSection("Intermediate/projector",column.sample_z_m,p_truth[1]))

In [ ]:
waist=1.2e-6; divergence=wavelength_m/(np.pi*waist)
ray_truth_parameters=np.asarray(physical_parameters if physical_global_fit is not None else p_truth); ray_initial_parameters=np.asarray(physical_fit_initial_parameters if physical_global_fit is not None else p_initial); ray_fit_parameters=np.asarray(physical_global_fit.parameters[:column.n_parameters] if physical_global_fit is not None else p_fitted); ray_fit_label=(("accepted 64-start astigmatic fit" if CONDENSER_ROUTE_BRANCH=="ASTIGMATIC" else "accepted 64-start full-column fit") if physical_global_fit is not None and physical_global_fit_ready else ("diagnostic 64-start candidate (gates failed)" if physical_global_fit is not None else "3-parameter response fit")); ray_reference_label=("Synthetic smoke truth" if physical_global_fit is None else ("Truth" if CONDENSER_ROUTE_IS_SYNTHETIC else "Reference/initial")); ray_reference_comparison_label=("synthetic truth" if ray_reference_label in {"Truth","Synthetic smoke truth"} else "reference")
truth_trace=hard_edge_basis_trace(ray_truth_parameters,nominal_dac,waist,divergence,f"{ray_reference_comparison_label} / nominal")
initial_trace=hard_edge_basis_trace(ray_initial_parameters,nominal_dac,waist,divergence,"initial")
fitted_trace=hard_edge_basis_trace(ray_fit_parameters,nominal_dac,waist,divergence,ray_fit_label)
minus_trace=hard_edge_basis_trace(ray_truth_parameters,minus_dac,waist,divergence,"CL3 − wobble")
plus_trace=hard_edge_basis_trace(ray_truth_parameters,plus_dac,waist,divergence,"CL3 + wobble")
truth_components,lens_labels,lens_roles=plotting_components(ray_truth_parameters,nominal_dac)
initial_components,_,_=plotting_components(ray_initial_parameters,nominal_dac); fitted_components,_,_=plotting_components(ray_fit_parameters,nominal_dac)
minus_components,_,_=plotting_components(ray_truth_parameters,minus_dac); plus_components,_,_=plotting_components(ray_truth_parameters,plus_dac)
fit_parameter_sets={ray_reference_label:ray_truth_parameters,"Initial":ray_initial_parameters,ray_fit_label:ray_fit_parameters}; fit_z_min=min(float(parameters[0]) for parameters in fit_parameter_sets.values()); fit_z_max=max(float(parameters[1]) for parameters in fit_parameter_sets.values()); fit_sections=(ColumnSection("Condenser",fit_z_min,column.sample_z_m-.004),ColumnSection("Objective and sample",column.sample_z_m-.032,column.sample_z_m+.036),ColumnSection("Intermediate/projector",column.sample_z_m,fit_z_max))
fig,axes=plot_neoarm_ray_diagram(fitted_components,fitted_trace,component_labels=lens_labels,component_roles=lens_roles,extra_elements=passive_elements_for(ray_fit_parameters),sections=fit_sections,title=f"NEOARM Gaussian two-basis solution — {ray_fit_label}",figsize=(16,11.5))
assert "larmor" in axes; events=basis_trace_events(fitted_trace)
print({name:[round(event.z,7) for event in values] for name,values in events.items()})
gaussian_exports=export_ray_figure(fig,GENERATED/"neoarm_gaussian_basis",dpi=300,metadata={"title":"NEOARM Gaussian basis rays","voltage_V":beam_cfg["voltage_v"],"sample_region":selected_label,"waist_m":waist,"divergence_rad":divergence,"fit_label":ray_fit_label,"full_physical_fit_used":physical_global_fit is not None,"selected_condenser_route":CONDENSER_ROUTE,"trace_and_lens_geometry_parameter_set":ray_fit_label,"corrector_enabled":False})
plt.show(); print({key:path.relative_to(REPO) for key,path in gaussian_exports.items()})

fit_scenarios={ray_reference_label:truth_trace,"Initial":initial_trace,ray_fit_label:fitted_trace}
fit_components={ray_reference_label:truth_components,"Initial":initial_components,ray_fit_label:fitted_components}
fit_extra_elements={name:passive_elements_for(parameters) for name,parameters in fit_parameter_sets.items()}; fit_position_indices=np.asarray((0,2))
def image_scale_rotation(parameters,dac_values):
    matrix=np.asarray(column.matrix(parameters,dac_values,segment="sample_to_detector")); real_map=np.asarray(detector.affine_xy)@matrix[np.ix_(fit_position_indices,fit_position_indices)]; affine=fit_fourier_affine(np.eye(2),real_map.T); return {"magnification":float(affine.isotropic_scale),"rotation_deg":float(np.rad2deg(affine.rotation_rad))}
def wrapped_degrees(value): return float((value+180.)%360.-180.)
fit_response_summaries={name:image_scale_rotation(parameters,nominal_dac) for name,parameters in fit_parameter_sets.items()}; fit_reference_response=fit_response_summaries[ray_reference_label]
fig_fit,fit_axes=plot_neoarm_ray_small_multiples(fit_components,fit_scenarios,component_labels=lens_labels,component_roles=lens_roles,extra_elements=fit_extra_elements,sections=fit_sections,title=f"{ray_reference_label}, initial, and {ray_fit_label} columns — shared union physical axis",figsize=(15.5,12))
for name,response in fit_response_summaries.items():
    delta_magnification=100*(response["magnification"]/fit_reference_response["magnification"]-1); delta_rotation=wrapped_degrees(response["rotation_deg"]-fit_reference_response["rotation_deg"]); fit_axes[name]["overview"].text(.018,.965,f"M={response['magnification']/1e3:.3f}k× · θ={response['rotation_deg']:+.4f}°\nvs {ray_reference_comparison_label}: ΔM={delta_magnification:+.4f}% · Δθ={delta_rotation:+.4f}°",transform=fit_axes[name]["overview"].transAxes,ha="left",va="top",fontsize=6.7,color="#1F2937",bbox={"facecolor":"white","edgecolor":"#CBD5E1","linewidth":.4,"alpha":.9,"pad":1.5},zorder=20)
fit_figure_caption=((f"Full synthetic physical candidate · accepted={physical_global_fit_ready} · data-only training WRMS {physical_training_data_wrms:.3f} sigma · practical rank {physical_global_fit.fit_rank}/{physical_identifiable_subspace.rank} · held-out gates {physical_held_out_statistical_gate}/{physical_held_out_engineering_gate}" if CONDENSER_ROUTE_IS_SYNTHETIC else f"Instrument candidate against declared reference · accepted={final_instrument_acceptance} · data-only training WRMS {physical_training_data_wrms:.3f} sigma · practical rank {physical_global_fit.fit_rank}/{physical_identifiable_subspace.rank}") if physical_global_fit is not None else f"Matched-model 3-parameter smoke fit · WRMS {active_weighted_rms:.3f} sigma · rank {smoke_fit.jacobian_rank}/{len(smoke_idx)} · max active error {100*np.max(smoke_relative_errors):.4f}%"); fig_fit.text(.065,.006,fit_figure_caption,ha="left",va="bottom",fontsize=7,color="#4B5563")
fit_export_stem=("neoarm_truth_initial_fitted" if ray_reference_label in {"Truth","Synthetic smoke truth"} else "neoarm_reference_initial_fitted"); fit_exports=export_ray_figure(fig_fit,GENERATED/fit_export_stem,dpi=300,metadata={"title":f"{ray_reference_label}, initial, and {ray_fit_label} NEOARM rays","reference_is_matched_synthetic_truth":ray_reference_label in {"Truth","Synthetic smoke truth"},"shared_union_extent":True,"scenario_source_detector_z_m":{name:[float(parameters[0]),float(parameters[1])] for name,parameters in fit_parameter_sets.items()},"sample_region":selected_label,"image_scale_rotation":fit_response_summaries,"fit_label":ray_fit_label,"full_physical_fit_used":physical_global_fit is not None,"fit_caption":fit_figure_caption,"selected_condenser_route":CONDENSER_ROUTE})
plt.show(); print({key:path.relative_to(REPO) for key,path in fit_exports.items()})

wobble_scenarios={"CL3 − wobble":minus_trace,"Nominal":truth_trace,"CL3 + wobble":plus_trace}
wobble_components={"CL3 − wobble":minus_components,"Nominal":truth_components,"CL3 + wobble":plus_components}
assert column.channel_names[1]=="CL3"; wobble_trace_map={"CL3 − wobble":minus_trace,"Nominal":truth_trace,"CL3 + wobble":plus_trace}; wobble_dac_offsets={"CL3 − wobble":-float(perturbations[1]),"Nominal":0.,"CL3 + wobble":float(perturbations[1])}
def trace_radius_at_z(trace,z_m): return float(np.interp(z_m,np.asarray(trace.z),np.asarray(trace.radius)))
wobble_response_summaries={name:{"delta_cl3_dac":wobble_dac_offsets[name],"sample_radius_m":trace_radius_at_z(trace,column.sample_z_m),"detector_radius_m":float(trace.radius[-1])} for name,trace in wobble_trace_map.items()}; wobble_nominal_response=wobble_response_summaries["Nominal"]
fig_wobble,wobble_axes=plot_neoarm_ray_small_multiples(wobble_components,wobble_scenarios,component_labels=lens_labels,component_roles=lens_roles,extra_elements=passive_elements,sections=sections,title="CL3 wobble response — identical physical extents",figsize=(15.5,12))
for name,response in wobble_response_summaries.items():
    delta_sample=100*(response["sample_radius_m"]/wobble_nominal_response["sample_radius_m"]-1); delta_detector=100*(response["detector_radius_m"]/wobble_nominal_response["detector_radius_m"]-1); wobble_axes[name]["overview"].text(.018,.965,f"ΔCL3={response['delta_cl3_dac']:+.4f} raw DAC\nw_sample={response['sample_radius_m']*1e9:.3f} nm ({delta_sample:+.3f}%)\nw_detector={response['detector_radius_m']*1e3:.3f} mm ({delta_detector:+.3f}%)",transform=wobble_axes[name]["overview"].transAxes,ha="left",va="top",fontsize=6.7,color="#1F2937",bbox={"facecolor":"white","edgecolor":"#CBD5E1","linewidth":.4,"alpha":.9,"pad":1.5},zorder=20)
wobble_centered_response=(np.log(wobble_response_summaries["CL3 + wobble"]["sample_radius_m"])-np.log(wobble_response_summaries["CL3 − wobble"]["sample_radius_m"]))/(2*perturbations[1]); fig_wobble.text(.065,.006,f"Centered finite-difference response: Δln(w_sample)/ΔCL3 DAC = {wobble_centered_response:+.3e} per raw unit",ha="left",va="bottom",fontsize=7,color="#4B5563")
wobble_exports=export_ray_figure(fig_wobble,GENERATED/"neoarm_cl3_wobble",dpi=300,metadata={"title":"Nominal and CL3 wobble NEOARM rays","shared_extents":True,"dac_step":perturbations[1],"gaussian_basis_width_response":wobble_response_summaries,"centered_log_sample_width_response_per_raw_dac":wobble_centered_response})
plt.show(); print({key:path.relative_to(REPO) for key,path in wobble_exports.items()})

In [ ]:
def source_states_for_calibrated_sample_basis(parameters,dacs,position_scale_m,tilt_scale_rad):
    desired=np.zeros((4,2)); desired[0,0]=position_scale_m; desired[1,1]=tilt_scale_rad  # pure lab-x position and tilt at the sample
    upstream=np.asarray(column.matrix(np.asarray(parameters),np.asarray(dacs),segment="source_to_sample"),float); source=np.linalg.solve(upstream,desired); recovered=upstream@source
    coordinate_scales=np.asarray((abs(position_scale_m),abs(tilt_scale_rad),abs(position_scale_m),abs(tilt_scale_rad)))[:,None]; relative_errors=np.max(np.abs((recovered-desired)/coordinate_scales),axis=0)
    if not np.all(np.isfinite(relative_errors)) or np.max(relative_errors)>1e-10: raise RuntimeError(f"Could not construct the calibrated sample-plane basis: relative errors={relative_errors}.")
    return source,desired,recovered,relative_errors,float(np.linalg.cond(upstream))
calibrated_reference_source,calibrated_sample_basis,calibrated_reference_recovered,calibrated_reference_errors,calibrated_reference_condition=source_states_for_calibrated_sample_basis(ray_truth_parameters,nominal_dac,plan.position_amplitude,plan.tilt_amplitude)
calibrated_fit_source,calibrated_fit_sample_basis,calibrated_fit_recovered,calibrated_fit_errors,calibrated_fit_condition=source_states_for_calibrated_sample_basis(ray_fit_parameters,nominal_dac,plan.position_amplitude,plan.tilt_amplitude)
np.testing.assert_allclose(calibrated_sample_basis,calibrated_fit_sample_basis,rtol=0.,atol=0.)
calibrated_reference=hard_edge_basis_trace(ray_truth_parameters,nominal_dac,plan.position_amplitude,plan.tilt_amplitude,f"calibrated {ray_reference_comparison_label}",source_basis_states=calibrated_reference_source,coordinate="x")
calibrated_fit=hard_edge_basis_trace(ray_fit_parameters,nominal_dac,plan.position_amplitude,plan.tilt_amplitude,ray_fit_label,source_basis_states=calibrated_fit_source,coordinate="x")
fig_cal,axes_cal=plot_neoarm_ray_diagram(fitted_components,calibrated_fit,component_labels=lens_labels,component_roles=lens_roles,extra_elements=passive_elements_for(ray_fit_parameters),sections=fit_sections,title=f"NEOARM calibrated lab-x position/tilt responses — {ray_fit_label}",trace_semantics="calibrated_response",figsize=(16,11.5))
assert "larmor" in axes_cal
calibrated_exports=export_ray_figure(fig_cal,GENERATED/"neoarm_calibrated_basis",dpi=300,metadata={"title":"NEOARM calibrated lab-x position and tilt responses at the sample","position_input_m":plan.position_amplitude,"tilt_input_rad":plan.tilt_amplitude,"input_plane":"sample","sample_basis_coordinate":"lab_x","trace_semantics":"independent_calibrated_responses; no Gaussian envelope","reference_source_basis_states":calibrated_reference_source,"fitted_source_basis_states":calibrated_fit_source,"reference_sample_recovery_relative_errors":calibrated_reference_errors,"fitted_sample_recovery_relative_errors":calibrated_fit_errors,"reference_source_to_sample_raw_coordinate_condition_number":calibrated_reference_condition,"fitted_source_to_sample_raw_coordinate_condition_number":calibrated_fit_condition,"raw_coordinate_condition_number_note":"unit-scaling dependent; reported only as numerical provenance","sample_region":selected_label,"fit_label":ray_fit_label,"full_physical_fit_used":physical_global_fit is not None,"selected_condenser_route":CONDENSER_ROUTE,"trace_and_lens_geometry_parameter_set":ray_fit_label,"reference_is_matched_synthetic_truth":ray_reference_label in {"Truth","Synthetic smoke truth"}})
plt.show(); print({key:path.relative_to(REPO) for key,path in calibrated_exports.items()})

## 7. Condenser gate without calibrated deflector angle or sample tilt

The paper-informed route uses only image-derived quantities: raw unnormalised C-aperture counts with aperture-out dose references, beam-radius/spatial-covariance wobble curves, and state-relative orientation of a deliberately astigmatic incident beam. The astigmatic tag must be generated upstream of every condenser lens being fitted; otherwise its stigmator plane and quadrupole/transport map must be explicit and it cannot identify upstream lenses. Its command strength need not be calibrated. The fitted entrance covariance contains the shared ellipse orientation and ellipticity; no deflector command is converted to radians. An ordinary image of the downstream specimen is not a condenser-rotation measurement. A defocused Ronchigram is usable only when its model explicitly transports the incident angle-to-sample map.

The audit below is deliberately conservative and fail-closed. It grants the route the richer existing translated-aperture Poisson block, removes all calibrated angular/deflector rows, releases source/Obj-prefield, entrance covariance, dose, aperture, and real-space detector-scale nuisances, and adds 32 separate astigmatic-tag projection exposures (one reference and 31 varied states). That edge block requires mechanical aperture translation or a calibrated upstream beam traversal; the 0.290/0.295 m deflector is downstream and cannot supply it. Paper-style centred throughput is weaker, so none of the conditional numbers below transfer if the traversal is unavailable. Position rows already contain full 2D spatial covariance, including ellipse orientation, so a real acquisition must either keep the tag exposure distinct or estimate one joint pixel/bootstrap covariance; the independent summary blocks below are an optimistic upper bound. Even this upper bound is rank 11/12: a q-free axial dilation changes the three effective lengths together and compensates their centres, so Larmor rotation cannot see it.

Indexed Si diffraction can supply an angular radius without commanded tilt via $\alpha=(\lambda/d)(r_{\rm disk}/r_g)$; camera length and pixel scale cancel. The remaining $\lambda/d$ uncertainty and the independent real-space atlas/downstream scale are explicit nuisance priors. The manifest compares unanchored scales, a Si-only anchor, fixed synthetic scales, and a stringent conditional case. Only the last reaches the local 1% gate, and only if both scale posteriors are at most 0.05% and the incident-ellipse orientation is measured to 1 mrad. That result is an optimistic conditional local upper bound until the image-level/joint bootstrap covariance is supplied; it is not an achieved NEOARM calibration.

A first exploratory full-angular-ellipse search remains rejected: its selector appended states after the physical eligibility filter, and its most repeated state produced approximately 3.57 and 2.70 micrometre sample widths—far outside the required 30–200 nm window. Its apparent sub-1% result is retained only as negative provenance. The checksum-bound replacement below reapplies that filter to every repeated acquisition, uses 13 distinct states expanded to 48 sample acquisitions, and profiles 31 nuisance columns. It reaches route-native local structural rank 12/12 with a worst scaled local uncertainty of 0.997%, but that is a knife-edge conditional design result rather than full-column practical rank or hardware evidence.

See `examples/lens_inversion/neoarm_condenser_no_tilt_manifest.json`.

In [ ]:
from temgym_core.condenser_no_tilt_audit import audit_no_tilt_route
no_tilt_cases={
    'A: unanchored scales':dict(angular_scale_mode='unanchored',rotation_sigma_rad=.02),
    'B: Si ratio only':dict(angular_scale_mode='si_ratio_prior',si_log_angular_scale_prior_sigma=.002,rotation_sigma_rad=.02),
    'C: fixed-scale synthetic upper bound':dict(angular_scale_mode='fixed',profile_realspace_scale=False,rotation_sigma_rad=.02),
    'D: conditional dual 0.05% scales + 1 mrad tag':dict(angular_scale_mode='si_ratio_prior',si_log_angular_scale_prior_sigma=5e-4,realspace_log_scale_prior_sigma=5e-4,rotation_sigma_rad=1e-3),
}
no_tilt_audits={name:audit_no_tilt_route(reusable_branch_B_synthetic_likelihood,**kwargs) for name,kwargs in no_tilt_cases.items()}
for name,audit in no_tilt_audits.items(): print(name,{'rank':f'{audit.target_rank}/12','smin_over_smax':audit.minimum_singular_value_ratio,'max_scaled_sigma':audit.maximum_scaled_standard_uncertainty,'local_1pct_gate':audit.practical_pass})
assert no_tilt_audits['A: unanchored scales'].target_rank==11
assert not no_tilt_audits['B: Si ratio only'].practical_pass and not no_tilt_audits['C: fixed-scale synthetic upper bound'].practical_pass
assert no_tilt_audits['D: conditional dual 0.05% scales + 1 mrad tag'].practical_pass
no_tilt_null=no_tilt_audits['A: unanchored scales'].null_family; print('q-free axial-dilation direction:',{key:round(value,7) for key,value in no_tilt_null.items() if abs(value)>1e-7})

### Strict deliberately astigmatic pilot

This is the executable replacement for the rejected search. It contains no calibrated deflector or commanded sample tilt. Each sample acquisition supplies the two real-space RMS widths and correlation, the two indexed-Si angular RMS widths and correlation, and the state-relative orientation of one deliberately astigmatic incident tag. The absolute stigmator command, ellipticity, and initial angle are not treated as calibrated: they are shared fitted nuisance quantities. The tag must be introduced upstream of CL1; if the actual stigmator is later in the column, its measured plane and quadrupole/transport map must enter the model and the tag has no sensitivity to lenses upstream of that plane.

The conservative accounting is 88 translated C-aperture frames plus three detector frames for each of 48 sample acquisitions, or 232 frames. It is 184 only if real-space ellipse shape and orientation are extracted from the same pixels and their complete joint image-bootstrap covariance is used. The selected normalized controls are a synthetic design, not NEOARM DAC settings. Even though the nominal 0.4 m diffraction geometry fits the Si {111} radius and ellipse support on a 256×256, 55 µm detector, hardware readiness remains false until MTF, SNR, saturation/deadtime, deblending, tag-plane/repeatability, nonlinear truth-envelope recovery, and held-out gates pass.

Centered partial clipping at the C aperture was also tested outside the paper's route. It is useful CL1/CL3 reconnaissance but has exactly zero direct C-mini derivative in the draft geometry and gave about 2.5% worst uncertainty, so it does not replace translated-edge anchoring. Signed Fresnel diversity retained formal rank but did not adequately constrain the dominant CL1-length/scale direction either.

In [ ]:
from temgym_core.condenser_astigmatic_design import (
    AstigmaticDesignAdmissionError, build_astigmatic_design_matrix,
    load_astigmatic_condenser_design, recompute_astigmatic_design_audit,
    require_astigmatic_hardware_readiness,
)
astigmatic_design=load_astigmatic_condenser_design(REPO/'examples/lens_inversion/neoarm_condenser_astigmatic_design.json',repository_root=REPO)
astigmatic_matrix,astigmatic_summaries=build_astigmatic_design_matrix(astigmatic_design)
astigmatic_audit=recompute_astigmatic_design_audit(astigmatic_design)
astigmatic_blockers=tuple(name for name,gate in astigmatic_design.hardware_gates.items() if not gate.get('verified_on_instrument',False))
print({'unique_states':len(astigmatic_design.controls),'sample_acquisitions':len(astigmatic_design.expanded_controls),'raw_frames_conservative':astigmatic_design.conservative_raw_frame_count,'raw_frames_joint_same_image':astigmatic_design.joint_covariance_raw_frame_count,'matrix_shape':astigmatic_matrix.matrix.shape,'profiled_rank':f'{astigmatic_audit.target_rank}/12','maximum_scaled_sigma':astigmatic_audit.maximum_scaled_standard_uncertainty,'Si_111_radius_px_at_0p4m':astigmatic_audit.si_111_radius_px,'angular_RMS_range_px':astigmatic_audit.angular_eigen_rms_range_px.tolist(),'nominal_geometry_only':astigmatic_audit.nominal_detector_geometry_pass,'hardware_ready':astigmatic_audit.hardware_ready})
print('Unverified astigmatic-pilot hardware gates:',astigmatic_blockers)
try:
    require_astigmatic_hardware_readiness(astigmatic_design,astigmatic_audit)
except AstigmaticDesignAdmissionError as error:
    print('Expected fail-closed admission:',error)
assert astigmatic_audit.target_rank==12 and astigmatic_audit.conditional_precision_pass and not astigmatic_audit.hardware_ready
assert astigmatic_audit.sample_equivalent_rms_range_m[0]>=30e-9 and astigmatic_audit.sample_equivalent_rms_range_m[1]<=200e-9
assert not any('deflector' in name.lower() for name in (*astigmatic_matrix.row_names,*astigmatic_matrix.column_names))

### Nonlinear no-commanded-tilt likelihood

The design matrix is now backed by the same nonlinear hard-edge forward model used for the proposed fit. Its 88 translated-aperture measurements use raw Poisson counts; its 48 sample acquisitions use one dense covariance over spatial widths/correlation, indexed-Si angular widths/correlation, and the $180^\circ$-periodic relative tag angle. The JAX Jacobian must reproduce the checksum-bound 428×43 design. After profiling 29 nuisance directions the 12 condenser targets are resolved in the route-native local structural audit, but the complete model has rank 41/43 because two nuisance-only gauges remain. The unified full-column likelihood applies its own finer mechanical scales and SNR threshold and need not retain all twelve as practical directions. A full nonlinear multistart test also finds a curved near-gauge valley, so this cell demonstrates an executable likelihood and local identifiability—not global uniqueness or instrument recovery.

In [ ]:
from temgym_core import make_astigmatic_condenser_likelihood, make_synthetic_astigmatic_observations
astigmatic_likelihood=make_astigmatic_condenser_likelihood(REPO/'examples/lens_inversion/neoarm_condenser_astigmatic_design.json',repository_root=REPO)
astigmatic_nonlinear_jacobian=astigmatic_likelihood.verify_nominal_design_jacobian(); astigmatic_rank=astigmatic_likelihood.local_rank_diagnostics()
astigmatic_synthetic_draw=make_synthetic_astigmatic_observations(astigmatic_likelihood.design,parameters=astigmatic_likelihood.nominal_parameters,sample_noise=True,seed=26090419)
astigmatic_synthetic_likelihood=make_astigmatic_condenser_likelihood(REPO/'examples/lens_inversion/neoarm_condenser_astigmatic_design.json',observations=astigmatic_synthetic_draw,repository_root=REPO)
astigmatic_synthetic_residual=np.asarray(astigmatic_synthetic_likelihood.residual(astigmatic_synthetic_likelihood.nominal_parameters))
print({'nonlinear_J_shape':astigmatic_nonlinear_jacobian.shape,'full_rank':astigmatic_rank.full_rank,'full_nullity':astigmatic_rank.full_nullity,'nuisance_rank':astigmatic_rank.nuisance_rank,'profiled_condenser_rank':f'{astigmatic_rank.profiled_target_rank}/12','synthetic_draw_WRMS_at_truth':float(np.sqrt(np.mean(astigmatic_synthetic_residual**2))),'observation_kind':astigmatic_synthetic_draw.kind,'hardware_ready':astigmatic_synthetic_likelihood.hardware_ready})
assert astigmatic_rank.full_rank==41 and astigmatic_rank.full_nullity==2 and astigmatic_rank.nuisance_rank==29 and astigmatic_rank.profiled_target_rank==12
assert not astigmatic_synthetic_likelihood.hardware_ready and not any('deflector' in name.lower() or 'K16' in name for name in (*astigmatic_synthetic_likelihood.parameter_names,*astigmatic_synthetic_likelihood.residual_names))

### What the deliberately astigmatic measurements look like

This planning figure makes the proposed observables concrete without pretending that an analytic ellipse is a GPU specimen-scattering validation. The left panel is the incident sample-plane covariance, the centre panel is the native 256×256 indexed-Si diffraction geometry at the planned 0.4 m camera length, and the right panel shows only state-relative, $180^\circ$-periodic ellipse orientation. The absolute astigmatism command and angle remain fitted nuisance quantities. The selected state obeys the strict beam-size and detector-support rules, but every hardware gate remains visibly false until measured data replace this preview.

In [ ]:
from temgym_core.condenser_astigmatic_visualization import (
    build_astigmatic_planning_preview, export_astigmatic_planning_preview,
    nominal_astigmatic_covariance_summaries,
)
astigmatic_preview=build_astigmatic_planning_preview(astigmatic_design,audit=astigmatic_audit,summaries=nominal_astigmatic_covariance_summaries(astigmatic_design))
astigmatic_preview_exports=export_astigmatic_planning_preview(astigmatic_preview,GENERATED/'condenser_astigmatic_no_tilt_planning_preview')
print({'selected_state':astigmatic_preview.selected_state_index,'repeat_count':astigmatic_preview.selected_repeat_count,'sample_RMS_axes_nm':(astigmatic_preview.selected_spatial_axis_rms_m*1e9).tolist(),'angular_RMS_axes_mrad':(astigmatic_preview.selected_angular_axis_rms_rad*1e3).tolist(),'Si_111_radius_px':astigmatic_preview.si_111_radius_px,'hardware_evidence':False,'exports':{key:str(value.relative_to(REPO)) for key,value in astigmatic_preview_exports.items()}})
plt.show()

## 8. Transfer to the instrument

1. Replace nominal geometry and certificate values; record detector dark/flat data, voltage, detector pose, and stage pose. Retain any independently calibrated sample-position covariance, but do not enter a commanded deflection angle or sample-plane tilt as known. Tilt and the raw-command-to-kick map remain fitted nuisance coordinates unless independently recovered from registered known-pattern image plus diffraction measurements.
2. Preserve raw, unnormalised intensity and first establish the upstream anchors: integrated intensity versus raw lens control through the physical C aperture, lens-wobble critical-setting invariance, and state-relative rotation of a deliberately astigmatic incident ellipse generated upstream of CL1. If the tag is generated later, include its physical stigmator plane and map and do not assign it sensitivity to upstream lenses. Fit the shared entrance ellipse orientation and ellipticity; no calibrated deflector command is used. An ordinary known-specimen Ronchigram is a downstream/projector observable, not a condenser-rotation measurement, unless the forward model explicitly transports its incident angle-to-sample map. Pure large beam-shift scans are not treated as angle calibration: they can be noise limited and introduce coma. Approach nominal and held-out DAC states from both directions; keep the corrector disabled and objective controls fixed.
3. Extract certified layer boundaries or lattice Fourier vectors, image affine scale/rotation, centroid/spatial covariance, upstream C-aperture absolute counts, and indexed diffraction vectors. Sample-frame transmitted totals remain validation-only until a calibrated incident-current model replaces the synthetic unit-current construction. The core no-commanded-tilt route obtains convergence from the Si disk-radius/Bragg-vector-radius ratio, so camera length and detector pixels cancel and only the certified wavelength/lattice uncertainty remains. Fitting sample tilt and a raw-command-to-kick map is an optional secondary calibration; it is not an input to this condenser gate and raw commands are never pre-converted to radians. Reject frames with drift, saturation, contamination, coma outside the model, or violated dose assumptions; do not silently normalize intensity.
4. Freeze one certified sample crop, cubic level, Gaussian topology, and interaction route per acquisition over its complete allowed parameter range. Refine until sample-plane complex error is noise-relative, detector disagreement is below 0.2 noise standard deviations RMS, observable changes are below 0.2 experimental uncertainty, and identifiable parameters change by less than 0.1 estimated standard deviations for two successive refinements.
5. Initialize with thin lenses and the configured 64-start transfer-response feasibility fit. Only after every material/diffraction topology has an independent signature-bound validation and the complete physical observation dataset is assembled, run the separate physical-image global fit with all requested parameters released. Report rank, profiles, local null families, distinct equivalent optima, and held-out results. Admit a Glaser shape parameter only when held-out predictions improve beyond uncertainty.
6. The preferred instrument route is now the fail-closed no-commanded-tilt gate above. With both scale anchors free its base audit is 11/12; the missing family is a q-free axial dilation, so the astigmatic rotation tag alone cannot close it. The stricter 48-acquisition replacement reaches route-native local structural rank 12/12 and 0.997% worst scaled uncertainty only with independently justified 0.1% real-space and indexed-Si angular log-scale priors, 0.2% covariance-summary errors, and 1 mrad tag orientation. Its executable nonlinear Jacobian is rank 41/43. In the current unified full-column likelihood, only six condenser combinations clear the finer SNR>75 practical gate; these stable combinations may be fitted, while a claim about all twelve named centre/length/strength coordinates remains false. Stronger axial-diversity/caustic measurements or credible centre/length priors are required to promote the weak directions. Two nuisance-only gauges and a curved near-gauge family also prevent a global-uniqueness claim. These precisions and the upstream-of-CL1 tag must be demonstrated, not assumed. Branch A and the checksum-bound 296-frame Branch B remain synthetic mathematical fallbacks for comparison, not prerequisites for the preferred experiment. None may claim an instrument fit until safe raw-command continuations, truth-envelope rank, joint nuisance-released multistart recovery, profile families, approach-direction repeats, and held-out validation pass. Downstream specimen images alone have exactly zero upstream-condenser sensitivity under a sample-plane input model.
7. If the weakest singular directions predict useful rank gain, add an eight-setting D-optimal batch. Otherwise report the non-identifiable physical family; more frames at the same states increase precision but do not create rank.

The end-to-end acceptance target is matched-model recovery within 1% for each identifiable parameter, explicit null/profile families for the others, and held-out errors below 0.5% magnification, 0.5° rotation, 0.5 pixel centroid, 2% beam diameter, and one pixel for indexed diffraction vectors. The implemented independent-reference influence gate covers the complete nuisance-profiled optical quotient, but the present two-successive-topology exercise refits only one conditional 150k `IL1.q_bias` proxy with every other optical and nuisance coordinate fixed. Consequently `final_instrument_acceptance` remains false by construction until two global refinement transitions demonstrate the same $<0.1\sigma$ stability for every admitted optical direction; the representative result must not be reported as an identifiable full-column-combination or literal all-parameter convergence. Publication figures are exported as SVG, PDF, and 300-dpi PNG with numerical provenance.